# Experimento independiente: preparación de una réplica T4

El autor eligió continuar con la T4 disponible, sin comprar más cómputo. La corrida A100 interrumpida se conserva íntegra. **Esta réplica reinicia los seis entrenamientos: no importa sus pesos ni continúa sus épocas.** El rechazo anterior de HBG semilla 11 permanece documentado.

**Primera ejecución: CPU, ejecutar todo con los valores iniciales.** Se verifican los archivos y se prepara la enmienda/protocolo. `RUN_TRAINING=False` es deliberado en este paso; no representa entrenamiento terminado. La prueba final permanece cerrada.

Después de revisar la salida de preparación, para iniciar el nuevo entrenamiento: seleccionar **GPU T4**, cambiar `RUN_TRAINING=True` en la única celda de configuración y ejecutar desde el inicio. El nuevo intento podrá recuperar sus propios checkpoints T4 con un entorno compatible. No volver a la corrida A100 para escoger la de mejores resultados.

Se mantienen población, métodos, tres semillas, controles, hiperparámetros, puertas de selección y escenarios. El presupuesto conserva hasta **72 horas de entrenamiento y 120 de búsqueda**, más evaluación y persistencia. Esos límites no garantizan acabar con la cuota disponible ni son una estimación medida en T4.

La preparación no verifica actividades de test fuera de los registros del pipeline. Las comprobaciones funcionales originales se recuperan si corresponden a las mismas fuentes; no se presentan como nuevas pruebas en T4. El gasto previo se conserva en la enmienda y debe sumarse explícitamente al informar el costo total del proyecto, pues el R existente no lo agrega por sí solo.


## Configuración y etapas

La primera ejecución prepara el protocolo con CPU. El entrenamiento se activa después en la celda siguiente.


In [37]:
import os, sys, json, subprocess, importlib.metadata
from pathlib import Path
try:
    import google.colab
except ImportError as exc:
    raise RuntimeError('10 sólo se ejecuta en Google Colab.') from exc
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
WORK = Path('/content/TESIS_FINAL_10')
RUNTIME = WORK / 'runtime'
(RUNTIME / 'svrplab').mkdir(parents=True, exist_ok=True)
(RUNTIME / 'tests').mkdir(exist_ok=True)
os.chdir(RUNTIME)
# Copiar ruta e identificador del acta de población generada por 07, sin buscar la más reciente.
POPULATION_RELATIVE_PATH = 'final_populations/final07_5abdec3f593b3050e3d0'
POPULATION_REPORT_SHA256 = 'ce05f08892e92f3cff035f887bdf4604e54f9ac124e4af9ae048eeaf2541acfe'
PREPARE_PROTOCOL = True
RUN_TRAINING = False
OPEN_FINAL_TEST = False
TRAIN_DEVICE = 'cuda'
DRIVE_ROOT = Path('/content/drive/MyDrive/TESIS-EHBG-FACS')
def installed(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
if installed('torch') is None:
    raise RuntimeError('Usar un runtime Colab con PyTorch; no reemplazar CUDA automáticamente.')
needed = ['numpy', 'pandas', 'scipy', 'matplotlib', 'pytest', 'ortools', 'gurobipy']
missing = [name for name in needed if installed(name) is None]
if missing:
    protected = ['numpy','pandas','matplotlib','scipy','protobuf','torch','pillow']
    constraints = RUNTIME / 'constraints.txt'
    constraints.write_text(''.join(f'{name}=={installed(name)}\n' for name in protected if installed(name)))
    requirements = ['gurobipy>=11,<14' if name == 'gurobipy' else 'ortools>=9.11,<10' if name == 'ortools' else name for name in missing]
    subprocess.run([sys.executable,'-m','pip','install','-q','-c',str(constraints),*requirements],check=True)
print('Entrenamiento:', RUN_TRAINING, '| abrir test:', OPEN_FINAL_TEST)


Entrenamiento: False | abrir test: False


In [38]:
CONFIG = {'schema_version': 'final-experiment-v1', 'role': 'confirmatory_holdout', 'execution_policy': 'colab_only', 'split_strategy': 'independent_train_validation_test_v1', 'objective': 'conservative_upper_travel_minutes_v1', 'time_scale': 1000, 'solver_seeds': [11, 29, 47], 'policies': ['fixed_order', 'causal_edd'], 'search_seconds': 120.0, 'methods': {'hbg_facs': {'ants': 16, 'iterations': 100, 'alpha': 1.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}, 'hbg_untrained': {'ants': 16, 'iterations': 100, 'alpha': 1.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}, 'hbg_no_pheromone': {'ants': 16, 'iterations': 100, 'alpha': 0.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'depot_guided': True, 'constructive_seed': True, 'local_search_moves': 100, 'constructive_seconds': 2.0}, 'pomo': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'pomo_untrained': {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False, 'all_starts': False, 'completion_search_nodes': 2048}, 'edd': {}, 'ortools_routing': {'capacity_scale': 1000}, 'aco': {'max_iterations': None, 'ants': 20, 'alpha': 1.0, 'beta': 2.0, 'evaporation': 0.1, 'q0': 0.2, 'local_search_seconds': 1.0, 'local_search_moves': 20}, 'tabu': {'max_iterations': None, 'neighborhood_samples': 80, 'tenure': 15, 'initial_restarts': 4, 'local_search_seconds': 1.0, 'local_search_moves': 20}, 'gurobi': {'threads': 1, 'mip_gap': 0.0}}, 'training': {'hbg_facs': {'schema_version': 'hybrid-05-v1', 'role': 'train_validation_for_final_holdout', 'replicate_id': 'independent_final_holdout_v1', 'split_strategy': 'independent_train_validation_test_v1', 'model': {'model_dim': 128, 'num_heads': 8, 'num_layers': 3, 'dropout': 0.0, 'tanh_clip': 10.0}, 'training': {'epochs': 30, 'steps_per_epoch': 400, 'gradient_accumulation': 4, 'learning_rate': 0.0001, 'weight_decay': 1e-06, 'gradient_clip_norm': 1.0, 'patience': 5, 'min_delta': 0.0001, 'seconds_per_model': 43200.0, 'permute_nodes': True, 'deterministic_algorithms': True, 'flow_learning_rate': 0.001, 'checkpoint_selection': 'prespecified_gate_then_validation_coverage_completion_fraction_cost_v1', 'require_gpu': True, 'geometry_sampling': 'shuffled_without_replacement', 'checkpoint_gate': 'one_complete_geometry_per_size'}, 'time_scale': 1000, 'objective': 'conservative_upper_travel_minutes_v1', 'hbg': {'ants': 4, 'iterations': 2, 'alpha': 1.0, 'evaporation': 0.1, 'temperature': 0.05, 'lambda_db': 0.5, 'constructive_seed': True, 'depot_guided': False, 'local_search_moves': 0, 'constructive_seconds': 2.0}, 'replay': {'capacity_per_instance': 32, 'samples_per_step': 2}}, 'pomo': {'schema_version': 'reinforcement-04-v1', 'role': 'train_validation_for_final_holdout', 'replicate_id': 'independent_final_holdout_v1', 'split_strategy': 'independent_train_validation_test_v1', 'model': {'model_dim': 128, 'num_heads': 8, 'num_layers': 3, 'dropout': 0.0, 'tanh_clip': 10.0}, 'training': {'epochs': 30, 'steps_per_epoch': 400, 'gradient_accumulation': 4, 'learning_rate': 0.0001, 'weight_decay': 1e-06, 'gradient_clip_norm': 1.0, 'patience': 5, 'min_delta': 0.0001, 'seconds_per_model': 43200.0, 'permute_nodes': True, 'deterministic_algorithms': True, 'checkpoint_selection': 'prespecified_gate_then_validation_coverage_completion_fraction_cost_v1', 'geometry_sampling': 'shuffled_without_replacement', 'checkpoint_gate': 'one_complete_geometry_per_size'}, 'time_scale': 1000, 'objective': 'conservative_upper_travel_minutes_v1', 'rollout': {'n_starts': 8}}}, 'require_cuda': True, 'training_geometry_minimum_observations': 1, 'training_failure_policy': 'record_rejected_validation_as_no_solution_all_seeds_retained', 'scenarios': {'master_seed': 2026091910, 'evaluation_count': 1000, 'trace_audit_first_scenarios': 2, 'export': 'scenario_metrics_csv_gzip_all_scenarios_with_fixed_trace_audit'}, 'resource_limits': {'maximum_planned_search_hours': 144, 'maximum_planned_training_hours': 96, 'maximum_scenario_rows': 10000000}, 'test_exposure_policy': 'single_population_claim_same_release_only_immutable_results_on_resume', 'analysis_export_schema': 'final-analysis-export-v1', 'open_test_default': False, 'train_default': False}
DESIGN = {'schema_version': 'research-design-v2', 'status': 'final_protocol_pending_colab_population_and_holdout_adapter', 'execution_policy': 'colab_only', 'sizes': [50, 100, 200, 300], 'split_seed': 20260919, 'geometry_quotas_per_size': {'train': 100, 'validation': 20, 'test': 30}, 'population': {'family': 'official_generated_geometry_conditioned_windows_v1', 'master_seed': 2026091907, 'candidate_multiplier': 4, 'map_size': [1000, 1000], 'demand_range': [0, 100], 'num_depots': 1, 'dynamic_customers': False, 'geometry_rule': 'fresh_official_base_generator_per_ticket', 'num_cities_rule': 'max_1_n_div_50', 'fleet_rule': 'official_24_bin_histogram_of_original_windows_before_conditioning', 'unknown_policy': 'stop_preserve_same_ticket_no_substitution', 'exhaustion_policy': 'stop_report_shortage_no_automatic_quota_change', 'duplicate_policy': 'archive_and_advance_without_counting_as_new_geometry', 'selection_rule': 'first_quota_certified_in_prefixed_role_stream_proven_rejections_only', 'certification_seconds_per_strategy': [120, 600, 1800], 'certification_continuation_policy': 'same_candidate_unknown_only_preserve_all_search_stages'}, 'fr_absolute_error': 0.05, 'fr_error_probability': 0.05, 'cvar_alpha': 0.95, 'minimum_expected_tail_observations': 50, 'primary_endpoint': 'geometry_mean_feasibility_first_conservative_cost_win_score', 'primary_policy': 'fixed_order', 'primary_contrasts': [['hbg_facs', 'hbg_untrained'], ['hbg_facs', 'hbg_no_pheromone']], 'primary_multiplicity': 'holm_two_sided_familywise_0.05', 'win_score': {'validity': 'complete_coverage_capacity_windows_and_return_under_common_conservative_matrix', 'complete_beats_failed': True, 'both_failed_score': 0, 'both_valid_rule': 'lower_conservative_travel_cost_wins', 'cost_tie_absolute_minutes': 1e-06, 'cost_tie_relative': 1e-09, 'seed_aggregation': 'mean_all_prefixed_seed_pairs_within_geometry', 'size_aggregation': 'equal_weight_four_size_strata', 'missing_artifact_policy': 'analysis_blocked_never_recode_as_solver_failure'}, 'secondary_endpoints': ['pipeline_strict_feasibility', 'conservative_valid_plan_fraction', 'conditional_conservative_gap_to_gurobi_bound_with_coverage', 'conditional_travel_on_common_valid_plan_cohort', 'conditional_duration_travel_plus_waiting_on_common_valid_plan_cohort', 'cvar_travel', 'mean_lateness_and_depot_overtime', 'recourse_reorders_and_delta_fixed_order_vs_causal_edd', 'total_compute_training_tuning_inference_evaluation_and_peak_memory'], 'secondary_comparators': ['gurobi', 'ortools_routing', 'aco', 'tabu', 'pomo', 'edd'], 'seed_ids': [11, 29, 47], 'search_seconds': 120.0, 'inference_scope': 'geometry_population_conditional_on_frozen_checkpoints_and_finite_seed_set', 'power_guaranteed': False, 'quotas_are_planning_targets_not_a_power_calculation': True, 'planning_paired_score_sd_sensitivity': [0.25, 0.5, 1.0], 'test_exposure_policy': 'once_after_population_config_code_models_and_analysis_frozen', 'failed_runs_policy': 'preserve_all_failures_missing_are_not_solver_failures', 'traffic_model_decision': 'retain_sealed_original_minute_implementation_no_intensity_change', 'unit_model_change_policy': 'separate_version_never_edit_accepted_bank', 'objective_scope': 'conservative_travel_with_risk_evaluated_separately_no_claim_c_plus_Q', 'confirmatory_runner_status': 'holdout_adapter_not_yet_validated', 'analysis_language': 'R_for_frozen_geometry_level_analysis_python_for_common_evaluation', 'primary_inference': {'method': 'stratified_geometry_weighted_hoeffding_mean_bound', 'score_range': [-1.0, 1.0], 'geometry_weight': '1 / (number_of_size_strata * n_test_geometries_in_size)', 'two_sided_pvalue': 'min(1, 2 * exp(-mean_score^2 / (2 * sum_geometry_weights_squared)))', 'pointwise_confidence': 0.95, 'simultaneous_two_contrast_interval_individual_confidence': 0.975, 'confidence_halfwidth': 'sqrt(2 * sum_geometry_weights_squared * log(2 / alpha))', 'interval_clipped_to': [-1.0, 1.0], 'assumptions': 'independent_test_geometries_conditional_on_frozen_models_and_finite_seed_set', 'limitation': 'finite_sample_distribution_free_but_conservative_no_power_guarantee', 'noninferiority_margin_fr': None, 'fr_interpretation_guard': 'always_report_pipeline_FR_difference_and_interval_no_global_robustness_claim_from_cost_win_score'}}
ANALYSIS_CONFIG = {'schema_version': 'analysis-08-v1', 'role': 'development_only', 'analysis_mode': 'descriptive_only', 'source_export_schema': 'analysis-export-08-v1', 'unit_of_independence': 'geometry_group_sha256', 'run_selection': 'explicit_report_path_and_sha256_no_latest_or_best', 'solver_replica_aggregation': 'all_scheduled_seeds_mean_within_instance', 'geometry_aggregation': 'equal_instance_weight_within_geometry', 'missing_success': 'bounded_unknown_no_imputation', 'no_solution_success': 'zero_only_when_recorded_no_solution_and_no_routes', 'cost_analysis': 'conditional_conservative_feasible_with_explicit_denominators', 'cvar_aggregation': 'mean_of_per_plan_cvar_not_pooled_scenarios', 'reference_method': 'gurobi', 'confirmation': {'enabled_for_verified_final_export': True, 'accepted_export_schema': 'final-analysis-export-v1', 'accepted_role': 'confirmatory_holdout', 'requires_verified_population_protocol_freeze_test_chain': True, 'requires_frozen_analysis_r_sha256': True, 'primary_inference': 'stratified_geometry_weighted_hoeffding_mean_bound', 'multiplicity': 'holm_two_sided_familywise_0.05', 'development_results_are_not_confirmatory_evidence': True, 'enabled': False, 'enabled_explanation': 'Legacy development branch remains descriptive; final10 branch is selected explicitly and verifies an independent locked release.'}}
ANALYSIS_CONFIG_SOURCE = '{\n  "schema_version": "analysis-08-v1",\n  "role": "development_only",\n  "analysis_mode": "descriptive_only",\n  "source_export_schema": "analysis-export-08-v1",\n  "unit_of_independence": "geometry_group_sha256",\n  "run_selection": "explicit_report_path_and_sha256_no_latest_or_best",\n  "solver_replica_aggregation": "all_scheduled_seeds_mean_within_instance",\n  "geometry_aggregation": "equal_instance_weight_within_geometry",\n  "missing_success": "bounded_unknown_no_imputation",\n  "no_solution_success": "zero_only_when_recorded_no_solution_and_no_routes",\n  "cost_analysis": "conditional_conservative_feasible_with_explicit_denominators",\n  "cvar_aggregation": "mean_of_per_plan_cvar_not_pooled_scenarios",\n  "reference_method": "gurobi",\n  "confirmation": {\n    "enabled_for_verified_final_export": true,\n    "accepted_export_schema": "final-analysis-export-v1",\n    "accepted_role": "confirmatory_holdout",\n    "requires_verified_population_protocol_freeze_test_chain": true,\n    "requires_frozen_analysis_r_sha256": true,\n    "primary_inference": "stratified_geometry_weighted_hoeffding_mean_bound",\n    "multiplicity": "holm_two_sided_familywise_0.05",\n    "development_results_are_not_confirmatory_evidence": true,\n    "enabled": false,\n    "enabled_explanation": "Legacy development branch remains descriptive; final10 branch is selected explicitly and verifies an independent locked release."\n  }\n}\n'


In [39]:
%%writefile svrplab/__init__.py
"""Infraestructura experimental de la tesis HBG-FACS."""


Overwriting svrplab/__init__.py


In [40]:
%%writefile svrplab/attention.py
"""Attention Model supervisado propio para 03; ejecución sólo en Google Colab.

Encoder de conjuntos sin posición y decoder pointer con atención y máscaras
conservadoras. Inspirado en la familia AM de Kool et al. (ICLR 2019), pero NO
reproduce su arquitectura exacta, REINFORCE ni sus resultados. Las etiquetas
proceden de búsquedas de desarrollo, nunca de certificados del banco.

Cada depósito seleccionado cierra el vehículo actual y avanza al siguiente.
La flota mantiene su orden original, incluso con capacidades heterogéneas y
rutas vacías. Las construcciones bloqueadas se registran. Durante inferencia,
la completación opcional explora prefijos desde cero, se declara por separado
y no sustituye rutas por etiquetas. No se consultan escenarios estocásticos.

Fuentes técnicas: https://arxiv.org/abs/1803.08475 y
https://docs.pytorch.org/docs/stable/generated/torch.nn.MultiheadAttention.html
"""
from __future__ import annotations

from collections.abc import Mapping
from dataclasses import asdict, dataclass
import math
import time

import numpy as np
import torch
from torch import nn

from .search import SearchProblem

IMPLEMENTATION = 'project_supervised_attention_v2'
INFERENCE_DEFAULTS = {'seconds': 120., 'n_samples': 16, 'permute_samples': False,
                      'completion_search_nodes': 2048}
MASK_VERSION = 'necessary_capacity_reachability_v2'
_TOL = 1e-9
_NODE_FEATURES = 9
_STATE_FEATURES = 6
_ACTION_FEATURES = 5


@dataclass(frozen=True)
class AttentionConfig:
    model_dim: int = 128
    num_heads: int = 8
    num_layers: int = 3
    dropout: float = .1
    tanh_clip: float = 10.

    def __post_init__(self):
        for name in ('model_dim', 'num_heads', 'num_layers'):
            value = getattr(self, name)
            if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer)) or value < 1:
                raise ValueError(f'{name} debe ser entero positivo.')
        if self.model_dim % self.num_heads:
            raise ValueError('model_dim debe ser divisible entre num_heads.')
        for name in ('dropout', 'tanh_clip'):
            value = getattr(self, name)
            if (isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.number))
                    or not math.isfinite(float(value))):
                raise ValueError(f'{name} debe ser un número finito.')
        if not 0 <= self.dropout < 1 or self.tanh_clip <= 0:
            raise ValueError('dropout debe pertenecer a [0,1), tanh_clip debe ser positivo.')

    def to_dict(self):
        return asdict(self)

    @classmethod
    def from_value(cls, config):
        if isinstance(config, cls):
            return config
        if config is None:
            return cls()
        if not isinstance(config, Mapping):
            raise ValueError('La configuración del modelo debe ser un mapping.')
        aliases = {'dim': 'model_dim', 'heads': 'num_heads', 'layers': 'num_layers'}
        values = dict(config)
        for short, full in aliases.items():
            if short in values:
                if full in values:
                    raise ValueError(f'No mezcle {short} y {full}.')
                values[full] = values.pop(short)
        unknown = set(values) - set(cls.__dataclass_fields__)
        if unknown:
            raise ValueError(f'Parámetros de modelo desconocidos: {sorted(unknown)}')
        return cls(**values)


def _node_order(problem, node_order):
    """Permuta sólo clientes; el token de depósito conserva la posición cero."""
    if node_order is None:
        return np.arange(problem.n + 1, dtype=np.int64)
    order = list(node_order)
    if (any(isinstance(j, (bool, np.bool_)) or not isinstance(j, (int, np.integer)) for j in order)
            or sorted(order) != list(range(1, problem.n + 1))):
        raise ValueError('node_order debe ser una permutación de los clientes 1..n, sin depósito.')
    return np.array([0, *order], dtype=np.int64)


def _scales(problem):
    xy = problem.x.locations - problem.x.locations[0]
    coord = max(1., float(np.max(np.abs(xy))))
    horizon = max(1., abs(float(problem.horizon)), float(np.max(np.abs(problem.close))),
                  float(np.max(np.abs(problem.open))))
    capacity = max(1., float(np.max(problem.capacities)))
    return coord, horizon, capacity


def _node_features(problem, order, scales):
    coord, horizon, capacity = scales
    locations = (problem.x.locations - problem.x.locations[0]) / coord
    features = np.column_stack((locations, problem.demands / capacity,
                                problem.open / horizon, problem.close / horizon,
                                problem.x.appear_times / horizon,
                                np.arange(problem.n + 1) == 0,
                                problem.travel[0] / horizon, problem.travel[:, 0] / horizon))
    if not np.isfinite(features).all():
        raise ValueError('Features del modelo no finitos.')
    return np.asarray(features[order], dtype=np.float32)


class AttentionPolicy(nn.Module):
    """Encoder permutation-equivariant y decoder pointer condicionado por flota."""
    def __init__(self, config=None):
        super().__init__()
        self.config = AttentionConfig.from_value(config)
        c = self.config
        self.node_embedding = nn.Linear(_NODE_FEATURES, c.model_dim)
        # Capas construidas separadamente: no comparten pesos ni inicialización.
        self.encoder = nn.ModuleList([
            nn.TransformerEncoderLayer(c.model_dim, c.num_heads,
                                       dim_feedforward=4 * c.model_dim,
                                       dropout=c.dropout, activation='gelu',
                                       batch_first=True, norm_first=True)
            for _ in range(c.num_layers)
        ])
        self.encoder_norm = nn.LayerNorm(c.model_dim)
        self.context = nn.Linear(2 * c.model_dim + _STATE_FEATURES, c.model_dim)
        self.action_embedding = nn.Linear(_ACTION_FEATURES, c.model_dim, bias=False)
        self.glimpse = nn.MultiheadAttention(c.model_dim, c.num_heads,
                                           dropout=c.dropout, batch_first=True)
        self.pointer_query = nn.Linear(c.model_dim, c.model_dim, bias=False)
        self.pointer_key = nn.Linear(c.model_dim, c.model_dim, bias=False)

    def encode(self, features):
        encoded = self.node_embedding(features).unsqueeze(0)
        for layer in self.encoder:
            encoded = layer(encoded)
        return self.encoder_norm(encoded).squeeze(0)

    def next_logits(self, encoded, current_local, state_features, action_features, blocked):
        if bool(blocked.all()):
            raise ValueError('No se calcula softmax de un estado sin acciones factibles.')
        query = self.context(torch.cat((encoded.mean(dim=0), encoded[current_local],
                                        state_features))).view(1, 1, -1)
        candidates = encoded + self.action_embedding(action_features)
        glimpse, _ = self.glimpse(query, candidates.unsqueeze(0), candidates.unsqueeze(0),
                                  key_padding_mask=blocked.unsqueeze(0), need_weights=False)
        logits = (self.pointer_key(candidates) * self.pointer_query(glimpse[0, 0])).sum(dim=-1)
        logits = self.config.tanh_clip * torch.tanh(logits / math.sqrt(self.config.model_dim))
        return logits.masked_fill(blocked, -torch.inf)


def _travel_lower_bound(problem):
    """Admissible all-pairs travel bound, including for nonmetric matrices.

    Intermediate vertices may already be visited: allowing them only relaxes
    the completion problem. Rounding each sum down avoids pruning a feasible
    completion due to numerical overestimation. This cache changes no data.
    """
    cached = getattr(problem, '_attention_travel_bound', None)
    if cached is not None:
        return cached
    lower = np.array(problem.travel, dtype=np.float64, copy=True)
    if not np.isfinite(lower).all() or (lower < 0).any():
        raise ValueError('La anticipación requiere tiempos finitos no negativos.')
    np.fill_diagonal(lower, 0.)
    for intermediate in range(problem.n + 1):
        via = np.nextafter(lower[:, intermediate, None] + lower[None, intermediate, :], -np.inf)
        np.minimum(lower, np.maximum(0., via), out=lower)
    lower.setflags(write=False)
    problem._attention_travel_bound = lower
    return lower


class _State:
    """Estado determinista CPU; sólo las features y la atención pasan al dispositivo."""
    def __init__(self, problem):
        self.problem = problem
        self.vehicle = 0
        self.node = 0
        self.clock = 0.
        self.load = 0.
        self.visited = np.zeros(problem.n + 1, dtype=bool)
        self.visited[0] = True
        self.routes = [[] for _ in range(problem.vehicles)]
        self.lower = _travel_lower_bound(problem)

    def clone(self):
        copied = object.__new__(type(self))
        copied.problem, copied.lower = self.problem, self.lower
        copied.vehicle, copied.node = self.vehicle, self.node
        copied.clock, copied.load = self.clock, self.load
        copied.visited = self.visited.copy()
        copied.routes = [list(route) for route in self.routes]
        return copied

    def _completion_possible_after(self, candidates, remaining, start):
        """Necessary conditions only, never a certificate of completion.

        Each pending customer must fit/reach either the current vehicle after
        the proposed action or a fresh future vehicle. Customers that cannot
        use any future vehicle must fit together in the current residual
        capacity. All conditions relax ordering and hence cannot exclude an
        actually feasible completion. Zero-demand customers remain mandatory.
        """
        p = self.problem
        pending = np.flatnonzero(remaining)
        if not len(candidates) or not len(pending):
            return np.ones(len(candidates), dtype=bool)
        future = p.capacities[self.vehicle + 1:]
        future_start = np.maximum(self.lower[0, pending], p.open[pending])
        future_ok = ((p.demands[pending] <= (float(future.max()) if len(future) else -1.) + _TOL)
                     & (future_start <= p.close[pending] + _TOL)
                     & (future_start + self.lower[pending, 0] <= p.horizon + _TOL))
        residual = p.capacities[self.vehicle] - self.load - p.demands[candidates]
        earliest = np.maximum(start[candidates, None] + self.lower[np.ix_(candidates, pending)],
                              p.open[pending][None, :])
        current_ok = ((p.demands[pending][None, :] <= residual[:, None] + _TOL)
                      & (earliest <= p.close[pending][None, :] + _TOL)
                      & (earliest + self.lower[pending, 0][None, :] <= p.horizon + _TOL))
        selected = candidates[:, None] == pending[None, :]
        individual = (selected | current_ok | future_ok[None, :]).all(axis=1)
        total_left = float(p.demands[pending].sum()) - p.demands[candidates]
        total_capacity = residual + float(future.sum())
        mandatory = (p.demands[pending] * ~future_ok)[None, :] * ~selected
        return (individual & (total_left <= total_capacity + (len(future) + 1) * _TOL)
                & (mandatory.sum(axis=1) <= residual + _TOL))

    @property
    def done(self):
        return self.vehicle == self.problem.vehicles

    def inputs(self, order, scales):
        p = self.problem
        _, horizon, capacity = scales
        remaining = ~self.visited
        arrival = self.clock + p.travel[self.node]
        start = np.maximum(arrival, p.open)
        returned = start + p.travel[:, 0]
        allowed = (remaining & (self.load + p.demands <= p.capacities[self.vehicle] + _TOL)
                   & (start <= p.close + _TOL)
                   & (start + self.lower[:, 0] <= p.horizon + _TOL))
        candidates = np.flatnonzero(allowed)
        allowed[candidates] &= self._completion_possible_after(candidates, remaining, start)
        # Cerrar el último vehículo no puede dejar clientes sin atender, ni de demanda cero.
        can_close = self.clock + p.travel[self.node, 0] <= p.horizon + _TOL
        if self.vehicle == p.vehicles - 1:
            can_close = can_close and not bool(remaining.any())
        elif remaining.any():
            future = p.capacities[self.vehicle + 1:]
            earliest = np.maximum(self.lower[0, remaining], p.open[remaining])
            can_close = (can_close and float(p.demands[remaining].sum()) <= float(future.sum()) + len(future) * _TOL
                         and float(p.demands[remaining].max()) <= float(future.max()) + _TOL
                         and bool((earliest <= p.close[remaining] + _TOL).all())
                         and bool((earliest + self.lower[remaining, 0] <= p.horizon + _TOL).all()))
        allowed[0] = can_close
        residual = float(p.capacities[self.vehicle]) - self.load
        state = np.array([self.clock / horizon, residual / capacity,
                          float(p.capacities[self.vehicle]) / capacity,
                          self.vehicle / max(1, p.vehicles - 1),
                          float(remaining.sum()) / max(1, p.n),
                          float(p.capacities[self.vehicle + 1:].sum()) /
                          max(1., float(p.capacities.sum()))], dtype=np.float32)
        action = np.column_stack((p.travel[self.node] / horizon, start / horizon,
                                  (p.close - start) / horizon, p.demands / capacity,
                                  returned / horizon))
        # El depósito significa regreso y cambio de vehículo, no servicio como cliente.
        action[0] = [p.travel[self.node, 0] / horizon,
                     (self.clock + p.travel[self.node, 0]) / horizon,
                     (p.horizon - self.clock - p.travel[self.node, 0]) / horizon,
                     0., (self.clock + p.travel[self.node, 0]) / horizon]
        return state, np.asarray(action[order], dtype=np.float32), ~allowed[order]

    def advance(self, customer):
        if customer == 0:
            self.vehicle += 1
            self.node, self.clock, self.load = 0, 0., 0.
        else:
            self.clock = max(self.clock + float(self.problem.travel[self.node, customer]),
                             float(self.problem.open[customer]))
            self.load += float(self.problem.demands[customer])
            self.node = customer
            self.visited[customer] = True
            self.routes[self.vehicle].append(customer)


def _tensors(state, order, scales, device):
    context, action, blocked = state.inputs(order, scales)
    return (torch.as_tensor(context, device=device), torch.as_tensor(action, device=device),
            torch.as_tensor(blocked, device=device))


def _prepare(model, problem, device, node_order):
    order = _node_order(problem, node_order)
    scales = _scales(problem)
    inverse = np.empty(problem.n + 1, dtype=np.int64)
    inverse[order] = np.arange(problem.n + 1)
    features = torch.as_tensor(_node_features(problem, order, scales), device=device)
    return model.encode(features), order, inverse, scales


def teacher_loss(model, problem, routes, device, node_order=None):
    """NLL media por decisión; el caller decide train/eval y ejecuta backward.

    Rechaza etiquetas sin cobertura/capacidad o no factibles en el modelo
    conservador. Devuelve (loss, {'decisions', 'nll_sum'}), con nll_sum numérico
    separado del grafo. Incluye un cierre por vehículo y clientes de demanda 0.
    No comprueba procedencia: el registro de profesores debe comprobarla antes.
    """
    violation, _ = problem.score(routes)
    if violation > _TOL:
        raise ValueError('La etiqueta no es factible bajo la matriz conservadora común.')
    encoded, order, inverse, scales = _prepare(model, problem, device, node_order)
    state = _State(problem)
    terms = []
    for route in routes:
        for customer in [*route, 0]:
            context, action, blocked = _tensors(state, order, scales, device)
            label = int(inverse[customer])
            if bool(blocked[label]):
                raise ValueError('Una acción del profesor está prohibida por las máscaras comunes.')
            logits = model.next_logits(encoded, int(inverse[state.node]), context, action, blocked)
            term = -torch.log_softmax(logits, dim=-1)[label]
            if not bool(torch.isfinite(term)):
                raise FloatingPointError('NLL no finita; no continuar el entrenamiento.')
            terms.append(term)
            state.advance(int(customer))
    if not state.done or not bool(state.visited.all()) or not terms:
        raise ValueError('Secuencia del profesor incompleta.')
    total = torch.stack(terms).sum()
    return total / len(terms), {'decisions': len(terms), 'nll_sum': float(total.detach().cpu())}


def _sync(device):
    device = torch.device(device)
    if device.type == 'cuda':
        torch.cuda.synchronize(device)


def _expired(deadline, device):
    if deadline is None:
        return False
    _sync(device)
    return time.perf_counter() >= deadline


def decode(model, problem, device, mode='greedy', generator=None, deadline=None, node_order=None):
    """Construye sin reparar; None significa bloqueo o tiempo, no infactibilidad probada.

    En sample, generator debe corresponder al dispositivo de PyTorch utilizado.
    deadline es un instante de perf_counter, compartido por todos los intentos.
    La sincronización CUDA incluye el trabajo asíncrono en dicho presupuesto.
    """
    if mode not in ('greedy', 'sample'):
        raise ValueError('mode debe ser greedy o sample.')
    was_training = model.training
    model.eval()
    try:
        with torch.inference_mode():
            if _expired(deadline, device):
                return None
            encoded, order, inverse, scales = _prepare(model, problem, device, node_order)
            state = _State(problem)
            while not state.done:
                if _expired(deadline, device):
                    return None
                context, action, blocked = _tensors(state, order, scales, device)
                if bool(blocked.all()):
                    return None
                logits = model.next_logits(encoded, int(inverse[state.node]), context, action, blocked)
                if not bool(torch.isfinite(logits[~blocked]).all()):
                    raise FloatingPointError('Logits no finitos en la inferencia.')
                if mode == 'greedy':
                    local = int(torch.argmax(logits).item())
                else:
                    probabilities = torch.softmax(logits, dim=-1)
                    local = int(torch.multinomial(probabilities, 1, generator=generator).item())
                state.advance(int(order[local]))
            # La validación independiente también se incluye en el tiempo de solve_attention.
            violation, _ = problem.score(state.routes)
            return state.routes if violation <= _TOL else None
    finally:
        model.train(was_training)


class UniformMaskedPolicy(nn.Module):
    """Untrained diagnostic: exactly uniform logits over the common mask."""
    def __init__(self, config=None):
        super().__init__()
        self.config = AttentionConfig.from_value(config)

    def encode(self, features):
        return features.new_zeros((len(features), 1))

    def next_logits(self, encoded, current_local, state_features, action_features, blocked):
        return encoded.new_zeros(len(blocked)).masked_fill(blocked, -torch.inf)


def policy_completion_search(model, problem, device, deadline, max_nodes=2048):
    """Policy-ordered depth-first completion, separately declared from a rollout.

    Restarts from the empty state, backtracks only through the shared action
    mask, and shares the caller's deadline. No external routes or solver are
    consulted. These search trajectories have no rollout probability and must
    not be added to on-policy training or replay as sampled trajectories.
    """
    if type(max_nodes) is not int or max_nodes < 0:
        raise ValueError('max_nodes debe ser entero no negativo.')
    stats = {'expanded_nodes': 0, 'backtracks': 0, 'termination': 'node_limit',
             'construction': 'policy_ordered_depth_first_completion',
             'is_policy_sample': False, 'mask_version': MASK_VERSION}
    was_training = model.training
    model.eval()
    try:
        with torch.inference_mode():
            if max_nodes == 0 or _expired(deadline, device):
                stats['termination'] = 'time_limit' if max_nodes else 'disabled'
                return None, stats
            encoded, order, inverse, scales = _prepare(model, problem, device, None)
            stack = [(_State(problem), None)]
            while stack:
                if _expired(deadline, device):
                    stats['termination'] = 'time_limit'
                    return None, stats
                state, choices = stack.pop()
                if state.done:
                    if bool(state.visited.all()) and problem.score(state.routes)[0] <= _TOL:
                        stats['termination'] = 'feasible_found'
                        return state.routes, stats
                    stats['backtracks'] += 1
                    continue
                if choices is None:
                    if stats['expanded_nodes'] >= max_nodes:
                        return None, stats
                    stats['expanded_nodes'] += 1
                    context, action, blocked = _tensors(state, order, scales, device)
                    if bool(blocked.all()):
                        stats['backtracks'] += 1
                        continue
                    logits = model.next_logits(encoded, int(inverse[state.node]), context, action, blocked)
                    if not bool(torch.isfinite(logits[~blocked]).all()):
                        raise FloatingPointError('Logits no finitos en búsqueda de completación.')
                    choices = [int(order[local]) for local in torch.argsort(
                        logits, descending=True, stable=True).cpu().tolist() if not bool(blocked[local])]
                if len(choices) > 1:
                    stack.append((state, choices[1:]))
                child = state.clone()
                child.advance(choices[0])
                stack.append((child, None))
            if not stack:
                stats['termination'] = 'tree_exhausted_under_mask'
            return None, stats
    finally:
        model.train(was_training)


def _inference_config(config):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración de inferencia debe ser un mapping.')
    unknown = set(config or {}) - set(INFERENCE_DEFAULTS)
    if unknown:
        raise ValueError(f'Parámetros de inferencia desconocidos: {sorted(unknown)}')
    result = INFERENCE_DEFAULTS | dict(config or {})
    seconds = result['seconds']
    if (isinstance(seconds, (bool, np.bool_)) or not isinstance(seconds, (int, float, np.number))
            or not math.isfinite(float(seconds)) or seconds < 0):
        raise ValueError('seconds debe ser finito y no negativo.')
    samples = result['n_samples']
    if isinstance(samples, (bool, np.bool_)) or not isinstance(samples, (int, np.integer)) or samples < 0:
        raise ValueError('n_samples debe ser entero no negativo.')
    if not isinstance(result['permute_samples'], bool):
        raise ValueError('permute_samples debe ser booleano.')
    nodes = result['completion_search_nodes']
    if type(nodes) is not int or nodes < 0:
        raise ValueError('completion_search_nodes debe ser entero no negativo.')
    return result | {'seconds': float(seconds), 'n_samples': int(samples)}


def solve_attention(problem: SearchProblem, seed, config, model, device):
    """Un greedy y hasta n_samples, todos dentro del mismo límite de inferencia."""
    if isinstance(seed, (bool, np.bool_)) or not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('La semilla debe ser un entero no negativo.')
    effective = _inference_config(config)
    device = torch.device(device)
    _sync(device)
    started = time.perf_counter()
    deadline = started + effective['seconds']
    generator = torch.Generator(device=device).manual_seed(int(seed) % (2**63 - 1))
    rng = np.random.default_rng(int(seed))
    best, best_score, history = None, (math.inf, math.inf), []
    attempts = completed = failed = 0
    termination = 'sample_limit'
    completion_search = {'termination': 'not_needed', 'expanded_nodes': 0, 'is_policy_sample': False}
    for attempt in range(effective['n_samples'] + 1):
        if _expired(deadline, device):
            termination = 'time_limit'
            break
        mode = 'greedy' if attempt == 0 else 'sample'
        order = (rng.permutation(np.arange(1, problem.n + 1))
                 if mode == 'sample' and effective['permute_samples'] else None)
        attempts += 1
        routes = decode(model, problem, device, mode=mode, generator=generator,
                        deadline=deadline, node_order=order)
        if routes is None:
            failed += 1
        else:
            score = problem.score(routes)
            completed += 1
            if score < best_score:
                best, best_score = routes, score
                _sync(device)
                history.append({'iteration': attempt, 'elapsed_seconds': time.perf_counter() - started,
                                'violation_minutes': float(score[0]), 'travel_minutes': float(score[1]),
                                'phase': mode})
        if _expired(deadline, device):
            termination = 'time_limit'
            break
    if best is None and effective['completion_search_nodes'] and not _expired(deadline, device):
        best, completion_search = policy_completion_search(
            model, problem, device, deadline, effective['completion_search_nodes'])
        if best is not None:
            best_score = problem.score(best)
            history.append({'iteration': attempts, 'elapsed_seconds': time.perf_counter() - started,
                            'violation_minutes': float(best_score[0]), 'travel_minutes': float(best_score[1]),
                            'phase': 'policy_completion_search'})
    if best is None and completion_search['termination'] == 'not_needed':
        completion_search['termination'] = ('no_remaining_budget' if effective['completion_search_nodes'] else 'disabled')
    if _expired(deadline, device):
        termination = 'time_limit'
    _sync(device)
    elapsed = time.perf_counter() - started
    result = problem.result(
        best, 'feasible' if best is not None else 'no_solution', elapsed, history,
        method='am_supervised', seed=int(seed), implementation=IMPLEMENTATION,
        model_config=model.config.to_dict(), effective_config=effective,
        termination=termination, attempts=attempts, completed_decodes=completed, failed_decodes=failed,
        raw_decode_complete_fraction=completed / attempts if attempts else None,
        completion_search=completion_search, mask_version=MASK_VERSION,
        solution_origin='policy_completion_search' if completion_search['termination'] == 'feasible_found' else
                        'raw_policy_decode' if best is not None else 'none',
        inference_device=str(device), budget_includes_construction=True,
        budget_includes_encoding=True, budget_overrun_seconds=max(0., elapsed - effective['seconds']),
        time_limit_granularity='synchronized_decoder_step_or_completed_validation',
        search_objective='conservative_upper_travel_minutes',
        nominal_scope='conservative_upper_mean_matrix_not_stochastic_feasibility',
        construction='from_scratch_without_certificates_or_teacher_fallback',
        seed_reproducibility='same_environment_and_sample_cutoff; wall_time_cutoff_may_differ',
        no_solution_scope='decoder_dead_end_or_budget_not_infeasibility_proof',
        training_included_in_solve_seconds=False)
    _sync(device)
    elapsed = time.perf_counter() - started
    result['solve_seconds'] = elapsed
    result['metadata']['budget_overrun_seconds'] = max(0., elapsed - effective['seconds'])
    return result


Overwriting svrplab/attention.py


In [41]:
%%writefile svrplab/bank_io.py
"""Copia del banco sellado desde Drive; sólo stdlib, sin métodos ni evaluador.

Se incrusta como bank_io.py en el runtime Colab antes de importar el banco.
No modifica Drive. Los errores de integridad no se consideran errores de conexión.
"""
from __future__ import annotations

import errno
import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import time
import uuid


_TRANSIENT = {errno.EIO, errno.EAGAIN, errno.EINTR, errno.ECONNABORTED,
              errno.ECONNRESET, errno.ETIMEDOUT, errno.ENOTCONN, errno.ESTALE}


def _retry_io(operation, label, delays, sleep, progress):
    """Reabre/repite la operación completa; no oculta errores de sello/permisos."""
    for attempt in range(len(delays) + 1):
        try:
            return operation()
        except OSError as exc:
            if exc.errno not in _TRANSIENT:
                raise
            if attempt == len(delays):
                raise ConnectionError(
                    f'Lectura de Drive interrumpida tras {attempt + 1} intentos: {label}. '
                    'La copia parcial local se conserva. Vuelve a ejecutar la celda de banco; '
                    'si persiste, activa FORCE_REMOUNT_DRIVE y vuelve a ejecutarla. '
                    'No regeneres el banco ni omitas sus hashes.') from exc
            delay = delays[attempt]
            progress(f'Drive: {label}: {type(exc).__name__} (errno={exc.errno}); '
                     f'reintento {attempt + 1}/{len(delays)} en {delay} s.')
            sleep(delay)


def _file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def _manifest(folder, reference):
    manifest = json.loads((folder / 'CHECKSUMS.json').read_text(encoding='utf-8'))
    manifest_sha = hashlib.sha256(json.dumps(
        manifest, sort_keys=True, separators=(',', ':'), ensure_ascii=False,
        allow_nan=False).encode('utf-8')).hexdigest()
    if manifest_sha != reference['manifest_sha256']:
        raise ValueError('El manifiesto no corresponde al banco aceptado en 00.')
    if not isinstance(manifest, dict) or not manifest:
        raise ValueError('Manifiesto inválido o vacío.')
    for name, digest in manifest.items():
        path = PurePosixPath(name)
        if (not name or path.is_absolute() or '..' in path.parts
                or str(path) != name or '__pycache__' in path.parts
                or path.name == 'CHECKSUMS.json'):
            raise ValueError(f'Ruta de manifiesto no permitida: {name!r}')
        if (not isinstance(digest, str) or len(digest) != 64
                or any(char not in '0123456789abcdef' for char in digest)):
            raise ValueError(f'SHA-256 inválido en manifiesto: {name}')
    return manifest


def _inside(folder, relative):
    path = folder / relative
    if not path.resolve().is_relative_to(folder.resolve()):
        raise ValueError(f'Ruta fuera del banco: {relative}')
    return path


def _inventory(folder):
    """os.walk propaga errores de Drive; no acepta un listado parcial silencioso."""
    def fail(exc):
        raise exc
    actual = set()
    for root, dirs, files in os.walk(folder, onerror=fail):
        dirs[:] = [name for name in dirs if name != '__pycache__']
        for name in dirs + files:
            if (Path(root) / name).is_symlink():
                raise ValueError('El banco no admite enlaces simbólicos.')
        for name in files:
            relative = str((Path(root) / name).relative_to(folder))
            if relative != 'CHECKSUMS.json':
                actual.add(relative)
    return actual


def _validate_local(folder, reference):
    manifest = _manifest(folder, reference)
    if _inventory(folder) != set(manifest):
        raise ValueError('Inventario local distinto del manifiesto; conservar e inspeccionar la copia.')
    for relative, expected in manifest.items():
        if _file_hash(_inside(folder, relative)) != expected:
            raise ValueError(f'Hash incorrecto: {relative}')
    acceptance = json.loads((folder / 'acceptance.json').read_text(encoding='utf-8'))
    config = json.loads((folder / 'config.json').read_text(encoding='utf-8'))
    if (acceptance.get('status') != reference['acceptance_status']
            or acceptance.get('protocol_version') != reference['protocol_version']
            or config.get('protocol_version') != reference['protocol_version']):
        raise ValueError('Acta/configuración distinta de la aceptación esperada.')
    return manifest


def _copy_file(source, target, expected):
    """El destino sólo cambia tras una lectura completa y SHA correcto."""
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_name(target.name + '.download-' + uuid.uuid4().hex)
    digest = hashlib.sha256()
    try:
        with source.open('rb') as src, temporary.open('xb') as dst:
            for block in iter(lambda: src.read(1024 * 1024), b''):
                digest.update(block)
                dst.write(block)
        if digest.hexdigest() != expected:
            raise ValueError(f'Hash incorrecto al copiar: {source.name}. No se aceptó el archivo.')
        temporary.replace(target)
    finally:
        # Sólo elimina el temporal local de esta operación; nunca el banco de Drive.
        temporary.unlink(missing_ok=True)


def copy_accepted_bank(remote, destination, reference, *, retry_delays=(1, 2, 4, 8),
                       sleep=time.sleep, progress=print):
    """Reutiliza una copia íntegra o completa .partial, con sello exacto de 00."""
    remote, destination = Path(remote), Path(destination)
    if remote.name != reference['bank_id'] or destination.name != reference['bank_id']:
        raise ValueError('Ruta de banco distinta del BANK_ID aceptado.')
    if destination.absolute().is_relative_to(remote.absolute()):
        raise ValueError('La copia de trabajo debe estar fuera del banco remoto.')
    if destination.exists():
        manifest = _validate_local(destination, reference)
        progress('Copia local completa: manifiesto y archivos verificados; no se releen escenarios de Drive.')
        return manifest

    def remote_io(operation, label):
        return _retry_io(operation, label, retry_delays, sleep, progress)

    # El sello fijado se comprueba ANTES de confiar en rutas o contenido del banco.
    manifest = remote_io(lambda: _manifest(remote, reference), 'CHECKSUMS.json')
    if remote_io(lambda: _inventory(remote), 'inventario del banco') != set(manifest):
        raise ValueError('Inventario de Drive distinto del manifiesto.')
    partial = destination.with_name(destination.name + '.partial')
    if partial.is_symlink():
        raise ValueError('La copia parcial no puede ser un enlace simbólico.')
    partial.mkdir(parents=True, exist_ok=True)
    # Los temporales de interrupciones abruptas quedan fuera del banco final.
    for relative in sorted(_inventory(partial)):
        path = partial / relative
        if '.download-' in path.name and relative not in manifest:
            preserved = partial.with_name(partial.name + '.interrupted')
            preserved.mkdir(exist_ok=True)
            path.rename(preserved / (uuid.uuid4().hex + '_' + path.name))
    extras = _inventory(partial) - set(manifest)
    if extras:
        raise ValueError(f'Archivos inesperados en copia parcial; conservar e inspeccionar: {sorted(extras)}')
    for index, (relative, expected) in enumerate(sorted(manifest.items()), 1):
        target = _inside(partial, relative)
        if not target.is_file() or _file_hash(target) != expected:
            remote_io(lambda: _copy_file(_inside(remote, relative), target, expected), relative)
        progress(f'Banco: {index}/{len(manifest)} archivos comprobados: {relative}')
    (partial / 'CHECKSUMS.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    _validate_local(partial, reference)
    partial.rename(destination)
    return manifest


Overwriting svrplab/bank_io.py


In [42]:
%%writefile svrplab/baselines.py
"""Independent conventional controls and paid construction/local search helpers.

EDD means earliest deadline first with feasible cheapest insertion. It is a
fast control, not a lower bound. OR-Tools Routing uses guided local search on
exactly the shared conservative travel grid. It is a heuristic reference, not
an optimal ceiling. No function reads certificates, scenarios or learned data.
All calls that construct/search routes are to be executed exclusively in Colab.

Primary API: https://developers.google.com/optimization/routing/vrptw and
https://developers.google.com/optimization/routing/routing_options .
"""
from __future__ import annotations

from collections.abc import Mapping
import importlib.metadata
import math
from numbers import Integral, Real
import time

import numpy as np

_TOL = 1e-9
_MAX_INT = 2**60
EDD_DEFAULTS = {'seconds': 120.}
ORTOOLS_DEFAULTS = {'seconds': 120., 'capacity_scale': 1000}


def _config(defaults, config):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración debe ser un mapping.')
    unknown = set(config or {}) - set(defaults)
    if unknown:
        raise ValueError(f'Parámetros desconocidos: {sorted(unknown)}')
    result = defaults | dict(config or {})
    seconds = result['seconds']
    if isinstance(seconds, bool) or not isinstance(seconds, Real) or not math.isfinite(seconds) or seconds < 0:
        raise ValueError('seconds debe ser finito y no negativo.')
    result['seconds'] = float(seconds)
    if 'capacity_scale' in result:
        value = result['capacity_scale']
        if isinstance(value, bool) or not isinstance(value, Integral) or value <= 0:
            raise ValueError('capacity_scale debe ser un entero positivo.')
        result['capacity_scale'] = int(value)
    return result


def _seed(seed):
    if isinstance(seed, bool) or not isinstance(seed, Integral) or seed < 0:
        raise ValueError('La semilla debe ser un entero no negativo.')
    return int(seed)


def _route_values(problem, route, vehicle):
    stats = problem.route_stats(route, vehicle)
    return (stats['lateness_minutes'] + stats['depot_overtime_minutes'], stats['travel_minutes'])


def _profile(problem, route):
    """Earliest prefix and latest suffix starts for constant-time insertions."""
    nodes = [0] + list(route) + [0]
    earliest = [0.]
    for previous, customer in zip(nodes[:-2], nodes[1:-1]):
        earliest.append(max(earliest[-1] + float(problem.travel[previous, customer]),
                            float(problem.open[customer])))
    latest = [0.] * len(nodes)
    latest[-1] = problem.horizon
    for position in range(len(nodes) - 2, 0, -1):
        customer, following = nodes[position:position + 2]
        latest[position] = min(float(problem.close[customer]),
                               latest[position + 1] - float(problem.travel[customer, following]))
    feasible = all(earliest[i] <= latest[i] + _TOL for i in range(1, len(nodes) - 1))
    return nodes, earliest, latest, feasible


def construct_insertion(problem, rng, deadline, pheromone=None, alpha=0., beta=2., q0=1.):
    """Complete EDD/RCL construction with hard capacity and paid soft repair.

    The deterministic case processes scarce-capacity customers, then deadlines.
    Stochastic construction samples among the three earliest eligible customers
    and feasible insertion positions, guided by travel and (optional) pheromone.
    If no temporal insertion exists, choose minimum total temporal violation;
    do not drop customers, add vehicles, or declare the instance infeasible.
    """
    routes = [[] for _ in range(problem.vehicles)]
    loads = np.zeros(problem.vehicles)
    remaining = set(range(1, problem.n + 1))
    profiles = [_profile(problem, route) for route in routes]
    while remaining:
        if time.perf_counter() >= deadline:
            return None, 'time_limit'
        availability = {j: int(np.count_nonzero(loads + problem.demands[j] <= problem.capacities + _TOL))
                        for j in remaining}
        if min(availability.values()) == 0:
            return None, 'capacity_dead_end'
        scarce = min(availability.values())
        ordered = sorted((j for j in remaining if availability[j] == scarce),
                         key=lambda j: (float(problem.close[j]), float(problem.open[j]), j))
        deterministic = q0 >= 1. or rng.random() < q0
        customer = ordered[0] if deterministic else ordered[int(rng.integers(min(3, len(ordered))))]
        feasible = []
        for vehicle, route in enumerate(routes):
            if time.perf_counter() >= deadline:
                return None, 'time_limit'
            if loads[vehicle] + problem.demands[customer] > problem.capacities[vehicle] + _TOL:
                continue
            nodes, earliest, latest, valid = profiles[vehicle]
            if not valid:
                continue
            for position in range(len(route) + 1):
                previous, following = nodes[position:position + 2]
                start = max(earliest[position] + float(problem.travel[previous, customer]),
                            float(problem.open[customer]))
                if (start > problem.close[customer] + _TOL
                        or start + problem.travel[customer, following] > latest[position + 1] + _TOL):
                    continue
                delta = float(problem.travel[previous, customer] + problem.travel[customer, following]
                              - problem.travel[previous, following])
                trail = (1. if pheromone is None else
                         math.sqrt(max(1e-12, float(pheromone[previous, customer]))
                                   * max(1e-12, float(pheromone[customer, following]))))
                weight = alpha * math.log(trail) - beta * math.log1p(max(0., delta))
                feasible.append((vehicle, position, delta, weight))
        if feasible:
            if deterministic:
                # ACO exploitation uses its pheromone; alpha=0 is cheapest EDD.
                choice = max(feasible, key=lambda row: (row[3], -row[2], -row[0], -row[1]))
            else:
                log_weights = np.array([row[3] for row in feasible])
                weights = np.exp(log_weights - np.max(log_weights))
                choice = feasible[int(rng.choice(len(feasible), p=weights / weights.sum()))]
            vehicle, position = choice[:2]
        else:
            # Preserve a complete diagnostic solution even when construction
            # cannot respect all windows. Full affected-route scores are paid.
            best = None
            for vehicle, route in enumerate(routes):
                if loads[vehicle] + problem.demands[customer] > problem.capacities[vehicle] + _TOL:
                    continue
                old = _route_values(problem, route, vehicle)
                for position in range(len(route) + 1):
                    if time.perf_counter() >= deadline:
                        return None, 'time_limit'
                    candidate = route[:position] + [customer] + route[position:]
                    new = _route_values(problem, candidate, vehicle)
                    row = (new[0] - old[0], new[1] - old[1], vehicle, position)
                    if best is None or row < best:
                        best = row
            if best is None:
                return None, 'capacity_dead_end'
            vehicle, position = best[2:]
        routes[vehicle].insert(position, customer)
        loads[vehicle] += problem.demands[customer]
        profiles[vehicle] = _profile(problem, routes[vehicle])
        remaining.remove(customer)
    return routes, 'complete'


def improve_feasible_routes(problem, routes, rng, deadline, max_moves=100):
    """Paid first-improving relocate repair/descent; retains the input incumbent.

    Return ``(routes, diagnostics)``. Coverage, capacity and fleet remain hard;
    score is lexicographic (temporal violation, conservative travel). An already
    feasible incumbent therefore stays feasible. Infeasible input is permitted
    for repair, but success is not guaranteed. No external routes/solver used.
    """
    if isinstance(max_moves, bool) or not isinstance(max_moves, Integral) or max_moves < 0:
        raise ValueError('max_moves debe ser entero no negativo.')
    current = [list(route) for route in routes]
    incumbent_score = problem.score(current)  # Common coverage/capacity contract before repair.
    stats = [_route_values(problem, route, v) for v, route in enumerate(current)]
    loads = [float(sum(problem.demands[j] for j in route)) for route in current]
    evaluated = accepted = 0
    termination = 'move_limit'
    for _ in range(max_moves):
        if time.perf_counter() >= deadline:
            termination = 'time_limit'
            break
        positions = [(v, i) for v, route in enumerate(current) for i in range(len(route))]
        # Repair the worst routes first; randomize ties with the method's RNG.
        if rng is not None:
            rng.shuffle(positions)
        positions.sort(key=lambda pair: -stats[pair[0]][0])
        chosen = None
        for origin, offset in positions:
            customer = current[origin][offset]
            reduced = current[origin][:offset] + current[origin][offset + 1:]
            reduced_stats = _route_values(problem, reduced, origin)
            for target, route in enumerate(current):
                if target != origin and loads[target] + problem.demands[customer] > problem.capacities[target] + _TOL:
                    continue
                base = reduced if target == origin else route
                for position in range(len(base) + 1):
                    if time.perf_counter() >= deadline:
                        termination = 'time_limit'
                        break
                    if target == origin and position == offset:
                        continue
                    inserted = base[:position] + [customer] + base[position:]
                    new_stats = _route_values(problem, inserted, target)
                    old_pair = stats[origin]
                    new_pair = new_stats
                    if target != origin:
                        old_pair = tuple(stats[origin][k] + stats[target][k] for k in (0, 1))
                        new_pair = tuple(reduced_stats[k] + new_stats[k] for k in (0, 1))
                    evaluated += 1
                    better = (new_pair[0] < old_pair[0] - _TOL or
                              (abs(new_pair[0] - old_pair[0]) <= _TOL
                               and new_pair[1] < old_pair[1] - _TOL
                               and (old_pair[0] > _TOL or new_pair[0] <= _TOL)))
                    if better:
                        candidate = [list(existing) for existing in current]
                        candidate[origin] = reduced
                        candidate[target] = inserted
                        candidate_score = problem.score(candidate)
                        if (candidate_score >= incumbent_score or
                                (incumbent_score[0] <= _TOL and candidate_score[0] > _TOL)):
                            continue
                        incumbent_score = candidate_score
                        chosen = (origin, target, customer, reduced, reduced_stats, inserted, new_stats)
                        break
                if chosen is not None or termination == 'time_limit':
                    break
            if chosen is not None or termination == 'time_limit':
                break
        if chosen is None:
            if termination != 'time_limit':
                termination = 'relocate_local_optimum'
            break
        origin, target, customer, reduced, reduced_stats, inserted, new_stats = chosen
        if origin != target:
            current[origin], stats[origin] = reduced, reduced_stats
            loads[origin] -= problem.demands[customer]
            loads[target] += problem.demands[customer]
        current[target], stats[target] = inserted, new_stats
        accepted += 1
    return current, {'evaluated_neighbors': evaluated, 'accepted_moves': accepted,
                     'termination': termination, 'operator': 'first_improving_relocate',
                     'objective': 'lexicographic_violation_then_conservative_travel'}


def _finish(problem, routes, started, cfg, seed, method, termination, history=(), **extra):
    score = None if routes is None else problem.score(routes)
    status = 'no_solution' if routes is None else ('feasible' if score[0] <= _TOL else 'conservative_infeasible')
    elapsed = time.perf_counter() - started
    result = problem.result(routes, status, elapsed, list(history), method=method, seed=seed,
                            effective_config=cfg, termination=termination,
                            implementation='independent_conventional_baselines_v2',
                            construction='from_scratch_without_certificates',
                            budget_includes_construction=True, budget_includes_repair=True,
                            surrogate_optimal=False, stochastic_optimal=False,
                            reference_is_bound=False, **extra)
    elapsed = time.perf_counter() - started
    result['solve_seconds'] = elapsed
    result['metadata']['budget_overrun_seconds'] = max(0., elapsed - cfg['seconds'])
    return result


def solve_edd(problem, seed, config=None):
    """Fast deterministic EDD insertion; ends after construction, without LS."""
    started = time.perf_counter()
    cfg, seed = _config(EDD_DEFAULTS, config), _seed(seed)
    routes, reason = construct_insertion(problem, np.random.default_rng(seed), started + cfg['seconds'])
    return _finish(problem, routes, started, cfg, seed, 'edd', reason,
                   seed_affects_search=False, construction_rule='EDD_feasible_cheapest_insertion',
                   budget_policy='upper_limit_fast_control_not_required_to_exhaust')


def _ortools():
    try:
        from ortools.constraint_solver import pywrapcp, routing_enums_pb2
    except ImportError as exc:
        raise RuntimeError('La referencia OR-Tools requiere instalar ortools en Colab.') from exc
    return pywrapcp, routing_enums_pb2


def _integer(value, scale):
    raw = float(value) * scale
    if not math.isfinite(raw) or raw < 0 or raw > _MAX_INT:
        raise ValueError('Recurso fuera del rango entero seguro.')
    rounded = round(raw)
    if not math.isclose(raw, rounded, rel_tol=0., abs_tol=1e-6):
        raise ValueError('El recurso no pertenece a la rejilla declarada; no se redondeará para favorecer al método.')
    return int(rounded)


def solve_ortools_routing(problem, seed, config=None):
    """Fixed-fleet Routing+GLS heuristic reference; no dropped customers."""
    started = time.perf_counter()
    cfg, seed = _config(ORTOOLS_DEFAULTS, config), _seed(seed)
    deadline = started + cfg['seconds']
    if time.perf_counter() >= deadline:
        return _finish(problem, None, started, cfg, seed, 'ortools_routing', 'time_limit')
    pywrapcp, routing_enums = _ortools()
    # The reference pays for the identical conventional feasible seed; its
    # absence never substitutes a certificate or changes the fixed fleet.
    routes, construction_reason = construct_insertion(problem, np.random.default_rng(seed), deadline)
    feasible_seed = routes is not None and problem.score(routes)[0] <= _TOL
    seed_score = None if routes is None else list(problem.score(routes))
    # An infeasible EDD diagnostic is not a Routing solution. Retain its score
    # for auditing, but never return its routes under the OR-Tools method name.
    routes = routes if feasible_seed else None
    origin = 'edd_paid_fallback' if feasible_seed else 'none'

    def finish(termination, **extra):
        return _finish(problem, routes, started, cfg, seed, 'ortools_routing', termination,
                       construction_reason=construction_reason,
                       edd_diagnostic_score=seed_score,
                       conventional_seed_used=bool(feasible_seed),
                       solution_origin=origin,
                       returned_paid_edd_fallback=(origin == 'edd_paid_fallback'),
                       seed_affects_search=False, **extra)

    if time.perf_counter() >= deadline:
        return finish('time_limit')
    scale = problem.time_scale
    if isinstance(scale, bool) or not isinstance(scale, Integral) or scale <= 0:
        raise ValueError('time_scale debe ser entero positivo.')
    matrix = [[_integer(value, scale) for value in row] for row in problem.travel]
    horizon = math.floor(float(problem.horizon) * scale)
    demands = [_integer(value, cfg['capacity_scale']) for value in problem.demands]
    capacities = [_integer(value, cfg['capacity_scale']) for value in problem.capacities]
    windows = [(math.ceil(float(problem.open[j]) * scale), min(horizon, math.floor(float(problem.close[j]) * scale)))
               for j in range(problem.n + 1)]
    if problem.vehicles < 1 or any(lo > hi for lo, hi in windows[1:]):
        return finish('incompatible_integer_model')
    manager = pywrapcp.RoutingIndexManager(problem.n + 1, problem.vehicles, 0)
    routing = pywrapcp.RoutingModel(manager)
    transit = routing.RegisterTransitCallback(lambda a, b: matrix[manager.IndexToNode(a)][manager.IndexToNode(b)])
    routing.SetArcCostEvaluatorOfAllVehicles(transit)
    routing.AddDimension(transit, horizon, horizon, True, 'Time')
    times = routing.GetDimensionOrDie('Time')
    for customer in range(1, problem.n + 1):
        times.CumulVar(manager.NodeToIndex(customer)).SetRange(*windows[customer])
    for vehicle in range(problem.vehicles):
        times.CumulVar(routing.Start(vehicle)).SetRange(0, 0)
        times.CumulVar(routing.End(vehicle)).SetRange(0, horizon)
    demand = routing.RegisterUnaryTransitCallback(lambda a: demands[manager.IndexToNode(a)])
    routing.AddDimensionWithVehicleCapacity(demand, 0, capacities, True, 'Capacity')
    params = pywrapcp.DefaultRoutingSearchParameters()
    params.first_solution_strategy = routing_enums.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
    params.local_search_metaheuristic = routing_enums.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    params.log_search = False
    # Import/build/seed/extraction are all part of the same end-to-end budget.
    remaining = deadline - time.perf_counter()
    if remaining <= 0:
        return finish('time_limit')
    params.time_limit.seconds = int(remaining)
    params.time_limit.nanos = int((remaining - int(remaining)) * 1e9)
    routing.CloseModelWithParameters(params)
    assignment = None
    if feasible_seed:
        assignment = routing.ReadAssignmentFromRoutes(routes, True)
    remaining = deadline - time.perf_counter()
    if remaining <= 0:
        return finish('time_limit')
    params.time_limit.seconds = int(remaining)
    params.time_limit.nanos = int((remaining - int(remaining)) * 1e9)
    assignment = (routing.SolveFromAssignmentWithParameters(assignment, params) if assignment is not None
                  else routing.SolveWithParameters(params))
    if assignment is not None:
        candidate = []
        for vehicle in range(problem.vehicles):
            index, route = routing.Start(vehicle), []
            while not routing.IsEnd(index):
                customer = manager.IndexToNode(index)
                if customer:
                    route.append(customer)
                index = assignment.Value(routing.NextVar(index))
            candidate.append(route)
        candidate_score = problem.score(candidate)
        if candidate_score[0] > _TOL:
            raise AssertionError('OR-Tools no satisface el contrato conservador compartido.')
        if routes is None or candidate_score <= problem.score(routes):
            routes = candidate
            origin = 'ortools_routing_assignment'
    termination = ('time_limit' if time.perf_counter() >= deadline else
                   ('routing_no_assignment' if assignment is None else 'routing_search_finished'))
    return finish(termination,
                   solver_status=int(routing.status()), ortools_version=importlib.metadata.version('ortools'),
                   routing_search='parallel_cheapest_insertion_plus_guided_local_search',
                   routing_assignment_found=assignment is not None,
                   capacity_scale=cfg['capacity_scale'], time_scale=int(scale),
                   bound_scope='no_optimality_bound_from_routing_search')


Overwriting svrplab/baselines.py


In [43]:
%%writefile svrplab/checkpoint_diagnostics.py
"""Cruce 2×2 de checkpoints históricos 03/04; sólo inferencia en Google Colab.

No reentrena, selecciona épocas, cambia corpus ni incorpora sus filas a 06.
Separa señal de entrenamiento de procedimiento de inferencia; uniforme es un
control adicional. Los tiempos miden presupuestos máximos, no trabajo idéntico.
"""
from dataclasses import asdict
from pathlib import Path
import math
import time

import torch

from . import pilot as p, experiments as e
from .attention import AttentionPolicy, UniformMaskedPolicy, solve_attention
from .pomo import solve_pomo
from .search import SearchProblem, OBJECTIVE
from .supervised import choose_device, synchronize, _runtime_signature
from .supervised_data import validate_fold


def _inside(root, relative):
    root, relative = Path(root), Path(relative)
    if relative.is_absolute() or '..' in relative.parts or not relative.parts:
        raise ValueError('Ruta relativa inválida en entrenamiento elegido.')
    path = (root / relative).resolve()
    if not path.is_relative_to(root.resolve()):
        raise ValueError('Ruta fuera del entrenamiento elegido.')
    return path


def _models(root, instances, spec, bank_sha, seeds, time_scale, signal):
    root = Path(root)
    meta_sha, index_sha = p.verify_bundle(root / 'meta'), p.verify_bundle(root / 'index')
    identity = e.read_json(root / 'meta/identity.json')
    prefix = {'supervised': 'attention03_', 'reinforcement': 'pomo04_'}[signal]
    if (identity['bank_sha256'] != bank_sha or root.name != prefix + p.digest(identity)[:20]
            or identity['evaluator_sha256'] != p.file_hash(p.__file__)
            or identity['seeds'] != seeds):
        raise ValueError('Identidad, banco, evaluador o semillas del entrenamiento incompatibles.')
    training_config = identity['config']
    recorded_scale = (training_config['teacher']['time_scale'] if signal == 'supervised'
                      else training_config['time_scale'])
    if recorded_scale != time_scale or (signal == 'reinforcement' and training_config['objective'] != OBJECTIVE):
        raise ValueError('El cruce debe conservar la escala y el objetivo conservador de entrenamiento.')
    if e.read_json(root / 'meta/folds.json') != identity['folds']:
        raise ValueError('Folds distintos de la identidad de entrenamiento.')
    for name, digest in identity['source_hashes'].items():
        if p.file_hash(_inside(root / 'meta/code', name)) != digest:
            raise ValueError('Fuente histórica distinta del hash declarado en entrenamiento.')
    for name in ['attention.py', 'search.py', *(['pomo.py'] if signal == 'reinforcement' else [])]:
        saved = root / 'meta/code' / name
        if not saved.is_file() or p.file_hash(saved) != p.file_hash(Path(__file__).with_name(name)):
            raise ValueError(f'Cambió {name}; el cruce histórico exige la implementación de inferencia original.')
    result = {}
    for record in e.read_json(root / 'index/models.json'):
        fold = record['fold']
        validate_fold(fold, instances)
        if (record['traffic'] != asdict(spec) or fold not in identity['folds']
                or record['model_config'] != training_config['model']
                or record['heldout_used_for_training_or_selection'] is not False):
            raise ValueError('Tráfico, arquitectura o fold del checkpoint incompatible con su identidad.')
        final = _inside(root, record['relative_directory'])
        if (p.verify_bundle(final) != record['bundle_sha256']
                or p.file_hash(final / 'best.pt') != record['model_checkpoint_sha256']):
            raise ValueError('Checkpoint histórico alterado.')
        p.verify_bundle(final.parent / 'meta')
        fold_identity = e.read_json(final.parent / 'meta/identity.json')
        if (p.digest(fold_identity) != record['identity_sha256'] or fold_identity['fold'] != fold
                or fold_identity['seed'] != record['training_seed'] or fold_identity['config'] != training_config):
            raise ValueError('Los pesos no corresponden a la identidad sellada del fold.')
        if e.read_json(final / 'summary.json') != {k: v for k, v in record.items()
                                                  if k not in ('relative_directory', 'bundle_sha256')}:
            raise ValueError('Resumen e índice de checkpoint discrepan.')
        for instance in instances:
            if instance.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups']:
                key = (instance.sha, record['training_seed'])
                if key in result:
                    raise ValueError('Dos modelos para la misma geometría y semilla.')
                result[key] = (final, record)
    expected = {(x.sha, seed) for x in instances for seed in seeds}
    if set(result) != expected:
        raise ValueError('Inventario histórico distinto del plan prefijado; no omitir modelos fallidos.')
    return result, {'training_id': root.name, 'meta_sha256': meta_sha, 'index_sha256': index_sha}


def run_checkpoint_crossover(bank, roots, config, work_root, drive_root, device='cuda'):
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Inferencia diagnóstica exclusivamente en Google Colab.') from exc
    if set(roots) != {'supervised', 'reinforcement'} or config['role'] != 'development_diagnostic_only':
        raise ValueError('Se requieren ambos entrenamientos históricos y rol diagnóstico.')
    if config['analysis_mode'] != 'descriptive_only':
        raise ValueError('El cruce no habilita pruebas confirmatorias.')
    if (not config['seeds'] or len(set(config['seeds'])) != len(config['seeds'])
            or any(type(seed) is not int or seed < 0 for seed in config['seeds'])
            or type(config['time_scale']) is not int or config['time_scale'] <= 0
            or isinstance(config['seconds_per_attempt'], bool)
            or not math.isfinite(config['seconds_per_attempt']) or config['seconds_per_attempt'] <= 0
            or set(config['decoders']) != {'am', 'pomo'}
            or any(Path(roots[signal]).name != config['training_ids'][signal] for signal in roots)):
        raise ValueError('Semillas, presupuesto, decoders o entrenamientos distintos del plan diagnóstico.')
    bank, device = Path(bank), choose_device(device)
    if device.type != 'cuda':
        raise RuntimeError('El cruce requiere GPU en Colab para mantener el recurso declarado.')
    bank_sha = p.verify_bundle(bank)
    instances = p.load_bank(bank)
    bank_config = e.read_json(bank / 'config.json')
    spec = p.Traffic(**bank_config['traffic'])
    models, model_identities = {}, {}
    for signal in ('supervised', 'reinforcement'):
        models[signal], model_identities[signal] = _models(roots[signal], instances, spec, bank_sha,
            config['seeds'], config['time_scale'], signal)
    for key in models['supervised']:
        a, b = models['supervised'][key][1], models['reinforcement'][key][1]
        # Historical 03 used dropout=.1 and 04 dropout=0. This changes training,
        # not eval-mode topology: insisting on complete config equality blocks
        # the actual requested historical crossover. All other fields must match.
        inference_a = {k: v for k, v in a['model_config'].items() if k != 'dropout'}
        inference_b = {k: v for k, v in b['model_config'].items() if k != 'dropout'}
        if a['fold'] != b['fold'] or inference_a != inference_b:
            raise ValueError('El cruce exige arquitectura de inferencia y folds iguales; sólo admite dropout distinto en entrenamiento.')
    identity = {'schema_version': 'checkpoint-crossover-v1', 'bank_sha256': bank_sha,
                'models': model_identities, 'config': config,
                'sources': {name: p.file_hash(Path(__file__).with_name(name)) for name in
                            ('checkpoint_diagnostics.py', 'attention.py', 'pomo.py', 'search.py',
                             'supervised.py', 'supervised_data.py', 'experiments.py')},
                'evaluator_sha256': p.file_hash(p.__file__), 'runtime_signature': _runtime_signature(device),
                'dropout_scope': 'historical_training_difference_disabled_by_model_eval',
                'no_retraining_or_checkpoint_selection': True,
                'independent_unit': 'geometry_group_sha256', 'native_decoders_common_cap_not_equal_work': True}
    identifier = 'crossover03_' + p.digest(identity)[:20]
    root, remote = [Path(base) / 'checkpoint_diagnostics' / bank.name / identifier
                    for base in (work_root, drive_root)]
    print('DIAGNOSTIC_ID =', identifier, flush=True)
    if not e._recover(root / 'meta', remote / 'meta'):
        e.write_json(root / 'meta/identity.json', identity)
        e.write_json(root / 'meta/environment.json', p.environment_report())
        e._commit(root / 'meta', remote / 'meta')
    if e.read_json(root / 'meta/identity.json') != identity:
        raise ValueError('Identidad de reanudación incompatible.')
    rows, bundles = [], {}
    for x in sorted(instances, key=lambda item: item.n):
        for seed in config['seeds']:
            for signal in ('supervised', 'reinforcement', 'uniform'):
                final, record = models['supervised' if signal == 'uniform' else signal][(x.sha, seed)]
                for decoder, solve in (('am', solve_attention), ('pomo', solve_pomo)):
                    relative = f'{signal}/{decoder}/{x.sha}/seed_{seed}'
                    directory = root / relative
                    if not e._recover(directory, remote / relative):
                        started = time.perf_counter()
                        model = None
                        try:
                            problem = SearchProblem(x, spec, config['time_scale'])
                            if signal == 'uniform':
                                model = UniformMaskedPolicy(record['model_config']).to(device)
                            else:
                                # The preceding bundle + file checks are repeated before deserialization.
                                if p.file_hash(final / 'best.pt') != record['model_checkpoint_sha256']:
                                    raise ValueError('Checkpoint cambió antes de cargarlo.')
                                model = AttentionPolicy(record['model_config']).to(device)
                                model.load_state_dict(torch.load(final / 'best.pt', map_location='cpu', weights_only=True))
                            model.eval()
                            synchronize(device)
                            remaining = max(0., config['seconds_per_attempt'] - (time.perf_counter() - started))
                            settings = dict(config['decoders'][decoder], seconds=remaining)
                            solution = solve(problem, seed, settings, model, device) if remaining else problem.result(None, 'time_limit_model_load', time.perf_counter() - started, [])
                            synchronize(device)
                            score = None if solution['routes'] is None else problem.score(solution['routes'])
                            if solution['routes'] is not None:
                                p.validate_routes(x, solution['routes'])
                            row = {'signal': signal, 'decoder': decoder, 'solver_seed': seed,
                                   'instance_sha256': x.sha, 'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                                   'n_customers': x.n, 'fold_id': record['fold']['fold_id'],
                                   'checkpoint_sha256': None if signal == 'uniform' else record['model_checkpoint_sha256'],
                                   'training_dropout': None if signal == 'uniform' else record['model_config']['dropout'],
                                   'dropout_active_at_inference': model.training,
                                   'objective_scope': OBJECTIVE, 'time_scale': config['time_scale'],
                                   'status': solution['status'], 'complete_feasible': score is not None and score[0] <= 1e-9,
                                   'conservative_travel_minutes': None if score is None else score[1],
                                   'search_violation_minutes': None if score is None else score[0],
                                   'seconds': time.perf_counter() - started,
                                   'cap_seconds': config['seconds_per_attempt'],
                                   'solution_origin': solution.get('metadata', {}).get('solution_origin'),
                                   'role': config['role']}
                        finally:
                            del model
                            torch.cuda.empty_cache()
                        # Unexpected technical failures stop; they never become scientific zeros.
                        e.write_json(directory / 'row.json', row)
                        e.write_json(directory / 'solution.json', solution)
                        e._commit(directory, remote / relative)
                    recorded = e.read_json(directory / 'row.json')
                    expected_row = {'signal': signal, 'decoder': decoder, 'solver_seed': seed,
                        'instance_sha256': x.sha, 'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                        'n_customers': x.n, 'fold_id': record['fold']['fold_id'],
                        'objective_scope': OBJECTIVE, 'time_scale': config['time_scale'],
                        'cap_seconds': config['seconds_per_attempt'], 'role': config['role']}
                    if any(recorded.get(key) != value for key, value in expected_row.items()):
                        raise ValueError('Resultado diagnóstico recuperado de otra identidad o decoder.')
                    rows.append(recorded)
                    bundles[relative] = p.verify_bundle(directory)
                    print(x.n, seed, signal, decoder, rows[-1]['status'], flush=True)
    expected = len(instances) * len(config['seeds']) * 6
    if len(rows) != expected:
        raise ValueError('Cruce incompleto; no publicar tabla parcial como completo.')
    # Detect source changes during the crossover; no implicit migration to new weights.
    for signal in ('supervised', 'reinforcement'):
        _, final_identity = _models(roots[signal], instances, spec, bank_sha,
            config['seeds'], config['time_scale'], signal)
        if final_identity != model_identities[signal]:
            raise ValueError('El entrenamiento fuente cambió durante el cruce.')
    if p.verify_bundle(bank) != bank_sha:
        raise ValueError('Banco alterado durante el cruce.')
    if not e._recover(root / 'report', remote / 'report'):
        e.write_csv(root / 'report/crossover.csv', rows)
        e.write_json(root / 'report/bundles.json', bundles)
        e.write_json(root / 'report/protocol.json', identity)
        e.write_json(root / 'report/status.json', {'status': 'complete_diagnostic', 'attempts': expected,
                     'confirmatory': False, 'retrained': False, 'new_independent_geometries': 0})
        e._commit(root / 'report', remote / 'report')
    if (e.read_json(root / 'report/bundles.json') != bundles
            or e.read_json(root / 'report/protocol.json') != identity):
        raise ValueError('El reporte recuperado no corresponde a los intentos completos actuales.')
    return root


Overwriting svrplab/checkpoint_diagnostics.py


In [44]:
%%writefile svrplab/comparison.py
"""06: lectura verificable de resultados y comparación descriptiva, sólo en Colab.

No resuelve, entrena, reevalúa escenarios ni accede a certificados. Reconciliar
los CSV es aritmética sobre mediciones ya exportadas por el evaluador sellado.
"""
from __future__ import annotations

import csv
import json
import math
import shutil
from pathlib import Path

from . import pilot as p
from . import experiments as e
from . import comparison_stats as stats

SCHEMA = 'comparison-06-v2'
LEARNED = {'am_supervised': 'supervised_training', 'pomo': 'rl_training',
           'hbg_facs': 'hbg_training', 'hbg_no_pheromone': 'hbg_training'}
FLOAT_FIELDS = ('travel_minutes', 'base_minutes', 'net_execution_delta_minutes',
                'waiting_minutes', 'lateness_minutes', 'depot_overtime_minutes',
                'customer_violation_fraction')
INT_FIELDS = ('late_customers', 'used_vehicles', 'recourse_reorders')
BOOL_FIELDS = ('strict_feasible', 'all_customers_served')


def _require_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Análisis y pruebas exclusivamente en Google Colab.') from exc


def validate_config(config):
    # Una sola definición del catálogo/métricas evita aceptar opciones que las
    # tablas rechazarían después de leer todas las exportaciones.
    stats._config(config)
    if (config.get('schema_version') != SCHEMA or config.get('role') != 'development_only'
            or config.get('analysis_mode') != 'descriptive_only'):
        raise ValueError('06 sólo implementa comparación descriptiva del piloto de desarrollo.')
    methods = config.get('expected_methods', [])
    allowed = set(stats.METHODS)
    if (not methods or len(set(methods)) != len(methods) or not set(methods) <= allowed
            or config.get('reference_method') not in methods):
        raise ValueError('Métodos o referencia inválidos.')
    metrics = config.get('metrics', [])
    if (not metrics or len(set(metrics)) != len(metrics)
            or not set(metrics) <= set(e.SUMMARY_METRICS + ['search_total_seconds'])):
        raise ValueError('Métricas inválidas.')
    e.validate_config(config['method_contract'])
    ref = config['accepted_bank']
    if (ref['role'] != 'development_only'
            or ref['acceptance_status'] != 'ready_for_method_development'
            or len(ref['manifest_sha256']) != 64):
        raise ValueError('Referencia de banco no aceptada para desarrollo.')


def _safe_relative(root, relative):
    relative = Path(relative)
    root = Path(root).resolve()
    if relative.is_absolute() or '..' in relative.parts or not relative.parts:
        raise ValueError('Ruta relativa inválida en inventario.')
    path = root / relative
    if not path.resolve().is_relative_to(root):
        raise ValueError('Ruta fuera del directorio sellado.')
    return path


def _verify(directory):
    directory = Path(directory)
    manifest = e.read_json(directory / 'CHECKSUMS.json')
    if not isinstance(manifest, dict) or not manifest:
        raise ValueError('Manifiesto vacío o inválido.')
    for relative in manifest:
        _safe_relative(directory, relative)
    return p.verify_bundle(directory)


def _bank_context(bank, config):
    validate_config(config)
    bank = Path(bank)
    ref = config['accepted_bank']
    digest = _verify(bank)
    if bank.name != ref['bank_id'] or digest != ref['manifest_sha256']:
        raise ValueError('Banco distinto de la referencia aceptada en Colab.')
    bc = e.read_json(bank / 'config.json')
    acceptance = e.read_json(bank / 'acceptance.json')
    if (bc['protocol_version'] != ref['protocol_version']
            or acceptance.get('status') != ref['acceptance_status']):
        raise ValueError('Acta o versión del banco incompatibles.')
    evaluator = p.file_hash(bank / 'code/svrplab_pilot.py')
    if evaluator != p.file_hash(p.__file__):
        raise ValueError('06 debe importar el evaluador guardado en el banco.')
    # Sólo deserializar instancias: load_bank ejecutaría auditoría de certificados.
    instances = [p.freeze_instance(e.read_json(path))
                 for path in sorted((bank / 'instances').glob('*.json'))]
    if sorted(x.n for x in instances) != sorted(bc['sizes']):
        raise ValueError('Registro de tamaños incompleto.')
    registry = [{'instance_id': x.instance_id, 'instance_sha256': x.sha,
                 'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                 'n_customers': x.n} for x in instances]
    if (acceptance.get('instance_hashes') != {str(x.n): x.sha for x in instances}
            or acceptance.get('geometry_groups') != {str(x.n): x.provenance['geometry_group_sha256'] for x in instances}
            or acceptance.get('ready_for_confirmatory_comparison') is not False):
        raise ValueError('Registro de instancias distinto del acta sellada.')
    scenarios = {}
    for x in instances:
        split = e.read_json(bank / f'scenarios/n{x.n:04d}.json')
        e.validate_scenarios(split, bc)
        scenarios[x.sha] = split['evaluation']
    protocol = e.comparison_protocol(digest, evaluator, bc, config['method_contract'])
    return {'bank_id': bank.name, 'bank_hash': digest, 'bank_config': bc,
            'instances': instances, 'registry': registry, 'scenarios': scenarios,
            'protocol': protocol, 'comparison_id': p.digest(protocol),
            'method_contract': config['method_contract']}


def _validate_method_settings(plan, expected):
    for method in plan['methods']:
        if method not in expected or plan['config']['methods'].get(method) != expected[method]:
            raise ValueError(f'Protocolo distinto: parámetros no preespecificados para {method}.')


def _controlled_contrast_audit(rows, diagnostics, registry, seeds, method_configs, source_by_method):
    """Ablation claims require identical inference and verified model provenance."""
    index = {(r['instance_sha256'], r['method'], r['solver_seed']): r for r in rows}
    meta = {(r['instance_sha256'], r['method'], r['solver_seed']):
            json.loads(r['solver_metadata_json']) for r in diagnostics}
    output = []
    for method, reference in stats.CONTROLLED_PAIRS:
        family = 'pomo' if method == 'pomo' else 'hbg'
        required_sources = ('attention.py', 'pomo.py', 'reinforcement.py') if family == 'pomo' else (
                            'attention.py', 'hbg.py', 'hybrid.py', 'baselines.py')
        for identity in registry:
            reasons = set()
            settings = [dict(method_configs.get(m, {})) for m in (method, reference)]
            if reference == 'hbg_no_pheromone':
                if settings[0].get('alpha') != 1. or settings[1].get('alpha') != 0.:
                    reasons.add('pheromone_alpha_not_predeclared_1_vs_0')
                for value in settings:
                    value.pop('alpha', None)
            if not all(settings) or settings[0] != settings[1]:
                reasons.add('inference_parameters_differ_or_missing')
            environments = [source_by_method.get(m, {}).get('__environment_sha256') for m in (method, reference)]
            if not environments[0] or environments[0] != environments[1]:
                reasons.add('execution_environment_differs_or_missing')
            for source in required_sources:
                versions = [source_by_method.get(m, {}).get(source) for m in (method, reference)]
                if not versions[0] or versions[0] != versions[1]:
                    reasons.add('implementation_sources_differ_or_missing')
            for seed in seeds:
                keys = [(identity['instance_sha256'], m, seed) for m in (method, reference)]
                records = [index.get(k) for k in keys]
                metadata = [meta.get(k) for k in keys]
                if not all(records) or not all(metadata):
                    reasons.add('attempt_or_metadata_missing')
                    continue
                if reference == 'hbg_no_pheromone':
                    for field in ('training_id', 'model_checkpoint_sha256', 'fold_id', 'training_seed',
                                  'offline_manifest_sha256'):
                        if records[0].get(field) is None or records[0].get(field) != records[1].get(field):
                            reasons.add('trained_checkpoint_or_training_identity_differs')
                # Shape, masking, and initialization are recorded by solver wrappers
                # even if no route is produced. Missing proof is not guessed from names.
                for field in ('model_config', 'mask_version', 'inference_device'):
                    values = [m.get(field) for m in metadata]
                    if values[0] is None or values[0] != values[1]:
                        reasons.add(f'{field}_differs_or_missing')
                if reference != 'hbg_no_pheromone':
                    fold = records[0].get('fold_id')
                    expected_seed = p.seed_for('reinforcement-04' if family == 'pomo' else 'hybrid-05', fold, seed) if fold else None
                    values = [m.get('initialization_seed') for m in metadata]
                    if expected_seed is None or values != [expected_seed, expected_seed]:
                        reasons.add('initialization_seed_differs_or_unverified')
                    if records[1].get('training_seconds', 0.) != 0.:
                        reasons.add('untrained_control_has_training_cost')
            output.append({**identity, 'method': method, 'reference_method': reference,
                'status': 'verified' if not reasons else 'unverified',
                'reason': ';'.join(sorted(reasons)) if reasons else 'same_inference_sources_and_required_model_provenance',
                'expected_solver_seeds': len(seeds),
                'attribution_scope': 'pheromone_with_same_weights' if reference == 'hbg_no_pheromone' else
                                     'learning_or_policy_control_with_same_inference_and_initialization',
            })
    return output


def _inspect_run(directory, bank_context):
    directory = Path(directory)
    meta_hash = _verify(directory / 'meta')
    report_hash = _verify(directory / 'report')
    plan = e.read_json(directory / 'meta/plan.json')
    if (plan['run_id'] != directory.name or plan['bank_id'] != bank_context['bank_id']
            or plan['role'] != 'development_only'):
        raise ValueError('Identidad del run inválida.')
    e.validate_config(plan['config'])
    expected = e.comparison_protocol(bank_context['bank_hash'],
        bank_context['protocol']['evaluator_sha256'], bank_context['bank_config'], plan['config'])
    if (plan['protocol'] != expected or plan['protocol'] != bank_context['protocol']
            or plan['comparison_id'] != p.digest(expected)):
        raise ValueError('Protocolo distinto: banco/evaluador/runner/objetivo/presupuesto/CRN.')
    _validate_method_settings(plan, bank_context['method_contract']['methods'])
    methods = plan['methods']
    if (not methods or len(set(methods)) != len(methods)
            or not set(methods) <= set(plan['config']['methods'])
            or set(plan['method_availability']) != set(methods)
            or not set(plan['method_availability'].values()) <= {'enabled', 'license_pending'}):
        raise ValueError('Inventario de métodos inválido.')
    signature = p.digest({key: plan[key] for key in
        ['protocol', 'config', 'methods', 'source_hashes', 'method_availability', 'method_provenance']})
    if plan['signature'] != signature:
        raise ValueError('Firma de ejecución inconsistente.')
    for name, digest in plan['source_hashes'].items():
        if p.file_hash(_safe_relative(directory / 'meta/code', name)) != digest:
            raise ValueError('Fuente guardada distinta del manifiesto.')
    for name, key in [('experiments.py', 'runner_sha256'), ('search.py', 'search_contract_sha256'),
                      ('pilot.py', 'evaluator_sha256')]:
        if plan['source_hashes'].get(name) != plan['protocol'][key]:
            raise ValueError('Fuentes del run no corresponden al protocolo.')
    validation = e.read_json(directory / 'report/validation.json')
    if (validation.get('status') != 'completed' or validation.get('bank_unchanged') is not True
            or validation.get('ready_for_confirmatory_comparison') is not False):
        raise ValueError('Ejecución sin validación completa de desarrollo.')
    bundles = e.read_json(directory / 'report/bundles.json')
    if bundles.get('meta') != meta_hash:
        raise ValueError('Índice de bundles no corresponde al meta.')
    return {'plan': plan, 'bundles': bundles, 'validation': validation,
            'report_sha256': report_hash, 'meta_sha256': meta_hash}


def discover_runs(bank, method_runs_root, config):
    """Inventario ligero: la carga final verifica además todos los bundles/CSV."""
    _require_colab()
    bc = _bank_context(bank, config)
    inventory = []
    root = Path(method_runs_root)
    for directory in sorted(root.iterdir()) if root.exists() else []:
        if not directory.is_dir():
            continue
        item = {'run_id': directory.name, 'methods': [], 'comparison_id': None,
                'status': 'incomplete', 'reason': 'Falta meta o reporte sellado.'}
        try:
            if not all((directory / part / 'CHECKSUMS.json').is_file() for part in ['meta', 'report']):
                inventory.append(item)
                continue
            plan = e.read_json(directory / 'meta/plan.json')
            item.update(methods=plan.get('methods', []), comparison_id=plan.get('comparison_id'))
            inspected = _inspect_run(directory, bc)
            item.update(status='compatible', reason='Meta/reporte verificados; bundles se verifican al cargar.',
                        report_sha256=inspected['report_sha256'])
        except (OSError, ValueError, KeyError, TypeError) as exc:
            item.update(status='incompatible' if 'Protocolo distinto' in str(exc) else 'invalid',
                        reason=str(exc))
        inventory.append(item)
    return inventory


def select_runs(inventory, config, selection=None):
    """No selecciona por fecha, rendimiento o éxito; exige resolver duplicados."""
    validate_config(config)
    selection = {} if selection is None else selection
    if not isinstance(selection, dict) or not set(selection) <= set(config['expected_methods']):
        raise ValueError('RUN_SELECTION debe mapear métodos esperados a run_id.')
    result, ids = {}, set()
    for method in config['expected_methods']:
        options = [r for r in inventory if r['status'] == 'compatible' and method in r['methods']]
        if method in selection:
            options = [r for r in options if r['run_id'] == selection[method]]
            if len(options) != 1:
                raise ValueError(f'Run seleccionado no compatible para {method}.')
        elif len(options) > 1:
            raise ValueError(f'Varios runs para {method}; fije RUN_SELECTION explícitamente: '
                             + ', '.join(r['run_id'] for r in options))
        if options:
            result[method] = options[0]['run_id']
            ids.add(options[0]['comparison_id'])
    if not result:
        raise ValueError('No hay runs compatibles. Ejecute 01–05 en Colab y revise el inventario.')
    if len(ids) != 1 or None in ids:
        raise ValueError('No se pueden mezclar comparison_id diferentes.')
    return result


def _finite(value, key, nonnegative=False):
    if isinstance(value, bool):
        raise ValueError(f'Número booleano en {key}.')
    try:
        number = float(value)
    except (TypeError, ValueError) as exc:
        raise ValueError(f'Número inválido en {key}.') from exc
    if not math.isfinite(number) or (nonnegative and number < 0):
        raise ValueError(f'Número no finito/negativo en {key}.')
    return number


def _read_scenarios(path, context, expected_seeds):
    records, observed = [], []
    required = {'scenario_seed', *FLOAT_FIELDS, *INT_FIELDS, *BOOL_FIELDS, *context}
    with Path(path).open(newline='', encoding='utf-8') as stream:
        reader = csv.DictReader(stream)
        if (not reader.fieldnames or len(set(reader.fieldnames)) != len(reader.fieldnames)
                or not required <= set(reader.fieldnames)):
            raise ValueError('Columnas de escenarios incompletas/duplicadas.')
        for row in reader:
            if None in row or any(value is None for value in row.values()):
                raise ValueError('Fila CSV malformada.')
            for key, value in context.items():
                if row[key] != ('' if value is None else str(value)):
                    raise ValueError(f'Escenario con identidad distinta: {key}.')
            seed = row['scenario_seed']
            if not seed.isascii() or not seed.isdecimal() or str(int(seed)) != seed:
                raise ValueError('Semilla no es decimal entero exacto.')
            observed.append(int(seed))
            for key in FLOAT_FIELDS:
                row[key] = _finite(row[key], key, key != 'net_execution_delta_minutes')
            for key in INT_FIELDS:
                if not row[key].isascii() or not row[key].isdecimal():
                    raise ValueError(f'Conteo inválido en {key}.')
                row[key] = int(row[key])
            for key in BOOL_FIELDS:
                if row[key] not in ('True', 'False'):
                    raise ValueError(f'Booleano inválido en {key}.')
                row[key] = row[key] == 'True'
            if row['customer_violation_fraction'] > 1 or not row['all_customers_served']:
                raise ValueError('Cobertura o fracción de violación inválida.')
            if (not math.isclose(row['net_execution_delta_minutes'],
                                  row['travel_minutes'] - row['base_minutes'], rel_tol=1e-9, abs_tol=1e-7)
                    or row['strict_feasible'] != (row['late_customers'] == 0 and row['depot_overtime_minutes'] <= 1e-9)):
                raise ValueError('Métricas de escenario internamente inconsistentes.')
            if 'n_customers' in context and (row['late_customers'] > context['n_customers']
                    or not math.isclose(row['customer_violation_fraction'],
                                        row['late_customers'] / context['n_customers'], abs_tol=1e-12)):
                raise ValueError('Fracción de clientes tardíos inconsistente.')
            # Restaurar tipos de identidad (solver_seed, n_customers, etc.).
            row.update(context)
            records.append(row)
    if observed != list(expected_seeds) or len(set(observed)) != len(observed):
        raise ValueError('Escenarios faltantes, duplicados, ajenos o reordenados.')
    return records


def _check_aggregates(rows, summary, alpha):
    if not rows or not 0 <= alpha < 1:
        raise ValueError('Muestra/alpha inválidos.')
    costs = [r['travel_minutes'] for r in rows]
    mean = math.fsum(costs) / len(costs)
    variance = math.fsum((v - mean) ** 2 for v in costs) / len(costs)
    descending = sorted(costs, reverse=True)
    mass = len(costs) * (1 - alpha)
    whole = min(len(costs), math.floor(mass))
    tail = math.fsum(descending[:whole])
    if whole < len(costs):
        tail += (mass - whole) * descending[whole]
    calculated = {'scenarios': len(rows), 'alpha': alpha, 'mean_travel_minutes': mean,
                  'cvar_travel_minutes': tail / mass, 'cost_variance_population': variance,
                  'std_travel_minutes': math.sqrt(variance),
                  'scenario_strict_feasibility': sum(r['strict_feasible'] for r in rows) / len(rows),
                  'used_vehicles': rows[0]['used_vehicles']}
    for raw in ['customer_violation_fraction', 'lateness_minutes', 'depot_overtime_minutes',
                'waiting_minutes', 'base_minutes', 'net_execution_delta_minutes',
                'late_customers', 'recourse_reorders']:
        calculated['mean_' + raw] = math.fsum(r[raw] for r in rows) / len(rows)
    if len({r['used_vehicles'] for r in rows}) != 1:
        raise ValueError('Cambió el número de vehículos entre escenarios.')
    for key, value in calculated.items():
        if not math.isclose(_finite(summary.get(key), key), value, rel_tol=1e-9, abs_tol=1e-7):
            raise ValueError(f'Resumen no reconcilia escenarios: {key}.')
    return calculated


def _load_run(directory, bc):
    inspected = _inspect_run(directory, bc)
    plan, bundles = inspected['plan'], inspected['bundles']
    for relative, digest in bundles.items():
        if _verify(_safe_relative(directory, relative)) != digest:
            raise ValueError(f'Bundle modificado: {relative}.')
    rows = e.read_json(Path(directory) / 'report/summary.json')
    if not isinstance(rows, list):
        raise ValueError('Resumen debe ser una lista de intentos.')
    indexed = {}
    for row in rows:
        key = (row['instance_sha256'], row['method'], row['solver_seed'], row['policy'])
        if key in indexed:
            raise ValueError('Intento/política duplicado en reporte.')
        indexed[key] = row
    expected_keys, expected_bundles, scenario_rows = set(), {'meta'}, []
    for x in bc['instances']:
        for method in plan['methods']:
            for seed in plan['config']['seeds']:
                job = f'{method}/n{x.n:04d}/seed_{seed}'
                expected_bundles.add(f'{job}/solution')
                solution_dir = _safe_relative(directory, f'{job}/solution')
                solution = e.read_json(solution_dir / 'solution.json')
                learned = plan['method_provenance'].get(method, {}).get(f'{x.sha}:{seed}', {})
                if method in LEARNED:
                    e.validate_learned_provenance(learned, x, seed)
                context = {'schema_version': e.SCHEMA, 'comparison_id': plan['comparison_id'],
                    'run_id': plan['run_id'], 'bank_id': bc['bank_id'],
                    'bank_manifest_sha256': bc['bank_hash'],
                    'evaluator_sha256': plan['protocol']['evaluator_sha256'], 'method': method,
                    'solver_seed': seed, 'instance_id': x.instance_id, 'instance_sha256': x.sha,
                    'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                    'n_customers': x.n, 'objective_scope': plan['protocol']['objective'],
                    'search_budget_seconds': plan['protocol']['search_seconds'],
                    'scenario_set_sha256': p.digest(bc['scenarios'][x.sha]),
                    **e.provenance_columns(learned)}
                if e.read_json(solution_dir / 'identity.json') != context:
                    raise ValueError('Identidad de solución inconsistente.')
                meta, nominal = solution['metadata'], solution['nominal']
                if plan['method_availability'][method] == 'license_pending' and (
                        solution['status'] != 'license_pending' or solution['routes'] is not None):
                    raise ValueError('Disponibilidad de licencia inconsistente.')
                base = {**context, 'solver_status': solution['status'],
                    'structural_valid': solution['routes'] is not None,
                    'nominal_feasible': nominal['strict_feasible'] if nominal else None,
                    'conservative_feasible': meta.get('conservative_feasible'),
                    'search_violation_minutes': meta.get('search_violation_minutes'),
                    'search_objective_minutes': meta.get('search_objective_minutes'),
                    'solve_seconds': solution['solve_seconds'],
                    'search_total_seconds': solution['search_total_seconds'],
                    'best_bound': meta.get('best_bound'), 'relative_gap': meta.get('relative_gap'),
                    'surrogate_optimal': meta.get('surrogate_optimal'),
                    'model_load_seconds': meta.get('model_load_seconds', 0.0),
                    'tuning_seconds_this_run': 0.0}
                for key in ['solve_seconds', 'search_total_seconds', 'model_load_seconds']:
                    _finite(base[key], key, True)
                for policy in bc['protocol']['policies']:
                    key = (x.sha, method, seed, policy)
                    expected_keys.add(key)
                    if key not in indexed:
                        raise ValueError('Falta una réplica/política esperada.')
                    if solution['routes'] is None:
                        expected_row = {**base, 'policy': policy, 'evaluation_status': 'no_solution',
                                        **dict.fromkeys(e.SUMMARY_METRICS)}
                    else:
                        p.validate_routes(x, solution['routes'])  # sólo cobertura/capacidad, sin simulación
                        expected_bundles.add(f'{job}/{policy}')
                        result_dir = _safe_relative(directory, f'{job}/{policy}')
                        evaluated = e.read_json(result_dir / 'summary.json')
                        if any(evaluated.get(k) != v for k, v in {**context, 'policy': policy}.items()):
                            raise ValueError('Evaluación de otra identidad/política.')
                        qc = e.read_json(result_dir / 'validation.json')
                        if qc.get('status') != 'passed' or any(qc.get(k) is not True for k in
                                ['structural_valid', 'instance_unchanged', 'trace_matches_evaluator',
                                 'recourse_preserves_vehicle_assignment', 'cvar_independently_checked']):
                            raise ValueError('Evaluación sin trazas validadas.')
                        if (qc['scenarios'] != len(bc['scenarios'][x.sha])
                                or qc['event_rows'] != qc['expected_event_rows']
                                or qc['realized_route_rows'] != qc['expected_realized_route_rows']
                                or qc['event_rows'] != len(bc['scenarios'][x.sha]) * (x.n + sum(bool(r) for r in solution['routes']))
                                or qc['realized_route_rows'] != len(bc['scenarios'][x.sha]) * (x.n + 2 * len(x.capacities))
                                or qc['nominal_feasible'] != nominal['strict_feasible']):
                            raise ValueError('Conteos de evaluación inconsistentes.')
                        records = _read_scenarios(result_dir / 'scenario_metrics.csv',
                            {**context, 'policy': policy}, bc['scenarios'][x.sha])
                        if records[0]['used_vehicles'] != sum(bool(route) for route in solution['routes']):
                            raise ValueError('Vehículos evaluados distintos de las rutas persistidas.')
                        _check_aggregates(records, evaluated, bc['protocol']['alpha'])
                        for time_key in ['evaluation_seconds', 'trace_validation_seconds']:
                            _finite(evaluated[time_key], time_key, True)
                        scenario_rows.extend(records)
                        expected_row = {**base, **evaluated, 'evaluation_status': 'validated'}
                    if indexed[key] != expected_row:
                        raise ValueError('Reporte distinto del resultado persistido de un intento.')
    if set(indexed) != expected_keys or set(bundles) != expected_bundles:
        raise ValueError('Reporte o inventario con intentos faltantes/extra.')
    validation = inspected['validation']
    if (validation['summary_rows'] != len(rows)
            or validation['attempts'] != len(rows) // len(bc['protocol']['policies'])
            or validation['unsolved_policy_rows'] != sum(r['evaluation_status'] != 'validated' for r in rows)):
        raise ValueError('Resumen de validación no corresponde a los intentos.')
    return rows, scenario_rows, inspected


def _offline_costs(rows, drive_root, bank_id):
    """Una fila por manifiesto de entrenamiento; corpus docente en tabla separada."""
    groups = {}
    for row in rows:
        if row['method'] not in LEARNED:
            continue
        key = (LEARNED[row['method']], row['offline_cost_id'])
        groups.setdefault(key, []).append(row)
    costs, teachers = [], {}
    for (namespace, cost_id), records in sorted(groups.items()):
        methods = sorted({r['method'] for r in records})
        method = '|'.join(methods)
        if not cost_id or Path(cost_id).name != cost_id or cost_id in ('.', '..'):
            raise ValueError('Identidad offline inválida.')
        manifests = {r['offline_manifest_sha256'] for r in records}
        if len(manifests) != 1:
            raise ValueError('Un mismo costo offline tiene manifiestos diferentes.')
        selected = {}
        for record in records:
            model_key = (record['fold_id'], record['training_seed'], record['model_checkpoint_sha256'])
            value = {k: record[k] for k in ['training_seconds', 'validation_seconds']}
            if model_key in selected and selected[model_key] != value:
                raise ValueError('Costo contradictorio para el mismo modelo.')
            selected[model_key] = value
        output = {'method': method, 'shared_by_methods': '|'.join(methods),
                  'charged_once_across_ablations': True, 'offline_cost_id': cost_id,
                  'offline_manifest_sha256': next(iter(manifests)),
                  'selected_models': len(selected),
                  'selected_training_seconds': sum(v['training_seconds'] for v in selected.values()),
                  'selected_validation_seconds_included': sum(v['validation_seconds'] for v in selected.values()),
                  'manifest_training_seconds': None, 'manifest_models_trained': None,
                  'teacher_corpus_id': None, 'status': 'manifest_missing',
                  'scope': 'selected models deduplicated; manifest totals include all trained models',
                  'validation_is_part_of_training': True}
        index = Path(drive_root) / namespace / bank_id / cost_id / 'index'
        if not index.exists():
            costs.append(output)
            continue
        if _verify(index) != output['offline_manifest_sha256']:
            raise ValueError('Manifiesto de costo offline modificado.')
        manifest = e.read_json(index / 'offline_costs.json')
        models = e.read_json(index / 'models.json')
        if manifest['offline_cost_id'] != cost_id:
            raise ValueError('Inventario offline corresponde a otro entrenamiento.')
        available = {(m['fold']['fold_id'], m['training_seed'], m['model_checkpoint_sha256']): m for m in models}
        if len(available) != len(models) or not set(selected) <= set(available):
            raise ValueError('Modelos seleccionados ausentes/duplicados en inventario offline.')
        for key, values in selected.items():
            if any(values[k] != available[key][k] for k in values):
                raise ValueError('Costos por modelo distintos de manifiesto offline.')
        training = sum(_finite(m['training_seconds'], 'training_seconds', True) for m in models)
        validation = sum(_finite(m['validation_seconds'], 'validation_seconds', True) for m in models)
        if (manifest['models_trained'] != len(models)
                or not math.isclose(training, _finite(manifest['training_seconds_unique'], 'training_total', True))
                or not math.isclose(validation, _finite(manifest['validation_seconds_included_in_training'], 'validation_total', True))):
            raise ValueError('Totales offline no reconcilian inventario de modelos.')
        output.update(status='verified', manifest_training_seconds=training,
                      manifest_models_trained=len(models), teacher_corpus_id=manifest.get('teacher_corpus_id'))
        if output['teacher_corpus_id']:
            teacher_id = output['teacher_corpus_id']
            teacher = {'teacher_corpus_id': teacher_id, **manifest['teacher_costs_unique']}
            for field, value in teacher.items():
                if field != 'teacher_corpus_id':
                    _finite(value, field, True)
            if teacher_id in teachers and teachers[teacher_id] != teacher:
                raise ValueError('Costos contradictorios del mismo corpus docente.')
            teachers[teacher_id] = teacher
        costs.append(output)
    return costs, list(teachers.values())


def _coverage(rows, bc, config, selection):
    result = []
    for method in config['expected_methods']:
        selected = [r for r in rows if r['method'] == method]
        expected = len(bc['registry']) * len(bc['protocol']['solver_seeds']) * len(bc['protocol']['policies'])
        evaluated = sum(r['evaluation_status'] == 'validated' for r in selected)
        license_pending = sum(r['solver_status'] == 'license_pending' for r in selected)
        result.append({'method': method, 'run_id': selection.get(method),
            'expected_policy_rows': expected, 'recorded_policy_rows': len(selected),
            'evaluated_policy_rows': evaluated, 'license_pending_policy_rows': license_pending,
            'failed_policy_rows': len(selected) - evaluated - license_pending,
            'missing_policy_rows': expected - len(selected),
            'complete_evaluated_cohort': evaluated == expected})
    return result


def _report_text(coverage, scope, selection):
    lines = ['# Comparación 06 — piloto de desarrollo', '',
        'Resultados descriptivos de las ejecuciones seleccionadas. No hay pruebas de significancia, '
        'ranking poblacional ni afirmación de superioridad. Semillas y escenarios no aumentan el número '
        'de geometrías independientes. Los folds comparten entrenamiento y no son pruebas externas independientes.', '',
        'Costo: minutos de viaje ejecutado; no es el costo completo de recurso c+Q. '
        'CVaR se calcula por plan y después se promedia entre réplicas. '
        'Costo comparativo se condiciona a planes conservadoramente factibles; FR condicional requiere evaluación. '
        'joint_performance también informa viaje de todas las evaluaciones y los denominadores. Interpretar con cobertura.', '',
        '| Método | Evaluaciones | Esperadas | Fallos | Licencia pendiente | Faltantes |',
        '|---|---:|---:|---:|---:|---:|']
    for r in coverage:
        lines.append(f"| {r['method']} | {r['evaluated_policy_rows']} | {r['expected_policy_rows']} | "
                     f"{r['failed_policy_rows']} | {r['license_pending_policy_rows']} | {r['missing_policy_rows']} |")
    lines += ['', 'Las filas anteriores cuentan intento × política; las dos políticas no son muestras independientes.', '',
        f"Diferencias frente a {scope['reference_method']} sólo para cohortes completas en ambos métodos; no se emparejan "
        'semillas de solver como si representaran el mismo flujo aleatorio. Las diferencias de escenarios '
        'sí usan los CRN del banco. Las cotas Gurobi pertenecen exclusivamente al objetivo conservador.', '',
        'Tiempos de entrenamiento y validación se deduplican por modelo; validación ya está incluida '
        'en entrenamiento. Los costos docentes se deduplican por corpus. Un manifiesto ausente impide '
        'verificar su total y se identifica como tal. No se amortizan costos offline sobre el piloto.', '',
        'Las comparaciones de tiempo requieren considerar el hardware de cada ejecución (input_environments.json). '
        '06 verifica hashes, identidades y agregados; no vuelve a simular las rutas.', '',
        'Selección de runs (no basada automáticamente en fecha o rendimiento):',
        '```json', json.dumps(selection, indent=2), '```', '',
        'Tablas adicionales: joint_performance informa éxito sobre todos los intentos y cotas '
        'de desconocimiento para faltantes; no son intervalos de confianza. paired_geometry_costs '
        'contrasta costos sólo en geometrías con todas las semillas factibles en ambos métodos y '
        'conserva el número y motivo de exclusiones. No existe una penalización de costo inventada. '
        'risk_diagnostics describe varianza, dispersión y exceso CVaR; recourse_differences contrasta '
        'causal_edd contra fixed_order sobre el mismo plan y CRN. budget_audit deduplica las políticas '
        'para no multiplicar tiempo de búsqueda. La comparación POMO/sin entrenar y HBG/sin feromona '
        'se fija antes de observar resultados. No se ha implementado una ejecución confirmatoria: '
        'inference_readiness enumera explícitamente los requisitos pendientes.', '',
        'Las figuras se exportan en PNG (300 dpi), PDF y SVG; costos condicionados y ausencia de '
        'réplicas permanecen visibles. No se ordenan métodos por desempeño.', '',
        'Alcance estadístico:', '```json', json.dumps(scope, indent=2, ensure_ascii=False), '```', '']
    return '\n'.join(lines)


def _discussion_table_text(tables):
    """Publication-oriented descriptive table with denominators beside costs."""
    output = ['# Tablas para discusión · piloto descriptivo', '',
        'Viaje y CVaR se condicionan a planes conservadoramente factibles. Éxito incluye '
        'todos los intentos previstos; si falta información se muestra un intervalo de '
        'desconocimiento, no un intervalo de confianza. Un guion indica ausencia de medición.', '']
    groups = {}
    for row in tables['joint_performance']:
        groups.setdefault((row['instance_id'], row['n_customers'], row['policy']), []).append(row)
    def number(value):
        return '—' if value is None else f'{value:.2f}'
    for (instance_id, n, policy), rows in groups.items():
        output += [f'## {instance_id} · n={n} · {policy}', '',
            '| Método | Evaluados / previstos | Factibles conservadores / previstos | Viaje (min) | CVaR por plan (min) | Éxito total (%) | Fallos | Faltantes/licencia |',
            '|---|---:|---:|---:|---:|---:|---:|---:|']
        for r in rows:
            lo, hi = 100 * r['scenario_success_lower_bound'], 100 * r['scenario_success_upper_bound']
            success = number(lo) if r['scenario_success_per_scheduled_attempt'] is not None else f'[{lo:.2f}, {hi:.2f}]'
            output.append(f"| {r['method']} | {r['evaluated_attempts']}/{r['expected_attempts']} | "
                f"{r['conservative_feasible_attempts']}/{r['expected_attempts']} | "
                f"{number(r['mean_travel_conditional_conservative_feasible'])} | "
                f"{number(r['cvar_travel_conditional_conservative_feasible'])} | {success} | "
                f"{r['failed_attempts']} | {r['missing_attempts']}/{r['license_pending_attempts']} |")
        output.append('')
    output += ['No hay selección de mejor semilla, ranking global ni prueba confirmatoria. '
               'Cada geometría cuenta una vez; escenarios y semillas no aumentan N.', '']
    return '\n'.join(output)


def _solution_diagnostics(solution):
    """Flatten recorded metadata without imputing iterations or attributing unknown stops."""
    meta = solution['metadata']
    raw = solution['status']
    normalized = ('time_budget_reached' if raw == 'interrupted' and
                  meta.get('budget_termination_requested') is True else raw)
    attempts, complete = meta.get('attempts'), meta.get('completed_decodes')
    return {'solver_status': raw, 'solver_status_normalized': normalized,
        'status_normalization_basis': 'recorded_budget_callback' if normalized != raw else 'original_status',
        **{key: meta.get(key) for key in (
            'termination', 'attempts', 'completed_decodes', 'failed_decodes', 'interrupted_decodes',
            'iterations', 'completed_iterations', 'constructed_solutions', 'initial_attempts',
            'evaluated_neighbors', 'accepted_moves', 'constructive_seed_count', 'neural_complete_count',
            'best_candidate_generation_source', 'implementation', 'construction',
            'inference_device', 'model_config', 'budget_termination_requested',
            'solution_origin', 'returned_paid_edd_fallback', 'routing_assignment_found')},
        'decode_completion_fraction': complete / attempts if type(attempts) is int and attempts > 0
            and type(complete) is int and 0 <= complete <= attempts else None,
        'solver_metadata_json': json.dumps(meta, sort_keys=True, ensure_ascii=False)}


def run_comparison(bank, method_runs_root, output_root, config, selection=None,
                   local_root=None, validation_artifacts=()):
    """Exporta un análisis sellado reproducible; nunca modifica los resultados fuente."""
    _require_colab()
    bc = _bank_context(bank, config)
    inventory = discover_runs(bank, method_runs_root, config)
    chosen = select_runs(inventory, config, selection)
    rows, scenarios, inputs, environments, diagnostics, sources_by_method = [], [], {}, {}, [], {}
    for run_id in sorted(set(chosen.values())):
        directory = _safe_relative(method_runs_root, run_id)
        summary, raw, inspected = _load_run(directory, bc)
        selected_methods = {m for m, selected_id in chosen.items() if selected_id == run_id}
        sources_by_method.update({m: {**inspected['plan']['source_hashes'],
            '__environment_sha256': p.digest(e.read_json(directory / 'meta/environment.json'))}
            for m in selected_methods})
        rows.extend(r for r in summary if r['method'] in selected_methods)
        scenarios.extend(r for r in raw if r['method'] in selected_methods)
        inputs[run_id] = {k: inspected[k] for k in ['report_sha256', 'meta_sha256']}
        environments[run_id] = e.read_json(directory / 'meta/environment.json')
        for method in sorted(selected_methods):
            for x in bc['instances']:
                for seed in bc['protocol']['solver_seeds']:
                    solution = e.read_json(directory / method / f'n{x.n:04d}' / f'seed_{seed}' / 'solution/solution.json')
                    meta = solution['metadata']
                    diagnostics.append({'run_id': run_id, 'method': method, 'solver_seed': seed,
                        'instance_sha256': x.sha, 'n_customers': x.n,
                        'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                        **_solution_diagnostics(solution)})
    contrast_audit = _controlled_contrast_audit(rows, diagnostics, bc['registry'],
        bc['protocol']['solver_seeds'], config['method_contract']['methods'], sources_by_method)
    tables = stats.build_tables(rows, scenarios, bc['registry'], bc['protocol']['policies'],
                               bc['protocol']['solver_seeds'], config, contrast_audit=contrast_audit)
    costs, teachers = _offline_costs(rows, Path(method_runs_root).parent.parent, bc['bank_id'])
    tables.update(offline_costs=costs, teacher_costs=teachers, solution_diagnostics=diagnostics)
    tables['method_provenance'] = [
        {'method': method, 'run_id': run_id, 'comparison_id': bc['comparison_id'],
         'objective': bc['protocol']['objective'],
         'search_budget_seconds': bc['protocol']['search_seconds'],
         'algorithm_config_json': json.dumps(e.read_json(_safe_relative(method_runs_root, run_id) / 'meta/plan.json')['config']['methods'][method], sort_keys=True),
         'role': config.get('method_roles', {}).get(method, 'comparator'),
         'environment_file': 'input_environments.json', 'environment_run_key': run_id,
         'same_budget_does_not_imply_same_hardware': True}
        for method, run_id in chosen.items()]
    coverage = _coverage(rows, bc, config, chosen)
    sources = {name: p.file_hash(Path(__file__).with_name(name)) for name in
               ['comparison.py', 'comparison_stats.py', 'experiments.py', 'search.py']}
    identity = {'schema_version': SCHEMA, 'config': config, 'comparison_id': bc['comparison_id'],
                'selected_runs': chosen, 'inputs': inputs, 'source_hashes': sources,
                'offline_costs_sha256': p.digest(costs), 'teacher_costs_sha256': p.digest(teachers),
                'validation_hashes': {Path(f).name: p.file_hash(f) for f in validation_artifacts}}
    analysis_id = 'comparison06_' + p.digest(identity)[:20]
    remote = Path(output_root) / analysis_id / 'report'
    local = Path(local_root or '/content/comparison06') / analysis_id / 'report'
    if local.resolve() == remote.resolve():
        raise ValueError('Staging local y destino Drive deben ser distintos.')
    for destination in [local, remote]:
        if any(destination.resolve().is_relative_to(Path(source).resolve())
               for source in [bank, method_runs_root]):
            raise ValueError('Las salidas deben quedar fuera del banco y las corridas fuente.')
    if not e._recover(local, remote):
        local.mkdir(parents=True, exist_ok=True)
        e.write_json(local / 'identity.json', identity)
        e.write_json(local / 'input_inventory.json', inventory)
        e.write_json(local / 'input_environments.json', environments)
        e.write_json(local / 'environment.json', p.environment_report())
        e.write_json(local / 'registry.json', bc['registry'])
        e.write_csv(local / 'registry.csv', bc['registry'])
        e.write_json(local / 'common_protocol.json', bc['protocol'])
        e.write_json(local / 'analysis_export_contract.json', {
            'schema_version': 'analysis-export-08-v1', 'comparison_schema': SCHEMA,
            'bank_id': bc['bank_id'], 'bank_manifest_sha256': bc['bank_hash'],
            'comparison_id': bc['comparison_id'], 'role': config['role'],
            'analysis_mode': config['analysis_mode'], 'unit_of_independence': 'geometry_group_sha256',
            'policies': bc['protocol']['policies'], 'solver_seeds': bc['protocol']['solver_seeds'],
            'expected_methods': config['expected_methods'], 'reference_method': config['reference_method'],
            'source_selection': chosen, 'source_selection_is_best_run_search': False,
            'required_files': ['identity.json', 'validation.json', 'attempts.csv', 'attempts.json',
                'registry.csv', 'registry.json', 'common_protocol.json', 'tables.json',
                'joint_performance.csv', 'case_summary.csv', 'paired_geometry_costs.csv',
                'paired_geometry_success.csv', 'gaps.csv', 'solution_diagnostics.csv',
                'statistical_scope.json', 'input_environments.json']})
        e.write_json(local / 'attempts.json', rows)
        e.write_csv(local / 'attempts.csv', rows)
        e.write_json(local / 'coverage.json', coverage)
        e.write_csv(local / 'coverage.csv', coverage)
        e.write_json(local / 'tables.json', tables)
        for name, data in tables.items():
            if isinstance(data, list):
                e.write_csv(local / f'{name}.csv', data, None if data else ['status'])
        e.write_json(local / 'statistical_scope.json', tables['statistical_scope'])
        for name in sources:
            destination = local / 'code' / name
            destination.parent.mkdir(exist_ok=True)
            shutil.copy2(Path(__file__).with_name(name), destination)
        for source in map(Path, validation_artifacts):
            destination = local / 'contract_validation' / source.name
            destination.parent.mkdir(exist_ok=True)
            shutil.copy2(source, destination)
        stats.save_plots(tables, local / 'figures')
        (local / 'report.md').write_text(_report_text(coverage, tables['statistical_scope'], chosen), encoding='utf-8')
        (local / 'discussion_tables.md').write_text(_discussion_table_text(tables), encoding='utf-8')
        # Revalidar los bundles fuente detecta cambios durante el análisis.
        for run_id, hashes in inputs.items():
            directory = _safe_relative(method_runs_root, run_id)
            for part in ['meta', 'report']:
                if _verify(directory / part) != hashes[f'{part}_sha256']:
                    raise ValueError('Un resultado fuente cambió durante el análisis.')
            for relative, digest in e.read_json(directory / 'report/bundles.json').items():
                if _verify(_safe_relative(directory, relative)) != digest:
                    raise ValueError('Un bundle fuente cambió durante el análisis.')
        if _verify(bank) != bc['bank_hash']:
            raise ValueError('El banco cambió durante el análisis.')
        final_costs, final_teachers = _offline_costs(rows, Path(method_runs_root).parent.parent, bc['bank_id'])
        if final_costs != costs or final_teachers != teachers:
            raise ValueError('Los manifiestos offline cambiaron durante el análisis.')
        e.write_json(local / 'validation.json', {'status': 'completed', 'bank_unchanged': True,
            'source_results_unchanged': True, 'offline_manifests_unchanged': True,
            'scenario_aggregates_reconciled': True,
            'no_simulations_or_solvers_run': True, 'ready_for_confirmatory_comparison': False,
            'selected_runs': chosen, 'summary_rows': len(rows), 'scenario_rows': len(scenarios)})
        e._commit(local, remote)
    if e.read_json(local / 'identity.json') != identity:
        raise ValueError('Reporte recuperado con identidad distinta.')
    files = [str(f.relative_to(remote)) for f in sorted(remote.rglob('*')) if f.is_file()]
    return {'report_dir': str(remote), 'comparison_run_id': analysis_id, 'coverage': coverage,
            'statistical_scope': tables['statistical_scope'], 'files': files, 'tables': tables}


Overwriting svrplab/comparison.py


In [45]:
%%writefile svrplab/comparison_stats.py
"""06: descripción del piloto por geometría; ejecución exclusivamente en Colab.

No calcula p-valores, intervalos poblacionales, bootstrap ni rankings globales.
Las semillas de solver no se emparejan como RNG equivalentes. Los escenarios
CRN se emparejan sólo después de promediar las réplicas de cada método y sólo
para cohortes completas. CVaR se promedia por plan; nunca se agrupan los costos
de planes distintos para fabricar un CVaR diferente.

El loader de 06 verifica procedencia, sellos, protocolo y evaluaciones; este
módulo verifica claves, cobertura y agregación. Una geometría sigue contando
una vez aunque tenga varias instancias, ventanas, réplicas o escenarios.
"""
from __future__ import annotations

from collections import defaultdict
from collections.abc import Mapping
import hashlib
import math
from pathlib import Path
import re
from statistics import fmean


SCHEMA = 'comparison-06-v2'
BASE_METHODS = ['gurobi', 'aco', 'tabu', 'am_supervised', 'pomo', 'hbg_facs']
CONTROLS = ['edd', 'ortools_routing', 'pomo_untrained', 'random_masked', 'hbg_no_pheromone', 'hbg_untrained']
METHODS = BASE_METHODS + CONTROLS
CONTROLLED_PAIRS = [('pomo', 'pomo_untrained'), ('pomo', 'random_masked'),
                    ('hbg_facs', 'hbg_no_pheromone'), ('hbg_facs', 'hbg_untrained')]
METRICS = ['scenario_strict_feasibility', 'mean_travel_minutes', 'cvar_travel_minutes',
           'mean_lateness_minutes', 'mean_depot_overtime_minutes', 'search_total_seconds']
SCENARIO_METRICS = {'scenario_strict_feasibility': 'strict_feasible',
                    'mean_travel_minutes': 'travel_minutes',
                    'mean_lateness_minutes': 'lateness_minutes',
                    'mean_depot_overtime_minutes': 'depot_overtime_minutes'}
_BOUND_REL_TOL, _BOUND_ABS_TOL = 1e-8, 1e-6


def _number(value, *, boolean=False):
    if value is None or value == '':
        return None
    if isinstance(value, bool):
        if boolean:
            return float(value)
        raise ValueError('Un booleano no representa una métrica numérica.')
    if boolean and value in ('True', 'False', 'true', 'false'):
        return float(value in ('True', 'true'))
    try:
        result = float(value)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError('Métrica no numérica.') from error
    if not math.isfinite(result):
        raise ValueError('Una medición disponible debe ser finita; ausente se representa por null.')
    return result


def _seed(value):
    # Nunca pasar las semillas de 64 bits por float/pandas inferencia numérica.
    if type(value) is int and value >= 0:
        return value
    if isinstance(value, str) and re.fullmatch(r'[0-9]+', value):
        return int(value)
    raise ValueError('Las semillas deben ser enteros exactos o cadenas decimales no negativas.')


def _config(config):
    if (not isinstance(config, Mapping) or config.get('schema_version') != SCHEMA
            or config.get('role') != 'development_only'
            or config.get('analysis_mode') != 'descriptive_only'):
        raise ValueError('06 sólo admite comparación descriptiva de desarrollo.')
    methods, metrics = config.get('expected_methods'), config.get('metrics')
    if (not isinstance(methods, list) or len(methods) != len(set(methods))
            or not set(BASE_METHODS) <= set(methods) or not set(methods) <= set(METHODS)
            or config.get('reference_method') not in ('aco', 'gurobi')):
        raise ValueError('Se requieren los seis métodos, controles reconocidos y referencia descriptiva ACO histórica o Gurobi declarada.')
    if not isinstance(metrics, list) or len(metrics) != len(METRICS) or set(metrics) != set(METRICS):
        raise ValueError('Las métricas descriptivas deben coincidir con el protocolo 06.')
    if (config.get('ready_for_confirmatory', False) is not False
            or config.get('future_confirmatory_design', {}).get('enabled', False) is not False):
        raise ValueError('El piloto no permite habilitar inferencia confirmatoria.')
    return list(methods), list(metrics)


def _identity(registry):
    result = {}
    for row in registry:
        if not isinstance(row, Mapping):
            raise ValueError('Registro de instancias inválido.')
        for key in ('instance_id', 'instance_sha256', 'geometry_group_sha256'):
            if not isinstance(row.get(key), str) or not row[key]:
                raise ValueError(f'Falta identidad: {key}.')
        if type(row.get('n_customers')) is not int or row['n_customers'] < 1:
            raise ValueError('n_customers debe ser entero positivo.')
        sha = row['instance_sha256']
        if sha in result:
            raise ValueError('Instancia duplicada en registry.')
        result[sha] = dict(row)
    if not result:
        raise ValueError('El registro de instancias no puede estar vacío.')
    return result


def _row_key(row, registry, policies, methods, seeds):
    if not isinstance(row, Mapping):
        raise ValueError('Fila de resultados inválida.')
    sha, policy, method = row.get('instance_sha256'), row.get('policy'), row.get('method')
    seed = _seed(row.get('solver_seed'))
    if sha not in registry or policy not in policies or method not in methods or seed not in seeds:
        raise ValueError('Fila ajena a la cohorte preespecificada.')
    identity = registry[sha]
    for key in ('instance_id', 'geometry_group_sha256', 'n_customers'):
        if key in row and row[key] != identity[key]:
            raise ValueError(f'Identidad de fila incompatible: {key}.')
    return sha, policy, method, seed


def _category(row):
    if row is None:
        return 'missing'
    if row.get('solver_status') == 'license_pending':
        return 'license_pending'
    if row.get('evaluation_status') == 'validated':
        return 'evaluated'
    return 'failed'


def _context(identity, policy, method):
    return {key: identity[key] for key in ('instance_id', 'instance_sha256', 'geometry_group_sha256',
                                          'n_customers')} | {'policy': policy, 'method': method}


def _case_tables(index, registry, policies, seeds, methods, metrics):
    output = []
    for sha, identity in registry.items():
        for policy in policies:
            for method in methods:
                rows = [index.get((sha, policy, method, seed)) for seed in seeds]
                counts = {name: sum(_category(row) == name for row in rows)
                          for name in ('evaluated', 'failed', 'license_pending', 'missing')}
                for metric in metrics:
                    runtime = metric == 'search_total_seconds'
                    feasible_cost = metric in ('mean_travel_minutes', 'cvar_travel_minutes')
                    eligible = [row for row in rows if row is not None and (runtime or (
                        _category(row) == 'evaluated' and (not feasible_cost or (
                            row.get('conservative_feasible') is True and row.get('structural_valid') is True))))]
                    values = [_number(row.get(metric)) for row in eligible]
                    values = [value for value in values if value is not None]
                    if any(value < 0 for value in values):
                        raise ValueError(f'Una métrica no negativa contiene un valor negativo: {metric}.')
                    if metric == 'scenario_strict_feasibility' and any(value > 1 for value in values):
                        raise ValueError('La fracción de factibilidad debe pertenecer a [0,1].')
                    output.append(_context(identity, policy, method) | {
                        'metric': metric, 'mean_conditional': fmean(values) if values else None,
                        'expected': len(seeds), **counts, 'metric_available': len(values),
                        'evaluated_fraction': counts['evaluated'] / len(seeds),
                        'metric_coverage_fraction': len(values) / len(seeds),
                        'cohort_complete': counts['evaluated'] == len(seeds) and len(values) == len(seeds),
                        'conditioning': ('recorded_attempts_with_runtime' if runtime else
                                         'validated_conservative_feasible_plans' if feasible_cost else
                                         'validated_solutions_with_metric'),
                        'aggregation': 'mean_over_available_expected_solver_replicas',
                        'cvar_scope': 'mean_of_per_plan_empirical_cvar' if metric == 'cvar_travel_minutes' else None,
                    })
    return output


def _geometry_tables(cases):
    groups = defaultdict(list)
    for row in cases:
        groups[(row['geometry_group_sha256'], row['policy'], row['method'], row['metric'])].append(row)
    output = []
    for (geometry, policy, method, metric), rows in groups.items():
        values = [row['mean_conditional'] for row in rows if row['mean_conditional'] is not None]
        counts = {name: sum(row[name] for row in rows)
                  for name in ('expected', 'evaluated', 'failed', 'license_pending', 'missing', 'metric_available')}
        output.append({
            'geometry_group_sha256': geometry, 'policy': policy, 'method': method, 'metric': metric,
            'n_customers': '|'.join(str(n) for n in sorted({row['n_customers'] for row in rows})),
            'expected_instances': len(rows), 'instances_with_measurement': len(values),
            'fully_evaluated_instances': sum(row['cohort_complete'] for row in rows),
            'mean_conditional': fmean(values) if values else None, **counts,
            'evaluated_fraction': counts['evaluated'] / counts['expected'],
            'metric_coverage_fraction': counts['metric_available'] / counts['expected'],
            'cohort_complete': all(row['cohort_complete'] for row in rows),
            'independent_geometry_count': 1, 'conditioning': rows[0]['conditioning'],
            'aggregation': 'equal_instance_weight_after_mean_over_solver_replicas',
            'cvar_scope': rows[0]['cvar_scope'],
        })
    return output


def _pairwise_tables(cases, reference):
    lookup = {(row['instance_sha256'], row['policy'], row['method'], row['metric']): row for row in cases}
    output = []
    for row in cases:
        if row['method'] == reference:
            continue
        ref = lookup[(row['instance_sha256'], row['policy'], reference, row['metric'])]
        complete = row['cohort_complete'] and ref['cohort_complete']
        delta = row['mean_conditional'] - ref['mean_conditional'] if complete else None
        ratio = 100. * delta / ref['mean_conditional'] if complete and ref['mean_conditional'] != 0 else None
        output.append({key: row[key] for key in ('instance_id', 'instance_sha256', 'geometry_group_sha256',
                                                 'n_customers', 'policy', 'method', 'metric')} | {
            'reference_method': reference, 'status': 'complete' if complete else 'incomplete_cohort',
            'method_mean': row['mean_conditional'] if complete else None,
            'reference_mean': ref['mean_conditional'] if complete else None,
            'delta_mean': delta, 'ratio_percent': ratio,
            'ratio_status': 'defined' if ratio is not None else ('zero_reference' if complete else 'incomplete_cohort'),
            'delta_percentage_points': 100 * delta if complete and row['metric'] == 'scenario_strict_feasibility' else None,
            'method_evaluated': row['evaluated'], 'reference_evaluated': ref['evaluated'],
            'expected_per_method': row['expected'],
            'direction': 'higher_is_better' if row['metric'] == 'scenario_strict_feasibility' else 'lower_is_better',
            'solver_seed_pairing': False,
            'aggregation': 'difference_of_complete_method_replica_means_within_case',
        })
    return output


def _scenario_tables(scenario_rows, index, registry, policies, seeds, methods, cases, reference):
    if not scenario_rows:
        return []
    by_run = defaultdict(dict)
    for row in scenario_rows:
        key = _row_key(row, registry, policies, methods, seeds)
        if _category(index.get(key)) != 'evaluated':
            raise ValueError('Escenarios sin una evaluación validated correspondiente.')
        scenario = _seed(row.get('scenario_seed'))
        if scenario in by_run[key]:
            raise ValueError('Escenario duplicado dentro de una réplica.')
        by_run[key][scenario] = row
    # Cada ejecución disponible debe conservar su evaluación íntegra. Los
    # escenarios omitidos no se convierten en una selección de casos fáciles.
    sets_by_case = defaultdict(list)
    for key, summary in index.items():
        if _category(summary) != 'evaluated':
            continue
        expected_count = _seed(summary.get('scenarios'))
        observed = by_run.get(key, {})
        if expected_count < 1 or len(observed) != expected_count:
            raise ValueError('Cohorte de escenarios incompleta frente al resumen validado.')
        sets_by_case[key[:2]].append(set(observed))
    for scenario_sets in sets_by_case.values():
        if any(scenarios != scenario_sets[0] for scenarios in scenario_sets[1:]):
            raise ValueError('Los métodos/réplicas no comparten los mismos escenarios CRN.')
    complete = {(row['instance_sha256'], row['policy'], row['method'], row['metric']): row['cohort_complete']
                for row in cases}
    output = []
    for sha, identity in registry.items():
        for policy in policies:
            scenario_sets = sets_by_case.get((sha, policy), [])
            if not scenario_sets:
                continue
            for method in methods:
                if method == reference:
                    continue
                for metric, source in SCENARIO_METRICS.items():
                    if not (complete[(sha, policy, method, metric)] and complete[(sha, policy, reference, metric)]):
                        continue
                    for scenario in sorted(scenario_sets[0]):
                        means = {}
                        for selected in (method, reference):
                            values = [_number(by_run[(sha, policy, selected, seed)][scenario].get(source),
                                              boolean=source == 'strict_feasible') for seed in seeds]
                            if any(value is None for value in values):
                                raise ValueError('Escenario validado con métrica ausente.')
                            if any(value < 0 or (source == 'strict_feasible' and value not in (0., 1.)) for value in values):
                                raise ValueError('Métrica de escenario fuera de su dominio.')
                            means[selected] = fmean(values)
                        delta = means[method] - means[reference]
                        output.append(_context(identity, policy, method) | {
                            'reference_method': reference, 'scenario_seed': str(scenario),
                            'metric': metric, 'scenario_metric': source,
                            'method_replica_mean': means[method], 'reference_replica_mean': means[reference],
                            'delta_mean': delta,
                            'delta_percentage_points': 100 * delta if source == 'strict_feasible' else None,
                            'replicas_per_method': len(seeds), 'pairing': 'scenario_seed_CRN_after_replica_means',
                            'solver_seed_pairing': False, 'independent_population_observation': False,
                        })
    return output


def _gap_tables(index, registry, policies, seeds, methods):
    bounds = defaultdict(list)
    for (sha, policy, method, seed), row in index.items():
        if method != 'gurobi' or row.get('solver_status') == 'license_pending':
            continue
        bound = _number(row.get('best_bound'))
        if bound is not None:
            bounds[sha].append((bound, seed, policy, row))
    output = []
    for sha, identity in registry.items():
        candidates = bounds.get(sha, [])
        source = max(candidates, key=lambda value: value[0]) if candidates else None
        lb = source[0] if source else None
        validated_optimum = False
        for bound, _, _, row in candidates:
            objective = _number(row.get('search_objective_minutes'))
            if (row.get('surrogate_optimal') is True and _category(row) == 'evaluated'
                    and row.get('structural_valid') is True and row.get('conservative_feasible') is True
                    and objective is not None and math.isclose(bound, objective,
                                                              rel_tol=_BOUND_REL_TOL, abs_tol=_BOUND_ABS_TOL)):
                validated_optimum = True
        for policy in policies:
            for method in methods:
                for seed in seeds:
                    row = index.get((sha, policy, method, seed))
                    candidate = _number(row.get('search_objective_minutes')) if row else None
                    eligible = bool(row and _category(row) == 'evaluated' and row.get('structural_valid') is True
                                    and row.get('conservative_feasible') is True and candidate is not None)
                    gap, clipped = None, False
                    if not eligible:
                        status = 'candidate_unavailable_or_not_conservative_feasible'
                    elif candidate < 0:
                        raise ValueError('Costo conservador negativo.')
                    elif lb is None:
                        status = 'no_gurobi_bound'
                    elif lb > candidate and not math.isclose(lb, candidate, rel_tol=_BOUND_REL_TOL, abs_tol=_BOUND_ABS_TOL):
                        status = 'bound_exceeds_candidate_inconsistent'
                    elif candidate == 0:
                        status = 'zero_candidate_ratio_undefined'
                    else:
                        clipped = lb > candidate
                        gap = max(0., (candidate - lb) / candidate * 100.)
                        status = 'available_upper_bound_on_relative_suboptimality'
                    output.append(_context(identity, policy, method) | {
                        'solver_seed': seed, 'candidate_cost_minutes': candidate if eligible else None,
                        'best_gurobi_lower_bound_minutes': lb, 'gap_upperbound_pct': gap, 'status': status,
                        'bound_aggregation': 'strongest_certified_bound_across_all_recorded_gurobi_attempts',
                        'bound_is_equal_single_attempt_runtime_comparison': False,
                        'bound_is_not_stochastic_mean_cost_reference': True,
                        'gap_denominator': 'candidate_conservative_objective_not_unknown_optimum',
                        'bound_source_solver_seed': source[1] if source else None,
                        'bound_source_policy': source[2] if source else None,
                        'gurobi_surrogate_optimum_validated': validated_optimum,
                        'candidate_surrogate_optimal_validated': bool(eligible and method == 'gurobi'
                            and row.get('surrogate_optimal') is True and lb is not None
                            and _number(row.get('best_bound')) is not None
                            and math.isclose(candidate, _number(row['best_bound']),
                                             rel_tol=_BOUND_REL_TOL, abs_tol=_BOUND_ABS_TOL)
                            and math.isclose(candidate, lb, rel_tol=_BOUND_REL_TOL, abs_tol=_BOUND_ABS_TOL)),
                        'numerical_equality_clipped_to_zero': clipped,
                        'scope': 'conservative_surrogate_same_instance_only_not_stochastic_optimality',
                        'numerical_equality_is_optimality_proof': False,
                    })
    return output


def _success_value(row):
    """Observed pipeline success; no-solution is zero, missing evaluation unknown."""
    if row is None or row.get('solver_status') == 'license_pending':
        return None
    if _category(row) == 'evaluated':
        return _number(row.get('scenario_strict_feasibility'))
    if row.get('evaluation_status') == 'no_solution' and row.get('structural_valid') is False:
        return 0.
    return None


def _joint_tables(index, registry, policies, seeds, methods):
    output = []
    for sha, identity in registry.items():
        for policy in policies:
            for method in methods:
                rows = [index.get((sha, policy, method, seed)) for seed in seeds]
                valid = [r for r in rows if _category(r) == 'evaluated']
                conservative = [r for r in valid if r.get('conservative_feasible') is True]
                values = [_success_value(r) for r in rows]
                observed = [v for v in values if v is not None]
                unresolved = len(rows) - len(observed)
                def mean(field, eligible):
                    data = [_number(r.get(field)) for r in eligible]
                    data = [v for v in data if v is not None]
                    return fmean(data) if data else None
                output.append(_context(identity, policy, method) | {
                    'expected_attempts': len(seeds),
                    'recorded_attempts': sum(r is not None for r in rows),
                    'evaluated_attempts': len(valid),
                    'conservative_feasible_attempts': len(conservative),
                    'conservative_feasible_per_scheduled_attempt': len(conservative) / len(seeds),
                    'nominal_feasible_attempts': sum(r.get('nominal_feasible') is True for r in valid),
                    'failed_attempts': sum(_category(r) == 'failed' for r in rows),
                    'missing_attempts': sum(r is None for r in rows),
                    'license_pending_attempts': sum(_category(r) == 'license_pending' for r in rows),
                    'scenario_success_known_attempts': len(observed),
                    'scenario_success_unresolved_attempts': unresolved,
                    'scenario_success_per_scheduled_attempt': sum(observed) / len(seeds) if not unresolved else None,
                    'scenario_success_lower_bound': sum(observed) / len(seeds),
                    'scenario_success_upper_bound': (sum(observed) + unresolved) / len(seeds),
                    'success_bounds_are_confidence_intervals': False,
                    'scenario_success_conditioning': 'all_scheduled_attempts_no_solution_zero_unresolved_bounded',
                    'mean_travel_conditional_evaluated': mean('mean_travel_minutes', valid),
                    'mean_travel_conditional_conservative_feasible': mean('mean_travel_minutes', conservative),
                    'cvar_travel_conditional_conservative_feasible': mean('cvar_travel_minutes', conservative),
                    'cost_denominator_evaluated': len(valid),
                    'cost_denominator_conservative_feasible': len(conservative),
                    'no_numeric_cost_assigned_to_failed_solution': True,
                })
    return output


def _contrasts(methods, config):
    pairs = [(m, config['reference_method']) for m in methods if m != config['reference_method']]
    controls = config.get('controlled_contrasts', [
        {'method': 'pomo', 'reference': 'pomo_untrained'},
        {'method': 'pomo', 'reference': 'random_masked'},
        {'method': 'hbg_facs', 'reference': 'hbg_no_pheromone'},
        {'method': 'hbg_facs', 'reference': 'hbg_untrained'},
        {'method': 'hbg_facs', 'reference': 'ortools_routing'},
        {'method': 'hbg_facs', 'reference': 'edd'},
    ])
    for contrast in controls:
        method, reference = contrast['method'], contrast['reference']
        if method not in METHODS or reference not in METHODS or method == reference:
            raise ValueError('Contraste controlado inválido.')
        if method in methods and reference in methods and (method, reference) not in pairs:
            pairs.append((method, reference))
    return pairs


def _paired_geometry_costs(index, registry, policies, seeds, methods, config):
    """Costs only on the same complete feasible geometries, all seeds retained.

    Selection here is explicit conditioning, never a claim over all geometries.
    Every expected geometry receives a row, including exclusions and their counts.
    """
    geometries = defaultdict(list)
    for sha, identity in registry.items():
        geometries[identity['geometry_group_sha256']].append(sha)
    output = []
    for method, reference in _contrasts(methods, config):
        for policy in policies:
            for metric in ('mean_travel_minutes', 'cvar_travel_minutes'):
                pair_rows = []
                for geometry, instances in geometries.items():
                    eligible, means, counts = {}, {}, {}
                    for selected in (method, reference):
                        per_instance = []
                        expected = len(instances) * len(seeds)
                        qualifying = 0
                        for sha in instances:
                            values = []
                            for seed in seeds:
                                r = index.get((sha, policy, selected, seed))
                                if (_category(r) == 'evaluated' and r.get('conservative_feasible') is True
                                        and r.get('structural_valid') is True):
                                    value = _number(r.get(metric))
                                    if value is not None:
                                        values.append(value)
                            qualifying += len(values)
                            if len(values) == len(seeds):
                                per_instance.append(fmean(values))
                        eligible[selected] = len(per_instance) == len(instances)
                        counts[selected] = qualifying
                        means[selected] = fmean(per_instance) if eligible[selected] else None
                    common = eligible[method] and eligible[reference]
                    pair_rows.append({
                        'geometry_group_sha256': geometry, 'policy': policy, 'metric': metric,
                        'method': method, 'reference_method': reference,
                        'status': 'common_complete_feasible' if common else 'excluded_incomplete_or_infeasible',
                        'expected_instances': len(instances), 'expected_attempts_per_method': expected,
                        'method_qualifying_attempts': counts[method], 'reference_qualifying_attempts': counts[reference],
                        'method_mean': means[method] if common else None,
                        'reference_mean': means[reference] if common else None,
                        'delta_mean': means[method] - means[reference] if common else None,
                        'conditional_on_both_methods_complete_conservative_feasibility': True,
                        'solver_seed_pairing': False, 'independent_geometry_count': 1,
                    })
                total = sum(r['status'] == 'common_complete_feasible' for r in pair_rows)
                for r in pair_rows:
                    r.update(common_feasible_geometry_count=total, expected_geometry_count=len(geometries))
                output.extend(pair_rows)
    return output


def _paired_geometry_success(index, registry, policies, seeds, methods, config):
    joint = _joint_tables(index, registry, policies, seeds, methods)
    lookup = {(r['instance_sha256'], r['policy'], r['method']): r for r in joint}
    geometries = defaultdict(list)
    for sha, identity in registry.items():
        geometries[identity['geometry_group_sha256']].append(sha)
    output = []
    for method, reference in _contrasts(methods, config):
        for policy in policies:
            for geometry, instances in geometries.items():
                values = {}
                for selected in (method, reference):
                    per_instance = [lookup[(sha, policy, selected)]['scenario_success_per_scheduled_attempt']
                                    for sha in instances]
                    values[selected] = fmean(per_instance) if all(v is not None for v in per_instance) else None
                complete = all(value is not None for value in values.values())
                output.append({'geometry_group_sha256': geometry, 'policy': policy,
                    'method': method, 'reference_method': reference,
                    'metric': 'scenario_success_per_scheduled_attempt',
                    'method_mean': values[method], 'reference_mean': values[reference],
                    'delta_percentage_points': 100 * (values[method] - values[reference]) if complete else None,
                    'status': 'complete_known_cohort' if complete else 'unresolved_attempts',
                    'failed_solutions_retained_as_zero_success': True,
                    'all_expected_solver_seeds_retained': True,
                    'expected_attempts_per_method': len(instances) * len(seeds),
                    'expected_instances': len(instances), 'independent_geometry_count': 1,
                    'aggregation': 'equal_instance_weight_after_all_expected_solver_seeds',
                })
    return output


def _risk_diagnostics(index):
    output = []
    for row in index.values():
        if _category(row) != 'evaluated':
            continue
        mean = _number(row.get('mean_travel_minutes'))
        cvar = _number(row.get('cvar_travel_minutes'))
        variance = _number(row.get('cost_variance_population'))
        std = _number(row.get('std_travel_minutes'))
        output.append({key: row.get(key) for key in (
            'instance_id', 'instance_sha256', 'geometry_group_sha256', 'n_customers',
            'policy', 'method', 'solver_seed', 'scenarios', 'alpha')} | {
            'mean_travel_minutes': mean, 'std_travel_minutes': std,
            'cost_variance_population': variance,
            'coefficient_of_variation': std / mean if std is not None and mean else None,
            'cvar_minus_mean_minutes': cvar - mean if cvar is not None and mean is not None else None,
            'cvar_excess_percent': 100 * (cvar - mean) / mean if cvar is not None and mean else None,
            'scope': 'conditional_scenario_variation_of_one_plan_not_geometries',
            'evidence_of_stochastic_optimality': False,
        })
    return output


def _recourse_differences(index):
    output = []
    for (sha, policy, method, seed), row in index.items():
        if policy != 'causal_edd':
            continue
        reference = index.get((sha, 'fixed_order', method, seed))
        available = _category(row) == _category(reference) == 'evaluated'
        for metric in ('scenario_strict_feasibility', 'mean_travel_minutes', 'cvar_travel_minutes',
                       'mean_lateness_minutes', 'mean_depot_overtime_minutes'):
            value = _number(row.get(metric)) if available else None
            base = _number(reference.get(metric)) if available else None
            output.append({key: row.get(key) for key in ('instance_id', 'instance_sha256',
                'geometry_group_sha256', 'n_customers', 'method', 'solver_seed')} | {
                'policy': policy, 'reference_policy': 'fixed_order', 'metric': metric,
                'status': 'paired_same_plan' if value is not None and base is not None else 'unavailable',
                'delta_policy_minus_fixed': value - base if value is not None and base is not None else None,
                'direction': 'higher_is_better' if metric == 'scenario_strict_feasibility' else 'lower_is_better',
                'same_plan_same_scenarios': True, 'independent_population_observation': False,
            })
    return output


def _budget_audit(index):
    attempts = {}
    for (sha, policy, method, seed), row in index.items():
        key = sha, method, seed
        fields = ('search_budget_seconds', 'search_total_seconds', 'solve_seconds', 'model_load_seconds',
                  'offline_cost_id', 'model_checkpoint_sha256', 'training_seconds', 'solver_status')
        values = {field: row.get(field) for field in fields}
        if key in attempts and any(attempts[key].get(k) != v for k, v in values.items()):
            raise ValueError('Costo/procedencia de un intento cambia entre políticas.')
        budget, total = _number(values['search_budget_seconds']), _number(values['search_total_seconds'])
        attempts[key] = {k: row.get(k) for k in ('instance_id', 'instance_sha256',
            'geometry_group_sha256', 'n_customers', 'method', 'solver_seed')} | values | {
            'budget_used_fraction': total / budget if total is not None and budget else None,
            'policy_rows_are_not_additional_searches': True,
            'same_seconds_do_not_imply_same_hardware_or_operations': True,
        }
    return list(attempts.values())


def _inference_readiness(registry, config):
    design = config.get('future_confirmatory_design', {})
    return [{
        'status': 'blocked_development_bank', 'current_role': config['role'],
        'independent_geometry_count': len({r['geometry_group_sha256'] for r in registry.values()}),
        'development_geometries_must_be_excluded_from_final_test': True,
        'minimum_independent_test_geometries': design.get('minimum_independent_test_geometries'),
        'sample_size_justification': design.get('sample_size_justification'),
        'preregistered_primary_estimand': design.get('primary_estimand'),
        'paired_resampling_unit': 'geometry_group_sha256',
        'requirements': 'sealed_disjoint_test; frozen_models_and_analysis_before_test; '
                        'independent_geometry_sample_size_justification; complete_all_seed_attempts; '
                        'predeclared_primary_estimand_and_contrasts; scenario_precision_plan',
        'bootstrap_or_tests_executed': False,
        'confirmatory_pipeline_implemented': False,
    }]


def build_tables(summary_rows, scenario_rows, registry, policies, seeds, config, *, contrast_audit=None):
    """Tablas largas descriptivas con cobertura explícita del diseño esperado.

    Ausentes son null, nunca cero. Tiempo se describe sobre intentos observados,
    incluidos fallos; FR se describe sobre evaluaciones válidas y costos sobre
    planes conservadoramente factibles. joint_performance conserva ambos denominadores.
    Las diferencias de métodos requieren todas las réplicas válidas de ambos.
    No se producen agregados globales que confundan tamaños con replicaciones.
    """
    methods, metrics = _config(config)
    registry = _identity(registry)
    policies, seeds = list(policies), [_seed(seed) for seed in seeds]
    if (not policies or any(not isinstance(policy, str) or not policy for policy in policies)
            or len(set(policies)) != len(policies) or not seeds or len(set(seeds)) != len(seeds)):
        raise ValueError('Políticas/semillas esperadas vacías o duplicadas.')
    index = {}
    for row in summary_rows:
        key = _row_key(row, registry, policies, methods, seeds)
        if key in index:
            raise ValueError('Réplica duplicada: no promediar dos reportes del mismo intento.')
        index[key] = dict(row)
    cases = _case_tables(index, registry, policies, seeds, methods, metrics)
    reference = config['reference_method']
    tables = {
        'case_summary': cases,
        'geometry_summary': _geometry_tables(cases),
        'pairwise': _pairwise_tables(cases, reference),
        'scenario_differences': _scenario_tables(scenario_rows, index, registry, policies, seeds, methods, cases, reference),
        'gaps': _gap_tables(index, registry, policies, seeds, methods),
        'joint_performance': _joint_tables(index, registry, policies, seeds, methods),
        'paired_geometry_costs': _paired_geometry_costs(index, registry, policies, seeds, methods, config),
        'paired_geometry_success': _paired_geometry_success(index, registry, policies, seeds, methods, config),
        'risk_diagnostics': _risk_diagnostics(index),
        'recourse_differences': _recourse_differences(index),
        'budget_audit': _budget_audit(index),
        'inference_readiness': _inference_readiness(registry, config),
        'controlled_contrast_audit': list(contrast_audit or []),
        'statistical_scope': {
            'schema_version': SCHEMA, 'role': 'development_only', 'analysis_mode': 'descriptive_only',
            'ready_for_confirmatory': False, 'no_pvalues': True, 'no_population_bootstrap': True,
            'no_population_confidence_intervals': True, 'no_global_ranking': True,
            'independent_geometry_count': len({row['geometry_group_sha256'] for row in registry.values()}),
            'instance_count': len(registry), 'expected_solver_replicas_per_case_method': len(seeds),
            'expected_methods': methods, 'reference_method': reference, 'metrics': metrics,
            'scenario_comparison_status': 'available' if scenario_rows else 'not_provided',
            'unit_of_independence': 'geometry_group_sha256',
            'reference_selection_note': config.get('reference_selection_note', 'historical_descriptive_reference'),
            'solver_seeds_are_equivalent_rng_pairs': False,
            'scenarios_or_seeds_increase_independent_geometry_count': False,
            'conditional_means_are_not_complete_cohort_rankings': True,
            'failed_attempts_count_in_end_to_end_success': True,
            'missing_or_unevaluated_success_is_bounded_not_imputed': True,
            'no_artificial_cost_penalty': True,
            'controlled_effects_require_verified_matched_design': True,
            'controls': [name for name in methods if name in CONTROLS],
            'cvar_aggregation': 'mean_of_per_plan_cvar_never_pooled_plan_scenarios',
            'gap_scope': 'conservative_surrogate_bound_only_not_stochastic_optimality',
            'bound_equality_relative_tolerance': _BOUND_REL_TOL,
            'bound_equality_absolute_tolerance_minutes': _BOUND_ABS_TOL,
            'numerical_bound_equality_alone_proves_optimality': False,
            'exact_equality_note': 'Exact equality of a certified lower bound and a feasible objective '
                                   'would prove surrogate optimality; a floating tolerance comparison is not that certificate.',
        },
    }

    for table_name in ('paired_geometry_costs', 'paired_geometry_success'):
        for row in tables[table_name]:
            if (row['method'], row['reference_method']) not in CONTROLLED_PAIRS:
                row['contrast_design_status'] = 'comparison_not_single_factor_ablation'
                continue
            matching = [r for r in tables['controlled_contrast_audit']
                        if r['geometry_group_sha256'] == row['geometry_group_sha256']
                        and r['method'] == row['method'] and r['reference_method'] == row['reference_method']]
            verified = (len(matching) == row['expected_instances'] and
                        len({r['instance_sha256'] for r in matching}) == row['expected_instances'] and
                        all(r['status'] == 'verified' for r in matching))
            row['contrast_design_status'] = 'verified' if verified else 'unverified'
            if not verified:
                row['data_cohort_status'] = row['status']
                row['status'] = 'unverified_control_design'
                for key in ('method_mean', 'reference_mean', 'delta_mean', 'delta_percentage_points'):
                    if key in row:
                        row[key] = None
        if table_name == 'paired_geometry_costs':
            counts = defaultdict(int)
            for row in tables[table_name]:
                if row['status'] == 'common_complete_feasible':
                    counts[(row['method'], row['reference_method'], row['policy'], row['metric'])] += 1
            for row in tables[table_name]:
                row['common_feasible_geometry_count'] = counts[(row['method'], row['reference_method'], row['policy'], row['metric'])]
    return tables


def save_plots(tables, directory):
    """PNG descriptivos por geometría/política, sin intervalos ni ranking global."""
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch

    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    rows = tables['geometry_summary']
    methods = tables['statistical_scope']['expected_methods']
    lookup = {(r['geometry_group_sha256'], r['policy'], r['method'], r['metric']): r for r in rows}
    groups = sorted({(r['geometry_group_sha256'], r['policy']) for r in rows})
    files = []
    for geometry, policy in groups:
        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        axes = axes.flatten()
        cost_colors = ['#3173a3', '#67a6c9']
        for i, (metric, label) in enumerate((('mean_travel_minutes', 'Media'), ('cvar_travel_minutes', 'CVaR por plan'))):
            for j, method in enumerate(methods):
                row = lookup[(geometry, policy, method, metric)]
                if row['mean_conditional'] is not None:
                    axes[0].bar(j + (i - .5) * .34, row['mean_conditional'], width=.32,
                                color=cost_colors[i], hatch=None if row['cohort_complete'] else '//',
                                edgecolor='#333333', linewidth=.5)
        for j, method in enumerate(methods):
            cost_rows = [lookup[(geometry, policy, method, metric)]
                         for metric in ('mean_travel_minutes', 'cvar_travel_minutes')]
            if all(row['mean_conditional'] is None for row in cost_rows):
                axes[0].text(j, .02, 'sin datos', ha='center', va='bottom', rotation=90,
                             fontsize=8, transform=axes[0].get_xaxis_transform())
            row = lookup[(geometry, policy, method, 'scenario_strict_feasibility')]
            if row['mean_conditional'] is None:
                axes[1].text(j, .02, 'sin datos', ha='center', va='bottom', rotation=90, fontsize=8)
            else:
                axes[1].bar(j, row['mean_conditional'], color='#3a8f79', edgecolor='#333333', linewidth=.5,
                            hatch=None if row['cohort_complete'] else '//')
        categories = [('evaluated', '#3a8f79', 'Evaluada'), ('failed', '#d57745', 'Fallo'),
                      ('license_pending', '#d3ac49', 'Licencia'), ('missing', '#c6c8cc', 'Faltante')]
        bottom = [0.] * len(methods)
        for key, color, label in categories:
            values = [lookup[(geometry, policy, method, 'scenario_strict_feasibility')][key] /
                      lookup[(geometry, policy, method, 'scenario_strict_feasibility')]['expected'] for method in methods]
            axes[2].bar(range(len(methods)), values, bottom=bottom, label=label, color=color)
            bottom = [old + value for old, value in zip(bottom, values)]
        for j, method in enumerate(methods):
            row = lookup[(geometry, policy, method, 'scenario_strict_feasibility')]
            axes[2].text(j, 1.02, f"{row['evaluated']}/{row['expected']}", ha='center', fontsize=8)
        axes[0].set(title='Viaje: condicionado a planes conservadores factibles', ylabel='Minutos')
        axes[0].legend(handles=[Patch(facecolor=cost_colors[0], label='Media'),
                                Patch(facecolor=cost_colors[1], label='CVaR por plan'),
                                Patch(facecolor='white', edgecolor='#333333', hatch='//', label='Cohorte incompleta')], fontsize=8)
        axes[1].set(title='Factibilidad condicional', ylabel='Fracción de escenarios', ylim=(0, 1.08))
        axes[2].set(title='Cobertura de réplicas esperadas', ylabel='Fracción de réplicas', ylim=(0, 1.12))
        axes[2].legend(fontsize=8, loc='lower left')
        joint = [r for r in tables['joint_performance'] if r['geometry_group_sha256'] == geometry and r['policy'] == policy]
        for j, method in enumerate(methods):
            selected = [r for r in joint if r['method'] == method]
            lo = fmean(r['scenario_success_lower_bound'] for r in selected)
            hi = fmean(r['scenario_success_upper_bound'] for r in selected)
            axes[3].bar(j, lo, color='#3173a3', edgecolor='#333333', linewidth=.5)
            axes[3].bar(j, hi - lo, bottom=lo, color='white', hatch='...', edgecolor='#777777', linewidth=.5)
        axes[3].set(title='Éxito incluyendo intentos sin solución', ylabel='Fracción de escenarios × intentos previstos', ylim=(0, 1.08))
        axes[3].legend(handles=[Patch(facecolor='#3173a3', label='Éxito observado / previstos'),
                                Patch(facecolor='white', edgecolor='#777777', hatch='...', label='Faltantes o evaluación desconocida')], fontsize=8)
        for axis in axes:
            axis.set_xticks(range(len(methods)), methods, rotation=45, ha='right', fontsize=8)
            axis.grid(axis='y', alpha=.18)
            axis.set_axisbelow(True)
        sizes = sorted({r['n_customers'] for r in rows if r['geometry_group_sha256'] == geometry})
        fig.suptitle(f'Geometría {geometry[:12]} · n={",".join(map(str, sizes))} · {policy} · N=1 geometría; desarrollo descriptivo', fontsize=11)
        fig.tight_layout(rect=(0, 0, 1, .94))
        token = hashlib.sha256(f'{geometry}|{policy}'.encode()).hexdigest()[:12]
        path = directory / f'geometry_{token}_{re.sub(r"[^A-Za-z0-9_-]", "_", policy)}.png'
        for extension in ('png', 'pdf', 'svg'):
            exported = path.with_suffix('.' + extension)
            fig.savefig(exported, dpi=300, bbox_inches='tight')
            files.append(exported.name)
        plt.close(fig)
    # Paired cost plots show each geometry as one point and retain excluded counts.
    paired = tables['paired_geometry_costs']
    for policy in sorted({r['policy'] for r in paired}):
        selected = [r for r in paired if r['policy'] == policy and r['metric'] == 'mean_travel_minutes']
        contrasts = list(dict.fromkeys((r['method'], r['reference_method']) for r in selected))
        fig, panels = plt.subplots(1, 2, figsize=(18, max(5, .55 * len(contrasts))), sharey=True)
        ax = panels[0]
        for j, (method, reference) in enumerate(contrasts):
            rows = [r for r in selected if r['method'] == method and r['reference_method'] == reference]
            values = [r['delta_mean'] for r in rows if r['delta_mean'] is not None]
            for i, value in enumerate(values):
                offset = (i - (len(values) - 1) / 2) * .07
                ax.scatter(value, j + offset, s=38, color='#3173a3', alpha=.85)
            ax.text(1.01, j, f"{len(values)}/{len(rows)} geom.", transform=ax.get_yaxis_transform(), va='center', fontsize=9)
        ax.axvline(0, color='#666666', linewidth=1)
        # Empty contrasts still occupy rows; autoscale from points alone hides them.
        ax.set_ylim(-.6, len(contrasts) - .4)
        ax.set_yticks(range(len(contrasts)), [f'{m} − {r}' for m, r in contrasts])
        ax.set(xlabel='Diferencia de viaje medio (minutos); valores negativos favorecen al primer método',
               title=f'{policy}: costo condicionado a geometrías completas y factibles en ambos métodos')
        ax.grid(axis='x', alpha=.2)
        for j, (method, reference) in enumerate(contrasts):
            rows = [r for r in tables['paired_geometry_success'] if r['policy'] == policy
                    and r['method'] == method and r['reference_method'] == reference]
            values = [r['delta_percentage_points'] for r in rows if r['delta_percentage_points'] is not None]
            for i, value in enumerate(values):
                panels[1].scatter(value, j + (i - (len(values) - 1) / 2) * .07, s=38, color='#3a8f79', alpha=.85)
            panels[1].text(1.01, j, f'{len(values)}/{len(rows)} geom.', transform=panels[1].get_yaxis_transform(), va='center', fontsize=9)
        panels[1].axvline(0, color='#666666', linewidth=1)
        panels[1].set(xlabel='Diferencia de éxito (pp); positivos favorecen al primer método',
                      title='Éxito con todas las semillas; sin solución cuenta como cero')
        panels[1].grid(axis='x', alpha=.2)
        fig.tight_layout()
        for extension in ('png', 'pdf', 'svg'):
            path = directory / f'paired_cost_{re.sub(r"[^A-Za-z0-9_-]", "_", policy)}.{extension}'
            fig.savefig(path, dpi=300, bbox_inches='tight')
            files.append(path.name)
        plt.close(fig)
    return files


Overwriting svrplab/comparison_stats.py


In [46]:
%%writefile svrplab/exact.py
"""Método 01: MIP del surrogate de tiempos conservadores, con Gurobi.

Importar este módulo no importa Gurobi ni solicita licencia. Toda construcción,
optimización y prueba funcional se realiza exclusivamente en Google Colab.
El óptimo del modelo no es un óptimo del problema estocástico; ver EXACT_MODEL.md.
"""
from __future__ import annotations

import math
from pathlib import Path
import time

from .search import OBJECTIVE as OBJECTIVE_SCOPE

_STATUS_NAMES = {
    1: 'LOADED', 2: 'OPTIMAL', 3: 'INFEASIBLE', 4: 'INF_OR_UNBD',
    5: 'UNBOUNDED', 6: 'CUTOFF', 7: 'ITERATION_LIMIT', 8: 'NODE_LIMIT',
    9: 'TIME_LIMIT', 10: 'SOLUTION_LIMIT', 11: 'INTERRUPTED', 12: 'NUMERIC',
    13: 'SUBOPTIMAL', 14: 'INPROGRESS', 15: 'USER_OBJ_LIMIT',
    16: 'WORK_LIMIT', 17: 'MEM_LIMIT',
}


class _BuildBudgetExpired(Exception):
    """Se agotó el presupuesto antes de disponer de un modelo completo."""


def _check_deadline(deadline):
    if deadline is not None and time.perf_counter() >= deadline:
        raise _BuildBudgetExpired


def _gurobi():
    try:
        import gurobipy as gp
    except ImportError:
        raise RuntimeError('Instale gurobipy en Colab para habilitar el método 01.') from None
    if gp.gurobi.version()[0] < 10:
        raise RuntimeError('El método 01 requiere Gurobi >=10; use una versión vigente en Colab.')
    return gp


def create_wls_env(credentials):
    """Inicia un entorno WLS explícito; el llamador debe cerrarlo con dispose().

    Las tres claves vienen de Colab Secrets: WLSACCESSID, WLSSECRET, LICENSEID.
    No se buscan archivos de credenciales ni variables del sistema como fallback,
    no se imprime el diccionario y no se escribe gurobi.lic ni un archivo .prm.
    La inicialización se mide aparte del presupuesto por instancia.
    """
    keys = ('WLSACCESSID', 'WLSSECRET', 'LICENSEID')
    if not hasattr(credentials, 'get') or any(not credentials.get(k) for k in keys):
        raise ValueError('Faltan WLSACCESSID, WLSSECRET o LICENSEID en Colab Secrets.')
    try:
        license_id = int(credentials['LICENSEID'])
        if isinstance(credentials['LICENSEID'], bool) or license_id <= 0:
            raise ValueError
    except (ValueError, TypeError, OverflowError):
        raise ValueError('LICENSEID debe ser un entero positivo.') from None
    if any(not isinstance(credentials[k], str) or not credentials[k].strip() for k in keys[:2]):
        raise ValueError('WLSACCESSID y WLSSECRET deben ser cadenas no vacías.')
    gp = _gurobi()
    env = None
    try:
        env = gp.Env(empty=True)
        env.setParam('OutputFlag', 0)
        env.setParam('LogFile', '')
        env.setParam('CSClientLog', 0)
        env.setParam('WLSACCESSID', credentials['WLSACCESSID'])
        env.setParam('WLSSECRET', credentials['WLSSECRET'])
        env.setParam('LICENSEID', license_id)
        env.start()
        return env
    except Exception:
        if env is not None:
            env.dispose()
        # El texto de una excepción externa puede contener datos de conexión.
        raise RuntimeError('No se inició WLS. Revise licencia, Secrets y conexión de Colab.') from None


def _settings(config, seed):
    config = {} if config is None else config
    seconds = float(config.get('seconds', 120))
    gap = float(config.get('mip_gap', 0.0))
    threads = config.get('threads', 1)
    if not math.isfinite(seconds) or seconds <= 0:
        raise ValueError('seconds debe ser finito y positivo.')
    if not math.isfinite(gap) or gap < 0:
        raise ValueError('mip_gap debe ser finito y no negativo.')
    if isinstance(threads, bool) or not isinstance(threads, int) or threads < 1:
        raise ValueError('threads debe ser un entero positivo.')
    if isinstance(seed, bool) or not isinstance(seed, int) or seed < 0:
        raise ValueError('seed debe ser un entero no negativo.')
    return {'seconds': seconds, 'mip_gap': gap, 'threads': threads,
            'gurobi_seed': seed % 2_000_000_000,
            'log_to_console': bool(config.get('log_to_console', False))}


def prepare_exact_data(problem, deadline=None):
    """Índices y cotas del modelo; no requiere gurobipy ni licencia.

    Sólo se eliminan arcos individualmente imposibles por ventanas/capacidad.
    No usa certificados, escenarios, información futura ni soluciones iniciales.
    """
    n, fleet = problem.n, problem.vehicles
    capacities = tuple(float(c) for c in problem.capacities)
    if n < 1 or fleet < 1 or len(capacities) != fleet:
        raise ValueError('Dimensiones de clientes/flota inválidas.')
    if any(not math.isfinite(c) or c <= 0 for c in capacities):
        raise ValueError('Capacidades inválidas.')
    homogeneous = all(c == capacities[0] for c in capacities)
    max_capacity = max(capacities)
    nodes = range(n + 1)
    arcs = []
    for i in nodes:
        _check_deadline(deadline)
        for j in nodes:
            if i == j:
                continue
            if ((i and problem.demands[i] > max_capacity)
                    or (j and problem.demands[j] > max_capacity)):
                continue
            earliest = 0.0 if i == 0 else float(problem.open[i])
            latest = float(problem.horizon if j == 0 else problem.close[j])
            if earliest + float(problem.travel[i, j]) <= latest:
                arcs.append((i, j))
    incoming = {j: [] for j in nodes}
    outgoing = {i: [] for i in nodes}
    for i, j in arcs:
        incoming[j].append(i)
        outgoing[i].append(j)
    vehicle_arcs = None
    if not homogeneous:
        vehicle_arcs = []
        for k, capacity in enumerate(capacities):
            _check_deadline(deadline)
            vehicle_arcs.extend((k, i, j) for i, j in arcs
                                if problem.demands[i] <= capacity
                                and problem.demands[j] <= capacity)
    return {'n': n, 'fleet': fleet, 'capacities': capacities, 'arcs': arcs,
            'incoming': incoming, 'outgoing': outgoing,
            'vehicle_arcs': vehicle_arcs, 'homogeneous': homogeneous,
            'max_capacity': max_capacity,
            'formulation': 'homogeneous_two_index_mtz' if homogeneous
                           else 'heterogeneous_vehicle_arcs_mtz'}


def build_exact_model(problem, env, config=None, seed=0, deadline=None):
    """Construye, sin optimizar; devuelve (model, arc_variables, data).

    Requiere un entorno explícito. El llamador libera el modelo recibido.
    La etiqueta branch-and-cut corresponde al motor de Gurobi: no se añaden
    cortes VRP personalizados, lazy constraints ni heurísticas con certificados.
    """
    if env is None:
        raise ValueError('Se requiere un entorno Gurobi explícito.')
    settings = _settings(config, seed)
    gp = _gurobi()
    data = prepare_exact_data(problem, deadline)
    n, fleet = data['n'], data['fleet']
    clients, vehicles = range(1, n + 1), range(fleet)
    model = None
    try:
        _check_deadline(deadline)
        model = gp.Model('vrptw_conservative_mean_surrogate', env=env)
        model.Params.OutputFlag = 0
        model.Params.LogFile = ''
        model.Params.LogToConsole = 0
        model.Params.Threads = settings['threads']
        model.Params.Seed = settings['gurobi_seed']
        model.Params.MIPGap = settings['mip_gap']
        model.Params.MIPGapAbs = 0.0
        # La revisión independiente exige 1e-9 en carga y en tiempos.
        model.Params.FeasibilityTol = 1e-9
        model.Params.IntFeasTol = 1e-9
        incoming, outgoing = data['incoming'], data['outgoing']
        if data['homogeneous']:
            arc = model.addVars(data['arcs'], vtype=gp.GRB.BINARY, name='arc')
            active = arc
            for j in clients:
                if j % 64 == 0:
                    _check_deadline(deadline)
                model.addConstr(gp.quicksum(arc[i, j] for i in incoming[j]) == 1,
                                name=f'visit_in[{j}]')
                model.addConstr(gp.quicksum(arc[j, i] for i in outgoing[j]) == 1,
                                name=f'visit_out[{j}]')
            departures = gp.quicksum(arc[0, j] for j in outgoing[0])
            model.addConstr(departures <= fleet, name='fleet')
            model.addConstr(departures == gp.quicksum(arc[i, 0] for i in incoming[0]),
                            name='depot_balance')
            assigned_capacity = {j: data['capacities'][0] for j in clients}
        else:
            arc = model.addVars(data['vehicle_arcs'], vtype=gp.GRB.BINARY, name='arc')
            active = {}
            for position, (i, j) in enumerate(data['arcs']):
                if position % 256 == 0:
                    _check_deadline(deadline)
                active[i, j] = gp.quicksum(arc[k, i, j] for k in vehicles if (k, i, j) in arc)
            visits = {}
            for k in vehicles:
                _check_deadline(deadline)
                departures = gp.quicksum(arc[k, 0, j] for j in outgoing[0]
                                         if (k, 0, j) in arc)
                model.addConstr(departures <= 1, name=f'one_route[{k}]')
                model.addConstr(departures == gp.quicksum(arc[k, i, 0] for i in incoming[0]
                                                         if (k, i, 0) in arc),
                                name=f'depot_balance[{k}]')
                for j in clients:
                    if j % 64 == 0:
                        _check_deadline(deadline)
                    visits[k, j] = gp.quicksum(arc[k, i, j] for i in incoming[j]
                                               if (k, i, j) in arc)
                    model.addConstr(visits[k, j] == gp.quicksum(arc[k, j, i] for i in outgoing[j]
                                                               if (k, j, i) in arc),
                                    name=f'vehicle_flow[{k},{j}]')
            for j in clients:
                model.addConstr(gp.quicksum(visits[k, j] for k in vehicles) == 1,
                                name=f'visit_once[{j}]')
            assigned_capacity = {j: gp.quicksum(data['capacities'][k] * visits[k, j]
                                                for k in vehicles) for j in clients}
        _check_deadline(deadline)
        arrival = model.addVars(clients, lb={j: float(problem.open[j]) for j in clients},
                                ub={j: float(problem.close[j]) for j in clients}, name='service')
        load = model.addVars(clients, lb={j: float(problem.demands[j]) for j in clients},
                             ub=data['max_capacity'], name='load')
        order = model.addVars(clients, lb=1.0, ub=float(n), name='order')
        for j in clients:
            model.addConstr(load[j] <= assigned_capacity[j], name=f'capacity[{j}]')
        for position, (i, j) in enumerate(data['arcs']):
            if position % 256 == 0:
                _check_deadline(deadline)
            used = active[i, j]
            travel = float(problem.travel[i, j])
            if i == 0:
                model.addConstr(arrival[j] >= travel * used, name=f'depart_at_zero[{j}]')
            elif j == 0:
                big_m = max(0.0, float(problem.close[i]) + travel - float(problem.horizon))
                model.addConstr(arrival[i] + travel <= float(problem.horizon) + big_m * (1 - used),
                                name=f'return[{i}]')
            else:
                big_m = max(0.0, float(problem.close[i]) + travel - float(problem.open[j]))
                model.addConstr(arrival[j] >= arrival[i] + travel - big_m * (1 - used),
                                name=f'time[{i},{j}]')
                model.addConstr(load[j] >= load[i] + float(problem.demands[j])
                                - data['max_capacity'] * (1 - used), name=f'load[{i},{j}]')
                # Orden independiente de demanda y duración: cubre ambas en cero.
                model.addConstr(order[j] >= order[i] + 1 - n * (1 - used),
                                name=f'no_subtour[{i},{j}]')
        _check_deadline(deadline)
        model.setObjective(gp.quicksum(float(problem.travel[i, j]) * active[i, j]
                                      for i, j in data['arcs']), gp.GRB.MINIMIZE)
        model.update()
        _check_deadline(deadline)
        return model, arc, data
    except BaseException:
        if model is not None:
            model.dispose()
        raise


def _extract_routes(selected, data):
    """Reconstruye y rechaza subtours, duplicados o clientes ausentes."""
    routes = [[] for _ in range(data['fleet'])]
    covered = set()

    def walk(edges, first):
        following = {}
        for i, j in edges:
            if i == 0:
                continue
            if i in following:
                raise ValueError('Incumbente con salidas duplicadas.')
            following[i] = j
        route, j = [], first
        while j:
            if j in covered or j not in following:
                raise ValueError('Incumbente contiene un ciclo o una ruta abierta.')
            covered.add(j)
            route.append(j)
            j = following[j]
        return route

    if data['homogeneous']:
        edges = list(selected)
        firsts = sorted(j for i, j in edges if i == 0)
        if len(firsts) > data['fleet']:
            raise ValueError('Incumbente excede la flota.')
        for k, first in enumerate(firsts):
            routes[k] = walk(edges, first)
    else:
        for k in range(data['fleet']):
            edges = [(i, j) for vehicle, i, j in selected if vehicle == k]
            firsts = [j for i, j in edges if i == 0]
            if len(firsts) > 1:
                raise ValueError('Incumbente usa un vehículo más de una vez.')
            if firsts:
                routes[k] = walk(edges, firsts[0])
            elif edges:
                raise ValueError('Incumbente contiene un subciclo sin depósito.')
    if covered != set(range(1, data['n'] + 1)):
        raise ValueError('Incumbente no cubre exactamente todos los clientes.')
    return routes


def _finite(value):
    if value is None:
        return None
    value = float(value)
    return value if math.isfinite(value) and abs(value) < 1e100 else None


def _safe_attr(model, name):
    try:
        return _finite(getattr(model, name))
    except Exception:
        # GurobiError(DATA_NOT_AVAILABLE) no hereda de RuntimeError.
        return None


def _safe_log_line(message):
    # Sólo se exporta el progreso del optimizador, sin líneas de licenciamiento.
    upper = message.upper()
    return not any(word in upper for word in ('WLS', 'LICENSE', 'ACCESSID', 'SECRET',
                                               'TOKEN', 'USERNAME', 'USER NAME'))


def solve_exact(problem, seed, config, env=None, output_dir=None):
    """Optimiza dentro del presupuesto (incluye construcción) y exporta auditoría.

    Sin entorno devuelve license_pending. Nunca crea una licencia limitada como
    fallback. Los artefactos y la evaluación común se cronometran por separado.
    """
    started = time.perf_counter()
    settings = _settings(config, seed)
    metadata = {'solver': 'Gurobi', 'objective_scope': OBJECTIVE_SCOPE,
                'surrogate_optimal': False, 'stochastic_optimal': False,
                'incumbent_objective': None, 'best_bound': None,
                'relative_gap': None, 'nodes': None, 'solver_runtime_seconds': None,
                'build_seconds': None, 'gurobi_seed': settings['gurobi_seed'],
                'requested_seconds': settings['seconds'], 'threads': settings['threads'],
                'requested_mip_gap': settings['mip_gap'], 'warm_start_used': False,
                'budget_termination_requested': False}
    if env is None:
        return problem.result(None, 'license_pending', time.perf_counter() - started, [], **metadata)
    deadline = started + settings['seconds']
    model, routes, history, log = None, None, [], []
    status = 'not_started'
    try:
        try:
            model, arc, data = build_exact_model(problem, env, config, seed, deadline)
        except _BuildBudgetExpired:
            metadata['build_seconds'] = time.perf_counter() - started
            return problem.result(None, 'time_limit_build', metadata['build_seconds'], [], **metadata)
        metadata.update(formulation=data['formulation'], gurobi_version=list(_gurobi().gurobi.version()),
                        model_variables=int(model.NumVars), model_constraints=int(model.NumConstrs),
                        build_seconds=time.perf_counter() - started)
        remaining = deadline - time.perf_counter()
        if remaining <= 0:
            return problem.result(None, 'time_limit_build', time.perf_counter() - started, [], **metadata)
        gp = _gurobi()
        last_recorded = -math.inf

        def progress(current, where):
            nonlocal last_recorded
            elapsed = time.perf_counter() - started
            if where == gp.GRB.Callback.MESSAGE:
                message = current.cbGet(gp.GRB.Callback.MSG_STRING)
                if _safe_log_line(message):
                    log.append(message)
                    if settings['log_to_console']:
                        print(message, end='')
            elif where == gp.GRB.Callback.MIP and elapsed - last_recorded >= 1.0:
                incumbent = _finite(current.cbGet(gp.GRB.Callback.MIP_OBJBST))
                history.append({'elapsed_seconds': elapsed, 'event': 'mip_progress',
                                'incumbent_objective': incumbent,
                                'best_bound': _finite(current.cbGet(gp.GRB.Callback.MIP_OBJBND)),
                                'nodes': _finite(current.cbGet(gp.GRB.Callback.MIP_NODCNT))})
                last_recorded = elapsed
            if elapsed >= settings['seconds'] and where in (gp.GRB.Callback.MIP, gp.GRB.Callback.MIPSOL):
                metadata['budget_termination_requested'] = True
                current.terminate()

        # OutputFlag permite callbacks de mensajes; no se habilita log nativo.
        model.Params.OutputFlag = 1
        model.Params.TimeLimit = max(0.0, deadline - time.perf_counter())
        metadata['optimizer_time_limit_seconds'] = float(model.Params.TimeLimit)
        model.optimize(progress)
        elapsed = time.perf_counter() - started
        code = int(model.Status)
        solver_status = _STATUS_NAMES.get(code, f'UNKNOWN_{code}')
        status = solver_status.lower()
        has_incumbent = int(model.SolCount) > 0
        metadata.update(solver_status_code=code, solver_status=solver_status,
                        solution_count=int(model.SolCount),
                        incumbent_objective=_safe_attr(model, 'ObjVal') if has_incumbent else None,
                        best_bound=_safe_attr(model, 'ObjBound'),
                        relative_gap=_safe_attr(model, 'MIPGap') if has_incumbent else None,
                        nodes=_safe_attr(model, 'NodeCount'),
                        solver_runtime_seconds=_safe_attr(model, 'Runtime'))
        if has_incumbent:
            try:
                selected = [key for key, var in arc.items() if var.X > .5]
                candidate = _extract_routes(selected, data)
                violation, travel = problem.score(candidate)
                if violation > 1e-9:
                    status = 'incumbent_rejected_validation'
                    metadata['rejected_violation_minutes'] = float(violation)
                elif metadata['incumbent_objective'] is None or not math.isclose(
                        travel, metadata['incumbent_objective'], rel_tol=1e-8, abs_tol=1e-6):
                    status = 'incumbent_rejected_objective'
                else:
                    routes = candidate
                    metadata['surrogate_optimal'] = (code == gp.GRB.OPTIMAL
                                                       and settings['mip_gap'] == 0.0)
                    metadata['solver_optimal_with_requested_gap'] = code == gp.GRB.OPTIMAL
            except ValueError:
                status = 'incumbent_rejected_validation'
        history.append({'elapsed_seconds': elapsed, 'event': 'terminated',
                        'status': status, 'incumbent_objective': metadata['incumbent_objective'],
                        'best_bound': metadata['best_bound'], 'relative_gap': metadata['relative_gap'],
                        'nodes': metadata['nodes']})
        metadata['artifact_seconds'] = 0.0
        if output_dir is not None:
            export_started = time.perf_counter()
            directory = Path(output_dir)
            directory.mkdir(parents=True, exist_ok=True)
            # LP/SOL contienen sólo el modelo y sus variables, nunca parámetros.
            model.write(str(directory / 'model.lp.gz'))
            if has_incumbent:
                model.write(str(directory / 'incumbent.sol'))
            (directory / 'gurobi.log').write_text(''.join(log), encoding='utf-8')
            metadata['artifact_seconds'] = time.perf_counter() - export_started
        return problem.result(routes, status, elapsed, history, **metadata)
    finally:
        if model is not None:
            model.dispose()


Overwriting svrplab/exact.py


In [47]:
%%writefile svrplab/experiments.py
"""Ejecución Colab, evaluación única y checkpoints inmutables para métodos 01–06.

No modifica el banco ni entrega certificados a los algoritmos. Cada resultado
se reevalúa desde sus rutas persistidas con el pilot.py guardado en el banco.
"""
from __future__ import annotations

import csv
import json
import math
import shutil
import time
import uuid
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from . import pilot as p
from .search import OBJECTIVE, SearchProblem

SCHEMA = 'methods-v2'
SUMMARY_METRICS = [
    'scenarios', 'alpha', 'mean_travel_minutes', 'cvar_travel_minutes',
    'cost_variance_population', 'std_travel_minutes', 'scenario_strict_feasibility',
    'mean_customer_violation_fraction', 'mean_lateness_minutes',
    'mean_depot_overtime_minutes', 'mean_waiting_minutes', 'mean_base_minutes',
    'mean_net_execution_delta_minutes', 'mean_late_customers', 'mean_recourse_reorders',
    'used_vehicles', 'evaluation_seconds', 'trace_validation_seconds',
]


def write_csv(path, rows, fields=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(fields or dict.fromkeys(k for row in rows for k in row))
    with path.open('w', newline='', encoding='utf-8') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def native(value):
    """Normaliza escalares del evaluador sellado sin modificar su implementación."""
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return [native(item) for item in value.tolist()]
    if isinstance(value, dict):
        return {key: native(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [native(item) for item in value]
    return value


def write_json(path, value):
    p.write_json(path, native(value))


def read_json(path):
    return json.loads(Path(path).read_text())


def validate_scenarios(scenarios, config):
    if set(scenarios) != set(config['scenarios']):
        raise ValueError('Splits del banco incompatibles.')
    seen = set()
    for split, count in config['scenarios'].items():
        seeds = scenarios[split]
        if (len(seeds) != count or len(set(seeds)) != count
                or any(isinstance(s, bool) or not isinstance(s, int) or s < 0 for s in seeds)
                or seen.intersection(seeds)):
            raise ValueError('Escenarios repetidos, solapados o con tamaño/tipo incorrecto.')
        seen.update(seeds)


def validate_config(config):
    if (config['schema_version'] != SCHEMA or config['role'] != 'development_only'
            or config['objective'] != OBJECTIVE or config['scenario_split'] != 'evaluation'
            or config['export_all_scenario_events'] is not True):
        raise ValueError('Contrato experimental incompatible.')
    if not math.isfinite(config['search_seconds']) or config['search_seconds'] <= 0:
        raise ValueError('Presupuesto debe ser positivo y finito.')
    seeds = config['seeds']
    if not seeds or len(set(seeds)) != len(seeds) or any(type(s) is not int or s < 0 for s in seeds):
        raise ValueError('Semillas de métodos inválidas.')
    if type(config['time_scale']) is not int or config['time_scale'] < 1:
        raise ValueError('Escala de tiempo inválida.')


def route_rows(x, routes, context):
    rows = []
    for vehicle, route in enumerate(routes):
        load = 0.0
        for position, node in enumerate([0, *route, 0]):
            load += float(x.demands[node])
            rows.append({**context, 'vehicle': vehicle, 'position': position,
                         'node': node, 'empty_vehicle': not bool(route),
                         'x': float(x.locations[node, 0]), 'y': float(x.locations[node, 1]),
                         'demand': float(x.demands[node]), 'cumulative_load': load,
                         'capacity': float(x.capacities[vehicle]),
                         'window_open': float(x.windows[node, 0]),
                         'window_close': float(x.windows[node, 1]),
                         'appear_time': float(x.appear_times[node])})
    return rows


def validate_trace(x, routes, trace, raw):
    for key, value in raw.items():
        actual = trace[key]
        if isinstance(value, float):
            if not math.isclose(actual, value, rel_tol=1e-12, abs_tol=1e-9):
                raise AssertionError(f'Traza distinta del evaluador: {key}')
        elif actual != value:
            raise AssertionError(f'Traza distinta del evaluador: {key}')
    p.validate_routes(x, trace['realized_routes'])
    if any(sorted(a) != sorted(b) for a, b in zip(routes, trace['realized_routes'])):
        raise AssertionError('El recurso cambió la asignación de vehículo.')
    events = trace['events']
    expected = x.n + sum(bool(r) for r in routes)
    if len(events) != expected:
        raise AssertionError('Eventos incompletos.')
    if any(type(e['vehicle']) is not int or not 0 <= e['vehicle'] < len(routes) for e in events):
        raise AssertionError('Vehículo inválido en eventos.')
    for vehicle, realized in enumerate(trace['realized_routes']):
        vehicle_events = [e for e in events if e['vehicle'] == vehicle]
        path = [0, *realized, 0] if realized else [0]
        if [(e['from'], e['to']) for e in vehicle_events] != list(zip(path, path[1:])):
            raise AssertionError('Arcos distintos de la ruta realizada.')
        previous = 0.0
        for event in vehicle_events:
            node = event['to']
            start = max(event['arrival'], x.windows[node, 0], x.appear_times[node]) if node else event['arrival']
            expected_late = max(0.0, start - x.windows[node, 1])
            if (not all(math.isfinite(event[k]) for k in ('departure', 'arrival', 'service_start', 'late'))
                    or not math.isclose(event['departure'], previous, abs_tol=1e-9)
                    or event['arrival'] < event['departure'] - 1e-9
                    or not math.isclose(event['service_start'], start, abs_tol=1e-9)
                    or not math.isclose(event['late'], expected_late, abs_tol=1e-9)):
                raise AssertionError('Horario de eventos incoherente.')
            previous = event['service_start']
    travel = sum(e['arrival'] - e['departure'] for e in events)
    waiting = sum(e['service_start'] - e['arrival'] for e in events)
    late = sum(e['late'] for e in events if e['to'] != 0)
    overtime = sum(e['late'] for e in events if e['to'] == 0)
    for value, key in [(travel, 'travel_minutes'), (waiting, 'waiting_minutes'),
                       (late, 'lateness_minutes'), (overtime, 'depot_overtime_minutes')]:
        if not math.isclose(value, raw[key], rel_tol=1e-10, abs_tol=1e-7):
            raise AssertionError(f'Eventos no reconcilian {key}.')


def evaluate_solution(x, solution, seeds, spec, policy, alpha, directory, context):
    """Nunca selecciona/tunea con evaluation; puntúa una solución ya congelada."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    write_json(directory / 'environment.json', p.environment_report())
    routes = solution['routes']
    before = x.sha
    p.validate_routes(x, routes)
    summary, records = p.evaluate(x, routes, seeds, spec, policy, alpha)
    if [r['scenario_seed'] for r in records] != seeds:
        raise AssertionError('El evaluador cambió el conjunto u orden de escenarios.')
    trace_started = time.perf_counter()
    event_count = route_count = 0
    # Streaming: no acumular millones de eventos en RAM para n=300.
    with (directory / 'events.csv').open('w', newline='', encoding='utf-8') as ef, \
            (directory / 'realized_routes.csv').open('w', newline='', encoding='utf-8') as rf:
        ew = rw = None
        for record in records:
            seed = record['scenario_seed']
            trace = p.simulate(x, routes, seed, spec, policy, trace=True)
            validate_trace(x, routes, trace, record)
            trace_context = {**context, 'policy': policy, 'scenario_seed': str(seed)}
            for event_index, event in enumerate(trace['events']):
                row = {**trace_context, 'event_index': event_index, **event}
                if ew is None:
                    ew = csv.DictWriter(ef, fieldnames=list(row)); ew.writeheader()
                ew.writerow(row)
                event_count += 1
            for row in route_rows(x, trace['realized_routes'], trace_context):
                if rw is None:
                    rw = csv.DictWriter(rf, fieldnames=list(row)); rw.writeheader()
                rw.writerow(row)
                route_count += 1
    summary['trace_validation_seconds'] = time.perf_counter() - trace_started
    costs = [r['travel_minutes'] for r in records]
    for source in ['waiting_minutes', 'base_minutes', 'net_execution_delta_minutes',
                   'late_customers', 'recourse_reorders']:
        summary['mean_' + source] = float(np.mean([r[source] for r in records]))
    summary['std_travel_minutes'] = float(np.std(costs, ddof=0))
    # CVaR por integración de masa en la cola: control independiente de la fórmula de pilot.
    descending = sorted(costs, reverse=True)
    mass = len(costs) * (1 - alpha)
    whole = min(len(costs), int(math.floor(mass)))
    tail_sum = sum(descending[:whole])
    if whole < len(costs):
        tail_sum += (mass - whole) * descending[whole]
    if not math.isclose(tail_sum / mass, summary['cvar_travel_minutes'], rel_tol=1e-10):
        raise AssertionError('CVaR no coincide con integración de la cola.')
    if x.sha != before:
        raise AssertionError('La instancia cambió durante evaluación.')
    raw_rows = [{**context, 'policy': policy, **r, 'scenario_seed': str(r['scenario_seed'])}
                for r in records]
    write_csv(directory / 'scenario_metrics.csv', raw_rows)
    write_json(directory / 'summary.json', {**context, **summary})
    qc = {'status': 'passed', 'structural_valid': True, 'scenarios': len(records),
          'event_rows': event_count, 'realized_route_rows': route_count,
          'expected_event_rows': len(seeds) * (x.n + solution['nominal']['used_vehicles']),
          'expected_realized_route_rows': len(seeds) * (x.n + 2 * len(x.capacities)),
          'instance_unchanged': x.sha == before, 'trace_matches_evaluator': True,
          'recourse_preserves_vehicle_assignment': True, 'cvar_independently_checked': True,
          'nominal_feasible': solution['nominal']['strict_feasible'],
          'stochastic_feasibility_is_a_measurement': True}
    if qc['event_rows'] != qc['expected_event_rows'] or qc['realized_route_rows'] != qc['expected_realized_route_rows']:
        raise AssertionError('Número de filas de exportación incorrecto.')
    # Releer CSVs comprueba la persistencia, las semillas de 64 bits y el número de filas.
    for name, expected_rows in [('events.csv', event_count), ('realized_routes.csv', route_count),
                                ('scenario_metrics.csv', len(seeds))]:
        with (directory / name).open(newline='', encoding='utf-8') as stream:
            rows = csv.DictReader(stream)
            count = 0
            observed = set()
            for row in rows:
                observed.add(int(row['scenario_seed'])); count += 1
            if count != expected_rows or observed != set(seeds):
                raise AssertionError(f'Exportación incompleta: {name}')
    write_json(directory / 'validation.json', qc)
    return {**context, **summary}


def save_plot(x, routes, path, title):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 6))
    for vehicle, route in enumerate(routes):
        if not route:
            continue
        coords = x.locations[[0, *route, 0]]
        ax.plot(coords[:, 0], coords[:, 1], '-o', markersize=2, linewidth=.8,
                label=f'V{vehicle}')
    ax.scatter(*x.locations[0], marker='*', s=160, color='black', zorder=5)
    ax.set(title=title, xlabel='x (unidad fuente)', ylabel='y (unidad fuente)', aspect='equal')
    if sum(bool(r) for r in routes) <= 15:
        ax.legend(fontsize=6, ncol=3)
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


def _commit(directory, remote):
    p.seal_bundle(directory)
    _sync(directory, remote)


def _sync(local, remote):
    """Conservar staging interrumpido; promoverlo sólo si su hash es idéntico."""
    staging = remote.with_name(remote.name + '.partial')
    if not remote.exists() and staging.exists():
        try:
            complete = p.verify_bundle(staging) == p.verify_bundle(local)
        except (ValueError, OSError, json.JSONDecodeError):
            complete = False
        if complete:
            staging.rename(remote)
        else:
            staging.rename(staging.with_name(staging.name + '.interrupted_' + uuid.uuid4().hex[:8]))
    p.sync_verified(local, remote)


def _recover(local, remote):
    if (local / 'CHECKSUMS.json').exists():
        p.verify_bundle(local)
        _sync(local, remote)
        return True
    if remote.exists():
        # Conservar la escritura local incompleta, sin sobrescribirla ni mezclarla.
        if local.exists():
            local.rename(local.with_name(local.name + '.interrupted_' + uuid.uuid4().hex[:8]))
        _sync(remote, local)
        return True
    if local.exists():
        local.rename(local.with_name(local.name + '.interrupted_' + uuid.uuid4().hex[:8]))
    return False


def comparison_protocol(bank_hash, evaluator_hash, bank_config, config):
    """Mismo problema/evaluación/presupuesto; algoritmos y entrenamiento pueden diferir."""
    return {'schema_version': SCHEMA, 'bank_manifest_sha256': bank_hash,
            'evaluator_sha256': evaluator_hash,
            'runner_sha256': p.file_hash(__file__),
            'search_contract_sha256': p.file_hash(Path(__file__).with_name('search.py')),
            'objective': config['objective'], 'time_scale': config['time_scale'],
            'role': config['role'], 'solver_seeds': config['seeds'],
            'search_seconds': config['search_seconds'], 'scenario_split': config['scenario_split'],
            'policies': bank_config['policies'], 'alpha': bank_config['alpha_cvar'],
            'export_all_scenario_events': config['export_all_scenario_events']}


def provenance_columns(provenance):
    fields = ['training_id', 'model_checkpoint_sha256', 'fold_id', 'training_seed',
              'train_geometry_groups', 'validation_geometry_groups', 'heldout_geometry_groups',
              'training_seconds', 'validation_seconds', 'teacher_seconds', 'offline_cost_id',
              'offline_manifest_sha256', 'checkpoint_selection', 'training_scope']
    result = {key: provenance.get(key) for key in fields}
    for key in ['training_seconds', 'validation_seconds', 'teacher_seconds']:
        result[key] = provenance.get(key, 0.0)
        if isinstance(result[key], bool) or not isinstance(result[key], (float, int)) or not math.isfinite(result[key]) or result[key] < 0:
            raise ValueError(f'Costo offline inválido: {key}')
    for key, value in result.items():
        if isinstance(value, (dict, list, tuple)):
            result[key] = json.dumps(value, sort_keys=True, separators=(',', ':'))
    return result


def validate_learned_provenance(provenance, x, seed):
    groups = [provenance.get(role) for role in
              ['train_geometry_groups', 'validation_geometry_groups', 'heldout_geometry_groups']]
    if any(not isinstance(group, list) or not group for group in groups):
        raise ValueError('Faltan grupos de entrenamiento/validación/heldout del modelo.')
    train, validation, heldout = map(set, groups)
    if (train & validation or train & heldout or validation & heldout
            or x.provenance['geometry_group_sha256'] not in heldout
            or provenance.get('training_seed') != seed):
        raise ValueError('Procedencia del modelo tiene fuga de geometría o semilla distinta.')
    checkpoint = provenance.get('model_checkpoint_sha256', '')
    if not provenance.get('training_id') or len(checkpoint) != 64 or any(c not in '0123456789abcdef' for c in checkpoint):
        raise ValueError('Falta identidad/hash de checkpoint del modelo aprendido.')


def run_methods(bank, config, solvers, work_root, drive_root, run_id=None, validation_artifacts=(),
                method_availability=None, method_provenance=None):
    """solvers: {method: callable(problem, seed, config, output_dir)}.

    Rerun con run_id explícito recupera checkpoints; sin ID crea otra réplica
    trazable. La comparación usa comparison_id, sin confundir réplicas con N.
    """
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Experimentación exclusivamente en Google Colab.') from exc
    validate_config(config)
    bank = Path(bank)
    instances = sorted(p.load_bank(bank), key=lambda x: x.n)
    bank_config = read_json(bank / 'config.json')
    bank_hash = p.verify_bundle(bank)
    spec = p.Traffic(**bank_config['traffic'])
    nominal_spec = p.Traffic(**(asdict(spec) | {'mode': 'nominal_mean'}))
    scenarios = {x.n: read_json(bank / f'scenarios/n{x.n:04d}.json') for x in instances}
    for value in scenarios.values():
        validate_scenarios(value, bank_config)
    if not solvers or not set(solvers).issubset(config['methods']):
        raise ValueError('Métodos solicitados incompatibles con configuración.')
    method_availability = method_availability or {name: 'enabled' for name in solvers}
    if set(method_availability) != set(solvers) or not set(method_availability.values()).issubset({'enabled', 'license_pending'}):
        raise ValueError('Disponibilidad de métodos inválida.')
    package = Path(__file__).parent
    source_hashes = {f.name: p.file_hash(f) for f in sorted(package.glob('*.py'))}
    method_provenance = method_provenance or {}
    protocol = comparison_protocol(bank_hash, p.file_hash(p.__file__), bank_config, config)
    comparison_id = p.digest(protocol)
    signature = p.digest({'protocol': protocol, 'config': config,
                          'methods': sorted(solvers), 'source_hashes': source_hashes,
                          'method_availability': method_availability, 'method_provenance': method_provenance})
    run_id = run_id or datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ_') + uuid.uuid4().hex[:8]
    if Path(run_id).name != run_id or run_id in ('.', '..'):
        raise ValueError('run_id debe ser un nombre de carpeta simple.')
    root = Path(work_root) / 'method_runs' / bank.name / run_id
    remote = Path(drive_root) / 'method_runs' / bank.name / run_id
    print(f'RUN_ID = {run_id}\nDrive: {remote}', flush=True)
    if _recover(root / 'meta', remote / 'meta'):
        if read_json(root / 'meta/plan.json')['signature'] != signature:
            raise ValueError('No se puede reanudar con otro banco/código/protocolo/configuración/disponibilidad de licencia. Use RUN_ID nuevo.')
    else:
        meta = root / 'meta'
        meta.mkdir(parents=True)
        write_json(meta / 'plan.json', {'run_id': run_id, 'signature': signature,
                     'comparison_id': comparison_id, 'protocol': protocol, 'config': config,
                     'methods': sorted(solvers), 'source_hashes': source_hashes,
                     'method_availability': method_availability,
                     'method_provenance': method_provenance,
                     'bank_id': bank.name, 'role': 'development_only',
                     'created_utc': datetime.now(timezone.utc).isoformat()})
        write_json(meta / 'environment.json', p.environment_report())
        for source in package.glob('*.py'):
            destination = meta / 'code' / source.name
            destination.parent.mkdir(exist_ok=True)
            shutil.copy2(source, destination)
        for source in map(Path, validation_artifacts):
            destination = meta / 'validation' / source.name
            destination.parent.mkdir(exist_ok=True)
            shutil.copy2(source, destination)
        write_json(meta / 'metrics_schema.json', metric_schema())
        _commit(meta, remote / 'meta')
    if _recover(root / 'report', remote / 'report'):
        for relative, expected in read_json(root / 'report/bundles.json').items():
            if p.verify_bundle(remote / relative) != expected:
                raise ValueError(f'Checkpoint remoto modificado: {relative}')
            if not _recover(root / relative, remote / relative):
                raise ValueError(f'Checkpoint ausente: {relative}')
        print('Reporte completo recuperado y verificado.', flush=True)
        return root
    summaries, failures = [], []
    for x in instances:
        for method, solver in solvers.items():
            for seed in config['seeds']:
                key = f'{method}/n{x.n:04d}/seed_{seed}'
                job, remote_job = root / key, remote / key
                learned = method_provenance.get(method, {}).get(f'{x.sha}:{seed}', {})
                if method in ('am_supervised', 'pomo', 'hbg_facs', 'hbg_no_pheromone') and not learned:
                    raise ValueError('Falta procedencia del modelo entrenado para este intento.')
                if method in ('am_supervised', 'pomo', 'hbg_facs', 'hbg_no_pheromone'):
                    validate_learned_provenance(learned, x, seed)
                context = {'schema_version': SCHEMA, 'comparison_id': comparison_id,
                    'run_id': run_id, 'bank_id': bank.name, 'bank_manifest_sha256': bank_hash,
                    'evaluator_sha256': p.file_hash(p.__file__), 'method': method,
                    'solver_seed': seed, 'instance_id': x.instance_id, 'instance_sha256': x.sha,
                    'geometry_group_sha256': x.provenance['geometry_group_sha256'],
                    'n_customers': x.n, 'objective_scope': OBJECTIVE,
                    'search_budget_seconds': config['search_seconds'],
                    'scenario_set_sha256': p.digest(scenarios[x.n]['evaluation']),
                    **provenance_columns(learned)}
                print(f'{key}: búsqueda / recuperación...', flush=True)
                solution_dir = job / 'solution'
                if not _recover(solution_dir, remote_job / 'solution'):
                    solution_dir.mkdir(parents=True)
                    started = time.perf_counter()
                    try:
                        problem = SearchProblem(x, spec, config['time_scale'])
                        preparation = time.perf_counter() - started
                        settings = {**config['methods'][method],
                                    'seconds': max(0.0, config['search_seconds'] - preparation)}
                        if settings['seconds'] <= 0:
                            solution = problem.result(None, 'time_limit_no_solution', preparation, [])
                        else:
                            solution = solver(problem, seed, settings, solution_dir)
                        solution['search_total_seconds'] = time.perf_counter() - started
                        solution['preparation_seconds'] = preparation
                        solution['nominal'] = None
                        if solution['routes'] is not None:
                            p.validate_routes(x, solution['routes'])
                            conservative = problem.score(solution['routes'])
                            solution['metadata'].update(search_violation_minutes=conservative[0],
                                search_objective_minutes=conservative[1],
                                conservative_feasible=conservative[0] <= 1e-9)
                            solution['nominal'] = p.simulate(x, solution['routes'], 0, nominal_spec,
                                                             'fixed_order', trace=True)
                            if conservative[0] <= 1e-9 and not solution['nominal']['strict_feasible']:
                                raise AssertionError('Solución conservadora viola el simulador nominal.')
                    except Exception as exc:
                        solution = {'routes': None, 'nominal': None, 'status': 'solver_error',
                                    'solve_seconds': time.perf_counter() - started,
                                    'search_total_seconds': time.perf_counter() - started,
                                    'history': [], 'metadata': {'error_type': type(exc).__name__,
                                    'error': str(exc), 'objective_scope': OBJECTIVE}}
                    write_json(solution_dir / 'solution.json', solution)
                    write_json(solution_dir / 'environment.json', p.environment_report())
                    # Read-back antes de evaluar: la solución persistida es la autoridad.
                    solution = read_json(solution_dir / 'solution.json')
                    write_json(solution_dir / 'identity.json', context)
                    write_csv(solution_dir / 'search_history.csv', solution['history'],
                              None if solution['history'] else ['iteration', 'elapsed_seconds'])
                    if solution['routes'] is not None:
                        write_csv(solution_dir / 'planned_routes.csv', route_rows(x, solution['routes'], context))
                        write_csv(solution_dir / 'nominal_events.csv',
                                  [{**context, **e} for e in solution['nominal']['events']])
                        save_plot(x, solution['routes'], solution_dir / 'routes.png',
                                  f'{method} · n={x.n} · seed={seed} · {solution["status"]}')
                    _commit(solution_dir, remote_job / 'solution')
                solution = read_json(solution_dir / 'solution.json')
                if read_json(solution_dir / 'identity.json') != context:
                    raise ValueError('Checkpoint de solución corresponde a otro intento.')
                meta = solution['metadata']
                base = {**context, 'solver_status': solution['status'],
                        'structural_valid': solution['routes'] is not None,
                        'nominal_feasible': solution['nominal']['strict_feasible'] if solution['nominal'] else None,
                        'conservative_feasible': meta.get('conservative_feasible'),
                        'search_violation_minutes': meta.get('search_violation_minutes'),
                        'search_objective_minutes': meta.get('search_objective_minutes'),
                        'solve_seconds': solution['solve_seconds'],
                        'search_total_seconds': solution['search_total_seconds'],
                        'best_bound': meta.get('best_bound'), 'relative_gap': meta.get('relative_gap'),
                        'surrogate_optimal': meta.get('surrogate_optimal'),
                        'model_load_seconds': meta.get('model_load_seconds', 0.0),
                        'tuning_seconds_this_run': 0.0}
                for policy in bank_config['policies']:
                    result_dir = job / policy
                    if solution['routes'] is None:
                        summary = {**base, 'policy': policy, 'evaluation_status': 'no_solution',
                                   **dict.fromkeys(SUMMARY_METRICS)}
                        failures.append({**context, 'policy': policy, 'status': solution['status'],
                                         'error': meta.get('error', '')})
                    else:
                        print(f'{key}: {policy}, {len(scenarios[x.n]["evaluation"])} escenarios...', flush=True)
                        if not _recover(result_dir, remote_job / policy):
                            evaluate_solution(x, solution, scenarios[x.n]['evaluation'], spec, policy,
                                              bank_config['alpha_cvar'], result_dir, context)
                            _commit(result_dir, remote_job / policy)
                        evaluated = read_json(result_dir / 'summary.json')
                        if any(evaluated.get(k) != v for k, v in context.items()) or evaluated.get('policy') != policy:
                            raise ValueError('Checkpoint de evaluación corresponde a otro intento o política.')
                        summary = {**base, **evaluated,
                                   'evaluation_status': 'validated'}
                    summaries.append(summary)
    if p.verify_bundle(bank) != bank_hash:
        raise AssertionError('El banco fue modificado.')
    report = root / 'report'
    report.mkdir(parents=True, exist_ok=True)
    write_csv(report / 'summary.csv', summaries)
    write_csv(report / 'failures.csv', failures, None if failures else ['method', 'n_customers', 'solver_seed', 'status'])
    write_json(report / 'summary.json', summaries)
    inventory = {'meta': p.verify_bundle(root / 'meta')}
    for x in instances:
        for method in solvers:
            for seed in config['seeds']:
                key = f'{method}/n{x.n:04d}/seed_{seed}'
                for part in ['solution', *bank_config['policies']]:
                    relative = f'{key}/{part}'
                    if (root / relative / 'CHECKSUMS.json').exists():
                        inventory[relative] = p.verify_bundle(root / relative)
                        if p.verify_bundle(remote / relative) != inventory[relative]:
                            raise ValueError(f'Copia Drive distinta: {relative}')
    write_json(report / 'bundles.json', inventory)
    save_summary_plot(summaries, report / 'metrics.png', bank_config['policies'])
    write_json(report / 'validation.json', {'status': 'completed', 'bank_unchanged': True,
        'attempts': len(instances) * len(solvers) * len(config['seeds']),
        'summary_rows': len(summaries), 'unsolved_policy_rows': len(failures),
        'not_a_claim_of_all_routes_feasible': True, 'ready_for_confirmatory_comparison': False})
    _commit(report, remote / 'report')
    print(f'COMPLETADO: exportaciones verificadas en {remote}', flush=True)
    return root


def save_summary_plot(summaries, path, policies):
    import matplotlib.pyplot as plt
    rows = [r for r in summaries if r['evaluation_status'] == 'validated']
    if not rows:
        return
    fig, axes = plt.subplots(len(policies), 2, figsize=(12, 4 * len(policies)), squeeze=False)
    for row, policy in enumerate(policies):
        for method in sorted({r['method'] for r in rows}):
            data = [r for r in rows if r['policy'] == policy and r['method'] == method]
            for column, metric in enumerate(['scenario_strict_feasibility', 'mean_travel_minutes']):
                axes[row, column].scatter([r['n_customers'] for r in data],
                                         [r[metric] for r in data], label=method, alpha=.65)
                axes[row, column].set(xlabel='Clientes (una geometría por tamaño)', ylabel=metric,
                                      title=f'{policy} · todas las semillas')
                axes[row, column].legend()
        axes[row, 0].set_ylim(-.03, 1.03)
    fig.suptitle('Piloto de desarrollo · costo de viaje y factibilidad deben interpretarse juntos')
    fig.tight_layout(); fig.savefig(path, dpi=160); plt.close(fig)


def metric_schema():
    return {'schema_version': SCHEMA, 'objective_scope': OBJECTIVE,
            'cost_scope': 'travel_only_not_full_recourse_cost',
            'scenario_strict_feasibility': 'Fracción de escenarios con cobertura, capacidad, ventanas y retorno válidos.',
            'cvar_travel_minutes': 'CVaR empírico con masa fraccionaria, alpha del banco.',
            'cost_variance_population': 'Varianza de viaje entre escenarios, ddof=0.',
            'mean_net_execution_delta_minutes': 'Viaje ejecutado menos viaje base planeado; puede ser negativo; no es Q óptimo.',
            'relative_gap': 'Gap Gurobi de la misma formulación conservadora; null si no disponible.',
            'search_total_seconds': 'Preparación y llamada al método; incluye construcción, extracción y artefactos propios del solver.',
            'evaluation_seconds': 'Tiempo informado por el evaluador común para Monte Carlo.',
            'trace_validation_seconds': 'Segunda pasada determinista, validación y escritura de trazas.',
            'missing_metrics': 'null en JSON / vacío en CSV cuando no hay solución; no convertir en cero.',
            'scenario_seed': 'Entero de 64 bits serializado en decimal: leer CSV como texto para evitar redondeo.',
            'independent_unit': 'geometry_group_sha256; semillas y escenarios no aumentan N.',
            'training_seconds': 'Costo total del modelo de ese fold/semilla, repetido por política; no sumar filas.',
            'teacher_seconds': 'Etiquetado compartido de geometrías train/validation; costos únicos en manifiesto offline.',
            'offline_cost_id': 'Identifica el inventario de costos únicos de entrenamiento/etiquetado.',
            'comparison_id': 'Mismo banco/evaluador/runner/objetivo/presupuesto/escenarios; no exige algoritmos iguales.',
            'solver_seed': 'En am_supervised/pomo/hbg_facs identifica entrenamiento y búsqueda; no equivale al mismo flujo RNG de ACO.',
            'comparison_keys': ['comparison_id', 'instance_sha256', 'solver_seed', 'policy'],
            'summary_metrics': SUMMARY_METRICS}


Overwriting svrplab/experiments.py


In [48]:
%%writefile svrplab/final_experiment.py
"""10: ejecución holdout independiente exclusivamente en Colab.

El verificador de exportaciones sólo usa stdlib y el evaluador auditado. Los
entrenadores/solvers se importan dentro de las acciones explícitas. No se consulta
ningún certificado al construir una solución. Esta vía no altera methods-v2.
"""
from __future__ import annotations

import csv
import gzip
import json
import math
import shutil
import time
from datetime import datetime, timezone
from pathlib import Path

from . import pilot as p

SCHEMA = 'final-experiment-v1'
EXPORT_SCHEMA = 'final-analysis-export-v1'
SPLIT = 'independent_train_validation_test_v1'


def _colab():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Entrenamiento, evaluación y pruebas exclusivamente en Google Colab.') from exc


def _read(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def _now():
    return datetime.now(timezone.utc).isoformat()


def _sources():
    return {path.name: p.file_hash(path) for path in sorted(Path(__file__).parent.glob('*.py'))}


def _safe(root, relative):
    root, relative = Path(root).resolve(), Path(relative)
    if relative.is_absolute() or '..' in relative.parts or not relative.parts:
        raise ValueError('Ruta relativa inválida en manifiesto.')
    target = root / relative
    if not target.resolve().is_relative_to(root):
        raise ValueError('Ruta fuera del artefacto autorizado.')
    return target


def _csv(path):
    with Path(path).open(newline='', encoding='utf-8') as stream:
        return list(csv.DictReader(stream))


def validate_config(config, design):
    from . import hybrid as hy
    from . import reinforcement as rl
    from .research_design import scenario_precision_plan
    if (config['schema_version'] != SCHEMA or config['role'] != 'confirmatory_holdout'
            or config['split_strategy'] != SPLIT or config['execution_policy'] != 'colab_only'
            or config['objective'] != 'conservative_upper_travel_minutes_v1'
            or config['solver_seeds'] != design['seed_ids']
            or config['search_seconds'] != design['search_seconds']
            or config['policies'] != ['fixed_order', 'causal_edd']):
        raise ValueError('Contrato final incompatible con el diseño sellado.')
    expected = {'hbg_facs', 'hbg_untrained', 'hbg_no_pheromone', 'pomo', 'pomo_untrained',
                'gurobi', 'edd', 'ortools_routing', 'aco', 'tabu'}
    if set(config['methods']) != expected or config['methods']['pomo'] != config['methods']['pomo_untrained']:
        raise ValueError('Inventario final incompleto o control POMO con búsqueda distinta.')
    hy.validate_inference_controls(config)
    hy.validate_training_config(config['training']['hbg_facs'])
    rl.validate_training_config(config['training']['pomo'])
    for settings in config['training'].values():
        if settings['time_scale'] != config['time_scale'] or settings['objective'] != config['objective']:
            raise ValueError('Entrenamiento e inferencia usan objetivos/escalas distintos.')
        if settings['training'].get('geometry_sampling') != 'shuffled_without_replacement':
            raise ValueError('El entrenamiento final requiere cobertura ordenada de geometrías.')
    scenarios = config['scenarios']
    precision = scenario_precision_plan(design)
    required = max(math.ceil(math.log(2 / design['fr_error_probability']) / (2 * design['fr_absolute_error'] ** 2)),
                   math.ceil(design['minimum_expected_tail_observations'] / (1 - design['cvar_alpha'])))
    if (type(scenarios['evaluation_count']) is not int or scenarios['evaluation_count'] < required
            or not 0 <= scenarios['trace_audit_first_scenarios'] <= scenarios['evaluation_count']
            or scenarios['export'] != 'scenario_metrics_csv_gzip_all_scenarios_with_fixed_trace_audit'):
        raise ValueError('Escenarios/exportación no cumplen precisión y auditoría prefijadas.')
    if config['training_geometry_minimum_observations'] < 1:
        raise ValueError('No se permite congelar entrenamiento sin exposición de geometrías.')
    return precision


def resource_plan(registry, config):
    test = [row for row in registry if row['role'] == 'test']
    attempts = len(test) * len(config['methods']) * len(config['solver_seeds'])
    rows = attempts * len(config['policies']) * config['scenarios']['evaluation_count']
    training = sum(value['training']['seconds_per_model'] for value in config['training'].values()) * len(config['solver_seeds'])
    hours = attempts * config['search_seconds'] / 3600
    limits = config['resource_limits']
    return {'test_instances': len(test), 'solver_attempts': attempts,
            'planned_search_budget_hours': hours, 'planned_training_budget_hours': training / 3600,
            'scenario_metric_rows_upper_bound': rows,
            'client_visit_event_rows_avoided': sum(row['n_customers'] for row in test) * len(config['methods']) * len(config['solver_seeds']) * len(config['policies']) * config['scenarios']['evaluation_count'],
            'passes_declared_resource_limits': hours <= limits['maximum_planned_search_hours'] and training / 3600 <= limits['maximum_planned_training_hours'] and rows <= limits['maximum_scenario_rows'],
            'limits_are_compute_budgets_not_wall_time_guarantees': True,
            'evaluation_io_validation_and_restart_overheads_are_additional': True}



def recover_contract_validation(directory, remote, source_sha256):
    """Recupera el acta original; sus duraciones/entorno no cambian al reanudar.

    Un acta fallida con las mismas fuentes se conserva y detiene el flujo. Una
    ausencia permite ejecutar pytest una vez y sellar su resultado en el caller.
    """
    from . import experiments as e
    directory, remote = Path(directory), Path(remote)
    if not e._recover(directory, remote):
        return False
    validation = _read(directory / 'validation.json')
    if validation.get('source_sha256') != source_sha256:
        raise ValueError('Acta Colab recuperada de otras fuentes; no sustituirla.')
    if validation.get('status') != 'passed' or validation.get('returncode') != 0:
        raise ValueError('La validación Colab previa falló; se conserva y no se repite bajo el mismo sello.')
    return True

def prepare_protocol(population_root, population_report_sha256, design, config, analysis_source,
                     analysis_config, validation_bundle, work_root, drive_root):
    """Verifica población y sella decisiones/código/R ANTES de entrenar o abrir test."""
    _colab()
    from .research_design import load_population
    from . import experiments as e
    validate_config(config, design)
    instances, registry, manifest = load_population(population_root, population_report_sha256, design)
    if manifest['evaluator_sha256'] != p.file_hash(p.__file__):
        raise ValueError('El evaluador activo no es el que certificó la población.')
    validation_bundle = Path(validation_bundle)
    validation_sha = p.verify_bundle(validation_bundle)
    validation = _read(validation_bundle / 'validation.json')
    sources = _sources()
    if validation.get('status') != 'passed' or validation.get('source_sha256') != sources:
        raise ValueError('Faltan pruebas Colab aprobadas para estas mismas fuentes.')
    if not Path(analysis_source).is_file() or not Path(analysis_config).is_file():
        raise ValueError('Faltan fuente R/configuración que deben congelarse antes del test.')
    plan = resource_plan(registry, config)
    if not plan['passes_declared_resource_limits']:
        raise ValueError('El plan excede los recursos prefijados; revisar ANTES de exposición final.')
    core = {'schema_version': SCHEMA, 'role': 'confirmatory_holdout', 'config': config, 'design': design,
            'population_sha256': manifest['population_sha256'], 'registry_sha256': p.digest(registry),
            'registry': registry, 'population_manifest': manifest, 'source_sha256': sources,
            'evaluator_sha256': p.file_hash(p.__file__), 'analysis_source_sha256': p.file_hash(analysis_source),
            'analysis_r_sha256': p.file_hash(analysis_source),
            'analysis_config_sha256': p.file_hash(analysis_config), 'validation_bundle_sha256': validation_sha,
            'resource_plan': plan, 'scenario_seeds': {row['instance_sha256']:
                [p.seed_for('final10-evaluation', config['scenarios']['master_seed'], row['instance_sha256'], i)
                 for i in range(config['scenarios']['evaluation_count'])] for row in registry if row['role'] == 'test'}}
    protocol_id = 'final10_' + p.digest(core)[:20]
    root, remote = Path(work_root) / manifest['population_sha256'] / protocol_id, Path(drive_root) / manifest['population_sha256'] / protocol_id
    if e._recover(root / 'protocol', remote / 'protocol'):
        existing = _read(root / 'protocol/protocol.json')
        if {key: value for key, value in existing.items() if key != 'sealed_at'} != core:
            raise ValueError('Protocolo existente distinto; no cambiar después de iniciarlo.')
    else:
        e.write_json(root / 'protocol/protocol.json', {**core, 'sealed_at': _now()})
        e.write_json(root / 'protocol/research_design.json', design)
        e.write_json(root / 'protocol/population.json', _read(Path(population_root) / 'reports' / population_report_sha256 / 'population.json'))
        shutil.copy2(analysis_source, root / 'protocol/confirmatory_results_08.R')
        shutil.copy2(analysis_config, root / 'protocol/analysis_08.json')
        e.write_json(root / 'protocol/environment.json', p.environment_report())
        e._commit(root / 'protocol', remote / 'protocol')
    return root, remote, instances


def _protocol(root):
    root = Path(root)
    p.verify_bundle(root / 'protocol')
    protocol = _read(root / 'protocol/protocol.json')
    if (protocol['source_sha256'] != _sources() or protocol['evaluator_sha256'] != p.file_hash(p.__file__)
            or p.file_hash(root / 'protocol/confirmatory_results_08.R') != protocol['analysis_r_sha256']
            or p.file_hash(root / 'protocol/analysis_08.json') != protocol['analysis_config_sha256']):
        raise ValueError('Código/evaluador cambió: no entrenar ni abrir test bajo este protocolo.')
    return protocol


def _holdout_fold(protocol):
    registry = protocol['registry']
    return {'fold_id': 'independent_holdout', 'split_strategy': SPLIT,
            'split_registry_sha256': p.digest(registry),
            **{destination: sorted({row['geometry_group_sha256'] for row in registry if row['role'] == source})
               for source, destination in [('train', 'train_geometry_groups'), ('validation', 'validation_geometry_groups'), ('test', 'heldout_geometry_groups')]}}


def train_and_freeze(root, remote, instances, device='cuda'):
    """Tres semillas por familia global; geometrías test nunca construidas por trainer."""
    _colab()
    from . import experiments as e, hybrid as hy, reinforcement as rl
    import torch
    root, remote = Path(root), Path(remote)
    protocol = _protocol(root)
    config, fold = protocol['config'], _holdout_fold(protocol)
    if sorted(x.sha for x in instances) != sorted(row['instance_sha256'] for row in protocol['registry']):
        raise ValueError('Instancias diferentes de la población sellada.')
    if config['require_cuda'] and (str(device) != 'cuda' or not torch.cuda.is_available()):
        raise RuntimeError('Entrenamiento final requiere GPU CUDA de Colab.')
    if e._recover(root / 'freeze', remote / 'freeze'):
        recovered = _read(root / 'freeze/freeze.json')
        for row in recovered['models']:
            relative = Path(row['relative_directory'])
            if not e._recover(_safe(root, relative), _safe(remote, relative)):
                raise ValueError('No se recuperó el modelo final desde Drive.')
            model_meta = relative.parent / 'meta'
            if not e._recover(_safe(root, model_meta), _safe(remote, model_meta)):
                raise ValueError('No se recuperó la identidad del entrenamiento desde Drive.')
            checkpoint = Path(row['checkpoint_relative_path'])
            if checkpoint.parent != relative and not e._recover(_safe(root, checkpoint.parent), _safe(remote, checkpoint.parent)):
                raise ValueError('Falta checkpoint diagnóstico rechazado en Drive.')
        _verify_freeze(root, protocol)
        return root / 'freeze'
    models = []
    spec = p.Traffic(**protocol['population_manifest']['traffic'])
    for method, trainer in [('hbg_facs', hy), ('pomo', rl)]:
        for seed in config['solver_seeds']:
            relative = Path('training') / method / f'seed_{seed}'
            try:
                final = trainer.train_fold(instances, spec, config['training'][method], seed, fold,
                    root / relative, remote / relative, device, split_strategy=SPLIT)
            except Exception as exc:
                failure = root / 'technical_failures' / method / f'seed_{seed}'
                if not (failure / 'CHECKSUMS.json').exists():
                    e.write_json(failure / 'failure.json', {'method': method, 'seed': seed,
                        'error_type': type(exc).__name__, 'message': str(exc), 'test_exposure_authorized': False})
                    e._commit(failure, remote / 'technical_failures' / method / f'seed_{seed}')
                raise
            summary = _read(final / 'summary.json')
            checkpoint = rl.selected_checkpoint_path(final, summary)
            models.append({'method': method, 'seed': seed, 'relative_directory': str(final.relative_to(root)),
                           'checkpoint_relative_path': str(checkpoint.relative_to(root)),
                           'bundle_sha256': p.verify_bundle(final), 'summary': summary})
    def visits_for(row):
        summary = row['summary']
        return (summary['selected_checkpoint_training_geometry_visits'] if summary['selection_status'] == 'selected_complete_validation'
                else summary['training_geometry_visits'])
    coverage = all(set(visits_for(row)) == set(fold['train_geometry_groups']) and
                   all(count >= config['training_geometry_minimum_observations'] for count in visits_for(row).values()) for row in models)
    if not coverage:
        failure = root / 'training_gate_failure'
        if not e._recover(failure, remote / 'training_gate_failure'):
            e.write_json(failure / 'failure.json', {'status': 'training_geometry_coverage_failed',
                         'test_opened': False, 'models': models})
            e._commit(failure, remote / 'training_gate_failure')
        raise ValueError('No se observó cada geometría train: aumentar presupuesto sólo antes de abrir test mediante otro protocolo.')
    freeze = {'schema_version': 'final-model-freeze-v1', 'frozen_at': _now(),
              'protocol_sha256': p.file_hash(root / 'protocol/protocol.json'),
              'population_sha256': protocol['population_sha256'], 'registry_sha256': protocol['registry_sha256'],
              'source_sha256': protocol['source_sha256'], 'analysis_source_sha256': protocol['analysis_source_sha256'],
              'analysis_r_sha256': protocol['analysis_r_sha256'],
              'analysis_config_sha256': protocol['analysis_config_sha256'], 'models': models,
              'all_training_geometries_observed': coverage, 'heldout_in_training_or_selection': False,
              'all_seeds_retained': True, 'rejected_checkpoints_are_failures_not_models': True}
    e.write_json(root / 'freeze/freeze.json', freeze)
    e._commit(root / 'freeze', remote / 'freeze')
    _verify_freeze(root, protocol)
    return root / 'freeze'


def _verify_freeze(root, protocol):
    root = Path(root)
    p.verify_bundle(root / 'freeze')
    freeze = _read(root / 'freeze/freeze.json')
    if (freeze['protocol_sha256'] != p.file_hash(root / 'protocol/protocol.json')
            or freeze['population_sha256'] != protocol['population_sha256']
            or freeze['source_sha256'] != protocol['source_sha256']):
        raise ValueError('Freeze no corresponde al protocolo/población/código.')
    if (freeze['all_training_geometries_observed'] is not True or freeze['heldout_in_training_or_selection'] is not False
            or freeze['analysis_r_sha256'] != protocol['analysis_r_sha256']
            or freeze['analysis_config_sha256'] != protocol['analysis_config_sha256']
            or datetime.fromisoformat(freeze['frozen_at']) < datetime.fromisoformat(protocol['sealed_at'])):
        raise ValueError('Freeze no acredita cobertura, orden temporal o análisis prefijado.')
    expected = {(method, seed) for method in protocol['config']['training'] for seed in protocol['config']['solver_seeds']}
    if {(row['method'], row['seed']) for row in freeze['models']} != expected or len(freeze['models']) != len(expected):
        raise ValueError('Inventario de entrenamiento final incompleto o duplicado.')
    for row in freeze['models']:
        final = _safe(root, row['relative_directory'])
        if p.verify_bundle(final) != row['bundle_sha256'] or _read(final / 'summary.json') != row['summary']:
            raise ValueError('Resumen de modelo alterado.')
        if p.file_hash(_safe(root, row['checkpoint_relative_path'])) != row['summary']['model_checkpoint_sha256']:
            raise ValueError('Pesos/checkpoint diagnóstico alterados después de freeze.')
        summary = row['summary']
        diagnostics = summary.get('best_validation_diagnostics') or {}
        if summary['selection_status'] not in ('selected_complete_validation', 'validation_gate_failed'):
            raise ValueError('Intento neuronal no tiene desenlace válido.')
        if summary['selection_status'] == 'selected_complete_validation' and not (diagnostics.get('checkpoint_eligible')
                and set(diagnostics.get('sizes_with_complete_geometry', {})) == {str(size) for size in protocol['design']['sizes']}
                and all(diagnostics['sizes_with_complete_geometry'].values())):
            raise ValueError('Modelo no cumple la puerta final por tamaños antes de abrir test.')
        p.verify_bundle(final.parent / 'meta')
        trained_identity = _read(final.parent / 'meta/identity.json')
        if trained_identity['config'] != protocol['config']['training'][row['method']] or trained_identity['split_strategy'] != SPLIT:
            raise ValueError('Se entrenó con otro protocolo/configuración final.')
        if row['summary']['fold'] != _holdout_fold(protocol) or row['summary']['split_strategy'] != SPLIT:
            raise ValueError('Un checkpoint se entrenó con otra partición.')
        visits = (row['summary']['selected_checkpoint_training_geometry_visits'] if row['summary']['selection_status'] == 'selected_complete_validation'
                  else row['summary']['training_geometry_visits'])
        if set(visits) != set(_holdout_fold(protocol)['train_geometry_groups']) or any(value < protocol['config']['training_geometry_minimum_observations'] for value in visits.values()):
            raise ValueError('Freeze no acredita cobertura completa de entrenamiento.')
    return freeze


def _make_solvers(root, protocol, freeze, device, gurobi_env):
    from . import hybrid as hy, reinforcement as rl, hbg, pomo, baselines, metaheuristics, exact
    from .attention import AttentionPolicy
    from .supervised import _load_checkpoint, synchronize
    import torch
    models = {(row['method'], row['seed']): row for row in freeze['models']}
    config = protocol['config']

    def neural(family, untrained=False):
        def solve(problem, seed, settings, output):
            started = time.perf_counter()
            row = models[(family, seed)]
            summary = row['summary']
            if not untrained and summary['selection_status'] == 'validation_gate_failed':
                return problem.result(None, 'validation_gate_failed', time.perf_counter() - started, [],
                    validation_gate_failed=True, heldout_inference_executed=False,
                    checkpoint_artifact_kind='unselected_epoch_state_for_audit_only')
            prefix = 'hybrid-05' if family == 'hbg_facs' else 'reinforcement-04'
            initialization_seed = p.seed_for(prefix, summary['fold']['fold_id'], seed)
            if untrained:
                torch.manual_seed(initialization_seed)
                torch.cuda.manual_seed_all(initialization_seed)
            model = (hbg.HBGPolicy if family == 'hbg_facs' else AttentionPolicy)(summary['model_config']).to(device)
            try:
                if untrained and family == 'hbg_facs' and hy.initialization_fingerprint(model) != summary['initialization_state_sha256']:
                    raise ValueError('Control sin entrenar usa inicialización distinta.')
                if not untrained:
                    model.load_state_dict(_load_checkpoint(_safe(root, row['checkpoint_relative_path'])))
                model.eval(); synchronize(device)
                loading = time.perf_counter() - started
                remaining = max(0., settings['seconds'] - loading)
                if remaining <= 0:
                    result = problem.result(None, 'time_limit_model_load', loading, [])
                else:
                    call = hbg.solve_hbg if family == 'hbg_facs' else pomo.solve_pomo
                    result = call(problem, seed, {**settings, 'seconds': remaining}, model, device)
                synchronize(device)
                result['solve_seconds'] = time.perf_counter() - started
                result['metadata'].update(model_load_seconds=loading, learned=not untrained,
                    initialization_seed=initialization_seed, selection_status=summary['selection_status'],
                    training_seconds=0. if untrained else summary['training_seconds'],
                    model_checkpoint_sha256=None if untrained else summary['model_checkpoint_sha256'])
                return result
            finally:
                del model
                if str(device).startswith('cuda'):
                    torch.cuda.empty_cache()
        return solve

    return {'hbg_facs': neural('hbg_facs'), 'hbg_no_pheromone': neural('hbg_facs'),
            'hbg_untrained': neural('hbg_facs', True), 'pomo': neural('pomo'),
            'pomo_untrained': neural('pomo', True),
            'gurobi': lambda problem, seed, settings, output: exact.solve_exact(problem, seed, settings, env=gurobi_env, output_dir=output),
            'edd': lambda problem, seed, settings, output: baselines.solve_edd(problem, seed, settings),
            'ortools_routing': lambda problem, seed, settings, output: baselines.solve_ortools_routing(problem, seed, settings),
            'aco': lambda problem, seed, settings, output: metaheuristics.solve_aco(problem, seed, settings),
            'tabu': lambda problem, seed, settings, output: metaheuristics.solve_tabu(problem, seed, settings)}


def _claim_test(root, remote, protocol, freeze, exposure_root):
    """Un reclamo por población: sólo reanuda la misma identidad completa."""
    from . import experiments as e
    root, remote, exposure_root = Path(root), Path(remote), Path(exposure_root)
    core = {'schema_version': 'single-population-test-exposure-v1', 'population_sha256': protocol['population_sha256'],
            'protocol_sha256': p.file_hash(root / 'protocol/protocol.json'),
            'freeze_sha256': p.file_hash(root / 'freeze/freeze.json'), 'release_id': root.name,
            'analysis_source_sha256': protocol['analysis_source_sha256']}
    claim = exposure_root / protocol['population_sha256'] / 'exposure.json'
    claim.parent.mkdir(parents=True, exist_ok=True)
    if claim.exists():
        existing = _read(claim)
        if {key: value for key, value in existing.items() if key != 'opened_at'} != core:
            raise ValueError('Esta población test ya fue expuesta con otra identidad; no permitir otro protocolo.')
    else:
        existing = {**core, 'opened_at': _now()}
        # Exclusión a nivel del filesystem montado: una sesión concurrente no reemplaza la reserva.
        with claim.open('x', encoding='utf-8') as stream:
            json.dump(existing, stream, ensure_ascii=False, indent=2)
    for group in _holdout_fold(protocol)['heldout_geometry_groups']:
        group_claim = exposure_root / 'geometries' / f'{group}.json'
        group_claim.parent.mkdir(parents=True, exist_ok=True)
        if group_claim.exists():
            if _read(group_claim) != existing:
                raise ValueError('Alguna geometría test ya está reservada por otra exposición.')
        else:
            with group_claim.open('x', encoding='utf-8') as stream:
                json.dump(existing, stream, ensure_ascii=False, indent=2)
    if datetime.fromisoformat(existing['opened_at']) < datetime.fromisoformat(freeze['frozen_at']):
        raise ValueError('El test fue abierto antes del freeze de modelos.')
    if e._recover(root / 'test', remote / 'test'):
        if _read(root / 'test/exposure.json') != existing:
            raise ValueError('Registro local/remoto de exposición distinto.')
    else:
        e.write_json(root / 'test/exposure.json', existing)
        e._commit(root / 'test', remote / 'test')
    return existing


def run_test(root, remote, instances, exposure_root, *, device='cuda', gurobi_env=None, open_test=False):
    """Exposición deliberada; cada ruta se sella antes de sus escenarios comunes."""
    _colab()
    if open_test is not True:
        raise ValueError('Test cerrado. Revisar recursos y fijar OPEN_FINAL_TEST=True expresamente en Colab.')
    from . import experiments as e
    from .search import SearchProblem
    import torch
    root, remote = Path(root), Path(remote)
    protocol = _protocol(root)
    freeze = _verify_freeze(root, protocol)
    config = protocol['config']
    if gurobi_env is None:
        raise ValueError('Falta entorno/licencia Gurobi verificado; detener antes de exponer test.')
    if config['require_cuda'] and (str(device) != 'cuda' or not torch.cuda.is_available()):
        raise ValueError('Dispositivo final diferente del protocolo; detener antes de exponer test.')
    expected = {row['instance_sha256'] for row in protocol['registry']}
    if {x.sha for x in instances} != expected:
        raise ValueError('Población distinta del protocolo.')
    exposure = _claim_test(root, remote, protocol, freeze, exposure_root)
    if e._recover(root / 'report', remote / 'report'):
        # En un runtime limpio no se copian millones de registros: verificar el
        # reporte remoto junto a sus bundles hermanos y devolver esa misma ruta.
        remote_report = remote / 'report'
        verify_analysis_export(remote_report, p.file_hash(root / 'report/CHECKSUMS.json'))
        return remote_report
    solvers = _make_solvers(root, protocol, freeze, device, gurobi_env)
    runtime_environment = p.environment_report()
    runtime_id = p.digest(runtime_environment)
    spec = p.Traffic(**protocol['population_manifest']['traffic'])
    nominal = p.Traffic(**{**protocol['population_manifest']['traffic'], 'mode': 'nominal_mean'})
    by_sha = {x.sha: x for x in instances}
    test_registry = [row for row in protocol['registry'] if row['role'] == 'test']
    rows, artifacts = [], {}
    for entry in test_registry:
        x = by_sha[entry['instance_sha256']]
        for method in config['methods']:
            for seed in config['solver_seeds']:
                relative = Path('attempts') / x.sha / method / f'seed_{seed}'
                job, remote_job = root / relative, remote / relative
                context = {'comparison_id': root.name, 'final_release_id': root.name,
                    'instance_id': x.instance_id, 'instance_sha256': x.sha, 'geometry_group_sha256': entry['geometry_group_sha256'],
                    'n_customers': x.n, 'method': method, 'solver_seed': seed, 'artifact_status': 'observed'}
                if not e._recover(job / 'solution', remote_job / 'solution'):
                    (job / 'solution').mkdir(parents=True, exist_ok=True)
                    started = time.perf_counter()
                    import resource
                    torch.cuda.reset_peak_memory_stats()
                    try:
                        problem = SearchProblem(x, spec, config['time_scale'])
                        remaining = max(0., config['search_seconds'] - (time.perf_counter() - started))
                        settings = {**config['methods'][method], 'seconds': remaining}
                        result = (solvers[method](problem, seed, settings, job / 'solution') if remaining > 0
                                  else problem.result(None, 'time_limit_preparation', 0., []))
                        if result.get('status') in {'license_pending', 'solver_error', 'not_available', 'unavailable'}:
                            raise RuntimeError('Método no disponible: resolver infraestructura antes de continuar.')
                        result['search_total_seconds'] = time.perf_counter() - started
                        result['metadata'].update(peak_cuda_allocated_bytes=int(torch.cuda.max_memory_allocated()),
                            process_max_rss_kib=int(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss),
                            cpu_memory_scope='process_lifetime_high_water_not_per_method_delta',
                            runtime_environment_sha256=runtime_id)
                        structure = valid = False
                        cost = None
                        if result['routes'] is not None:
                            p.validate_routes(x, result['routes']); structure = True
                            violation, cost = problem.score(result['routes'])
                            valid = violation <= 1e-9
                            if valid and not p.simulate(x, result['routes'], 0, nominal)['strict_feasible']:
                                raise AssertionError('Solución conservadora incompatible con evaluador nominal.')
                        result.update(structural_valid=structure, conservative_feasible=valid,
                                      search_objective_minutes=cost, context=context)
                    except Exception as exc:
                        failure = job / ('technical_failure_' + str(time.time_ns()))
                        e.write_json(failure / 'failure.json', {**context, 'error_type': type(exc).__name__,
                            'message': str(exc), 'counts_as_algorithm_failure': False,
                            'resume_same_release_after_infrastructure_recovery': True})
                        e._commit(failure, remote_job / failure.name)
                        raise RuntimeError('Fallo técnico registrado; reporte final bloqueado, no imputar derrota del método.') from exc
                    e.write_json(job / 'solution/environment.json', runtime_environment)
                    e.write_json(job / 'solution/solution.json', result)
                    e._commit(job / 'solution', remote_job / 'solution')
                result = _read(job / 'solution/solution.json')
                if result['context'] != context:
                    raise ValueError('Solución recuperada de otro intento.')
                artifacts[str(relative / 'solution')] = p.verify_bundle(job / 'solution')
                for policy in config['policies']:
                    if not e._recover(job / policy, remote_job / policy):
                        summary = {**dict.fromkeys(e.SUMMARY_METRICS), **context, 'policy': policy, 'solver_status': result.get('status'),
                            'structural_valid': result['structural_valid'], 'conservative_feasible': result['conservative_feasible'],
                            'search_objective_minutes': result['search_objective_minutes'],
                            'search_total_seconds': result['search_total_seconds'],
                            'evaluation_status': 'no_solution', 'scenario_strict_feasibility': None,
                            'scenario_success_per_scheduled_attempt': 0., 'model_selection_status': result['metadata'].get('selection_status'),
                            'model_checkpoint_sha256': result['metadata'].get('model_checkpoint_sha256'),
                            'relative_gap': result['metadata'].get('relative_gap'), 'objective_bound': result['metadata'].get('best_bound'), 'best_bound': result['metadata'].get('best_bound'),
                            'surrogate_optimal': result['metadata'].get('surrogate_optimal'),
                            'peak_cuda_allocated_bytes': result['metadata'].get('peak_cuda_allocated_bytes'),
                            'process_max_rss_kib': result['metadata'].get('process_max_rss_kib'),
                            'inference_environment_sha256': result['metadata'].get('runtime_environment_sha256'),
                            'evaluation_environment_sha256': runtime_id}
                        if result['routes'] is not None and result['structural_valid']:
                            seeds = protocol['scenario_seeds'][x.sha]
                            metrics, records = p.evaluate(x, result['routes'], seeds, spec, policy, protocol['design']['cvar_alpha'])
                            records_by_seed = {int(record['scenario_seed']): record for record in records}
                            if set(records_by_seed) != set(seeds):
                                raise ValueError('Escenarios incompletos/repetidos en el evaluador común.')
                            (job / policy).mkdir(parents=True, exist_ok=True)
                            with gzip.open(job / policy / 'scenario_metrics.csv.gz', 'wt', encoding='utf-8', newline='') as stream:
                                writer = csv.DictWriter(stream, fieldnames=list(records[0])); writer.writeheader(); writer.writerows(records)
                            audits = []
                            trace_started = time.perf_counter()
                            for scenario in seeds[:config['scenarios']['trace_audit_first_scenarios']]:
                                trace = p.simulate(x, result['routes'], scenario, spec, policy, trace=True)
                                e.validate_trace(x, result['routes'], trace, records_by_seed[scenario])
                                audits.append(trace)
                            e.write_json(job / policy / 'trace_audit.json', audits)
                            summary.update(metrics, evaluation_status='validated',
                                std_travel_minutes=math.sqrt(metrics['cost_variance_population']),
                                trace_validation_seconds=time.perf_counter() - trace_started,
                                scenario_success_per_scheduled_attempt=metrics['scenario_strict_feasibility'],
                                mean_waiting_minutes=sum(row['waiting_minutes'] for row in records) / len(records),
                                mean_recourse_reorders=sum(row['recourse_reorders'] for row in records) / len(records),
                                mean_net_execution_delta_minutes=sum(row['net_execution_delta_minutes'] for row in records) / len(records))
                        e.write_json(job / policy / 'environment.json', runtime_environment)
                        e.write_json(job / policy / 'summary.json', summary)
                        e._commit(job / policy, remote_job / policy)
                    rows.append(_read(job / policy / 'summary.json'))
                    artifacts[str(relative / policy)] = p.verify_bundle(job / policy)
                print(x.instance_id, method, seed, result.get('status'), flush=True)
    return _export(root, remote, protocol, freeze, exposure, rows, artifacts)


def _export(root, remote, protocol, freeze, exposure, rows, artifacts):
    from . import experiments as e
    config = protocol['config']
    registry = [row for row in protocol['registry'] if row['role'] == 'test']
    expected = {(row['instance_sha256'], method, seed, policy) for row in registry
                for method in config['methods'] for seed in config['solver_seeds'] for policy in config['policies']}
    observed = {(row['instance_sha256'], row['method'], row['solver_seed'], row['policy']) for row in rows}
    if observed != expected or len(rows) != len(expected):
        raise ValueError('No exportar análisis con intentos ausentes o duplicados.')
    report = root / 'report'
    runtimes = {}
    for relative in artifacts:
        environment = _read(root / relative / 'environment.json')
        runtimes[p.digest(environment)] = environment
    e.write_json(report / 'runtime_environments.json', runtimes)
    e.write_csv(report / 'runtime_summary.csv', [{'environment_sha256': sha, 'gpu': env.get('gpu'),
        'cpu_count': env.get('cpu_count'), 'platform': env.get('platform'), 'python': env.get('python')}
        for sha, env in sorted(runtimes.items())])
    e.write_csv(report / 'attempts.csv', rows)
    e.write_csv(report / 'registry.csv', registry)
    e.write_json(report / 'research_design.json', protocol['design'])
    test = {**exposure, 'completed_at': _now(), 'artifact_bundle_sha256': artifacts,
            'expected_policy_rows': len(expected), 'observed_policy_rows': len(rows),
            'complete_inventory': True, 'result_replacement_allowed': False}
    source_objects = {'population': _read(root / 'protocol/population.json'), 'protocol': protocol,
                      'freeze': freeze, 'test': test}
    source_hashes = {}
    for name, obj in source_objects.items():
        path = report / 'source_manifests' / f'{name}.json'
        e.write_json(path, obj); source_hashes[name] = p.file_hash(path)
    release = {'schema_version': 'final-release-v1', 'role': 'confirmatory_holdout', 'final_release_id': root.name,
        'comparison_id': root.name, 'population_sha256': protocol['population_sha256'],
        'protocol_sha256': source_hashes['protocol'], 'model_freeze_sha256': source_hashes['freeze'],
        'test_exposure_sha256': p.file_hash(root / 'test/exposure.json'),
        'model_freeze_before_test': datetime.fromisoformat(freeze['frozen_at']) <= datetime.fromisoformat(exposure['opened_at']),
        'independent_geometry_split_verified': True, 'test_evaluation_once': True,
        'test_once_scope': 'one_logged_exposure_identity_per_population_same_identity_resumption_only',
        'analysis_source_sha256': protocol['analysis_source_sha256'], 'analysis_r_sha256': protocol['analysis_r_sha256'],
        'analysis_config_sha256': protocol['analysis_config_sha256'], 'complete_attempt_inventory': True,
        'source_manifest_sha256': source_hashes, 'distinct_runtime_environments': len(runtimes),
        'timing_requires_hardware_stratification': len(runtimes) > 1}
    e.write_json(report / 'final_release.json', release)
    contract = {'schema_version': EXPORT_SCHEMA, 'role': 'confirmatory_holdout', 'analysis_mode': 'confirmatory',
        'unit': 'geometry_group_sha256', 'final_release_id': root.name, 'comparison_id': root.name,
        'expected_methods': list(config['methods']), 'solver_seeds': config['solver_seeds'], 'policies': config['policies'],
        'source_manifest_sha256': source_hashes, 'analysis_source_sha256': protocol['analysis_source_sha256'],
        'analysis_r_sha256': protocol['analysis_r_sha256'],
        'analysis_config_sha256': protocol['analysis_config_sha256'],
        'all_failures_observed_missing_never_imputed': True, 'events_scope': config['scenarios']['export']}
    e.write_json(report / 'analysis_export_contract.json', contract)
    e._commit(report, remote / 'report')
    verify_analysis_export(report, p.file_hash(report / 'CHECKSUMS.json'))
    print('EXPORT FINAL:', remote / 'report', '\nCHECKSUMS_SHA256:', p.file_hash(report / 'CHECKSUMS.json'), flush=True)
    return report


def verify_analysis_export(directory, expected_manifest_sha256):
    """Sólo lectura: verifica cadena y matriz completa antes de análisis R.

    expected_manifest_sha256 es SHA-256 de los bytes CHECKSUMS.json copiado de 10,
    no el hash canónico que retorna verify_bundle. No ejecuta fuentes desde Drive.
    """
    directory = Path(directory)
    if p.file_hash(directory / 'CHECKSUMS.json') != expected_manifest_sha256:
        raise ValueError('Hash externo del reporte final distinto.')
    manifest = _read(directory / 'CHECKSUMS.json')
    for name in manifest:
        _safe(directory, name)
    p.verify_bundle(directory)
    contract = _read(directory / 'analysis_export_contract.json')
    if (contract.get('schema_version') != EXPORT_SCHEMA or contract.get('role') != 'confirmatory_holdout'
            or contract.get('analysis_mode') != 'confirmatory' or contract.get('unit') != 'geometry_group_sha256'):
        raise ValueError('Exportación no confirmatoria o de otro contrato.')
    chain = {}
    for name in ['population', 'protocol', 'freeze', 'test']:
        path = directory / 'source_manifests' / f'{name}.json'
        if p.file_hash(path) != contract['source_manifest_sha256'][name]:
            raise ValueError('Cadena de fuentes alterada.')
        chain[name] = _read(path)
    population, protocol, freeze, test = (chain[name] for name in ['population', 'protocol', 'freeze', 'test'])
    release = _read(directory / 'final_release.json')
    if (p.digest({key: value for key, value in population.items() if key != 'population_sha256'}) != population['population_sha256']
            or population['status'] != 'nominal_population_accepted'
            or protocol['population_sha256'] != population['population_sha256']
            or freeze['population_sha256'] != population['population_sha256']
            or test['population_sha256'] != population['population_sha256']
            or release['population_sha256'] != population['population_sha256']):
        raise ValueError('Población no aceptada o cadena de población distinta.')
    if (freeze['protocol_sha256'] != contract['source_manifest_sha256']['protocol']
            or test['protocol_sha256'] != freeze['protocol_sha256']
            or test['freeze_sha256'] != contract['source_manifest_sha256']['freeze']
            or release['protocol_sha256'] != freeze['protocol_sha256']
            or release['model_freeze_sha256'] != test['freeze_sha256']
            or release['source_manifest_sha256'] != contract['source_manifest_sha256']):
        raise ValueError('Protocolo, freeze y exposición no forman la misma cadena.')
    if not datetime.fromisoformat(protocol['sealed_at']) <= datetime.fromisoformat(freeze['frozen_at']) <= datetime.fromisoformat(test['opened_at']) <= datetime.fromisoformat(test['completed_at']):
        raise ValueError('El orden sellado->freeze->test->completo es inválido.')
    if (contract['analysis_source_sha256'] != protocol['analysis_source_sha256']
            or contract['analysis_config_sha256'] != protocol['analysis_config_sha256']
            or freeze['analysis_source_sha256'] != protocol['analysis_source_sha256']
            or release['analysis_source_sha256'] != protocol['analysis_source_sha256']):
        raise ValueError('Código R/configuración cambiaron después de congelar el protocolo.')
    if (contract['expected_methods'] != list(protocol['config']['methods'])
            or contract['solver_seeds'] != protocol['config']['solver_seeds']
            or contract['policies'] != protocol['config']['policies']
            or contract['analysis_r_sha256'] != protocol['analysis_r_sha256']
            or freeze['analysis_r_sha256'] != protocol['analysis_r_sha256']
            or freeze['all_training_geometries_observed'] is not True
            or freeze['heldout_in_training_or_selection'] is not False
            or freeze['source_sha256'] != protocol['source_sha256']):
        raise ValueError('Métodos, R, fuentes o comprobaciones de freeze difieren del protocolo.')
    registry = protocol['registry']
    if p.digest(registry) != protocol['registry_sha256']:
        raise ValueError('Registro de geometrías alterado.')
    role_groups = {role: {row['geometry_group_sha256'] for row in registry if row['role'] == role}
                   for role in ['train', 'validation', 'test']}
    excluded = set(protocol['population_manifest']['excluded_development_groups'])
    if (any(not values for values in role_groups.values()) or role_groups['train'] & role_groups['validation']
            or role_groups['train'] & role_groups['test'] or role_groups['validation'] & role_groups['test']
            or set.union(*role_groups.values()) & excluded):
        raise ValueError('La partición no es independiente del desarrollo.')
    fold = _holdout_fold(protocol)
    expected_models = {(name, seed) for name in protocol['config']['training'] for seed in contract['solver_seeds']}
    if {(row['method'], row['seed']) for row in freeze['models']} != expected_models or len(freeze['models']) != len(expected_models):
        raise ValueError('Faltan semillas/modelos en freeze.')
    for row in freeze['models']:
        summary = row['summary']
        if summary['fold'] != fold or summary['heldout_used_for_training_or_selection'] is not False:
            raise ValueError('Modelo con fuga de test o partición distinta.')
        if summary['selection_status'] not in ['selected_complete_validation', 'validation_gate_failed']:
            raise ValueError('Intento neuronal sin desenlace auditable.')
        visits = (summary['selected_checkpoint_training_geometry_visits'] if summary['selection_status'] == 'selected_complete_validation'
                  else summary['training_geometry_visits'])
        if set(visits) != set(fold['train_geometry_groups']) or any(value < protocol['config']['training_geometry_minimum_observations'] for value in visits.values()):
            raise ValueError('Falta cobertura de entrenamiento de todas las geometrías.')
        diagnostics = summary.get('best_validation_diagnostics') or {}
        if summary['selection_status'] == 'selected_complete_validation' and not (diagnostics.get('checkpoint_eligible') and set(diagnostics.get('sizes_with_complete_geometry', {})) == {str(size) for size in protocol['design']['sizes']} and all(diagnostics['sizes_with_complete_geometry'].values())):
            raise ValueError('Se congeló un modelo que no pasó completitud.')
    design = _read(directory / 'research_design.json')
    if design != protocol['design'] or contract['solver_seeds'] != design['seed_ids']:
        raise ValueError('Diseño/R no corresponden al protocolo congelado.')
    test_registry = [row for row in registry if row['role'] == 'test']
    exported_registry = _csv(directory / 'registry.csv')
    if {row['instance_sha256'] for row in exported_registry} != {row['instance_sha256'] for row in test_registry} or len(exported_registry) != len(test_registry):
        raise ValueError('Registro test exportado incompleto o duplicado.')
    by_export_sha = {row['instance_sha256']: row for row in exported_registry}
    for row in test_registry:
        if by_export_sha[row['instance_sha256']] != {key: str(value) for key, value in row.items()}:
            raise ValueError('Registro exportado no conserva todos los campos del protocolo.')
    expected = {(row['instance_sha256'], method, str(seed), policy) for row in test_registry
                for method in contract['expected_methods'] for seed in contract['solver_seeds'] for policy in contract['policies']}
    attempts = _csv(directory / 'attempts.csv')
    observed = {(row['instance_sha256'], row['method'], row['solver_seed'], row['policy']) for row in attempts}
    if observed != expected or len(attempts) != len(expected) or test['observed_policy_rows'] != len(expected):
        raise ValueError('Matriz de intentos incompleta o repetida; no imputar como fallos.')
    by_sha = {row['instance_sha256']: row for row in test_registry}
    for row in attempts:
        origin = by_sha[row['instance_sha256']]
        if (row['artifact_status'] != 'observed' or row['geometry_group_sha256'] != origin['geometry_group_sha256']
                or int(row['n_customers']) != origin['n_customers']):
            raise ValueError('Identidad de intento incorrecta.')
    expected_artifacts = {str(Path('attempts') / row['instance_sha256'] / method / f'seed_{seed}' / part)
                          for row in test_registry for method in contract['expected_methods']
                          for seed in contract['solver_seeds'] for part in ['solution', *contract['policies']]}
    if set(test['artifact_bundle_sha256']) != expected_artifacts:
        raise ValueError('Inventario de soluciones/evaluaciones incompleto.')
    for relative, expected_hash in test['artifact_bundle_sha256'].items():
        artifact = _safe(directory.parent, relative)
        if p.verify_bundle(artifact) != expected_hash:
            raise ValueError('Solución o evaluación cambiada después de exportar.')
    for row in attempts:
        relative = Path('attempts') / row['instance_sha256'] / row['method'] / f'seed_{row["solver_seed"]}'
        persisted = _read(_safe(directory.parent, relative / row['policy'] / 'summary.json'))
        expected_row = {name: '' if value is None else str(value) for name, value in persisted.items()}
        expected_row = {name: expected_row.get(name, '') for name in row}
        if row != expected_row:
            raise ValueError('CSV final difiere del resumen por intento sellado.')
        solution = _read(_safe(directory.parent, relative / 'solution' / 'solution.json'))
        for record in (persisted, solution):
            if (type(record.get('structural_valid')) is not bool or type(record.get('conservative_feasible')) is not bool
                    or record['conservative_feasible'] and not record['structural_valid']):
                raise ValueError('Flags de validez nulos, no booleanos o contradictorios.')
        if (persisted.get('evaluation_status') not in ('validated', 'no_solution')
                or not isinstance(persisted.get('solver_status'), str) or not persisted['solver_status'].strip()
                or persisted.get('artifact_status') != 'observed'
                or persisted['solver_status'] != solution.get('status')):
            raise ValueError('Estado de solver/evaluación ausente o no admitido.')
        if persisted['structural_valid'] != solution['structural_valid'] or persisted['conservative_feasible'] != solution['conservative_feasible']:
            raise ValueError('La factibilidad exportada difiere de la solución congelada.')
        if solution['routes'] is None and (persisted['evaluation_status'] != 'no_solution' or persisted['scenario_success_per_scheduled_attempt'] != 0.):
            raise ValueError('Intento sin rutas presentado como evaluado.')
    return {**contract, 'registry': test_registry, 'design': design, 'source_chain': chain}


Overwriting svrplab/final_experiment.py


In [49]:
%%writefile svrplab/hbg.py
"""05: GFlowNet TB/DB y colonia neuronal propios; ejecutar sólo en Google Colab.

Adaptación inspirada en HBG (https://arxiv.org/abs/2510.04792) y GFACS
(https://proceedings.mlr.press/v258/kim25a.html), NO réplica de esos modelos.
No implementa ENN, RL4CO, backward aprendido ni energy reshaping. La variante v2
permite semilla EDD pagada, feromona compartida en flota homogénea, inferencia
guiada por depósito y búsqueda local pagada, con ablaciones explícitas.

El estado matemático es el prefijo ORDENADO completo de acciones, incluyendo
cierres de vehículos. Es un árbol: cada hijo tiene un único padre, P_B=1.
Un GRU del prefijo condiciona logF; logZ(x)=logF(x,prefijo vacío). El decoder
AM comparte máscaras conservadoras con 03/04; incluye TODAS las decisiones,
también el primer cliente. Sus estadísticas son una parametrización compartida
de los estados del árbol, no una identificación de prefijos distintos.

R=exp(-cost_normalized/T)>0, manejada en dominio logarítmico. Cada bloqueo
tiene su propio sumidero y una transición determinista con P_F=P_B=1.
Completas: costo <=1; sumideros: costo >1. Los límites de tiempo no generan
terminales ni datos de entrenamiento. L=(1-lambda_db)*TB+lambda_db*DB, donde
DB promedia residuos de aristas y TB usa la política neuronal pura, incluso
para muestras fuera de política de la colonia y del replay. No es el esquema
adaptativo/local-energy original HBG. El objetivo es conservador, no CVaR;
el riesgo sólo se mide posteriormente por el evaluador común.
"""
from __future__ import annotations

from collections.abc import Mapping
import copy
import math
import time

import numpy as np
import torch
from torch import nn

from .attention import AttentionConfig, AttentionPolicy, _State, _expired, _prepare, _scales, _sync, _tensors
from .pomo import PENALTY, _cost_scale, _state_cost
from .search import SearchProblem


IMPLEMENTATION = 'project_hbg_tw_prefix_tree_v3_completion_audit'
HBG_DEFAULTS = {'ants': 4, 'iterations': 2, 'alpha': 1., 'evaporation': .1,
                'temperature': 1., 'lambda_db': .5, 'depot_guided': False,
                'constructive_seed': False, 'constructive_seconds': 2., 'local_search_moves': 0}
INFERENCE_DEFAULTS = HBG_DEFAULTS | {'ants': 16, 'iterations': 100, 'seconds': 120.}
REWARD = 'exp_negative_normalized_conservative_cost_v1'
_TAU_MIN, _TAU_MAX = 1e-3, 10.


def _integer(value, name, minimum=1):
    if (isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, np.integer))
            or value < minimum):
        raise ValueError(f'{name} debe ser entero >= {minimum}.')
    return int(value)


def _number(value, name):
    if (isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.number))
            or not math.isfinite(float(value))):
        raise ValueError(f'{name} debe ser un número finito.')
    return float(value)


def validate_hbg_params(params=None):
    """Parámetros normalizados; no modifica el mapping recibido."""
    if params is not None and not isinstance(params, Mapping):
        raise ValueError('Los parámetros HBG deben ser un mapping.')
    unknown = set(params or {}) - set(HBG_DEFAULTS)
    if unknown:
        raise ValueError(f'Parámetros HBG desconocidos: {sorted(unknown)}')
    result = HBG_DEFAULTS | dict(params or {})
    for name in ('ants', 'iterations'):
        result[name] = _integer(result[name], name)
    result['local_search_moves'] = _integer(result['local_search_moves'], 'local_search_moves', minimum=0)
    for name in ('depot_guided', 'constructive_seed'):
        if type(result[name]) is not bool:
            raise ValueError(f'{name} debe ser booleano.')
    for name in ('alpha', 'evaporation', 'temperature', 'lambda_db', 'constructive_seconds'):
        result[name] = _number(result[name], name)
    if result['constructive_seconds'] <= 0:
        raise ValueError('constructive_seconds debe ser positivo.')
    if result['alpha'] < 0:
        raise ValueError('alpha debe ser no negativo.')
    if not 0 < result['evaporation'] < 1:
        raise ValueError('evaporation debe pertenecer a (0,1).')
    if result['temperature'] <= 0:
        raise ValueError('temperature debe ser positiva.')
    if not 0 <= result['lambda_db'] <= 1:
        raise ValueError('lambda_db debe pertenecer a [0,1].')
    return result


def _inference_config(config):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración de inferencia debe ser un mapping.')
    unknown = set(config or {}) - set(INFERENCE_DEFAULTS)
    if unknown:
        raise ValueError(f'Parámetros de inferencia desconocidos: {sorted(unknown)}')
    result = INFERENCE_DEFAULTS | dict(config or {})
    seconds = _number(result.pop('seconds'), 'seconds')
    if seconds < 0:
        raise ValueError('seconds debe ser no negativo.')
    return validate_hbg_params(result) | {'seconds': seconds}


class HBGPolicy(AttentionPolicy):
    """AM para P_F, GRU del prefijo ordenado y head logF condicionado por instancia."""
    def __init__(self, config=None):
        if config is None:
            config = AttentionConfig(dropout=0.)
        config = AttentionConfig.from_value(config)
        if config.dropout != 0:
            raise ValueError('HBG requiere dropout=0 para recomputar la política fuera de línea.')
        super().__init__(config)
        d = self.config.model_dim
        self.prefix_encoder = nn.GRUCell(d, d)
        self.flow_head = nn.Sequential(nn.Linear(3 * d + 6, d), nn.GELU(), nn.Linear(d, 1))

    def initial_prefix(self, encoded):
        return encoded.new_zeros(self.config.model_dim)

    def advance_prefix(self, encoded, local_action, hidden):
        return self.prefix_encoder(encoded[local_action], hidden)

    def log_flow(self, encoded, current_local, state_features, prefix_hidden):
        features = torch.cat((encoded.mean(dim=0), encoded[current_local], prefix_hidden,
                              state_features))
        return self.flow_head(features).squeeze(-1)


def _require_policy(model):
    if not isinstance(model, HBGPolicy):
        raise ValueError('Se requiere HBGPolicy, no un checkpoint AM/POMO.')
    if model.config.dropout != 0:
        raise ValueError('HBG requiere dropout=0.')


def _terminal_record(state, actions, temperature):
    """Los prefijos truncados nunca se aceptan como terminales."""
    problem = state.problem
    scale = _cost_scale(problem)
    cost, missing, complete = _state_cost(state, scale)
    if not complete:
        if state.done or not bool(state.inputs(np.arange(problem.n + 1), _scales(problem))[2].all()):
            raise ValueError('El prefijo incompleto no es un bloqueo terminal.')
        if cost <= 1:
            raise ValueError('El costo de un sumidero debe superar toda solución completa.')
    log_reward = -cost / temperature
    if not math.isfinite(log_reward):
        raise FloatingPointError('logR no finito; revise temperature.')
    return {'format_version': 1, 'instance_sha': problem.x.sha,
            'actions': [int(action) for action in actions],
            'termination': 'complete' if complete else 'dead_end',
            'routes': copy.deepcopy(state.routes) if complete else None,
            'cost': float(cost), 'unserved': missing, 'log_reward': float(log_reward),
            'temperature': float(temperature), 'normalization_scale_minutes': scale}


def _record_equal(record, verified):
    for name in ('format_version', 'unserved'):
        _integer(record.get(name), name, minimum=0)
    for name in ('format_version', 'instance_sha', 'actions', 'termination', 'routes', 'unserved'):
        if record.get(name) != verified[name]:
            raise ValueError(f'Trayectoria inválida: {name} no coincide con su reconstrucción.')
    for name in ('cost', 'log_reward', 'temperature', 'normalization_scale_minutes'):
        value = _number(record.get(name), name)
        if not math.isclose(value, verified[name], rel_tol=1e-8, abs_tol=1e-10):
            raise ValueError(f'Trayectoria inválida: {name} fue alterado o usa otra recompensa.')


def validate_record(problem, record, params=None):
    """Reconstituye y comprueba un registro/replay sin cargar ni consultar modelo."""
    effective = validate_hbg_params(params)
    if not isinstance(record, Mapping) or record.get('instance_sha') != problem.x.sha:
        raise ValueError('La trayectoria debe pertenecer exactamente a esta instancia.')
    actions = record.get('actions')
    if not isinstance(actions, list) or len(actions) > problem.n + problem.vehicles:
        raise ValueError('Lista de acciones inválida.')
    state = _State(problem)
    order, scales = np.arange(problem.n + 1), _scales(problem)
    for customer in actions:
        _integer(customer, 'action', minimum=0)
        if customer > problem.n or state.done:
            raise ValueError('ID fuera de instancia o acción posterior al terminal.')
        if bool(state.inputs(order, scales)[2][customer]):
            raise ValueError('Acción incompatible con las máscaras comunes.')
        state.advance(customer)
    verified = _terminal_record(state, actions, effective['temperature'])
    _record_equal(record, verified)
    return verified


def trajectory_loss(model, problem, device, record, params=None, node_order=None):
    """Revalida una trayectoria propia y recompone TB/DB con P_F actual, sin tau.

    El caller controla backward/optimizer. PB=1 por construcción del árbol.
    Cierres y primer cliente sí contribuyen al logPF; un bloqueo añade una arista
    determinista a su sumidero. El trainer aporta sólo construcciones propias;
    aquí se rechazan prefijos truncados, otra instancia o temperatura.
    """
    _require_policy(model)
    effective = validate_hbg_params(params)
    verified = validate_record(problem, record, effective)
    actions = verified['actions']
    device = torch.device(device)
    encoded, order, inverse, scales = _prepare(model, problem, device, node_order)
    state, hidden = _State(problem), model.initial_prefix(encoded)
    flows, logprobs = [], []
    for customer in actions:
        if state.done:
            raise ValueError('La trayectoria continúa después del último vehículo.')
        context, action_features, blocked = _tensors(state, order, scales, device)
        local = int(inverse[customer])
        if bool(blocked[local]):
            raise ValueError('Acción incompatible con las máscaras comunes.')
        flows.append(model.log_flow(encoded, int(inverse[state.node]), context, hidden))
        logits = model.next_logits(encoded, int(inverse[state.node]), context, action_features, blocked)
        if not bool(torch.isfinite(logits[~blocked]).all()):
            raise FloatingPointError('Logits HBG no finitos.')
        logprobs.append(torch.log_softmax(logits, dim=-1)[local])
        hidden = model.advance_prefix(encoded, local, hidden)
        state.advance(customer)
    if verified['termination'] == 'dead_end':
        # Prefijo bloqueado -> sumidero único; NO softmax de acciones prohibidas.
        context, _, _ = _tensors(state, order, scales, device)
        flows.append(model.log_flow(encoded, int(inverse[state.node]), context, hidden))
        logprobs.append(encoded.sum() * 0.)
    if not flows:
        raise ValueError('Una trayectoria terminal necesita al menos una arista.')
    flow_values = torch.stack(flows)
    probability_values = torch.stack(logprobs)
    reward = flow_values.new_tensor(verified['log_reward'])
    successor_flows = torch.cat((flow_values[1:], reward.reshape(1)))
    residuals = flow_values + probability_values - successor_flows
    tb_residual = flow_values[0] + probability_values.sum() - reward
    tb, db = tb_residual.square(), residuals.square().mean()
    weight = effective['lambda_db']
    loss = (1. - weight) * tb + weight * db
    if not bool(torch.isfinite(loss)):
        raise FloatingPointError('Loss HBG no finita; detener entrenamiento.')
    cost = verified['cost']
    return loss, {'tb_loss': float(tb.detach().cpu()), 'db_loss': float(db.detach().cpu()),
                  'tb_residual': float(tb_residual.detach().cpu()),
                  'db_residual_sum': float(residuals.detach().sum().cpu()),
                  'mean_cost': cost, 'penalized_cost_minutes': cost * _cost_scale(problem),
                  'complete': verified['termination'] == 'complete',
                  'decisions': len(actions), 'balance_edges': len(flows),
                  'logZ': float(flow_values[0].detach().cpu()),
                  'logPF': float(probability_values.detach().sum().cpu()),
                  'log_reward': verified['log_reward'], 'unserved': verified['unserved'],
                  'backward_policy': 'unique_parent_prefix_tree', 'penalty': PENALTY}


def _sample_ant(model, problem, device, prepared, tau, rng, params, deadline):
    """No grafo; la primera acción procede de la misma P_F que las siguientes."""
    encoded, order, inverse, scales = prepared
    state, actions = _State(problem), []
    while not state.done:
        if _expired(deadline, device):
            return None
        context, action_features, blocked = _tensors(state, order, scales, device)
        if bool(blocked.all()):
            break
        logits = model.next_logits(encoded, int(inverse[state.node]), context, action_features, blocked)
        pheromone = torch.as_tensor(np.log(tau[0 if tau.shape[0] == 1 else state.vehicle, state.node, order]),
                                    dtype=logits.dtype, device=device)
        # q(a|s,tau) ∝ PF_theta(a|s) * tau(vehicle,node,a)^alpha.
        # q es conducta de muestreo, no sustituye PF en TB/DB.
        sampling_logits = logits + params['alpha'] * pheromone
        if not bool(torch.isfinite(sampling_logits[~blocked]).all()):
            raise FloatingPointError('Logits de colonia no finitos.')
        probabilities = torch.softmax(sampling_logits, dim=-1).cpu().numpy().astype(np.float64)
        probabilities /= probabilities.sum()
        local = (int(np.argmax(probabilities)) if params['depot_guided'] and state.node != 0
                 else int(rng.choice(len(order), p=probabilities)))
        customer = int(order[local])
        actions.append(customer)
        state.advance(customer)
    record = _terminal_record(state, actions, params['temperature'])
    record['generation_source'] = 'neural_pheromone_colony'
    return record



def _routes_record(problem, routes, params):
    """Convierte una construcción convencional pagada; sólo prefijos admisibles.

    No transforma reparaciones en acciones neuronales observadas. La procedencia
    queda explícita y TB/DB recomputan PF actual fuera de política.
    """
    state = _State(problem)
    actions = [action for route in routes for action in [*route, 0]]
    order, scales = np.arange(problem.n + 1), _scales(problem)
    for action in actions:
        if state.done or bool(state.inputs(order, scales)[2][action]):
            return None
        state.advance(action)
    record = _terminal_record(state, actions, params['temperature'])
    record['generation_source'] = 'paid_edd_constructive_seed'
    return record

def _reinforce(tau, complete_records, evaporation):
    """Evapora una vez por iteración y deposita sólo en la mejor completa."""
    tau *= 1. - evaporation
    if complete_records:
        best = min(complete_records, key=lambda record: (record['cost'], record['actions']))
        delta = 1. / (1. + best['cost'])
        vehicle, node = 0, 0
        for customer in best['actions']:
            # Empty-vehicle closures are bookkeeping, not travelled arcs.
            # Pooling them across homogeneous vehicles would bias 0→0.
            if node != 0 or customer != 0:
                tau[0 if tau.shape[0] == 1 else vehicle, node, customer] += delta
            if customer == 0:
                vehicle, node = vehicle + 1, 0
            else:
                node = customer
    np.clip(tau, _TAU_MIN, _TAU_MAX, out=tau)


def collect_colony(model, problem, device, rng, params=None, deadline=None):
    """Colonias secuenciales sin gradiente, tau nueva por instancia/llamada.

    Incluye codificación, todas las hormigas, evaporación y comprobación de
    soluciones en un único deadline absoluto perf_counter. Regresa registros
    JSON de completas y bloqueos; excluye intentos interrumpidos por tiempo.
    Restaura train/eval y no modifica pesos, datos, flota ni estado del replay.
    """
    _require_policy(model)
    effective = validate_hbg_params(params)
    if not isinstance(rng, np.random.Generator):
        raise ValueError('rng debe ser un Generator NumPy explícito.')
    if deadline is not None:
        deadline = _number(deadline, 'deadline')
    device = torch.device(device)
    _sync(device)
    started = time.perf_counter()
    records, history = [], []
    attempts = interrupted = iterations = completed = 0
    constructive_count = pheromone_deposit_count = pheromone_guided_attempts = 0
    termination, best = 'iteration_limit', math.inf
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            if _expired(deadline, device):
                termination = 'time_limit'
            else:
                tau_groups = 1 if np.all(problem.capacities == problem.capacities[0]) else problem.vehicles
                tau = np.ones((tau_groups, problem.n + 1, problem.n + 1), dtype=np.float64)
                if effective['constructive_seed']:
                    # Construcción pagada en esta llamada; nunca certificados del banco.
                    from .baselines import solve_edd
                    seed_budget = effective['constructive_seconds'] if deadline is None else max(
                        0., min(effective['constructive_seconds'], deadline - time.perf_counter()))
                    candidate = solve_edd(problem, int(rng.integers(2**31)), {'seconds': seed_budget})
                    if candidate.get('routes') is not None and problem.score(candidate['routes'])[0] <= 1e-9:
                        seeded = _routes_record(problem, candidate['routes'], effective)
                        if seeded is not None:
                            records.append(seeded)
                            constructive_count = 1
                            completed += 1
                            attempts += 1
                            best = seeded['cost']
                            _reinforce(tau, [seeded], effective['evaporation'])
                            pheromone_deposit_count += 1
                            history.append({'iteration': -1, 'phase': 'paid_edd_seed', 'attempt': attempts,
                                            'elapsed_seconds': time.perf_counter() - started,
                                            'violation_minutes': 0.,
                                            'travel_minutes': best * seeded['normalization_scale_minutes']})
                prepared = None
                if _expired(deadline, device):
                    termination = 'time_limit'
                else:
                    prepared = _prepare(model, problem, device, None)
                for iteration in range(effective['iterations']):
                    if _expired(deadline, device):
                        termination = 'time_limit'
                        break
                    iteration_complete = []
                    for ant in range(effective['ants']):
                        if _expired(deadline, device):
                            termination = 'time_limit'
                            break
                        attempts += 1
                        pheromone_guided_attempts += int(effective['alpha'] > 0 and pheromone_deposit_count > 0)
                        record = _sample_ant(model, problem, device, prepared, tau, rng, effective, deadline)
                        if record is None:
                            interrupted += 1
                            termination = 'time_limit'
                            break
                        records.append(record)
                        if record['termination'] == 'complete':
                            completed += 1
                            iteration_complete.append(record)
                            if record['cost'] < best:
                                best = record['cost']
                                _sync(device)
                                history.append({'iteration': iteration, 'ant': ant,
                                                'attempt': attempts, 'phase': 'colony',
                                                'elapsed_seconds': time.perf_counter() - started,
                                                'violation_minutes': 0.,
                                                'travel_minutes': best * record['normalization_scale_minutes']})
                        if _expired(deadline, device):
                            termination = 'time_limit'
                            break
                    if termination == 'time_limit':
                        break
                    _reinforce(tau, iteration_complete, effective['evaporation'])
                    pheromone_deposit_count += int(bool(iteration_complete))
                    iterations += 1
                    if _expired(deadline, device):
                        termination = 'time_limit'
                        break
    finally:
        model.train(was_training)
    _sync(device)
    costs = [record['cost'] for record in records]
    stats = {'mean_cost': float(np.mean(costs)) if costs else None,
             'constructive_seed_count': constructive_count,
             'pheromone_deposit_count': pheromone_deposit_count,
             'pheromone_guided_attempts': pheromone_guided_attempts,
             'pheromone_behavior_active': pheromone_guided_attempts > 0,
             'neural_complete_count': completed - constructive_count,
             'neural_complete_fraction': (completed - constructive_count) / max(1, len(records) - constructive_count),
             'complete_fraction': completed / len(records) if records else 0.,
             'rollout_count': len(records), 'complete_count': completed,
             'decisions': sum(len(record['actions']) for record in records),
             'attempts': attempts, 'interrupted_count': interrupted,
             'completed_iterations': iterations, 'termination': termination,
             'normalization_scale_minutes': _cost_scale(problem), 'penalty': PENALTY,
             'history': history, 'elapsed_seconds': time.perf_counter() - started}
    return records, stats


def insert_elite(replay, records, capacity):
    """Retiene completas únicas por (sha,prefijo), menor costo primero; JSON puro.

    El trainer separa buffers por instancia de desarrollo. Este helper no
    reconstruye rutas: trajectory_loss siempre revalida antes del backward.
    No acepta bloqueos o muestras truncadas como élites.
    """
    capacity = _integer(capacity, 'capacity')
    by_key = {}
    for record in [*replay, *records]:
        if not isinstance(record, Mapping) or record.get('termination') != 'complete':
            continue
        if not isinstance(record.get('instance_sha'), str) or not isinstance(record.get('actions'), list):
            raise ValueError('Registro replay inválido.')
        for action in record['actions']:
            _integer(action, 'action', minimum=0)
        cost = _number(record.get('cost'), 'cost')
        if not 0 <= cost <= 1. + 1e-9:
            raise ValueError('Costo élite fuera del rango completo [0,1].')
        key = (record['instance_sha'], tuple(record['actions']))
        existing = by_key.get(key)
        if existing is not None:
            # A complete prefix may arise both from paid EDD and the neural
            # colony. Its sampling provenance may differ; its state/reward may
            # not. Keep all observed sources without assigning EDD to a net.
            provenance = {'generation_source', 'generation_sources'}
            core = {name: value for name, value in record.items() if name not in provenance}
            previous_core = {name: value for name, value in existing.items() if name not in provenance}
            if core != previous_core:
                raise ValueError('Mismo prefijo replay con recompensa/metadatos incompatibles.')
            sources = set()
            for item in (existing, record):
                declared = item.get('generation_sources')
                if declared is None:
                    declared = [item.get('generation_source', 'neural_pheromone_colony')]
                if not isinstance(declared, list) or any(not isinstance(source, str) or not source for source in declared):
                    raise ValueError('Procedencia de generación inválida.')
                sources.update(declared)
            merged = copy.deepcopy(dict(existing))
            merged['generation_sources'] = sorted(sources)
            merged['generation_source'] = next(iter(sources)) if len(sources) == 1 else 'multiple_generation_sources'
            by_key[key] = merged
        else:
            by_key[key] = copy.deepcopy(dict(record))
    return sorted(by_key.values(), key=lambda row: (row['cost'], row['instance_sha'], row['actions']))[:capacity]


def sample_replay(replay, count, rng):
    """Muestreo uniforme sin reemplazo del buffer élite; no altera sus registros."""
    count = _integer(count, 'count', minimum=0)
    if not isinstance(rng, np.random.Generator):
        raise ValueError('rng debe ser un Generator NumPy explícito.')
    indices = rng.choice(len(replay), size=min(count, len(replay)), replace=False)
    return [copy.deepcopy(replay[int(index)]) for index in indices]


def validation_score(model, problem, device, params=None, seed=0):
    """Mejor costo de colonia con semilla fija; no usa ni actualiza replay."""
    seed = _integer(seed, 'seed', minimum=0)
    records, stats = collect_colony(model, problem, device, np.random.default_rng(seed), params)
    return {'score': min(record['cost'] for record in records),
            'complete_fraction': stats['complete_fraction'], 'rollout_count': len(records),
            'best_complete': stats['complete_count'] > 0,
            'normalization_scale_minutes': stats['normalization_scale_minutes'], 'penalty': PENALTY}


def solve_hbg(problem: SearchProblem, seed, config, model, device):
    """Inferencia con presupuesto común: sólo devuelve construcciones completas."""
    seed = _integer(seed, 'seed', minimum=0)
    effective = _inference_config(config)
    device = torch.device(device)
    _sync(device)
    started = time.perf_counter()
    deadline = started + effective['seconds']
    params = {key: value for key, value in effective.items() if key != 'seconds'}
    colony_deadline = deadline - (.1 * effective['seconds'] if effective['local_search_moves'] else 0.)
    records, stats = collect_colony(model, problem, device, np.random.default_rng(seed), params, colony_deadline)
    complete = [record for record in records if record['termination'] == 'complete']
    best = min(complete, key=lambda record: record['cost']) if complete else None
    local_search = {'evaluated_neighbors': 0, 'accepted_moves': 0, 'termination': 'disabled'}
    routes = best['routes'] if best else None
    if routes is not None and effective['local_search_moves'] and not _expired(deadline, device):
        from .baselines import improve_feasible_routes
        routes, local_search = improve_feasible_routes(problem, routes, np.random.default_rng(seed),
                                                       deadline, max_moves=effective['local_search_moves'])
    _sync(device)
    elapsed = time.perf_counter() - started
    result = problem.result(
        routes, 'feasible' if best else 'no_solution', elapsed,
        stats['history'], method='hbg_facs', seed=seed, implementation=IMPLEMENTATION,
        model_config=model.config.to_dict(), effective_config=effective,
        termination=stats['termination'], attempts=stats['attempts'],
        completed_decodes=stats['complete_count'],
        failed_decodes=stats['attempts'] - stats['complete_count'],
        interrupted_decodes=stats['interrupted_count'], completed_iterations=stats['completed_iterations'],
        constructive_seed_count=stats['constructive_seed_count'],
        pheromone_deposit_count=stats['pheromone_deposit_count'],
        pheromone_guided_attempts=stats['pheromone_guided_attempts'],
        pheromone_behavior_active=stats['pheromone_behavior_active'],
        ablation_scope='inference_pheromone_only_same_frozen_weights' if effective['alpha'] == 0 else None,
        neural_complete_count=stats['neural_complete_count'], local_search=local_search,
        best_candidate_generation_source=(best or {}).get('generation_source', 'neural_pheromone_colony') if best else None,
        local_search_improved=local_search.get('accepted_moves', 0) > 0,
        pheromone_scope='shared_homogeneous_fleet_otherwise_vehicle_specific',
        inference_device=str(device), budget_includes_construction=True, budget_includes_encoding=True,
        budget_includes_pheromone=True, time_limit_granularity='synchronized_decoder_step_or_completed_validation',
        search_objective='conservative_upper_travel_minutes', reward=REWARD,
        nominal_scope='conservative_upper_mean_matrix_not_stochastic_feasibility',
        construction='declared_paid_edd_seed_neural_colony_and_optional_local_search',
        no_solution_scope='decoder_dead_end_or_budget_not_infeasibility_proof',
        seed_reproducibility='same_environment_and_iteration_cutoff; wall_time_cutoff_may_differ',
        training_included_in_solve_seconds=False, pheromone_reset_per_solve=True)
    _sync(device)
    elapsed = time.perf_counter() - started
    result['solve_seconds'] = elapsed
    result['metadata']['budget_overrun_seconds'] = max(0., elapsed - effective['seconds'])
    return result


Overwriting svrplab/hbg.py


In [50]:
%%writefile svrplab/hybrid.py
"""05: HBG-FACS conservador, replay por geometría y checkpoints recuperables.

Ejecución sólo Colab. Sin profesores, certificados ni escenarios de evaluación
para entrenar o seleccionar modelos. Objetivo conservador de desarrollo, no c+Q.
"""
from __future__ import annotations

import hashlib
import math
import random
import time
import uuid
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch

from . import experiments as e
from . import pilot as p
from .attention import AttentionConfig, MASK_VERSION
from .hbg import (HBGPolicy, collect_colony, trajectory_loss, validation_score,
                  solve_hbg, validate_hbg_params, insert_elite, sample_replay, validate_record)
from .reinforcement import (CHECKPOINT_SELECTION, FINAL_CHECKPOINT_SELECTION, validation_candidate_is_better, selected_checkpoint_path)
from .search import OBJECTIVE, SearchProblem
from .supervised_data import make_geometry_folds, validate_fold, validate_holdout_fold
from .supervised import (require_colab, synchronize, choose_device, _cpu_state,
                         _load_checkpoint, _runtime_signature)


def validate_training_config(config):
    if (config['schema_version'] != 'hybrid-05-v1' or
            (config['role'], config['split_strategy']) not in {
                ('development_only', 'leave_one_geometry_out_with_inner_validation'),
                ('train_validation_for_final_holdout', 'independent_train_validation_test_v1')}):
        raise ValueError('Configuración híbrida incompatible.')
    if config['objective'] != OBJECTIVE or type(config['time_scale']) is not int or config['time_scale'] < 1:
        raise ValueError('Objetivo o escala temporal incompatible.')
    if not isinstance(config.get('replicate_id'), str) or not config['replicate_id'].strip():
        raise ValueError('replicate_id debe identificar una réplica con texto no vacío.')
    model = AttentionConfig.from_value(config['model'])
    if model.dropout != 0.:
        raise ValueError('HBG requiere dropout=0 para recalcular probabilidades y flujos.')
    validate_hbg_params(config['hbg'])
    replay = config['replay']
    if set(replay) != {'capacity_per_instance', 'samples_per_step'}:
        raise ValueError('Configuración de replay incompatible.')
    if type(replay['capacity_per_instance']) is not int or replay['capacity_per_instance'] < 1:
        raise ValueError('Capacidad de replay debe ser entera y positiva.')
    if type(replay['samples_per_step']) is not int or not 0 <= replay['samples_per_step'] <= replay['capacity_per_instance']:
        raise ValueError('Muestras de replay debe ser entero entre cero y la capacidad.')
    settings = config['training']
    if type(settings.get('require_gpu', False)) is not bool:
        raise ValueError('require_gpu debe ser booleano.')
    if settings.get('checkpoint_gate', 'all_validation_instances') not in ('all_validation_instances', 'one_complete_geometry_per_size'):
        raise ValueError('Puerta de completitud desconocida.')
    if config['role'] == 'development_only' and settings.get('checkpoint_gate', 'all_validation_instances') != 'all_validation_instances':
        raise ValueError('Los folds históricos requieren completitud de cada instancia de validación.')
    if settings.get('geometry_sampling', 'iid') not in ('iid', 'shuffled_without_replacement'):
        raise ValueError('Muestreo de geometrías desconocido.')
    expected_selection = CHECKPOINT_SELECTION if config['role'] == 'development_only' else FINAL_CHECKPOINT_SELECTION
    if settings.get('checkpoint_selection', expected_selection) != expected_selection:
        raise ValueError('Regla de selección debe priorizar completitud en toda validación.')
    for name in ['epochs', 'steps_per_epoch', 'gradient_accumulation', 'patience']:
        if type(settings[name]) is not int or settings[name] < 1:
            raise ValueError(f'{name} debe ser entero positivo.')
    for name in ['learning_rate', 'gradient_clip_norm', 'seconds_per_model']:
        if isinstance(settings[name], bool) or not math.isfinite(settings[name]) or settings[name] <= 0:
            raise ValueError(f'{name} debe ser finito y positivo.')
    flow_lr = settings.get('flow_learning_rate', settings['learning_rate'])
    if isinstance(flow_lr, bool) or not math.isfinite(flow_lr) or flow_lr <= 0:
        raise ValueError('flow_learning_rate debe ser positiva y finita.')
    for name in ['weight_decay', 'min_delta']:
        if isinstance(settings[name], bool) or not math.isfinite(settings[name]) or settings[name] < 0:
            raise ValueError(f'{name} debe ser finito y no negativo.')
    for name in ['permute_nodes', 'deterministic_algorithms']:
        if type(settings[name]) is not bool:
            raise ValueError(f'{name} debe ser booleano.')
    return True



def validate_inference_controls(common):
    """El control sin entrenar cambia pesos; alpha=0 sólo elimina guía de tau.

    Tienen la misma inicialización, reglas constructivas, búsqueda local, límites
    de colonias y presupuesto de reloj. El cómputo efectivamente terminado puede
    variar y se exporta; no se presenta esta ablación como reentrenamiento sin ACO.
    """
    methods = common['methods']
    base = methods['hbg_facs']
    if methods['hbg_untrained'] != base:
        raise ValueError('hbg_untrained debe conservar toda la configuración de inferencia HBG.')
    if methods['hbg_no_pheromone'] != {**base, 'alpha': 0.}:
        raise ValueError('hbg_no_pheromone sólo puede cambiar alpha a cero.')
    if base['alpha'] <= 0:
        raise ValueError('La condición principal requiere feromona activa.')
    return True

def _source_hashes():
    directory = Path(__file__).parent
    return {name: p.file_hash(directory / name) for name in
            ('hybrid.py', 'hbg.py', 'baselines.py', 'reinforcement.py', 'pomo.py', 'supervised.py', 'supervised_data.py',
             'attention.py', 'search.py', 'metaheuristics.py', 'experiments.py')}


def _group_instances(instances, groups):
    grouped = {group: [] for group in groups}
    for x in instances:
        group = x.provenance['geometry_group_sha256']
        if group in grouped:
            grouped[group].append(x.sha)
    if not grouped or any(not values for values in grouped.values()):
        raise ValueError('Faltan instancias de alguna geometría autorizada.')
    return {group: sorted(shas) for group, shas in grouped.items()}


def _training_device(config, requested):
    device = choose_device(str(requested))
    if config['training'].get('require_gpu', False) and device.type != 'cuda':
        raise RuntimeError('Esta réplica de 05 requiere GPU CUDA en Colab; seleccione GPU antes de entrenar.')
    return device


def initialization_fingerprint(model):
    """Huella estable de todos los tensores, además de semilla/arquitectura."""
    return p.digest({name: {'shape': list(tensor.shape), 'dtype': str(tensor.dtype),
                           'sha256': hashlib.sha256(tensor.detach().cpu().contiguous().numpy().tobytes()).hexdigest()}
                     for name, tensor in model.state_dict().items()})


def validation_cost(model, problems, grouped, groups, device, params, return_diagnostics=False, selection_gate='all_validation_instances'):
    """Validación neuronal con semilla fija, sin semilla EDD, replay ni búsqueda local.

    Cada instancia debe tener alguna construcción neural completa; una ruta EDD
    pagada durante entrenamiento/inferencia no puede habilitar el checkpoint.
    """
    if set(grouped) != set(groups) or any(not values for values in grouped.values()):
        raise ValueError('Geometrías de validación ausentes o ajenas.')
    model.eval()
    losses, completeness, successes, sizes_completed = {}, {}, {}, {}
    validation_params = {**params, 'constructive_seed': False, 'local_search_moves': 0}
    with torch.no_grad():
        for group in sorted(groups):
            records = [validation_score(model, problems[sha], device, validation_params,
                                        seed=p.seed_for('hbg05-validation', sha)) for sha in grouped[group]]
            if selection_gate == 'one_complete_geometry_per_size':
                for sha, record in zip(grouped[group], records):
                    size = str(problems[sha].n)
                    sizes_completed[size] = sizes_completed.get(size, False) or bool(record['best_complete'])
            values = [float(record['score']) for record in records]
            if not all(math.isfinite(value) for value in values):
                raise ValueError('Costo de validación no finito.')
            losses[group] = float(np.mean(values))
            completeness[group] = float(np.mean([record['complete_fraction'] for record in records]))
            successes[group] = float(np.mean([record['best_complete'] for record in records]))
    mean = float(np.mean(list(losses.values())))
    if not return_diagnostics:
        return mean, losses
    eligible = (all(sizes_completed.values()) and bool(sizes_completed) if selection_gate == 'one_complete_geometry_per_size'
                else all(value == 1. for value in successes.values()))
    return mean, losses, {'checkpoint_eligible': eligible, 'selection_gate': selection_gate,
                         'sizes_with_complete_geometry': sizes_completed, 'complete_fraction': float(np.mean(list(completeness.values()))),
                         'instances_with_completion_fraction': float(np.mean(list(successes.values()))),
                         'complete_fraction_by_geometry': completeness,
                         'instances_with_completion_fraction_by_geometry': successes,
                         'scope': 'neural_colony_without_edd_seed_or_local_search'}


def validate_replay(replay, problems, grouped, params, capacity):
    """El buffer sólo puede contener prefijos generados para instancias train.

    Revalida cada trayectoria recuperada, su recompensa y terminal con la
    instancia sellada; un hash de archivo no sustituye esta comprobación.
    """
    authorized = {sha for shas in grouped.values() for sha in shas}
    if not isinstance(replay, dict) or set(replay) != authorized:
        raise ValueError('Replay contiene instancias fuera de entrenamiento o faltantes.')
    for sha, records in replay.items():
        if not isinstance(records, list) or len(records) > capacity:
            raise ValueError('Capacidad o estructura del replay inválida.')
        seen = set()
        for record in records:
            validate_record(problems[sha], record, params)
            if record['termination'] != 'complete':
                raise ValueError('El replay élite sólo conserva construcciones completas.')
            key = tuple(record['actions'])
            if key in seen:
                raise ValueError('Replay contiene prefijos duplicados.')
            seen.add(key)
    return True


def _check_resume(progress, identity, device):
    if progress['identity_sha256'] != p.digest(identity):
        raise ValueError('Checkpoint de otro fold, código o configuración; no se cargaron pesos.')
    current = _runtime_signature(device)
    saved = progress['runtime_signature']
    if saved != current:
        differences = '; '.join(
            f'{key}: guardado={saved.get(key)!r}, actual={current.get(key)!r}'
            for key in sorted(set(saved) | set(current)) if saved.get(key) != current.get(key))
        raise ValueError('Entorno incompatible para reanudar: ' + differences
                         + '. Ejecute train_crossfit para usar una carpeta separada por entorno. '
                           'El checkpoint anterior se conserva.')


def train_fold(instances, spec, config, seed, fold, directory, remote, device, split_strategy='development_crossfit'):
    require_colab()
    validate_training_config(config)
    if type(seed) is not int or seed < 0:
        raise ValueError('La semilla de entrenamiento debe ser un entero no negativo.')
    if split_strategy == 'development_crossfit':
        validate_fold(fold, instances)
    elif split_strategy == 'independent_train_validation_test_v1':
        validate_holdout_fold(fold, instances)
    else:
        raise ValueError('Estrategia de partición no admitida.')
    directory, remote = Path(directory), Path(remote)
    device = _training_device(config, device)
    started = time.perf_counter()
    grouped = _group_instances(instances, fold['train_geometry_groups'])
    validation = _group_instances(instances, fold['validation_geometry_groups'])
    identity = {'schema_version': config['schema_version'], 'fold': fold, 'seed': seed,
                'split_strategy': split_strategy,
                'config': config, 'source_hashes': _source_hashes(), 'traffic': asdict(spec),
                'train_instances': grouped, 'validation_instances': validation,
                'evaluator_sha256': p.file_hash(p.__file__)}
    if e._recover(directory / 'meta', remote / 'meta'):
        if e.read_json(directory / 'meta/identity.json') != identity:
            raise ValueError('Checkpoint de entrenamiento incompatible con instancias, código, fold o parámetros.')
    else:
        e.write_json(directory / 'meta/identity.json', identity)
        e.write_json(directory / 'meta/environment.json', p.environment_report())
        e._commit(directory / 'meta', remote / 'meta')
    if e._recover(directory / 'final', remote / 'final'):
        recovered_summary = e.read_json(directory / 'final/summary.json')
        if recovered_summary.get('selection_status') == 'validation_gate_failed':
            rejected_epoch = f'epoch_{recovered_summary["epochs_completed"]:04d}'
            if not e._recover(directory / rejected_epoch, remote / rejected_epoch):
                raise ValueError('Falta el checkpoint diagnóstico del modelo rechazado.')
        selected_checkpoint_path(directory / 'final', recovered_summary)
        if recovered_summary['identity_sha256'] != p.digest(identity):
            raise ValueError('Modelo final de otro entrenamiento.')
        return directory / 'final'
    instances_by_sha = {x.sha: x for x in instances}
    selected = {sha for groups in (grouped, validation) for shas in groups.values() for sha in shas}
    # No se construye ni inspecciona el problema de la geometría heldout.
    problems = {sha: SearchProblem(instances_by_sha[sha], spec, config['time_scale'])
                for sha in sorted(selected)}
    settings = config['training']
    derived_seed = p.seed_for('hybrid-05', fold['fold_id'], seed)
    random.seed(derived_seed)
    torch.manual_seed(derived_seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(derived_seed)
    torch.use_deterministic_algorithms(settings['deterministic_algorithms'])
    torch.backends.cudnn.benchmark = False
    rng = np.random.default_rng(derived_seed)
    model = HBGPolicy(config['model']).to(device)
    initial_state_sha256 = initialization_fingerprint(model)
    flow_parameters = list(model.flow_head.parameters()) + list(model.prefix_encoder.parameters())
    flow_ids = {id(value) for value in flow_parameters}
    policy_parameters = [value for value in model.parameters() if id(value) not in flow_ids]
    optimizer = torch.optim.AdamW([
        {'params': policy_parameters, 'lr': settings['learning_rate']},
        {'params': flow_parameters, 'lr': settings.get('flow_learning_rate', settings['learning_rate'])}],
        weight_decay=settings['weight_decay'])
    best_state, best_loss, best_epoch, stale = None, None, 0, 0
    best_diagnostics, best_geometry_visits = None, None
    history, elapsed_total, validation_total, last_epoch, terminal = [], 0.0, 0.0, 0, None
    geometry_visits, geometry_queue = {group: 0 for group in grouped}, []
    replay = {sha: [] for shas in grouped.values() for sha in shas}
    checkpoint_numbers = set()
    for location in (directory, remote):
        if location.exists():
            checkpoint_numbers.update(int(child.name[6:]) for child in location.glob('epoch_*')
                                      if child.name[6:].isdigit() and (child / 'CHECKSUMS.json').exists())
    if checkpoint_numbers and sorted(checkpoint_numbers) != list(range(1, max(checkpoint_numbers) + 1)):
        raise ValueError('Faltan épocas selladas en la secuencia de checkpoints.')
    for epoch in sorted(checkpoint_numbers):
        if not e._recover(directory / f'epoch_{epoch:04d}', remote / f'epoch_{epoch:04d}'):
            raise ValueError('Checkpoint de época no recuperable.')
    initial_validation, initial_diagnostics = None, None
    if checkpoint_numbers:
        last_epoch = max(checkpoint_numbers)
        latest = directory / f'epoch_{last_epoch:04d}'
        progress = e.read_json(latest / 'progress.json')
        _check_resume(progress, identity, device)
        state = _load_checkpoint(latest / 'state.pt')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        best_state = state['best_model']
        best_loss, best_epoch, stale = progress['best_validation_score'], progress['best_epoch'], progress['stale_epochs']
        best_diagnostics = progress['best_validation_diagnostics']
        best_geometry_visits = progress['selected_checkpoint_training_geometry_visits']
        geometry_visits, geometry_queue = progress['training_geometry_visits'], progress['geometry_queue']
        if progress['initialization_state_sha256'] != initial_state_sha256:
            raise ValueError('La inicialización reconstruida no coincide con el checkpoint.')
        history = progress['history']
        elapsed_total, validation_total = progress['training_seconds'], progress['validation_seconds']
        rng.bit_generator.state = progress['numpy_generator_state']
        random.setstate((progress['python_rng_state'][0], tuple(progress['python_rng_state'][1]), progress['python_rng_state'][2]))
        torch.set_rng_state(state['torch_rng'])
        if device.type == 'cuda':
            torch.cuda.set_rng_state_all(state['cuda_rng'])
        replay = progress['replay']
        validate_replay(replay, problems, grouped, config['hbg'], config['replay']['capacity_per_instance'])
        terminal = progress['stop_reason']
        initial_validation = progress.get('initial_validation_score')
        initial_diagnostics = progress['initial_validation_diagnostics']
    elif time.perf_counter() - started < settings['seconds_per_model']:
        initial_started = time.perf_counter()
        initial_validation, _, initial_diagnostics = validation_cost(
            model, problems, validation, fold['validation_geometry_groups'], device, config['hbg'], return_diagnostics=True, selection_gate=settings.get('checkpoint_gate', 'all_validation_instances'))
        synchronize(device)
        validation_total += time.perf_counter() - initial_started
    synchronize(device)
    elapsed_total += time.perf_counter() - started
    groups = sorted(grouped)
    for epoch in range(last_epoch + 1, settings['epochs'] + 1):
        if terminal:
            break
        if elapsed_total >= settings['seconds_per_model']:
            terminal = 'time_limit'
            break
        epoch_started = time.perf_counter()
        model.train()
        values, costs, fractions, rollouts, updates, decisions = [], [], [], 0, 0, 0
        tb_values, db_values, replay_samples = [], [], 0
        neural_fractions, constructive_seeds = [], 0
        pheromone_deposits = pheromone_attempts = 0
        for offset in range(0, settings['steps_per_epoch'], settings['gradient_accumulation']):
            if elapsed_total + time.perf_counter() - epoch_started >= settings['seconds_per_model']:
                break
            batch_size = min(settings['gradient_accumulation'], settings['steps_per_epoch'] - offset)
            optimizer.zero_grad(set_to_none=True)
            batch_backwards = 0
            for _ in range(batch_size):
                if elapsed_total + time.perf_counter() - epoch_started >= settings['seconds_per_model']:
                    break
                if settings.get('geometry_sampling', 'iid') == 'shuffled_without_replacement':
                    if not geometry_queue:
                        geometry_queue = [str(group) for group in rng.permutation(groups)]
                    group = geometry_queue.pop()
                else:
                    group = groups[int(rng.integers(len(groups)))]
                items = grouped[group]
                sha = items[int(rng.integers(len(items)))]
                problem = problems[sha]
                order = rng.permutation(np.arange(1, problem.n + 1)).tolist() if settings['permute_nodes'] else None
                deadline = epoch_started + max(0., settings['seconds_per_model'] - elapsed_total)
                fresh, stats = collect_colony(model, problem, device, rng, config['hbg'], deadline=deadline)
                if not fresh:
                    break
                old = sample_replay(replay[sha], config['replay']['samples_per_step'], rng)
                examples = fresh + old
                replay_samples += len(old)
                # Cada backward libera su grafo antes de construir la siguiente trayectoria.
                group_losses, group_tb, group_db = [], [], []
                for record in examples:
                    if elapsed_total + time.perf_counter() - epoch_started >= settings['seconds_per_model']:
                        break
                    loss, metrics = trajectory_loss(model, problem, device, record, config['hbg'], node_order=order)
                    if not bool(torch.isfinite(loss).item()):
                        raise ValueError('Pérdida de entrenamiento no finita.')
                    (loss / (batch_size * len(examples))).backward()
                    batch_backwards += 1
                    group_losses.append(float(loss.detach().cpu()))
                    group_tb.append(metrics['tb_loss'])
                    group_db.append(metrics['db_loss'])
                    decisions += metrics['decisions']
                if not group_losses:
                    break
                replay[sha] = insert_elite(replay[sha], fresh, config['replay']['capacity_per_instance'])
                geometry_visits[group] += 1
                values.append(float(np.mean(group_losses)))
                tb_values.append(float(np.mean(group_tb)))
                db_values.append(float(np.mean(group_db)))
                costs.append(stats['mean_cost'])
                fractions.append(stats['complete_fraction'])
                neural_fractions.append(stats.get('neural_complete_fraction', stats['complete_fraction']))
                constructive_seeds += stats.get('constructive_seed_count', 0)
                pheromone_deposits += stats.get('pheromone_deposit_count', 0)
                pheromone_attempts += stats.get('pheromone_guided_attempts', 0)
                rollouts += stats['rollout_count']
            if not batch_backwards or elapsed_total + time.perf_counter() - epoch_started >= settings['seconds_per_model']:
                # No aplicar un batch truncado ni iniciar otro backward fuera de presupuesto.
                optimizer.zero_grad(set_to_none=True)
                break
            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), settings['gradient_clip_norm'], error_if_nonfinite=True)
            if not math.isfinite(float(norm)):
                raise ValueError('Gradientes no finitos.')
            optimizer.step(); updates += 1
        if not values or updates == 0:
            elapsed_total += time.perf_counter() - epoch_started
            terminal = 'time_limit'
            break
        synchronize(device)
        val_started = time.perf_counter()
        val_loss, group_losses, val_diagnostics = validation_cost(
            model, problems, validation, fold['validation_geometry_groups'], device, config['hbg'], return_diagnostics=True, selection_gate=settings.get('checkpoint_gate', 'all_validation_instances'))
        synchronize(device)
        val_seconds = time.perf_counter() - val_started
        if split_strategy == 'independent_train_validation_test_v1':
            val_diagnostics['training_geometry_coverage'] = all(value >= 1 for value in geometry_visits.values())
            val_diagnostics['checkpoint_eligible'] &= val_diagnostics['training_geometry_coverage']
        if validation_candidate_is_better(val_loss, val_diagnostics, best_loss, best_diagnostics, settings['min_delta']):
            best_loss, best_epoch, best_state, stale = val_loss, epoch, _cpu_state(model), 0
            best_diagnostics = val_diagnostics
            best_geometry_visits = dict(geometry_visits)
        else:
            stale += 1
        elapsed_total += time.perf_counter() - epoch_started
        validation_total += val_seconds
        terminal = ('patience' if stale >= settings['patience'] else
                    'time_limit' if elapsed_total >= settings['seconds_per_model'] else
                    'epochs' if epoch == settings['epochs'] else None)
        history.append({'epoch': epoch, 'train_loss': float(np.mean(values)), 'train_cost': float(np.mean(costs)),
                        'train_complete_fraction': float(np.mean(fractions)), 'rollout_count': rollouts,
                        'validation_score': val_loss,
                        'validation_complete_fraction': val_diagnostics['complete_fraction'],
                        'validation_instances_with_completion_fraction': val_diagnostics['instances_with_completion_fraction'],
                        'validation_checkpoint_eligible': val_diagnostics['checkpoint_eligible'],
                        'best_validation_score': best_loss, 'best_epoch': best_epoch,
                        'instance_groups_sampled': len(values), 'decisions': decisions, 'optimizer_updates': updates,
                        'neural_complete_fraction': float(np.mean(neural_fractions)),
                        'constructive_seed_count': constructive_seeds,
                        'pheromone_deposit_count': pheromone_deposits,
                        'pheromone_guided_attempts': pheromone_attempts,
                        'train_tb_loss': float(np.mean(tb_values)), 'train_db_loss': float(np.mean(db_values)),
                        'replay_samples': replay_samples, 'replay_entries': sum(map(len, replay.values())),
                        'training_seconds': elapsed_total, 'validation_seconds': validation_total})
        target = directory / f'epoch_{epoch:04d}'
        # Preservar una escritura interrumpida de esta época antes de repetirla.
        if e._recover(target, remote / target.name):
            raise ValueError('La época nueva ya existe; no sobrescribir un checkpoint.')
        target.mkdir(parents=True, exist_ok=True)
        torch.save({'model': _cpu_state(model), 'optimizer': optimizer.state_dict(), 'best_model': best_state,
                    'torch_rng': torch.get_rng_state(),
                    'cuda_rng': torch.cuda.get_rng_state_all() if device.type == 'cuda' else []}, target / 'state.pt')
        e.write_json(target / 'progress.json', {'epoch': epoch, 'identity_sha256': p.digest(identity),
                     'runtime_signature': _runtime_signature(device), 'initial_validation_score': initial_validation,
                     'best_validation_score': best_loss,
                     'best_epoch': best_epoch, 'best_validation_diagnostics': best_diagnostics,
                     'initialization_state_sha256': initial_state_sha256,
                     'initial_validation_diagnostics': initial_diagnostics,
                     'stale_epochs': stale, 'history': history,
                     'training_geometry_visits': geometry_visits, 'geometry_queue': geometry_queue,
                     'selected_checkpoint_training_geometry_visits': best_geometry_visits,
                     'training_seconds': elapsed_total, 'validation_seconds': validation_total,
                     'numpy_generator_state': rng.bit_generator.state, 'python_rng_state': random.getstate(),
                     'validation_by_geometry': group_losses, 'replay': replay, 'stop_reason': terminal})
        e.write_json(target / 'environment.json', p.environment_report())
        e._commit(target, remote / target.name)
        print(f'{fold["fold_id"]} seed={seed} epoch={epoch}: train={history[-1]["train_loss"]:.5f}, '
              f'val={val_loss:.5f}, best={best_epoch}, '
              f'completos val={val_diagnostics["complete_fraction"]:.1%}, '
              f'instancias completas={val_diagnostics["instances_with_completion_fraction"]:.1%}', flush=True)
    if not history:
        raise RuntimeError('No hay checkpoint entrenado; no evaluar pesos aleatorios.')
    final = directory / 'final'
    final.mkdir(parents=True, exist_ok=True)
    selection_status = 'selected_complete_validation' if best_state is not None else 'validation_gate_failed'
    checkpoint = final / 'best.pt' if best_state is not None else directory / f'epoch_{len(history):04d}/state.pt'
    if best_state is not None:
        torch.save(best_state, checkpoint)
    summary = {'identity_sha256': p.digest(identity), 'fold': fold, 'training_seed': seed,
               'training_seconds': elapsed_total, 'validation_seconds': validation_total,
               'training_budget_seconds': settings['seconds_per_model'],
               'training_budget_overrun_seconds': max(0., elapsed_total - settings['seconds_per_model']),
               'training_time_scope': 'setup_and_epoch_compute_validation_excludes_epoch_and_final_checkpoint_persistence',
               'teacher_seconds': 0.0,
               'selection_status': selection_status, 'best_validation_diagnostics': best_diagnostics,
               'split_strategy': split_strategy, 'training_geometry_visits': geometry_visits,
               'selected_checkpoint_training_geometry_visits': best_geometry_visits,
               'initialization_state_sha256': initial_state_sha256,
               'initial_validation_diagnostics': initial_diagnostics,
               'best_epoch': best_epoch, 'best_validation_score': best_loss, 'epochs_completed': len(history),
               'initial_validation_score': initial_validation,
               'improved_over_initial_validation': best_loss is not None and initial_validation is not None and best_loss < initial_validation - settings['min_delta'],
               'optimizer_updates': sum(row['optimizer_updates'] for row in history),
               'constructive_seed_count': sum(row.get('constructive_seed_count', 0) for row in history),
               'pheromone_deposit_count': sum(row['pheromone_deposit_count'] for row in history),
               'pheromone_guided_attempts': sum(row['pheromone_guided_attempts'] for row in history),
               'neural_complete_fraction_mean': float(np.mean([row.get('neural_complete_fraction', 0.) for row in history])),
               'ready_for_confirmatory_comparison': False,
               'stop_reason': terminal, 'model_config': config['model'], 'traffic': asdict(spec),
               'time_scale': config['time_scale'], 'objective': OBJECTIVE,
               'checkpoint_selection': settings.get('checkpoint_selection', CHECKPOINT_SELECTION) if best_state is not None else 'rejected_no_complete_validation',
               'checkpoint_artifact_kind': 'selected_weights' if best_state is not None else 'unselected_epoch_state_for_audit_only',
               'training_scope': ('development_geometry_crossfit' if split_strategy == 'development_crossfit' else 'independent_train_validation_for_final_holdout') + '_conservative_hbg_facs',
               'replay_entries': sum(map(len, replay.values())),
               'replay_samples': sum(row['replay_samples'] for row in history),
               'hbg_config': config['hbg'], 'replay_config': config['replay'],
               'instance_groups_sampled': sum(row['instance_groups_sampled'] for row in history),
               'train_instances': sum(map(len, grouped.values())),
               'validation_instances': sum(map(len, validation.values())),
               'train_geometries': len(grouped), 'heldout_used_for_training_or_selection': False,
               'model_checkpoint_sha256': p.file_hash(checkpoint)}
    e.write_json(final / 'summary.json', summary)
    e.write_json(final / 'history.json', history)
    e.write_json(final / 'replay.json', replay)
    e.write_csv(final / 'history.csv', history)
    save_learning_plot(history, final / 'learning.png', fold['fold_id'], seed)
    e._commit(final, remote / 'final')
    del model, optimizer
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return final


def save_learning_plot(history, path, fold_id, seed):
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs = [row['epoch'] for row in history]
    for metric in ['train_cost', 'validation_score']:
        axes[0].plot(epochs, [row[metric] for row in history], label=metric)
    for metric in ['train_loss', 'train_tb_loss', 'train_db_loss']:
        axes[1].plot(epochs, [row[metric] for row in history], label=metric)
    axes[0].set(title='Costo conservador penalizado', xlabel='Época', ylabel='Costo normalizado')
    axes[1].set(title='Balance de trayectoria y detallado', xlabel='Época', ylabel='Pérdida')
    for ax in axes:
        ax.legend()
    fig.suptitle(f'{fold_id} · seed={seed} · HBG-FACS')
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def train_crossfit(instances, spec, config, seeds, work_root, drive_root, bank_id,
                   bank_sha256, evaluator_sha256, device='auto'):
    require_colab()
    validate_training_config(config)
    if (not seeds or any(type(seed) is not int or seed < 0 for seed in seeds)
            or len(set(seeds)) != len(seeds)):
        raise ValueError('Semillas de entrenamiento vacías, repetidas o inválidas.')
    if evaluator_sha256 != p.file_hash(p.__file__):
        raise ValueError('Evaluador de entrenamiento distinto del módulo sellado activo.')
    device = _training_device(config, device)
    folds = make_geometry_folds(instances)
    identity = {'config': config, 'seeds': seeds, 'folds': folds,
                'runtime_signature': _runtime_signature(device),
                'bank_sha256': bank_sha256,
                'instance_sha256': sorted(x.sha for x in instances),
                'evaluator_sha256': evaluator_sha256, 'source_hashes': _source_hashes()}
    training_id = 'hbg05_' + p.digest(identity)[:20]
    root = Path(work_root) / 'hbg_training' / bank_id / training_id
    remote = Path(drive_root) / 'hbg_training' / bank_id / training_id
    print('TRAINING_ID =', training_id, '| dispositivo:', device, flush=True)
    print('Entorno de entrenamiento:', identity['runtime_signature'], flush=True)
    print('Carpeta por código/configuración/entorno:', remote,
          '| se recuperan checkpoints compatibles; los demás se conservan.', flush=True)
    if e._recover(root / 'meta', remote / 'meta'):
        if e.read_json(root / 'meta/identity.json') != identity:
            raise ValueError('Identidad de entrenamiento inconsistente.')
    else:
        e.write_json(root / 'meta/identity.json', identity)
        e.write_json(root / 'meta/folds.json', folds)
        for name in _source_hashes():
            target = root / 'meta/code' / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(Path(__file__).with_name(name).read_bytes())
        e._commit(root / 'meta', remote / 'meta')
    models = []
    for fold in folds:
        for seed in seeds:
            relative = Path(fold['fold_id']) / f'seed_{seed}'
            attempt_started = time.perf_counter()
            try:
                final = train_fold(instances, spec, config, seed, fold,
                                   root / relative, remote / relative, device)
            except Exception as exc:
                failure = root / 'failures' / (fold['fold_id'] + f'_seed_{seed}_' + uuid.uuid4().hex[:8])
                e.write_json(failure / 'failure.json', {'fold': fold, 'seed': seed, 'training_id': training_id,
                             'error_type': type(exc).__name__, 'error': str(exc),
                             'observed_call_wall_seconds_including_io': time.perf_counter() - attempt_started,
                             'completed_checkpoints_preserved': True})
                e.write_json(failure / 'environment.json', p.environment_report())
                e._commit(failure, remote / 'failures' / failure.name)
                raise
            models.append({**e.read_json(final / 'summary.json'), 'relative_directory': str(relative / 'final'),
                           'bundle_sha256': p.verify_bundle(final)})
    # Toda selección terminó antes de que el notebook invoque evaluate sobre heldout.
    if not e._recover(root / 'index', remote / 'index'):
        e.write_json(root / 'index/models.json', models)
        e.write_json(root / 'index/offline_costs.json', {'offline_cost_id': training_id,
                     'teacher_seconds_unique': 0.0,
                     'training_seconds_unique': sum(row['training_seconds'] for row in models),
                     'validation_seconds_included_in_training': sum(row['validation_seconds'] for row in models),
                     'models_trained': len(models),
                     'models_selected': sum(row['selection_status'] == 'selected_complete_validation' for row in models),
                     'validation_gate_failures': sum(row['selection_status'] == 'validation_gate_failed' for row in models),
                     'geometry_count': len(folds),
                     'summary_times_must_not_be_summed_across_policies_or_scenarios': True})
        e.write_csv(root / 'index/training_summary.csv', [{k: v for k, v in row.items()
                                                         if not isinstance(v, (dict, list))} for row in models])
        e._commit(root / 'index', remote / 'index')
    if e.read_json(root / 'index/models.json') != models:
        raise ValueError('Inventario de modelos distinto de los checkpoints.')
    return root


def make_solver(training_root, instances, spec, device='auto', expected_bank_sha256=None):
    """Carga sólo el modelo que dejó fuera la geometría de la instancia solicitada."""
    root, device = Path(training_root), choose_device(device)
    p.verify_bundle(root / 'meta')
    identity = e.read_json(root / 'meta/identity.json')
    if expected_bank_sha256 is not None and identity['bank_sha256'] != expected_bank_sha256:
        raise ValueError('Entrenamiento vinculado a otro banco.')
    validate_training_config(identity['config'])
    if (identity['evaluator_sha256'] != p.file_hash(p.__file__)
            or identity['source_hashes'] != _source_hashes()):
        raise ValueError('Código o evaluador distintos de los usados para entrenar.')
    if identity['instance_sha256'] != sorted(x.sha for x in instances):
        raise ValueError('Instancias distintas del registro de entrenamiento.')
    if identity['folds'] != make_geometry_folds(instances):
        raise ValueError('Folds distintos de los prefijados por geometría.')
    p.verify_bundle(root / 'index')
    index = e.read_json(root / 'index/models.json')
    registry, provenance = {}, {}
    expected_pairs = {(fold['fold_id'], seed) for fold in identity['folds'] for seed in identity['seeds']}
    seen_pairs = set()
    for model in index:
        relative = Path(model['relative_directory'])
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Ruta de checkpoint fuera del entrenamiento.')
        final = root / relative
        if p.verify_bundle(final) != model['bundle_sha256']:
            raise ValueError('Checkpoint de modelo modificado.')
        summary = e.read_json(final / 'summary.json')
        selected_checkpoint_path(final, summary)
        if model != {**summary, 'relative_directory': str(relative), 'bundle_sha256': model['bundle_sha256']}:
            raise ValueError('Inventario de modelos no coincide con el resumen sellado.')
        if model['traffic'] != asdict(spec):
            raise ValueError('Modelo entrenado con otro tráfico conocido.')
        fold = model['fold']
        validate_fold(fold, instances)
        pair = (fold['fold_id'], model['training_seed'])
        if pair not in expected_pairs or pair in seen_pairs or fold not in identity['folds']:
            raise ValueError('Fold o semilla fuera del plan de entrenamiento, o repetidos.')
        seen_pairs.add(pair)
        if (model['time_scale'] != identity['config']['time_scale'] or model['objective'] != OBJECTIVE
                or model['model_config'] != identity['config']['model']
                or model['hbg_config'] != identity['config']['hbg']
                or model['replay_config'] != identity['config']['replay']):
            raise ValueError('Modelo y configuración de entrenamiento incompatibles.')
        p.verify_bundle(final.parent / 'meta')
        fold_identity = e.read_json(final.parent / 'meta/identity.json')
        if (p.digest(fold_identity) != model['identity_sha256']
                or fold_identity['fold'] != fold or fold_identity['seed'] != model['training_seed']
                or fold_identity['config'] != identity['config']
                or fold_identity['source_hashes'] != identity['source_hashes']
                or fold_identity['traffic'] != asdict(spec)
                or fold_identity['evaluator_sha256'] != identity['evaluator_sha256']
                or fold_identity['train_instances'] != _group_instances(instances, fold['train_geometry_groups'])
                or fold_identity['validation_instances'] != _group_instances(instances, fold['validation_geometry_groups'])):
            raise ValueError('Identidad del modelo no corresponde a su fold y datos autorizados.')
        for x in instances:
            if x.provenance['geometry_group_sha256'] not in fold['heldout_geometry_groups']:
                continue
            key = f'{x.sha}:{model["training_seed"]}'
            if key in registry:
                raise ValueError('Dos modelos compiten por una observación heldout.')
            registry[key] = (final, model)
            provenance[key] = {key_: model[key_] for key_ in ['training_seed', 'training_seconds',
                'validation_seconds', 'teacher_seconds', 'model_checkpoint_sha256', 'checkpoint_selection', 'training_scope']}
            provenance[key].update(training_id=root.name, fold_id=fold['fold_id'],
                train_geometry_groups=fold['train_geometry_groups'],
                validation_geometry_groups=fold['validation_geometry_groups'],
                heldout_geometry_groups=fold['heldout_geometry_groups'], offline_cost_id=root.name,
                offline_manifest_sha256=p.verify_bundle(root / 'index'),
                selection_status=model['selection_status'], initialization_state_sha256=model['initialization_state_sha256'])
    if seen_pairs != expected_pairs:
        raise ValueError('Faltan modelos para completar el plan de entrenamiento.')

    def solver(problem, seed, config, output_dir):
        started = time.perf_counter()
        key = f'{problem.x.sha}:{seed}'
        if key not in registry:
            raise ValueError('No existe modelo fuera de geometría para esta instancia/semilla.')
        final, record = registry[key]
        for name in ('temperature', 'lambda_db'):
            if config.get(name, record['hbg_config'][name]) != record['hbg_config'][name]:
                raise ValueError('Recompensa o mezcla de balance distinta del modelo entrenado.')
        config = {**record['hbg_config'], **config}
        if problem.time_scale != record['time_scale']:
            raise ValueError('Escala temporal de inferencia distinta del entrenamiento.')
        selected_checkpoint_path(final, record)
        if record['selection_status'] == 'validation_gate_failed':
            return problem.result(None, 'validation_gate_failed', time.perf_counter() - started, [],
                method='hbg_facs', learned=False, validation_gate_failed=True, heldout_inference_executed=False,
                model_checkpoint_sha256=record['model_checkpoint_sha256'], fold_id=record['fold']['fold_id'],
                initialization_state_sha256=record['initialization_state_sha256'],
                checkpoint_artifact_kind='unselected_epoch_state_for_audit_only',
                no_solution_scope='no_checkpoint_met_prespecified_validation_completion_gate')
        model = HBGPolicy(record['model_config']).to(device)
        model.load_state_dict(_load_checkpoint(final / 'best.pt'))
        model.eval(); synchronize(device)
        loading = time.perf_counter() - started
        remaining = config['seconds'] - loading
        try:
            if remaining <= 0:
                result = problem.result(None, 'time_limit_model_load', loading, [])
            else:
                result = solve_hbg(problem, seed, {**config, 'seconds': remaining}, model, device)
            result['metadata'].update(model_load_seconds=loading, inference_device=str(device),
                                      model_config=record['model_config'], mask_version=MASK_VERSION,
                                      initialization_seed=p.seed_for('hybrid-05', record['fold']['fold_id'], seed),
                                      initialization_state_sha256=record['initialization_state_sha256'],
                                      selection_status=record['selection_status'],
                                      model_checkpoint_sha256=record['model_checkpoint_sha256'],
                                      fold_id=record['fold']['fold_id'])
            synchronize(device)
            result['solve_seconds'] = time.perf_counter() - started
            return result
        finally:
            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
    return solver, {'hbg_facs': provenance}


def make_untrained_solver(instances, config, device='auto', expected_initializations=None):
    """Control de inicialización con la MISMA colonia/EDD/LS y presupuesto del 05."""
    require_colab()
    validate_training_config(config)
    device = choose_device(device)
    folds = make_geometry_folds(instances)
    by_group = {group: fold for fold in folds for group in fold['heldout_geometry_groups']}

    def solver(problem, seed, settings, output_dir):
        started = time.perf_counter()
        fold = by_group[problem.x.provenance['geometry_group_sha256']]
        derived_seed = p.seed_for('hybrid-05', fold['fold_id'], seed)
        torch.manual_seed(derived_seed)
        if device.type == 'cuda':
            torch.cuda.manual_seed_all(derived_seed)
        model = HBGPolicy(config['model']).to(device)
        try:
            fingerprint = initialization_fingerprint(model)
            key = f'{problem.x.sha}:{seed}'
            if expected_initializations is not None and expected_initializations.get(key) != fingerprint:
                raise ValueError('El control sin entrenar difiere de la inicialización del entrenamiento.')
            model.eval(); synchronize(device)
            loading = time.perf_counter() - started
            effective = {**config['hbg'], **settings, 'seconds': max(0., settings['seconds'] - loading)}
            result = solve_hbg(problem, seed, effective, model, device)
            result['metadata'].update(method='hbg_untrained', learned=False,
                                      model_config=config['model'], mask_version=MASK_VERSION,
                                      initialization_seed=derived_seed, initialization_state_sha256=fingerprint,
                                      fold_id=fold['fold_id'],
                                      training_seconds=0., model_load_seconds=loading,
                                      control='same_initialization_paid_edd_colony_local_search_without_training')
            synchronize(device)
            result['solve_seconds'] = time.perf_counter() - started
            return result
        finally:
            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
    return solver


Overwriting svrplab/hybrid.py


In [51]:
%%writefile svrplab/metaheuristics.py
"""Transparent classical comparators for the conservative fixed-fleet problem.

Own ACO and Tabu variants, not official reproductions of a named paper. Both pay
for EDD feasible insertion, repair and local search within their declared wall
clock budget. ACO guides feasible insertions with shared directed pheromone and
improves constructed solutions by relocate descent. Tabu uses relocate, swap,
2-opt and inter-route 2-opt-star. Certificates and scenarios are never read.

max_iterations=None means time-limited search; an integer remains available for
controlled reproducibility tests. Every completed incumbent survives timeout;
unsuccessful constructions are recorded, not labelled proof of infeasibility.
Execute construction/search exclusively in Colab.
"""
from __future__ import annotations

import math
import itertools
import time
from collections.abc import Mapping

import numpy as np

from .search import SearchProblem
from .baselines import construct_insertion, improve_feasible_routes


ACO_DEFAULTS = {
    'seconds': 120., 'max_iterations': None, 'ants': 20,
    'alpha': 1., 'beta': 2., 'evaporation': .1, 'q0': .2,
    'local_search_seconds': 1., 'local_search_moves': 20,
}
TABU_DEFAULTS = {
    'seconds': 120., 'max_iterations': None, 'neighborhood_samples': 80,
    'tenure': 15, 'initial_restarts': 4,
    'local_search_seconds': 1., 'local_search_moves': 20,
}
_TOL = 1e-9


def _configuration(defaults, config):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración debe ser un mapping.')
    unknown = set(config or {}) - set(defaults)
    if unknown:
        raise ValueError(f'Parámetros desconocidos: {sorted(unknown)}')
    result = defaults | dict(config or {})
    for name, value in result.items():
        if name == 'max_iterations' and value is None:
            continue
        if isinstance(value, (bool, np.bool_)) or not isinstance(value, (int, float, np.integer, np.floating)):
            raise ValueError(f'Parámetro numérico inválido: {name}')
        if not math.isfinite(float(value)):
            raise ValueError(f'Parámetro no finito: {name}')
    for name in ('max_iterations', 'ants', 'neighborhood_samples', 'tenure', 'initial_restarts', 'local_search_moves'):
        if name in result and result[name] is not None:
            if int(result[name]) != result[name] or result[name] < 1:
                raise ValueError(f'{name} debe ser entero positivo.')
            result[name] = int(result[name])
    if result['seconds'] < 0 or result['local_search_seconds'] < 0:
        raise ValueError('Los presupuestos deben ser no negativos.')
    for name in ('alpha', 'beta'):
        if name in result and result[name] < 0:
            raise ValueError(f'{name} debe ser no negativo.')
    if 'evaporation' in result and not 0 < result['evaporation'] <= 1:
        raise ValueError('evaporation debe pertenecer a (0, 1].')
    if 'q0' in result and not 0 <= result['q0'] <= 1:
        raise ValueError('q0 debe pertenecer a [0, 1].')
    return {k: (v if v is None or isinstance(v, int) else float(v)) for k, v in result.items()}


def _seed_value(seed):
    if isinstance(seed, (bool, np.bool_)) or not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('La semilla debe ser un entero no negativo.')
    return int(seed)


def _capacity_screen(problem):
    """Condiciones necesarias; pasarlas no demuestra que exista partición."""
    if problem.vehicles == 0:
        return problem.n == 0
    return (float(np.sum(problem.demands[1:])) <= float(np.sum(problem.capacities)) + _TOL
            and (problem.n == 0 or float(np.max(problem.demands[1:]))
                 <= float(np.max(problem.capacities)) + _TOL))


def _history_row(started, iteration, score, phase, **extra):
    return {'iteration': int(iteration), 'elapsed_seconds': time.perf_counter() - started,
            'violation_minutes': float(score[0]), 'travel_minutes': float(score[1]),
            'phase': phase, **extra}


def _finish(problem, routes, score, started, config, seed, history, method,
            termination, iterations, **extra):
    elapsed = time.perf_counter() - started
    status = ('no_solution' if routes is None else
              'feasible' if score[0] <= _TOL else 'conservative_infeasible')
    result = problem.result(
        routes, status, elapsed, history, method=method, seed=seed,
        effective_config=config, termination=termination, iterations=int(iterations),
        implementation='project_own_insertion_local_search_v2',
        construction='from_scratch_without_certificates',
        search_objective='lexicographic_temporal_violation_then_conservative_travel_minutes',
        nominal_scope='conservative_upper_mean_matrix_not_stochastic_feasibility',
        budget_includes_construction=True, budget_includes_repair=True,
        initialization='paid_edd_feasible_cheapest_insertion',
        budget_overrun_seconds=max(0., elapsed - config['seconds']),
        time_limit_granularity='construction_step_or_neighborhood_candidate',
        seed_reproducibility='same_environment_and_iteration_cutoff; wall_time_cutoff_may_differ',
        **extra)
    # Incluir también la validación final del contrato común en el tiempo medido.
    elapsed = time.perf_counter() - started
    result['solve_seconds'] = elapsed
    result['metadata']['budget_overrun_seconds'] = max(0., elapsed - config['seconds'])
    return result


def _construct(problem, rng, deadline, pheromone, alpha, beta, q0):
    return construct_insertion(problem, rng, deadline, pheromone, alpha, beta, q0)


def _iterations(limit):
    return itertools.count(1) if limit is None else range(1, limit + 1)


def _improve(problem, routes, rng, deadline, config):
    """Per-construction local search remains inside the shared search deadline."""
    if time.perf_counter() >= deadline or config['local_search_seconds'] <= 0:
        return routes, {'evaluated_neighbors': 0, 'accepted_moves': 0}
    local_deadline = min(deadline, time.perf_counter() + config['local_search_seconds'])
    return improve_feasible_routes(problem, routes, rng, local_deadline,
                                   max_moves=config['local_search_moves'])


def _arcs(routes):
    """Atributos dirigidos con vehículo, sin arco artificial 0→0 para vacíos."""
    return {(v, int(a), int(b)) for v, route in enumerate(routes) if route
            for a, b in zip([0] + list(route), list(route) + [0])}


def _reinforce(pheromone, routes, weight):
    for route in routes:
        if route:
            for a, b in zip([0] + list(route), list(route) + [0]):
                pheromone[a, b] += weight


def solve_aco(problem: SearchProblem, seed, config=None):
    """EDD seed + pheromone-guided feasible insertion + paid relocate descent.

    Rank deposits: global best 1 and iteration best .5 after evaporation.
    The method remains a declared project variant, not a reproduction claim.
    """
    started = time.perf_counter()
    cfg = _configuration(ACO_DEFAULTS, config)
    seed = _seed_value(seed)
    deadline = started + cfg['seconds']
    rng = np.random.default_rng(seed)
    best, best_score, history = None, None, []
    completed_iterations = constructed = capacity_dead_ends = 0
    local_evaluated = local_moves = 0
    termination = 'iteration_limit'
    if not _capacity_screen(problem):
        return _finish(problem, best, best_score, started, cfg, seed, history, 'aco',
                       'capacity_screen_failed', 0, constructed_solutions=0)
    pheromone = np.ones((problem.n + 1, problem.n + 1))
    if time.perf_counter() < deadline:
        routes, reason = _construct(problem, rng, deadline, pheromone, 0., cfg['beta'], 1.)
        if routes is not None:
            routes, diagnostics = _improve(problem, routes, rng, deadline, cfg)
            local_evaluated += diagnostics['evaluated_neighbors']
            local_moves += diagnostics['accepted_moves']
            best, best_score = routes, problem.score(routes)
            constructed += 1
            history.append(_history_row(started, 0, best_score, 'paid_edd_initialization'))
        elif reason == 'capacity_dead_end':
            capacity_dead_ends += 1
    for iteration in _iterations(cfg['max_iterations']):
        if time.perf_counter() >= deadline:
            termination = 'time_limit'
            break
        iteration_best, iteration_score = None, None
        for _ in range(cfg['ants']):
            if time.perf_counter() >= deadline:
                termination = 'time_limit'
                break
            routes, reason = _construct(problem, rng, deadline, pheromone,
                                        cfg['alpha'], cfg['beta'], cfg['q0'])
            if routes is None:
                if reason == 'time_limit':
                    termination = 'time_limit'
                    break
                capacity_dead_ends += 1
                continue
            routes, diagnostics = _improve(problem, routes, rng, deadline, cfg)
            local_evaluated += diagnostics['evaluated_neighbors']
            local_moves += diagnostics['accepted_moves']
            score = problem.score(routes)
            constructed += 1
            if iteration_score is None or score < iteration_score:
                iteration_best, iteration_score = routes, score
            if best_score is None or score < best_score:
                best, best_score = routes, score
                history.append(_history_row(started, iteration, best_score, 'incumbent',
                                            constructed_solutions=constructed))
        if termination == 'time_limit':
            break
        pheromone *= 1. - cfg['evaporation']
        if iteration_best is not None:
            _reinforce(pheromone, iteration_best, .5)
        if best is not None:
            _reinforce(pheromone, best, 1.)
        np.clip(pheromone, 1e-6, 1e6, out=pheromone)
        completed_iterations = iteration
        if best_score is not None:
            history.append(_history_row(started, iteration, best_score, 'iteration',
                                        constructed_solutions=constructed))
    if time.perf_counter() >= deadline:
        termination = 'time_limit'
    return _finish(problem, best, best_score, started, cfg, seed, history, 'aco',
                   termination, completed_iterations, constructed_solutions=constructed,
                   capacity_dead_ends=capacity_dead_ends,
                   local_search_evaluated_neighbors=local_evaluated, local_search_accepted_moves=local_moves,
                   construction_rule='EDD_RCL_feasible_insertion_with_pheromone',
                   pheromone_rule='global_best_1_plus_iteration_best_0.5_after_evaporation')


def _apply_move(routes, kind, first, second):
    """Transformación pura; posiciones de relocate se refieren tras extracción."""
    candidate = [list(route) for route in routes]
    a, i = first
    b, j = second
    if kind == 'relocate':
        customer = candidate[a].pop(i)
        candidate[b].insert(j, customer)
    elif kind == 'swap':
        candidate[a][i], candidate[b][j] = candidate[b][j], candidate[a][i]
    elif kind == '2opt_star':
        if a == b:
            raise ValueError('2-opt-star requiere rutas distintas.')
        candidate[a], candidate[b] = candidate[a][:i] + candidate[b][j:], candidate[b][:j] + candidate[a][i:]
    elif kind == '2opt':
        if a != b:
            raise ValueError('2-opt requiere una sola ruta.')
        lo, hi = sorted((i, j))
        candidate[a][lo:hi + 1] = reversed(candidate[a][lo:hi + 1])
    else:
        raise ValueError(f'Movimiento desconocido: {kind}')
    return candidate


def _random_neighbor(routes, rng):
    positions = [(v, i) for v, route in enumerate(routes) for i in range(len(route))]
    if not positions:
        return None
    kind = ('relocate', 'swap', '2opt', '2opt_star')[int(rng.integers(4))]
    if kind == 'relocate':
        first = positions[int(rng.integers(len(positions)))]
        target = int(rng.integers(len(routes)))
        target_length = len(routes[target]) - int(target == first[0])
        second = (target, int(rng.integers(target_length + 1)))
    elif kind == '2opt_star':
        if len(routes) < 2:
            return None
        a, b = map(int, rng.choice(len(routes), size=2, replace=False))
        first = (a, int(rng.integers(len(routes[a]) + 1)))
        second = (b, int(rng.integers(len(routes[b]) + 1)))
    elif kind == 'swap':
        if len(positions) < 2:
            return None
        left, right = rng.choice(len(positions), size=2, replace=False)
        first, second = positions[int(left)], positions[int(right)]
    else:
        eligible = [v for v, route in enumerate(routes) if len(route) >= 2]
        if not eligible:
            return None
        vehicle = int(rng.choice(eligible))
        left, right = rng.choice(len(routes[vehicle]), size=2, replace=False)
        first, second = (vehicle, int(left)), (vehicle, int(right))
    candidate = _apply_move(routes, kind, first, second)
    return None if candidate == routes else (candidate, kind)


def _tabu_admissible(added_arcs, tabu_until, iteration, score, best_score):
    # Aspiración estricta: únicamente superar al mejor global lexicográfico.
    return score < best_score or not any(iteration < tabu_until.get(arc, 0) for arc in added_arcs)


def _capacity_valid(problem, routes):
    return all(sum(float(problem.demands[j]) for j in route) <= problem.capacities[v] + _TOL
               for v, route in enumerate(routes))


def solve_tabu(problem: SearchProblem, seed, config=None):
    """Búsqueda tabú con multiaranque propio y vecinos de capacidad válida.

    Acepta deterioro y violaciones temporales para explorar. Un arco eliminado
    queda prohibido para reinserción durante las próximas ``tenure`` iteraciones;
    así la reversión de relocate, swap y 2-opt es tabú, salvo aspiración. Si el
    muestreo no encuentra movimiento admisible se avanza la iteración (expira la
    memoria) y se vuelve a muestrear, sin declarar óptimo local ni infactibilidad.
    """
    started = time.perf_counter()
    cfg = _configuration(TABU_DEFAULTS, config)
    seed = _seed_value(seed)
    deadline = started + cfg['seconds']
    rng = np.random.default_rng(seed)
    best, best_score, history = None, None, []
    restarts = capacity_dead_ends = iterations = accepted = evaluated = 0
    local_evaluated = local_moves = 0
    termination = 'iteration_limit'
    if not _capacity_screen(problem):
        return _finish(problem, best, best_score, started, cfg, seed, history, 'tabu',
                       'capacity_screen_failed', 0, initial_attempts=0)
    # Construcción determinista inicial, seguida de intentos estocásticos;
    # todos se cobran al mismo reloj y ninguno usa una solución de otro método.
    neutral_pheromone = np.ones((problem.n + 1, problem.n + 1))
    for restart in range(cfg['initial_restarts']):
        if time.perf_counter() >= deadline:
            termination = 'time_limit'
            break
        restarts += 1
        routes, reason = _construct(problem, rng, deadline, neutral_pheromone,
                                    alpha=0., beta=2., q0=1. if restart == 0 else .5)
        if routes is None:
            if reason == 'time_limit':
                termination = 'time_limit'
                break
            capacity_dead_ends += 1
            continue
        routes, diagnostics = _improve(problem, routes, rng, deadline, cfg)
        local_evaluated += diagnostics['evaluated_neighbors']
        local_moves += diagnostics['accepted_moves']
        score = problem.score(routes)
        if best_score is None or score < best_score:
            best, best_score = routes, score
            history.append(_history_row(started, 0, best_score, 'initialization', restart=restart))
    if best is None:
        if termination != 'time_limit':
            termination = 'construction_failed'
        return _finish(problem, best, best_score, started, cfg, seed, history, 'tabu',
                       termination, 0, initial_attempts=restarts, capacity_dead_ends=capacity_dead_ends)
    current = [list(route) for route in best]
    current_score = best_score
    tabu_until = {}
    for iteration in _iterations(cfg['max_iterations']):
        if time.perf_counter() >= deadline:
            termination = 'time_limit'
            break
        current_arcs = _arcs(current)
        chosen = chosen_score = chosen_removed = chosen_kind = None
        seen = set()
        for _ in range(cfg['neighborhood_samples']):
            if time.perf_counter() >= deadline:
                termination = 'time_limit'
                break
            proposed = _random_neighbor(current, rng)
            if proposed is None:
                continue
            candidate, kind = proposed
            key = tuple(tuple(route) for route in candidate)
            if key in seen:
                continue
            seen.add(key)
            if not _capacity_valid(problem, candidate):
                continue
            score = problem.score(candidate)
            evaluated += 1
            candidate_arcs = _arcs(candidate)
            added = candidate_arcs - current_arcs
            if not _tabu_admissible(added, tabu_until, iteration, score, best_score):
                continue
            if chosen_score is None or score < chosen_score:
                chosen, chosen_score = candidate, score
                chosen_removed, chosen_kind = current_arcs - candidate_arcs, kind
        if chosen is not None:
            current, current_score = chosen, chosen_score
            accepted += 1
            for arc in chosen_removed:
                tabu_until[arc] = iteration + cfg['tenure'] + 1
            if current_score < best_score:
                best, best_score = [list(route) for route in current], current_score
        tabu_until = {arc: expiry for arc, expiry in tabu_until.items() if expiry > iteration}
        iterations = iteration
        history.append(_history_row(started, iteration, best_score, 'iteration',
                                    current_violation_minutes=float(current_score[0]),
                                    current_travel_minutes=float(current_score[1]),
                                    accepted_move=chosen_kind, evaluated_neighbors=evaluated))
        if termination == 'time_limit':
            break
    if time.perf_counter() >= deadline:
        termination = 'time_limit'
    return _finish(problem, best, best_score, started, cfg, seed, history, 'tabu',
                   termination, iterations, initial_attempts=restarts,
                   capacity_dead_ends=capacity_dead_ends, evaluated_neighbors=evaluated,
                   accepted_moves=accepted, local_search_evaluated_neighbors=local_evaluated,
                   local_search_accepted_moves=local_moves, tabu_rule='removed_directed_vehicle_arcs_forbid_readdition',
                   aspiration='strict_global_best_lexicographic_improvement')


Overwriting svrplab/metaheuristics.py


In [52]:
%%writefile svrplab/pilot.py
"""Notebook 00: datos, auditoría, CRN y evaluación independiente del solver.

Adaptación del generador de tráfico de SVRPBench (MIT, Ahmed Heakl, 2024).
Ver docs/SVRPBench-LICENSE.txt y docs/NOTEBOOK_00.md para las diferencias.
No contiene entrenamiento HBG/AM/POMO ni demuestra superioridad estadística.
"""
from __future__ import annotations

import hashlib
import importlib
import importlib.metadata
import json
import math
import os
from pathlib import Path
import platform
import random
import shutil
import subprocess
import sys
import time
import urllib.request
import warnings
from contextlib import contextmanager
from dataclasses import asdict, dataclass

import numpy as np

UPSTREAM_SHA = '9fd5737ae598e1667ae94907f9c45224c501c84b'
HF_REVISION = '4a79ade38f663edaefa62e34e7c2ea330a2e15da'
ROWS_SHA = '9c055992902ddf9c6133c438bc35e09623d59b0a376b623cbadd3375bbd36147'
SOURCE_FILES = ('city.py', 'constants.py', 'common.py', 'time_windows_generator.py',
                'travel_time_generator.py', 'real_twcvrp.py')
SOURCE_HASHES = {'city.py': 'c55ef26cfbd24fa705afd6223aa45cf53094168ba02d95552562c9a66dc09e9e', 'constants.py': '28cca2aecf61bad8e4fb75dc6807f18f200998eb8a767a62e425c50641f25758', 'common.py': '8400c0e767c189987ccf26dee9688117ee2deb4e14f3c8ce516dd30e70ea5f5b', 'time_windows_generator.py': '90b19f40fdef99604345bf5935ceb40b54f10e5d869c77974bcef736a26d2afb', 'travel_time_generator.py': 'a0f6251321bcf053f251c58dd6d7ce207b7c2f66595444fdface1d54127d6e5e', 'real_twcvrp.py': '3530823c42ef6480b6d0aa63d3535b8e8313e20d82b904647653cf0c5da9de07'}
PROTOCOL_VERSION = 'pilot-00-v4'
WINDOW_POLICY = 'official_type_width_conditioned_reachability_v1'
JOINT_WINDOW_POLICY = 'joint_official_draws_proven_rejection_v1'
LEGACY_WINDOW_POLICY = 'balanced_insertion_mean_time_projection_v1'
# Candidato rechazado en Colab, comunicado por el autor; NO banco aceptado.
COLAB_REJECTED_V3_SHA = {50: 'e7390617fa2471dc120f03182eeeb9d6ea081a2ce44b8e1b390fd095e980531d'}
# Instancias validadas (master_seed, n) -> sha256. Coinciden con Python 3.12 + versiones
# fijadas y con el entorno Colab (Python 3.13, numpy 2.1.3, scikit-learn 1.6.1).
# Garantizan reproducibilidad por contenido, no por versión de biblioteca.
REFERENCE_INSTANCE_SHA = {
    (20260916, 50): '17dc741073c2b524d3bfbd97912fd4abd9ab033267faf2bc629265de9fb1f0f9',
    (20260916, 100): '540bb7187a90ef06f56f90c4a5e0cb0438bfc902d2077975fb3faa7fe914319f',
    (20260916, 200): 'fb3e12f54bd1003e00ed1910601935fe714aa561ef2feadeae90221592644445',
    (20260916, 300): '3e7cf85cf9c76b4fb10b73d013ffbc4b6ee7017d777ad1c093928be45877eec8',
}
LEGACY_V2_REFERENCE_SHA = {
    50: 'cb57f1068a3c7b173dbd087154f8276a8ba98df7cb45cc542fc8a3f5ef12b94f',
    100: '3e70717e9302373d12ddf25c613ef3c84522ba6c4f8a735c3368684db60eccff',
    200: '387948482f58d7f1e8d6c109f5edcbcc59ea318660b166119e76dc5b3f6c422c',
    300: 'd2c4b2457c08cf9fa7981fa13a85bd0637ea914ef9eb281f62acdc96f3f3b231',
}


def canonical_bytes(value):
    return json.dumps(value, sort_keys=True, separators=(',', ':'),
                      ensure_ascii=False, allow_nan=False).encode('utf-8')


def digest(value):
    return hashlib.sha256(canonical_bytes(value)).hexdigest()


def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + '\n')
    temporary.replace(path)


def seed_for(*parts):
    return int.from_bytes(hashlib.sha256(canonical_bytes(parts)).digest()[:8], 'little')


@contextmanager
def legacy_seed(seed):
    """Aísla las RNG globales que usa el generador oficial."""
    py, np_state = random.getstate(), np.random.get_state()
    random.seed(seed)
    np.random.seed(seed % 2**32)
    try:
        yield
    finally:
        random.setstate(py)
        np.random.set_state(np_state)


def download(url, destination):
    """Descarga atómica, tiempo limitado; no ejecuta contenido descargado."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    request = urllib.request.Request(url, headers={'User-Agent': 'SVRPLab-thesis/1.0'})
    with urllib.request.urlopen(request, timeout=90) as response:
        payload = response.read()
    temporary = destination.with_suffix(destination.suffix + '.download')
    temporary.write_bytes(payload)
    temporary.replace(destination)
    return destination


def fetch_upstream(root):
    root = Path(root)
    for name in SOURCE_FILES:
        path = root / 'vrp_bench' / name
        if not path.exists():
            download(f'https://raw.githubusercontent.com/yehias21/svrpbench/{UPSTREAM_SHA}/vrp_bench/{name}', path)
    if not (root / 'LICENSE').exists():
        download(f'https://raw.githubusercontent.com/yehias21/svrpbench/{UPSTREAM_SHA}/LICENSE', root / 'LICENSE')
    for name, expected in SOURCE_HASHES.items():
        if file_hash(root / 'vrp_bench' / name) != expected:
            raise ValueError(f'Código upstream modificado: {name}')
    # Hashes verificados; el commit completo fija las URLs.
    return {str(p.relative_to(root)): file_hash(p)
            for p in sorted(root.rglob('*')) if p.is_file() and p.suffix == '.py'}


def load_catalog(cache):
    """Parquet fijado; fallback oficial Viewer sólo si coincide el hash de 560 filas.

    La revisión del Viewer no se puede seleccionar: se identifica como snapshot
    verificado por contenido, nunca como descarga de la revisión fijada.
    """
    cache = Path(cache)
    cache.mkdir(parents=True, exist_ok=True)
    snapshot = cache / 'catalog_rows.json'
    meta_file = cache / 'catalog_provenance.json'
    if snapshot.exists():
        rows = json.loads(snapshot.read_text())
        meta = json.loads(meta_file.read_text()) if meta_file.exists() else {'transport': 'local_verified_snapshot'}
    else:
        rows = None
        try:
            from huggingface_hub import hf_hub_download
            import pyarrow.parquet as pq
            path = hf_hub_download(repo_id='MBZUAI/svrp-bench',
                filename='data/test-00000-of-00001.parquet', repo_type='dataset',
                revision=HF_REVISION, local_dir=str(cache / 'hf'))
            rows = pq.read_table(path).to_pylist()
            meta = {'transport': 'pinned_parquet', 'revision': HF_REVISION,
                    'file_sha256': file_hash(path)}
            # El hash fijado se obtuvo del Viewer; si la representación Parquet difiere
            # (tipos numéricos), se intenta el Viewer antes de declarar el catálogo distinto.
            if digest(sorted(rows, key=lambda r: (r['subset_name'], r['instance_id']))) != ROWS_SHA:
                warnings.warn('Parquet no coincide con el hash fijado; comprobando snapshot oficial Viewer.')
                rows = None
        except Exception as exc:
            warnings.warn(f'Parquet no disponible ({type(exc).__name__}); comprobando snapshot oficial Viewer.')
        if rows is None:
            rows = []
            for offset in range(0, 560, 100):
                url = ('https://datasets-server.huggingface.co/rows?dataset=MBZUAI%2Fsvrp-bench'
                       f'&config=default&split=test&offset={offset}&length=100')
                path = download(url, cache / f'viewer_{offset}.json')
                page = json.loads(path.read_text())
                if page.get('partial') or page['num_rows_total'] != 560:
                    raise ValueError('El catálogo cambió o está incompleto. Auditar antes de continuar.')
                for item in page['rows']:
                    if item.get('truncated_cells'):
                        raise ValueError('Viewer truncó celdas. Se necesita el Parquet completo.')
                    rows.append(item['row'])
            meta = {'transport': 'viewer_content_verified', 'revision': None,
                    'snapshot_observed': '2026-09-16', 'requested_revision': HF_REVISION}
    rows = sorted(rows, key=lambda r: (r['subset_name'], r['instance_id']))
    if len(rows) != 560 or digest(rows) != ROWS_SHA:
        raise ValueError('Hash del catálogo distinto. No seleccionar datos nuevos silenciosamente.')
    meta.update(dataset='MBZUAI/svrp-bench', split='test', rows_sha256=ROWS_SHA,
                rows=len(rows), role='development_only_never_final_test')
    write_json(snapshot, rows)
    write_json(meta_file, meta)
    return rows, meta


def upstream_modules(root):
    directory = str((Path(root) / 'vrp_bench').resolve())
    if directory not in sys.path:
        sys.path.insert(0, directory)
    for name in ('constants', 'city', 'common', 'time_windows_generator'):
        mod = importlib.import_module(name)
        if Path(mod.__file__).resolve().parent != Path(directory):
            raise RuntimeError(f'Módulo {name} procede de otra ruta; reinicie el entorno.')
    return sys.modules['common'], sys.modules['time_windows_generator']


@dataclass(frozen=True)
class Instance:
    instance_id: str
    locations: np.ndarray
    demands: np.ndarray
    capacities: np.ndarray
    windows: np.ndarray
    appear_times: np.ndarray
    provenance: dict

    @property
    def n(self):
        return len(self.demands) - 1

    def payload(self):
        return {'instance_id': self.instance_id, 'depot_indices': [0],
                **{key: getattr(self, key).tolist() for key in
                   ('locations', 'demands', 'capacities', 'windows', 'appear_times')},
                'provenance': self.provenance}

    @property
    def sha(self):
        return digest(self.payload())


def freeze_instance(payload):
    if payload.get('depot_indices') != [0]:
        raise ValueError('Este piloto admite exactamente el depósito 0.')
    arrays = {}
    for field in ('locations', 'demands', 'capacities', 'windows', 'appear_times'):
        arrays[field] = np.array(payload[field], dtype=np.float64, copy=True)
        arrays[field].setflags(write=False)
    x = Instance(payload['instance_id'], **arrays, provenance=payload['provenance'])
    n = x.n + 1
    if n < 2 or x.locations.shape != (n, 2) or x.windows.shape != (n, 2):
        raise ValueError('Dimensiones inválidas.')
    if x.demands.shape != (n,) or x.appear_times.shape != (n,) or x.capacities.ndim != 1:
        raise ValueError('Dimensiones inválidas en demanda/flota/arribos.')
    if not all(np.isfinite(v).all() for v in arrays.values()):
        raise ValueError('Datos no finitos.')
    if len(x.capacities) == 0 or np.any(x.capacities <= 0) or np.any(x.demands < 0):
        raise ValueError('Capacidades/demandas inválidas.')
    if x.demands[0] != 0 or np.any(x.appear_times < 0) or np.any(x.windows[:, 0] > x.windows[:, 1]):
        raise ValueError('Depósito, aparición o ventanas inválidos.')
    return x


def make_instance(n, rows, root, master_seed=20260916):
    common, tw = upstream_modules(root)
    subset = f'twvrp_{n}_single_depot'
    candidates = sorted([r for r in rows if r['subset_name'] == subset], key=lambda r: r['instance_id'])
    seed = seed_for(master_seed, 'geometry', n)
    if candidates:
        row = candidates[0]  # elección anterior a ejecutar cualquier solver
        raw = row
        xy, demand, appears = row['locations'], row['demands'], row['appear_times']
        caps = row['vehicle_capacities']
        if len(caps) == 1:
            caps = caps * int(row['num_vehicles'])
        if len(caps) != int(row['num_vehicles']):
            raise ValueError('Flota inconsistente; no reparar de manera implícita.')
        origin = {'kind': 'hf_geometry_demands_fleet_plus_derived_windows',
                  'subset_name': subset, 'source_instance_id': int(row['instance_id']),
                  'source_row_sha256': digest(row)}
    else:
        if n != 300:
            raise ValueError(f'Tamaño {n} sin datos publicados y sin derivación aprobada.')
        from threadpoolctl import threadpool_limits
        with legacy_seed(seed), threadpool_limits(limits=1):
            base = common.generate_base_instance(n, (1000, 1000), max(1, n // 50), 1, (0, 100), False)
        xy, demand, appears = base['locations'].tolist(), base['demands'].tolist(), base['appear_time'].tolist()
        caps = None
        raw = {'locations': xy, 'demands': demand, 'appear_times': appears, 'seed': seed}
        origin = {'kind': 'derived_official_base_generator', 'geometry_seed': seed,
                  'generator': 'common.generate_base_instance', 'num_cities': max(1, n // 50)}
    if len(xy) != n + 1:
        raise ValueError('Número de nodos inconsistente; no inferir depósitos por demanda cero.')
    windows_seed = seed_for(master_seed, 'time_windows', n)
    # Replica la regla de real_twcvrp: tipo elegido con random.randint(0,1),
    # no la proporción 60/40 descrita en el artículo. Se preserva esa diferencia.
    with legacy_seed(windows_seed):
        windows = [tw.sample_time_window(random.randint(0, 1), a) for a in appears]
    windows[0] = (0, 1440)
    # El dataset original cuantiza ventanas a uint16; se mantiene en derivados.
    windows = np.array(windows, dtype=np.uint16).tolist()
    if caps is None:
        # Equivalente a plt.hist(..., bins=24, range=(0,1440)) de get_num_vehicles.
        count = int(np.histogram(np.asarray(windows).ravel(), bins=24, range=(0, 1440))[0].max())
        caps = [float(sum(demand))] * max(1, count)
    origin.update(upstream_commit=UPSTREAM_SHA, windows_seed=windows_seed,
                  windows='derived_NOT_original_HF_windows', time_unit='minute',
                  coordinate_unit='source_map_unit', service_minutes=0,
                  customer_ids='1..n_including_zero_demands', depot_ids=[0],
                  capacity_modified=False)
    payload = dict(instance_id=f'dev_tw_n{n}_i0', depot_indices=[0], locations=xy,
                   demands=demand, capacities=caps, windows=windows, appear_times=appears,
                   provenance=origin)
    instance = freeze_instance(payload)
    expected = REFERENCE_INSTANCE_SHA.get((master_seed, n))
    if expected and instance.sha != expected:
        raise ValueError(f'Instancia n={n} distinta de la validada (probable cambio de numpy/scikit-learn). '
                         'No se sustituye en silencio: active STRICT_PINS = True en la celda 1 y ejecute de nuevo.')
    return instance, raw


def distances(x):
    # Euclídea float64: coherente con VRPSolverBase; Location.distance del generador
    # trunca a entero. Se declara la diferencia, no se mezcla entre métodos.
    return np.linalg.norm(x.locations[:, None, :] - x.locations[None, :, :], axis=-1)


def audit_instance(x, spec=None, certificate=None):
    """Cotas necesarias con viaje mínimo d/v; pasan a ser prueba sólo si fallan.

    Incluye apertura y regreso: un cliente puede ser alcanzable antes del cierre
    y aun así hacer imposible el retorno, incluso visitándolo solo.
    """
    spec = spec or Traffic()
    d = distances(x)
    earliest = np.maximum.reduce([d[0] / spec.velocity, x.windows[:, 0], x.appear_times])
    impossible = np.flatnonzero(earliest > x.windows[:, 1] + 1e-9)
    impossible = [int(j) for j in impossible if j != 0]
    return_impossible = [int(j) for j in range(1, x.n + 1)
                         if earliest[j] + d[j, 0] / spec.velocity > x.windows[0, 1] + 1e-9]
    oversized = np.flatnonzero(x.demands > x.capacities.max()).tolist()
    capacity_ok = bool(x.capacities.sum() >= x.demands.sum())
    certified = certificate is not None and verify_certificate(x, certificate, spec)['strict_feasible']
    return {'instance_id': x.instance_id, 'n_customers': x.n, 'n_depots': 1,
            'available_vehicles': len(x.capacities), 'total_demand': float(x.demands.sum()),
            'zero_demand_customers': int(np.sum(x.demands[1:] == 0)),
            'capacity_min': float(x.capacities.min()),
            'capacity_nonbinding_per_vehicle': bool(np.all(x.capacities >= x.demands.sum())),
            'unreachable_by_window_lower_bound': impossible,
            'impossible_depot_return_lower_bound': return_impossible,
            'oversized_demands': oversized,
            'aggregate_capacity_sufficient': capacity_ok,
            'proven_infeasible_by_lower_bound': bool(impossible or return_impossible or oversized or not capacity_ok),
            'source_kind': x.provenance['kind'], 'sha256': x.sha,
            'feasibility_certified': bool(certified),
            'certificate_scope': 'nominal_mean_only' if certified else 'none'}


@dataclass(frozen=True)
class Traffic:
    velocity: float = 1.0
    noise_scale: float = 1.0
    accident_scale: float = 1.0
    bucket_minutes: float = 1.0
    mode: str = 'stochastic'  # stochastic / no_noise / nominal_mean (no escenarios)

    def __post_init__(self):
        if not all(math.isfinite(v) for v in (self.velocity, self.bucket_minutes, self.noise_scale, self.accident_scale)) or self.velocity <= 0 or self.bucket_minutes <= 0 or self.noise_scale < 0 or self.accident_scale < 0:
            raise ValueError('Parámetros de tráfico fuera del dominio.')
        if self.mode not in ('stochastic', 'no_noise', 'nominal_mean'):
            raise ValueError('Modo de tráfico desconocido.')


def normal_pdf(t, mu, sigma):
    return math.exp(-0.5 * ((t - mu) / sigma)**2) / (sigma * math.sqrt(2 * math.pi))


def traffic_parameters(t, dist, spec):
    peak = normal_pdf(t, 480, 90) + normal_pdf(t, 1020, 90)
    base = 0.25 * (0.5 + 2 * peak) * (1 - math.exp(-dist / 50))
    mu, sigma = 0.1 * peak, (0.3 + 0.2 * peak) * spec.noise_scale
    rate = 0.05 * normal_pdf(t, 1260, 120) * spec.accident_scale
    return base, mu, sigma, rate


def expected_time(dist, t, spec):
    if dist == 0 or spec.mode == 'no_noise':
        return dist / spec.velocity
    base, mu, sigma, rate = traffic_parameters(t, dist, spec)
    return dist / spec.velocity + base * math.exp(mu + sigma * sigma / 2) + rate * 75


def travel_time(dist, t, i, j, instance_id, scenario_seed, spec):
    if dist == 0 or i == j or spec.mode == 'no_noise':
        return dist / spec.velocity
    if spec.mode == 'nominal_mean':
        return expected_time(dist, t, spec)
    base, mu, sigma, rate = traffic_parameters(t, dist, spec)
    # Flujo RNG local: método, orden de evaluación y ruta no figuran en la clave.
    # La misma arista/minuto/escenario recibe los mismos números primitivos.
    bucket = math.floor(t / spec.bucket_minutes)
    rng = np.random.default_rng(seed_for('traffic-v1', instance_id, int(scenario_seed), i, j, bucket))
    factor = math.exp(mu + sigma * rng.standard_normal())
    accidents = rng.poisson(rate)
    accident_delay = float(rng.uniform(30, 120, size=accidents).sum())
    return dist / spec.velocity + base * factor + accident_delay


def scenario_seeds(master_seed=20260916, instance_id='', split='evaluation', count=256):
    if split not in ('train', 'selection', 'evaluation') or count < 1:
        raise ValueError('Split/count inválidos.')
    return [seed_for(master_seed, 'scenario', instance_id, split, k) for k in range(count)]


def validate_routes(x, routes):
    if not isinstance(routes, (list, tuple)) or len(routes) != len(x.capacities):
        raise ValueError('Una lista por vehículo, incluidas listas vacías; nodos sin depósito.')
    flat = []
    for v, route in enumerate(routes):
        if not isinstance(route, (list, tuple)):
            raise ValueError('Cada ruta debe ser una lista de índices enteros.')
        if any(isinstance(j, bool) or not isinstance(j, (int, np.integer)) or j < 1 or j > x.n for j in route):
            raise ValueError('Cliente fuera de rango o depósito dentro de la ruta.')
        if sum(x.demands[j] for j in route) > x.capacities[v] + 1e-9:
            raise ValueError('Capacidad excedida en una ruta.')
        flat.extend(route)
    if sorted(flat) != list(range(1, x.n + 1)):
        raise ValueError('Todos los clientes, incluso demanda cero, deben aparecer exactamente una vez.')
    return True


def _reference_schedule(x, route, d, spec):
    """Horario nominal sin las ventanas antiguas, con apariciones y regreso."""
    t, i, starts = 0., 0, {}
    for j in route:
        t = max(t + expected_time(d[i, j], t, spec), float(x.appear_times[j]))
        starts[j] = t
        i = j
    return starts, t + expected_time(d[i, 0], t, spec) if route else 0.


def derive_feasible_windows(parent, spec=Traffic(), margin_fraction=.25):
    """Archivo v2, sólo regresión/diagnóstico; no usar para el banco de la tesis.

    Inserción balanceada: clientes de mayor a menor distancia al depósito;
    entre inserciones que respetan capacidad y horizonte, elige menor tiempo
    de retorno nominal, luego menor incremento, vehículo y posición. No usa
    ventanas de clientes ni escenarios ni resultados de los métodos.

    Proyecta cada apertura al intervalo más cercano que sitúa el horario
    testigo en el interior de la ventana, conservando su duración entera.
    El margen predefinido es 1/4 de la duración, limitado por [0, horizonte].
    Si no logra construir/certificar, falla: no prueba infactibilidad global.
    """
    if not 0 <= margin_fraction < .5:
        raise ValueError('Margen fuera de [0, .5).')
    if parent.windows[0, 0] != 0:
        raise ValueError('Este protocolo exige apertura del depósito en 0.')
    nominal = Traffic(**(asdict(spec) | {'mode': 'nominal_mean'}))
    d, horizon = distances(parent), float(parent.windows[0, 1])
    routes = [[] for _ in parent.capacities]
    loads, returns = np.zeros(len(routes)), np.zeros(len(routes))
    for j in sorted(range(1, parent.n + 1), key=lambda k: (-d[0, k], k)):
        choices = []
        for v, route in enumerate(routes):
            if loads[v] + parent.demands[j] > parent.capacities[v] + 1e-9:
                continue
            for pos in range(len(route) + 1):
                candidate = route[:pos] + [j] + route[pos:]
                _, end = _reference_schedule(parent, candidate, d, nominal)
                if end <= horizon + 1e-9:
                    choices.append(((end, end - returns[v], v, pos), v, candidate, end))
        if not choices:
            raise ValueError(f'No se pudo construir testigo para cliente {j}; no cambiar flota/horizonte. '
                             'No constituye prueba de infactibilidad global.')
        _, v, routes[v], returns[v] = min(choices)
        loads[v] += parent.demands[j]
    validate_routes(parent, routes)
    windows = parent.windows.copy()
    for route in routes:
        starts, _ = _reference_schedule(parent, route, d, nominal)
        for j, t in starts.items():
            width = float(parent.windows[j, 1] - parent.windows[j, 0])
            if width < 2 or width != int(width) or width > horizon or horizon != int(horizon):
                raise ValueError('Proyección requiere duración entera >=2 y <= horizonte entero.')
            margin = min(margin_fraction * width, t, horizon - t)
            lower = max(0, math.ceil(t - width + margin - 1e-9))
            upper = min(int(horizon - width), math.floor(t - margin + 1e-9))
            if lower > upper:
                raise ValueError('No hay proyección entera admisible; no ensanchar ventanas.')
            opening = min(max(int(parent.windows[j, 0]), lower), upper)
            windows[j] = [opening, opening + width]
    shifts = windows[1:, 0] - parent.windows[1:, 0]
    transformation = {'policy': LEGACY_WINDOW_POLICY, 'margin_fraction': margin_fraction,
                      'parent_instance_sha256': parent.sha, 'nominal_traffic': asdict(nominal),
                      'changed_windows': int(np.count_nonzero(shifts)),
                      'mean_absolute_shift_minutes': float(np.abs(shifts).mean()),
                      'max_absolute_shift_minutes': float(np.abs(shifts).max()),
                      'window_widths_preserved': True, 'geometry_demands_fleet_preserved': True,
                      'uses_evaluation_scenarios': False}
    payload = parent.payload()
    payload.update(instance_id=parent.instance_id + '_fw_v2', windows=windows.tolist(),
                   provenance=dict(parent.provenance, kind='nominal_feasible_derived_windows',
                                   parent_kind=parent.provenance['kind'],
                                   windows='derived_projected_NOT_original_HF_windows',
                                   transformation=transformation))
    instance = freeze_instance(payload)
    if (parent.sha == REFERENCE_INSTANCE_SHA.get((20260916, parent.n))
            and spec == Traffic() and margin_fraction == .25
            and instance.sha != LEGACY_V2_REFERENCE_SHA[parent.n]):
        raise ValueError('Derivado v2 distinto de la referencia; banco rechazado. '
                         'Compruebe código/entorno; STRICT_PINS permite reproducir el entorno validado.')
    certificate = {'scope': 'nominal_mean_only', 'not_a_competitive_baseline': True,
                   'not_solver_input': True, 'policy': LEGACY_WINDOW_POLICY,
                   'parent_instance_sha256': parent.sha, 'instance_sha256': instance.sha,
                   'traffic': asdict(nominal), 'routes': routes}
    certificate['verification'] = verify_certificate(instance, certificate, spec)
    return instance, certificate


def verify_certificate(x, certificate, spec=Traffic()):
    """Recalcula el testigo con el evaluador común; no confía en un booleano guardado."""
    nominal = Traffic(**(asdict(spec) | {'mode': 'nominal_mean'}))
    if certificate.get('instance_sha256') != x.sha or certificate.get('traffic') != asdict(nominal):
        raise ValueError('Certificado corresponde a otra instancia o parametrización de tráfico.')
    if certificate.get('scope') != 'nominal_mean_only':
        raise ValueError('Alcance de certificado no reconocido.')
    if certificate.get('policy') == WINDOW_POLICY:
        verify_conservative_routes(x, certificate['routes'], nominal,
                                   certificate['time_scale'])
    result = simulate(x, certificate['routes'], 0, nominal, 'fixed_order', trace=True)
    if not result['strict_feasible']:
        raise ValueError('El testigo no satisface ventanas/regreso nominales; banco rechazado.')
    slack = [x.windows[e['to'], 1] - e['service_start'] for e in result['events'] if e['to'] != 0]
    return {'strict_feasible': True, 'used_vehicles': result['used_vehicles'],
            'minimum_customer_slack_minutes': float(min(slack)),
            'maximum_return_minutes': float(max(e['arrival'] for e in result['events'] if e['to'] == 0)),
            'travel_minutes': result['travel_minutes'], 'waiting_minutes': result['waiting_minutes']}


def nominal_upper_matrix(x, spec=Traffic(), time_scale=1000):
    """Cota superior de la MEDIA por arco para toda hora, no del ruido aleatorio.

    Cada densidad normal <=1/(sigma*sqrt(2*pi)); sustituir ambos picos
    por sus máximos acota base, mu, sigma y tasa de accidentes. Se redondea
    hacia arriba a milésimas de minuto para el certificador entero.
    """
    if not isinstance(time_scale, int) or isinstance(time_scale, bool) or time_scale < 1:
        raise ValueError('time_scale debe ser entero positivo.')
    d = distances(x)
    peak_max = 2 / (90 * math.sqrt(2 * math.pi))
    sigma_max = (.3 + .2 * peak_max) * spec.noise_scale
    delay = .25 * (.5 + 2 * peak_max) * (1 - np.exp(-d / 50))
    delay *= math.exp(.1 * peak_max + sigma_max**2 / 2)
    delay += 75 * .05 * spec.accident_scale / (120 * math.sqrt(2 * math.pi))
    upper = d / spec.velocity + (delay if spec.mode != 'no_noise' else 0)
    upper[d == 0] = 0
    return np.ceil(upper * time_scale).astype(np.int64)


def official_customer_types(parent, root):
    """Reproduce etiquetas latentes sin cambiar el generador ni los hashes v1."""
    _, tw = upstream_modules(root)
    types, windows = [], []
    with legacy_seed(parent.provenance['windows_seed']):
        for appearance in parent.appear_times:
            kind = random.randint(0, 1)
            types.append(kind)
            windows.append(tw.sample_time_window(kind, appearance))
    windows[0] = (0, 1440)
    if not np.array_equal(np.asarray(windows, dtype=np.uint16), parent.windows):
        raise ValueError('La reconstrucción de tipos no coincide con las ventanas v1.')
    types[0] = -1  # depósito, no cliente residencial/comercial
    return types


def make_conditioned_candidate(parent, root, spec=Traffic(), max_draws_per_customer=10000,
                               time_scale=1000, joint_attempt=0):
    """Intento 0 reproduce v3; los siguientes remuestrean TODAS las ventanas.

    Todo horario nuevo es una salida íntegra del generador oficial (uint16),
    del mismo tipo y duración que v1. Nunca se desplaza, recorta ni ensancha.
    Tipo, duración, geometría y flota son fijos para todos los intentos.
    La distribución resultante está condicionada por alcanzabilidad individual;
    esto NO prueba factibilidad global, que requiere un certificado separado.
    """
    if max_draws_per_customer < 1:
        raise ValueError('Presupuesto de muestras inválido.')
    if not isinstance(joint_attempt, int) or isinstance(joint_attempt, bool) or joint_attempt < 0:
        raise ValueError('Índice de remuestreo conjunto inválido.')
    _, tw = upstream_modules(root)
    types = official_customer_types(parent, root)
    upper = nominal_upper_matrix(parent, spec, time_scale)
    windows, ledger = parent.windows.copy(), []
    horizon = math.floor(parent.windows[0, 1] * time_scale)
    def acceptable(j, pair):
        start = max(int(upper[0, j]), math.ceil(max(pair[0], parent.appear_times[j]) * time_scale))
        return start <= math.floor(pair[1] * time_scale) and start + int(upper[j, 0]) <= horizon
    for j in range(1, parent.n + 1):
        if int(upper[0, j]) + int(upper[j, 0]) > horizon:
            raise ValueError(f'Geometría n={parent.n}, cliente {j}: no pasa el filtro individual. '
                             'No cambiar velocidad, flota ni horizonte automáticamente.')
        old = parent.windows[j].tolist()
        width = old[1] - old[0]
        if joint_attempt == 0 and acceptable(j, old):
            ledger.append({'customer': j, 'type': types[j], 'draw': 0, 'seed': None,
                           'window': old, 'accepted': True, 'origin': 'preserved_v1'})
            continue
        for draw in range(1, max_draws_per_customer + 1):
            seed = (seed_for(parent.provenance['windows_seed'], WINDOW_POLICY, j, draw) if joint_attempt == 0
                    else seed_for(parent.provenance['windows_seed'], JOINT_WINDOW_POLICY, joint_attempt, j, draw))
            with legacy_seed(seed):
                proposed = np.asarray(tw.sample_time_window(types[j], parent.appear_times[j]),
                                      dtype=np.uint16).tolist()
            width_ok = proposed[1] - proposed[0] == width
            reachable = acceptable(j, proposed)
            accepted = width_ok and reachable
            ledger.append({'customer': j, 'type': types[j], 'draw': draw, 'seed': seed,
                           'window': proposed, 'accepted': accepted, 'width_matches': width_ok,
                           'individual_screen_passed': reachable, 'origin': 'official_generator'})
            if accepted:
                windows[j] = proposed
                break
        else:
            raise ValueError(f'Presupuesto de muestreo agotado en n={parent.n}, cliente {j}; '
                             'candidato no certificado. No desplazar ventanas como fallback.')
    shifts = windows[1:, 0] - parent.windows[1:, 0]
    change = {'policy': WINDOW_POLICY, 'parent_instance_sha256': parent.sha,
              'changed_windows': int(np.count_nonzero(shifts)),
              'mean_absolute_shift_minutes': float(np.abs(shifts).mean()),
              'max_absolute_shift_minutes': float(np.abs(shifts).max()),
              'window_widths_preserved': True, 'customer_types_preserved': True,
              'geometry_demands_fleet_preserved': True,
              'conditioned_distribution_not_original_benchmark': True,
              'uses_evaluation_scenarios': False, 'time_scale': time_scale,
              'nominal_traffic': asdict(Traffic(**(asdict(spec) | {'mode': 'nominal_mean'}))),
              'sampling_ledger_sha256': digest(ledger)}
    if joint_attempt:
        change.update(joint_policy=JOINT_WINDOW_POLICY, joint_attempt=joint_attempt)
    payload = parent.payload()
    suffix = f'_joint{joint_attempt:03d}' if joint_attempt else ''
    payload.update(instance_id=parent.instance_id + '_cw_v3' + suffix, windows=windows.tolist(),
        provenance=dict(parent.provenance, kind='official_conditioned_windows',
                        windows='official_draws_conditioned_NOT_original_HF_windows',
                        customer_types=types,
                        geometry_group_sha256=digest({'locations': parent.locations.tolist(), 'depot_indices': [0]}),
                        transformation=change))
    candidate = freeze_instance(payload)
    if (joint_attempt == 0 and parent.sha == REFERENCE_INSTANCE_SHA.get((20260916, parent.n))
            and spec == Traffic() and time_scale == 1000 and parent.n in COLAB_REJECTED_V3_SHA
            and candidate.sha != COLAB_REJECTED_V3_SHA[parent.n]):
        raise ValueError('El intento 0 difiere del candidato v3 observado en Colab. Revisar código/entorno; '
                         'no sustituir silenciosamente el caso rechazado.')
    return candidate, ledger


def verify_conservative_routes(x, routes, spec=Traffic(), time_scale=1000):
    """Chequeo determinista independiente de los horarios que devuelve OR-Tools."""
    validate_routes(x, routes)
    upper = nominal_upper_matrix(x, spec, time_scale)
    horizon = math.floor(x.windows[0, 1] * time_scale)
    for route in routes:
        t, i = 0, 0
        for j in route:
            t = max(t + int(upper[i, j]), math.ceil(max(x.windows[j, 0], x.appear_times[j]) * time_scale))
            if t > math.floor(x.windows[j, 1] * time_scale):
                raise ValueError('Testigo viola una ventana en el verificador conservador.')
            i = j
        if t + int(upper[i, 0]) > horizon:
            raise ValueError('Testigo viola el retorno en el verificador conservador.')
    return True


def fleet_compatibility_diagnostic(x, spec=Traffic(), tolerance=1e-8):
    """Cota de vehículos por clientes mutuamente incompatibles, incluso sin ruido.

    E_i=max(apertura_i,aparición_i,d_0i/v), L_i=min(cierre_i,H-d_i0/v).
    Si E_i+d_ij/v>L_j, i no puede preceder a j, tampoco con visitas intermedias
    (desigualdad triangular). Si ambas direcciones fallan, no comparten vehículo.
    Una clique de k clientes exige al menos k vehículos. La clique hallada es
    una cota, no necesariamente máxima; pasar este chequeo no prueba factibilidad.
    """
    if not math.isfinite(tolerance) or tolerance < 0:
        raise ValueError('Tolerancia inválida.')
    d = distances(x) / spec.velocity
    earliest = np.maximum.reduce([d[0], x.windows[:, 0], x.appear_times])
    latest = np.minimum(x.windows[:, 1], x.windows[0, 1] - d[:, 0])
    impossible_single = [j for j in range(1, x.n + 1) if earliest[j] > latest[j] + tolerance]
    bad_order = earliest[:, None] + d > latest[None, :] + tolerance
    incompatible = bad_order & bad_order.T
    np.fill_diagonal(incompatible, False)
    incompatible[0, :] = incompatible[:, 0] = False
    neighbors = {i: set(np.flatnonzero(incompatible[i]).tolist()) for i in range(1, x.n + 1)}
    best = []
    for first in sorted(neighbors, key=lambda i: (-len(neighbors[i]), i)):
        clique, pending = [first], set(neighbors[first])
        while pending:
            j = min(pending, key=lambda i: (-len(neighbors[i] & pending), i))
            clique.append(j)
            pending &= neighbors[j]
        if len(clique) > len(best):
            best = clique
    proof_pairs = []
    for a, i in enumerate(best):
        for j in best[a + 1:]:
            ij = float(earliest[i] + d[i, j] - latest[j])
            ji = float(earliest[j] + d[j, i] - latest[i])
            if ij <= tolerance or ji <= tolerance:
                raise AssertionError('Cota de clique no verificada.')
            proof_pairs.append({'i': i, 'j': j, 'i_before_j_impossible_by_minutes': ij,
                                'j_before_i_impossible_by_minutes': ji})
    fleet_insufficient = len(best) > len(x.capacities)
    return {'instance_sha256': x.sha, 'scope': 'optimistic_free_flow_necessary_conditions',
            'velocity': spec.velocity, 'tolerance_minutes': tolerance,
            'available_vehicles': len(x.capacities),
            'individually_impossible_clients': impossible_single,
            'vehicles_lower_bound': len(best), 'mutually_incompatible_clients': best,
            'clique_is_proven_maximum': False, 'pair_witnesses': proof_pairs,
            'proves_insufficient_fleet': fleet_insufficient,
            'proves_infeasibility': bool(impossible_single or fleet_insufficient),
            'passing_is_not_feasibility_certificate': True}


def certification_model_data(x, spec=Traffic(), time_scale=1000, optimistic=False):
    """Datos enteros reproducibles; el modelo optimista es una RELAJACIÓN.

    Bajar cada tránsito y abrir hacia afuera los dominios preserva toda solución
    continua al tomar floor(t*S). INFEASIBLE en esta relajación prueba imposibilidad
    real. La cota superior nominal hace lo contrario: sólo certifica testigos.
    """
    if not isinstance(time_scale, int) or isinstance(time_scale, bool) or time_scale < 1:
        raise ValueError('time_scale debe ser entero positivo.')
    opening = np.maximum(x.windows[:, 0], x.appear_times)
    if optimistic:
        # Margen hacia afuera para que error flotante no endurezca la relajación.
        eps = 1e-8
        matrix = np.floor(np.maximum(0, distances(x) / spec.velocity - eps) * time_scale).astype(np.int64)
        lower = np.floor(np.maximum(0, opening - eps) * time_scale).astype(np.int64)
        upper = np.ceil((x.windows[:, 1] + eps) * time_scale).astype(np.int64)
        horizon = math.ceil((x.windows[0, 1] + eps) * time_scale)
    else:
        matrix = nominal_upper_matrix(x, spec, time_scale)
        lower = np.ceil(opening * time_scale).astype(np.int64)
        upper = np.floor(x.windows[:, 1] * time_scale).astype(np.int64)
        horizon = math.floor(x.windows[0, 1] * time_scale)
    return {'formulation': 'mandatory_multiple_circuit_time_windows_v1',
            'scope': 'optimistic_free_flow_relaxation' if optimistic else 'conservative_nominal_mean',
            'instance_sha256': x.sha, 'time_scale': time_scale, 'fleet': len(x.capacities),
            'matrix': matrix.tolist(), 'lower': lower.tolist(), 'upper': upper.tolist(),
            'horizon': horizon, 'capacity': 'nonbinding_checked_before_solve'}


def solve_certification_model(x, data, seconds, workers):
    """CP-SAT de factibilidad, sin objetivo ni eliminación de clientes.

    No se distinguen vehículos: esto elimina simetrías y requiere capacidad no
    vinculante. Cada circuito pasa por el depósito; el número de salidas <= K.
    """
    from ortools.sat.python import cp_model
    model = cp_model.CpModel()
    lower, upper, horizon = data['lower'], data['upper'], data['horizon']
    matrix = data['matrix']
    times = {}
    for j in range(1, x.n + 1):
        if lower[j] > upper[j]:
            model.add(False)
        times[j] = model.new_int_var(min(lower[j], upper[j]), upper[j], f't{j}')
    arcs, variables = [], {}
    # Incluso un nodo aislado queda explícitamente en el circuito; su self-loop
    # es falso, por lo que no se permite omitirlo de la solución.
    for j in range(1, x.n + 1):
        arcs.append((j, j, model.new_constant(0)))
    for i in range(x.n + 1):
        for j in range(x.n + 1):
            if i == j:
                continue
            start = 0 if i == 0 else lower[i]
            deadline = horizon if j == 0 else upper[j]
            if start + matrix[i][j] > deadline:
                continue
            arc = model.new_bool_var(f'a{i}_{j}')
            variables[i, j] = arc
            arcs.append((i, j, arc))
            if i == 0:
                model.add(times[j] >= matrix[i][j]).only_enforce_if(arc)
            elif j == 0:
                model.add(times[i] + matrix[i][j] <= horizon).only_enforce_if(arc)
            else:
                model.add(times[j] >= times[i] + matrix[i][j]).only_enforce_if(arc)
    model.add_multiple_circuit(arcs)
    model.add(sum(arc for (i, j), arc in variables.items() if i == 0) <= len(x.capacities))
    validation = model.validate()
    if validation:
        return None, {'status_name': 'MODEL_INVALID', 'validation_error': validation,
                      'seconds': 0.0, 'solution_returned': False, 'model_sha256': digest(data)}
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = float(seconds)
    solver.parameters.num_search_workers = workers
    solver.parameters.random_seed = 0
    started = time.perf_counter()
    status = solver.solve(model)
    attempt = {'status_name': solver.status_name(status), 'cp_sat_status': int(status),
               'seconds': time.perf_counter() - started, 'model_sha256': digest(data),
               'solution_returned': status in (cp_model.FEASIBLE, cp_model.OPTIMAL),
               'scope': data['scope'], 'time_scale': data['time_scale'],
               'workers': workers, 'random_seed': 0, 'limit_seconds': seconds,
               'ortools_version': importlib.metadata.version('ortools'),
               'response_stats': solver.response_stats()}
    if not attempt['solution_returned']:
        return None, attempt
    selected = [(i, j) for (i, j), arc in variables.items() if solver.boolean_value(arc)]
    successor = {i: j for i, j in selected if i != 0}
    routes = []
    seen = set()
    for first in sorted(j for i, j in selected if i == 0):
        route, j = [], first
        while j:
            if j in seen:
                raise ValueError('Circuito devuelto no es una partición de los clientes.')
            seen.add(j); route.append(j); j = successor[j]
        routes.append(route)
    routes += [[] for _ in range(len(x.capacities) - len(routes))]
    validate_routes(x, routes)
    return routes, attempt


def certify_candidate(x, spec=Traffic(), seconds_per_strategy=120, time_scale=1000, progress=False):
    """Prueba negativa optimista o testigo positivo conservador, nunca reparar datos.

    INFEASIBLE conservador y UNKNOWN no permiten rechazar/remuestrear. El modelo
    sólo cubre flotas cuyas capacidades individuales son >= demanda total.
    """
    if not math.isfinite(seconds_per_strategy) or seconds_per_strategy < 1:
        raise ValueError('Presupuesto de certificación inválido.')
    diagnostic = fleet_compatibility_diagnostic(x, spec)
    base = {'instance_sha256': x.sha, 'diagnostic': diagnostic,
            'is_proof_of_infeasibility': False, 'attempts': []}
    if progress:
        print(f"Cota necesaria de vehículos: >= {diagnostic['vehicles_lower_bound']}; "
              f"disponibles: {len(x.capacities)}.", flush=True)
    if diagnostic['proves_infeasibility']:
        return None, dict(base, status='proven_infeasible_free_flow_bound', is_proof_of_infeasibility=True)
    if np.any(x.capacities < x.demands.sum()):
        raise ValueError('CP-SAT 00 requiere capacidad no vinculante en cada vehículo; no ignorar esta restricción.')
    workers = min(8, os.cpu_count() or 1)
    attempts = base['attempts']
    for stage, optimistic in enumerate((True, False), 1):
        data = certification_model_data(x, spec, time_scale, optimistic)
        if progress:
            print(f"[{stage}/2] CP-SAT {data['scope']}; límite {seconds_per_strategy} s.", flush=True)
        routes, attempt = solve_certification_model(x, data, seconds_per_strategy, workers)
        attempts.append(attempt)
        if progress:
            print(f"[{stage}/2] {attempt['status_name']} en {attempt['seconds']:.2f} s.", flush=True)
        if attempt['status_name'] == 'MODEL_INVALID':
            return None, dict(base, status='certifier_model_invalid')
        if optimistic and attempt['status_name'] == 'INFEASIBLE':
            return None, dict(base, status='proven_infeasible_optimistic_model', is_proof_of_infeasibility=True)
        if routes is None:
            continue
        try:
            verify_conservative_routes(x, routes, spec, time_scale)
        except ValueError:
            attempt['conservative_witness_verified'] = False
            if not optimistic:
                raise  # un testigo del modelo superior debe pasar el verificador
            continue
        attempt['conservative_witness_verified'] = True
        certificate = {'scope': 'nominal_mean_only', 'policy': WINDOW_POLICY,
                       'instance_sha256': x.sha,
                       'parent_instance_sha256': x.provenance['transformation']['parent_instance_sha256'],
                       'traffic': asdict(Traffic(**(asdict(spec) | {'mode': 'nominal_mean'}))),
                       'routes': routes, 'time_scale': time_scale,
                       'not_solver_input': True, 'not_a_competitive_baseline': True,
                       'certifier': 'OR-Tools CP-SAT feasibility; independently verified upper-mean witness',
                       'ortools_version': importlib.metadata.version('ortools'), 'attempts': attempts}
        certificate['verification'] = verify_certificate(x, certificate, spec)
        return certificate, dict(base, status='certified_nominal')
    return None, dict(base, status='unknown_no_certificate')


def validate_sampling_ledger(x, ledger):
    """Integridad y cobertura de aceptación; reproducción oficial en celda Colab."""
    change = x.provenance['transformation']
    if change.get('policy') != WINDOW_POLICY or digest(ledger) != change['sampling_ledger_sha256']:
        raise ValueError('Política o registro de muestreo no corresponde al candidato.')
    accepted = [entry for entry in ledger if entry['accepted']]
    if sorted(entry['customer'] for entry in accepted) != list(range(1, x.n + 1)):
        raise ValueError('Debe haber exactamente una ventana aceptada por cliente.')
    for entry in accepted:
        j = entry['customer']
        if entry['type'] != x.provenance['customer_types'][j] or not np.array_equal(entry['window'], x.windows[j]):
            raise ValueError('Ventana/tipo aceptado no coincide con la instancia.')
    return True


def proven_rejection(x, report, spec=Traffic(), time_scale=1000):
    """Sólo dos fuentes admitidas de prueba negativa; timeout nunca es rechazo.

    La clique se recalcula aquí. Para CP-SAT se verifica que INFEASIBLE corresponde
    a la relajación reconstruida; su prueba depende del solver, no de reejecutarlo.
    """
    if report.get('instance_sha256') != x.sha or report.get('is_proof_of_infeasibility') is not True:
        return False
    if report.get('status') == 'proven_infeasible_free_flow_bound':
        diagnostic = fleet_compatibility_diagnostic(x, spec)
        return diagnostic['proves_infeasibility'] and report.get('diagnostic') == diagnostic
    if report.get('status') == 'proven_infeasible_optimistic_model':
        fingerprint = digest(certification_model_data(x, spec, time_scale, optimistic=True))
        return any(a.get('scope') == 'optimistic_free_flow_relaxation'
                   and a.get('status_name') == 'INFEASIBLE' and a.get('cp_sat_status') == 3
                   and a.get('model_sha256') == fingerprint and not a.get('solution_returned')
                   for a in report.get('attempts', []))
    return False


def validate_admission_history(x, history, spec=Traffic(), time_scale=1000):
    """No puede faltar un intento ni descartarse un resultado desconocido."""
    if history.get('policy') != JOINT_WINDOW_POLICY or not history.get('attempts'):
        raise ValueError('Falta historial de admisión conjunta.')
    records = history['attempts']
    if len(records) > history['max_joint_candidates']:
        raise ValueError('Se excedió el presupuesto preregistrado de candidatos.')
    for index, record in enumerate(records):
        candidate = freeze_instance(record['candidate'])
        transformation = candidate.provenance['transformation']
        if (record['joint_attempt'] != index or transformation.get('joint_attempt', 0) != index
                or (index > 0 and transformation.get('joint_policy') != JOINT_WINDOW_POLICY)
                or transformation['parent_instance_sha256'] != x.provenance['transformation']['parent_instance_sha256']):
            raise ValueError('Intentos omitidos, desordenados o de otro origen.')
        for field in ('locations', 'demands', 'capacities', 'appear_times'):
            if not np.array_equal(getattr(candidate, field), getattr(x, field)):
                raise ValueError('Geometría, demandas, flota o apariciones cambiaron entre intentos.')
        if (candidate.provenance['customer_types'] != x.provenance['customer_types']
                or not np.array_equal(candidate.windows[:, 1] - candidate.windows[:, 0],
                                      x.windows[:, 1] - x.windows[:, 0])
                or not np.array_equal(candidate.windows[0], x.windows[0])):
            raise ValueError('Tipos, duraciones o depósito cambiaron entre intentos.')
        validate_sampling_ledger(candidate, record['ledger'])
        if index < len(records) - 1:
            if record['certificate'] is not None or not proven_rejection(candidate, record['certification'], spec, time_scale):
                raise ValueError('Se saltó un candidato sin prueba de imposibilidad.')
        else:
            if (candidate.sha != x.sha or record['certification'].get('status') != 'certified_nominal'
                    or record['certification'].get('instance_sha256') != x.sha):
                raise ValueError('La instancia final no coincide con el primer certificado.')
            verify_certificate(candidate, record['certificate'], spec)
    return True


def admit_joint_windows(parent, root, config, attempts_dir, remote_dir=None, environment=None, progress=True):
    """Secuencia prefijada, checkpoints sellados por intento; se detiene en UNKNOWN.

    Un checkpoint desconocido se conserva también al reanudar. Ampliar su búsqueda
    requiere una revisión explícita del presupuesto, nunca omitirlo o borrarlo.
    """
    if (config['protocol_version'] != PROTOCOL_VERSION
            or config['windows']['joint_policy'] != JOINT_WINDOW_POLICY):
        raise ValueError('Protocolo conjunto incompatible.')
    attempts_dir = Path(attempts_dir)
    remote_dir = Path(remote_dir) if remote_dir else None
    spec = Traffic(**config['traffic'])
    scale = config['certification']['time_scale']
    limit = config['windows']['max_joint_candidates']
    if not isinstance(limit, int) or isinstance(limit, bool) or limit < 1:
        raise ValueError('Presupuesto de candidatos inválido.')
    history = {'policy': JOINT_WINDOW_POLICY, 'max_joint_candidates': limit, 'attempts': []}
    for index in range(limit):
        if progress:
            print(f'Intento conjunto {index + 1}/{limit} (índice {index}); geometría y flota fijas.', flush=True)
        candidate, ledger = make_conditioned_candidate(parent, root, spec,
            config['windows']['max_draws_per_customer'], scale, joint_attempt=index)
        validate_sampling_ledger(candidate, ledger)
        checkpoint = attempts_dir / f'attempt{index:03d}'
        remote = remote_dir / checkpoint.name if remote_dir else None
        if remote and remote.exists() and not checkpoint.exists():
            sync_verified(remote, checkpoint)
        if checkpoint.exists():
            verify_bundle(checkpoint)
            record = json.loads((checkpoint / 'record.json').read_text())
            if (record['candidate'] != candidate.payload() or record['ledger'] != ledger
                    or record['joint_attempt'] != index
                    or json.loads((checkpoint / 'config.json').read_text()) != config):
                raise ValueError('Checkpoint distinto de la secuencia/configuración regenerada.')
            certificate, report = record['certificate'], record['certification']
            if progress:
                print('Intento recuperado:', report['status'], flush=True)
        else:
            certificate, report = certify_candidate(candidate, spec,
                config['certification']['seconds_per_strategy'], scale, progress=progress)
            record = {'joint_attempt': index, 'candidate': candidate.payload(), 'ledger': ledger,
                      'certificate': certificate, 'certification': report}
            write_json(checkpoint / 'record.json', record)
            write_json(checkpoint / 'config.json', config)
            write_json(checkpoint / 'environment.json', environment or {})
            seal_bundle(checkpoint)
        if remote:
            sync_verified(checkpoint, remote)
        history['attempts'].append(record)
        if certificate is not None:
            validate_admission_history(candidate, history, spec, scale)
            return candidate, ledger, certificate, history
        if not proven_rejection(candidate, report, spec, scale):
            raise RuntimeError(f'n={parent.n}: SIN CERTIFICADO ({report["status"]}). '
                f'No se remuestrea ni se autoriza el banco. Conservar y revisar: {checkpoint}/record.json')
        if progress:
            print('INFACTIBILIDAD PROBADA:', report['status'],
                  '→ rechazo registrado; siguiente muestra conjunta oficial.', flush=True)
    raise RuntimeError(f'n={parent.n}: agotados {limit} candidatos con pruebas negativas. '
                       f'Banco no autorizado; revisar protocolo. Registro: {attempts_dir}')


def acceptance_report(instances, certificates, ledgers, config, histories=None):
    if config['protocol_version'] != PROTOCOL_VERSION or config['windows']['policy'] != WINDOW_POLICY:
        raise ValueError('Protocolo incompatible.')
    if config['role'] != 'development_only' or config['traffic']['mode'] != 'stochastic':
        raise ValueError('00 sólo certifica desarrollo bajo tráfico estocástico declarado.')
    if config['sizes'] != [50, 100, 200, 300] or sorted(x.n for x in instances) != config['sizes']:
        raise ValueError('Se requieren los cuatro tamaños del piloto.')
    if (config.get('depot_index') != 0 or config.get('service_time_minutes') != 0
            or config.get('global_start_time_minutes') != 0 or config.get('instances_per_size') != 1):
        raise ValueError('Convenciones configuradas no implementadas por el evaluador de 00.')
    spec = Traffic(**config['traffic'])
    for x in instances:
        if x.provenance.get('kind') != 'official_conditioned_windows':
            raise ValueError('Ventanas proyectadas o de otro origen no admitidas en v4.')
        validate_sampling_ledger(x, ledgers[x.n])
        if histories is None or x.n not in histories:
            raise ValueError('Falta historial conjunto para aceptar el banco v4.')
        history = histories[x.n]
        if (config['windows']['joint_policy'] != JOINT_WINDOW_POLICY
                or history['max_joint_candidates'] != config['windows']['max_joint_candidates']):
            raise ValueError('Política o presupuesto conjunto inconsistente.')
        validate_admission_history(x, history, spec, config['certification']['time_scale'])
        if history['attempts'][-1]['certificate'] != certificates[x.n]:
            raise ValueError('Certificado seleccionado distinto del historial.')
        if certificates[x.n].get('policy') != WINDOW_POLICY:
            raise ValueError('Certificado de otra política de generación.')
        kinds = np.asarray(x.provenance['customer_types'])
        widths = x.windows[:, 1] - x.windows[:, 0]
        if (kinds.shape != (x.n + 1,) or kinds[0] != -1 or not np.isin(kinds[1:], [0, 1]).all()
                or not np.isin(widths[1:], [60, 120]).all() or np.any(widths[kinds == 1] != 60)
                or not np.array_equal(x.windows[0], [0, 1440]) or np.any(x.appear_times != 0)):
            raise ValueError('Tipos, duraciones, depósito o apariciones fuera del alcance declarado.')
        audit = audit_instance(x, spec, certificates[x.n])
        if audit['proven_infeasible_by_lower_bound'] or not audit['feasibility_certified']:
            raise ValueError('No se cumplen las condiciones nominales del banco.')
    return {'status': 'ready_for_method_development', 'protocol_version': PROTOCOL_VERSION,
            'instance_hashes': {str(x.n): x.sha for x in instances},
            'geometry_groups': {str(x.n): x.provenance['geometry_group_sha256'] for x in instances},
            'all_nominal_certificates_verified': True,
            'all_windows_from_official_generator_conditioned': True,
            'all_rejections_proven_infeasible': True,
            'admission_history_hashes': {str(x.n): digest(histories[x.n]) for x in instances},
            'ready_for_confirmatory_comparison': False,
            'limitations': ['one_development_geometry_per_size', 'conditioned_window_distribution',
                            'fixed_known_demands_and_zero_service', 'nonbinding_capacity_in_source',
                            'traffic_is_dynamic_not_online_customer_revelation',
                            'no_almost_sure_stochastic_feasibility_guarantee',
                            'complete_recourse_cost_objective_pending_before_training']}


def diagnostic_greedy(x, spec=Traffic()):
    """Heurística de arranque, no baseline del artículo ni óptimo.

    Ordena clientes por cierre; asigna a la ruta con menor tardanza proyectada,
    incremento de distancia y llegada, en ese orden lexicográfico. Sin escenarios.
    """
    d = distances(x)
    routes = [[] for _ in x.capacities]
    times = np.zeros(len(routes)); loads = np.zeros(len(routes))
    for j in sorted(range(1, x.n + 1), key=lambda k: (x.windows[k, 1], k)):
        choices = []
        for v, route in enumerate(routes):
            if loads[v] + x.demands[j] > x.capacities[v] + 1e-9:
                continue
            i = route[-1] if route else 0
            arrival = max(times[v] + expected_time(d[i, j], times[v], spec),
                          x.windows[j, 0], x.appear_times[j])
            late = max(0., arrival - x.windows[j, 1])
            depot_late = max(0., arrival + expected_time(d[j, 0], arrival, spec) - x.windows[0, 1])
            choices.append(((late + depot_late, d[i, j] + d[j, 0] - d[i, 0], arrival, v), v, arrival))
        if not choices:
            raise ValueError(f'Heurística no pudo asignar cliente {j}. No prueba infactibilidad global.')
        _, v, arrival = min(choices)
        routes[v].append(j); times[v] = arrival; loads[v] += x.demands[j]
    validate_routes(x, routes)
    return routes


def simulate(x, routes, scenario_seed, spec=Traffic(), policy='fixed_order', trace=False):
    """Evalúa sin filtrar incumplimientos. Recurso sólo usa reloj y medias conocidas.

    causal_edd: si siguiente cliente ya se proyecta tarde, elige entre los pendientes
    aún alcanzables el de cierre más próximo; no cambia asignación de vehículo.
    El servicio tardío se registra y continúa, no se convierte en factibilidad.
    """
    validate_routes(x, routes)
    if policy not in ('fixed_order', 'causal_edd'):
        raise ValueError('Política de recurso desconocida.')
    d = distances(x)
    travel = waiting = late_sum = overtime = base_cost = 0.
    late_nodes, changed, actual_routes, events = set(), 0, [], []
    for v, route in enumerate(routes):
        path = [0] + list(route) + [0]
        base_cost += sum(d[a, b] / spec.velocity for a, b in zip(path, path[1:]))
        pending, actual, i, t = list(route), [], 0, 0.
        while pending:
            j = pending[0]
            if policy == 'causal_edd':
                forecast = max(t + expected_time(d[i, j], t, spec), x.windows[j, 0], x.appear_times[j])
                if forecast > x.windows[j, 1]:
                    possible = [k for k in pending if max(t + expected_time(d[i, k], t, spec),
                                x.windows[k, 0], x.appear_times[k]) <= x.windows[k, 1]]
                    if possible:
                        chosen = min(possible, key=lambda k: (x.windows[k, 1], d[i, k], k))
                        changed += int(chosen != j)
                        j = chosen
            duration = travel_time(d[i, j], t, i, j, x.instance_id, scenario_seed, spec)
            arrival = t + duration
            start = max(arrival, x.windows[j, 0], x.appear_times[j])
            late = max(0., start - x.windows[j, 1])
            if late > 1e-9:
                late_nodes.add(j)
            travel += duration; waiting += start - arrival; late_sum += late
            if trace:
                events.append({'vehicle': v, 'from': i, 'to': j, 'departure': t,
                               'arrival': arrival, 'service_start': start, 'late': late})
            pending.remove(j); actual.append(j); i, t = j, start
        if route:
            duration = travel_time(d[i, 0], t, i, 0, x.instance_id, scenario_seed, spec)
            travel += duration
            overtime += max(0., t + duration - x.windows[0, 1])
            if trace:
                events.append({'vehicle': v, 'from': i, 'to': 0, 'departure': t,
                               'arrival': t + duration, 'service_start': t + duration,
                               'late': max(0., t + duration - x.windows[0, 1])})
        actual_routes.append(actual)
    result = {'scenario_seed': int(scenario_seed), 'travel_minutes': travel,
              'base_minutes': base_cost, 'net_execution_delta_minutes': travel - base_cost,
              'waiting_minutes': waiting, 'lateness_minutes': late_sum,
              'depot_overtime_minutes': overtime, 'late_customers': len(late_nodes),
              'customer_violation_fraction': len(late_nodes) / x.n,
              'strict_feasible': len(late_nodes) == 0 and overtime <= 1e-9,
              'all_customers_served': True, 'used_vehicles': sum(bool(r) for r in routes),
              'recourse_reorders': changed}
    if trace:
        result.update(events=events, realized_routes=actual_routes)
    return result


def empirical_cvar(values, alpha=.95):
    values = np.asarray(values, dtype=np.float64)
    if values.ndim != 1 or len(values) == 0 or not np.isfinite(values).all() or not 0 <= alpha < 1:
        raise ValueError('Muestra/alpha inválidos para CVaR.')
    z = float(np.sort(values)[max(0, math.ceil(alpha * len(values)) - 1)])
    return z + float(np.maximum(values - z, 0).sum()) / (len(values) * (1 - alpha))


def evaluate(x, routes, seeds, spec=Traffic(), policy='fixed_order', alpha=.95):
    before = x.sha
    started = time.perf_counter()
    records = [simulate(x, routes, s, spec, policy) for s in seeds]
    if not records or len(set(seeds)) != len(seeds):
        raise ValueError('Escenarios vacíos o repetidos.')
    assert x.sha == before, 'La instancia fue alterada.'
    costs = np.array([r['travel_minutes'] for r in records])
    summary = {'instance_id': x.instance_id, 'instance_sha256': before, 'policy': policy,
               'scenarios': len(records), 'alpha': alpha,
               'mean_travel_minutes': float(costs.mean()),
               'cvar_travel_minutes': empirical_cvar(costs, alpha),
               'cost_variance_population': float(costs.var(ddof=0)),
               'scenario_strict_feasibility': float(np.mean([r['strict_feasible'] for r in records])),
               'mean_customer_violation_fraction': float(np.mean([r['customer_violation_fraction'] for r in records])),
               'mean_lateness_minutes': float(np.mean([r['lateness_minutes'] for r in records])),
               'mean_depot_overtime_minutes': float(np.mean([r['depot_overtime_minutes'] for r in records])),
               'used_vehicles': records[0]['used_vehicles'],
               'evaluation_seconds': time.perf_counter() - started}
    return summary, records


def environment_report():
    versions = {}
    for pkg in ('numpy', 'pandas', 'matplotlib', 'scipy', 'scikit-learn', 'pillow', 'pyarrow', 'huggingface-hub', 'ortools', 'torch'):
        try:
            versions[pkg] = importlib.metadata.version(pkg)
        except importlib.metadata.PackageNotFoundError:
            versions[pkg] = None
    resolved = {}
    for d in importlib.metadata.distributions():
        try:
            name = d.metadata['Name']
        except Exception:  # dist-info residual de pip sin METADATA
            continue
        if name:
            resolved.setdefault(name, d.version)  # la primera en sys.path es la importada
    report = {'resolved_packages': resolved, 'python': sys.version, 'platform': platform.platform(),
              'cpu_count': os.cpu_count(), 'versions': versions}
    if shutil.which('nvidia-smi'):
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
                                 '--format=csv,noheader'], capture_output=True, text=True, timeout=15)
        report['gpu'] = result.stdout.strip()
    else:
        report['gpu'] = None
    return report


def bundle_files(directory):
    """Archivos sellables; excluye bytecode que Python crea al importar bank/code."""
    return [p for p in sorted(directory.rglob('*')) if p.is_file() and p.name != 'CHECKSUMS.json'
            and '__pycache__' not in p.relative_to(directory).parts]


def seal_bundle(directory):
    directory = Path(directory)
    manifest = {str(p.relative_to(directory)): file_hash(p) for p in bundle_files(directory)}
    write_json(directory / 'CHECKSUMS.json', manifest)
    return digest(manifest)


def verify_bundle(directory):
    directory = Path(directory)
    manifest = json.loads((directory / 'CHECKSUMS.json').read_text())
    for name, expected in manifest.items():
        path = directory / name
        if not path.is_file() or file_hash(path) != expected:
            raise ValueError(f'Archivo ausente/modificado: {name}')
    actual = {str(p.relative_to(directory)) for p in bundle_files(directory)}
    if actual != set(manifest):
        raise ValueError('Archivos extra en banco sellado.')
    return digest(manifest)


def load_bank(directory):
    """Punto de entrada común para notebooks 01–06."""
    directory = Path(directory)
    verify_bundle(directory)
    frozen_code = directory / 'code' / 'svrplab_pilot.py'
    if not frozen_code.exists() or file_hash(frozen_code) != file_hash(__file__):
        raise ValueError('Evaluador diferente al banco. Importe el módulo guardado en bank/code.')
    config = json.loads((directory / 'config.json').read_text())
    if config.get('protocol_version') != PROTOCOL_VERSION:
        raise ValueError('Versión del banco incompatible; no mezclar bancos v1, v2, v3 y v4.')
    instances = [freeze_instance(json.loads(p.read_text())) for p in sorted((directory / 'instances').glob('*.json'))]
    if sorted(x.n for x in instances) != sorted(config['sizes']):
        raise ValueError('Tamaños del banco no coinciden con configuración.')
    certificates, ledgers, histories = {}, {}, {}
    for x in instances:
        parent = freeze_instance(json.loads((directory / f'sources/reconstructed_v1_n{x.n:04d}.json').read_text()))
        if parent.sha != x.provenance['transformation']['parent_instance_sha256']:
            raise ValueError('Fuente reconstruida distinta del padre declarado.')
        for field in ('locations', 'demands', 'capacities', 'appear_times'):
            if not np.array_equal(getattr(parent, field), getattr(x, field)):
                raise ValueError('Derivado alteró geometría, demanda, flota o apariciones de su fuente.')
        if (not np.array_equal(parent.windows[:, 1] - parent.windows[:, 0], x.windows[:, 1] - x.windows[:, 0])
                or not np.array_equal(parent.windows[0], x.windows[0])):
            raise ValueError('Derivado alteró duración de ventanas o depósito.')
        certificate = json.loads((directory / f'certificates/n{x.n:04d}.json').read_text())
        certificates[x.n] = certificate
        ledgers[x.n] = json.loads((directory / f'sources/sampling_ledger_n{x.n:04d}.json').read_text())
        histories[x.n] = json.loads((directory / f'sources/admission_n{x.n:04d}.json').read_text())
    report = acceptance_report(instances, certificates, ledgers, config, histories)
    stored = json.loads((directory / 'acceptance.json').read_text())
    if stored != report:
        raise ValueError('Acta de aceptación distinta de la validación recalculada.')
    return instances


def sync_verified(source, destination):
    """Copia comprobada, sin borrar origen ni sobrescribir un banco diferente."""
    source, destination = Path(source), Path(destination)
    expected = verify_bundle(source)
    if destination.exists():
        if verify_bundle(destination) != expected:
            raise ValueError(f'Ya existe otro contenido en {destination}; use otro ID.')
        return destination
    staging = destination.with_name(destination.name + '.partial')
    if staging.exists():
        raise ValueError(f'Copia interrumpida en {staging}; conservar e inspeccionar antes de reintentar.')
    staging.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source, staging)
    if verify_bundle(staging) != expected:
        raise IOError('Copia no verificada; origen conservado.')
    staging.rename(destination)
    return destination


Overwriting svrplab/pilot.py


In [53]:
%%writefile svrplab/pomo.py
"""04: REINFORCE multistart propio inspirado en POMO; ejecutar sólo en Colab.

Reutiliza AttentionPolicy y sus máscaras conservadoras sin etiquetas, escenarios
ni certificados. No reproduce el POMO original ni RL4CO: ventanas y vehículos
heterogéneos no garantizan simetría entre primeros clientes. Los rollouts del
grupo comparten una codificación y se decodifican secuencialmente.

POMO: Kwon et al., https://arxiv.org/html/2010.16011v3, secciones 4.1–4.3.
La primera acción impuesta queda fuera del logprob y el baseline es la media
detached del costo del grupo de la misma instancia. Se minimiza costo, de modo
que loss = mean((cost - mean(cost)).detach() * sum(logprob)).

Adaptaciones del proyecto: si un vehículo inicial no admite ningún cliente se
cierran vehículos vacíos sólo cuando la máscara común lo permite. Se fuerza un
cliente del primer vehículo con clientes admisibles; no se exploran prefijos
vacíos alternativos cuando éste sí admite clientes. Un único inicio se duplica
para dos continuaciones muestreadas en entrenamiento. Un bloqueo sin acciones
produce loss cero y se registra, sin afirmar infactibilidad de la instancia.
"""
from __future__ import annotations

from collections.abc import Mapping
from dataclasses import dataclass
import math
import time

import numpy as np
import torch

from .attention import (_State, _TOL, _expired, _prepare, _scales, _sync, _tensors,
                        MASK_VERSION, policy_completion_search)
from .search import SearchProblem


IMPLEMENTATION = 'project_reinforce_pomo_v2'
POMO_DEFAULTS = {'n_starts': 8}
INFERENCE_DEFAULTS = {'n_starts': 16, 'sampling_rounds': 1, 'permute_samples': False,
                      'all_starts': False, 'completion_search_nodes': 2048}
PENALTY = 'bounded_unserved_then_partial_travel_v1'


def validate_pomo_params(params=None):
    """Configuración de grupos de entrenamiento/validación; retorna copia normalizada."""
    if params is not None and not isinstance(params, Mapping):
        raise ValueError('Los parámetros POMO deben ser un mapping.')
    unknown = set(params or {}) - set(POMO_DEFAULTS)
    if unknown:
        raise ValueError(f'Parámetros POMO desconocidos: {sorted(unknown)}')
    result = POMO_DEFAULTS | dict(params or {})
    starts = result['n_starts']
    if isinstance(starts, (bool, np.bool_)) or not isinstance(starts, (int, np.integer)) or starts < 2:
        raise ValueError('n_starts debe ser entero >=2 para el baseline compartido.')
    return {'n_starts': int(starts)}


def _inference_config(config):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración de inferencia debe ser un mapping.')
    allowed = set(INFERENCE_DEFAULTS) | {'seconds'}
    unknown = set(config or {}) - allowed
    if unknown:
        raise ValueError(f'Parámetros de inferencia desconocidos: {sorted(unknown)}')
    result = {'seconds': 120.} | INFERENCE_DEFAULTS | dict(config or {})
    result.update(validate_pomo_params({'n_starts': result['n_starts']}))
    seconds, rounds = result['seconds'], result['sampling_rounds']
    if (isinstance(seconds, (bool, np.bool_)) or not isinstance(seconds, (int, float, np.number))
            or not math.isfinite(float(seconds)) or seconds < 0):
        raise ValueError('seconds debe ser finito y no negativo.')
    if isinstance(rounds, (bool, np.bool_)) or not isinstance(rounds, (int, np.integer)) or rounds < 0:
        raise ValueError('sampling_rounds debe ser entero no negativo.')
    if type(result['permute_samples']) is not bool:
        raise ValueError('permute_samples debe ser booleano.')
    if type(result['all_starts']) is not bool:
        raise ValueError('all_starts debe ser booleano.')
    if type(result['completion_search_nodes']) is not int or result['completion_search_nodes'] < 0:
        raise ValueError('completion_search_nodes debe ser entero no negativo.')
    return result | {'seconds': float(seconds), 'sampling_rounds': int(rounds)}


def _cost_scale(problem):
    """Todo recorrido completo tiene <= n+K arcos y costo <= U <= S."""
    upper = (problem.n + problem.vehicles) * float(np.max(problem.travel))
    if not math.isfinite(upper) or upper < 0 or bool((problem.travel < 0).any()):
        raise ValueError('Se requiere una matriz conservadora finita y no negativa.')
    return max(1., upper)


def _starting_customers(problem):
    """Prefijo vacío legal y clientes originales admisibles, sin consultar el modelo."""
    state = _State(problem)
    order, scales = np.arange(problem.n + 1), _scales(problem)
    while not state.done:
        blocked = state.inputs(order, scales)[2]
        customers = np.flatnonzero(~blocked[1:]) + 1
        if len(customers):
            return state.vehicle, customers
        if blocked[0]:
            break
        state.advance(0)
    return state.vehicle, np.array([], dtype=np.int64)


def _select_starts(customers, limit, rng=None, duplicate_single=False):
    if not len(customers):
        return [None, None] if duplicate_single else [None]
    if len(customers) == 1 and duplicate_single:
        return [int(customers[0]), int(customers[0])]
    count = min(limit, len(customers))
    selected = customers[:count] if rng is None else rng.choice(customers, count, replace=False)
    return [int(customer) for customer in selected]


@dataclass
class _Rollout:
    routes: list | None
    logprob: torch.Tensor
    decisions: int
    cost: float
    unserved: int
    termination: str


def _state_cost(state, scale):
    """Costo normalizado completo <=1; bloqueado en (1, 2+1/(n+1)].

    Para m clientes pendientes: 1 + m/n + P/[S(n+1)], donde P es el viaje
    de rutas parciales cerradas al depósito, S=max(1,(n+K)*max(travel)).
    Como P<=S, servir un cliente domina cualquier cambio en P. El término
    sólo entrena/selecciona checkpoint; nunca es una métrica del evaluador.
    """
    problem = state.problem
    missing = int((~state.visited).sum())
    if state.done and not missing:
        violation, travel = problem.score(state.routes)
        if violation > _TOL:
            raise ValueError('La construcción viola las máscaras conservadoras comunes.')
        return float(travel / scale), missing, True
    partial = sum(problem.route_stats(route, vehicle)['travel_minutes']
                  for vehicle, route in enumerate(state.routes))
    # La pausa por presupuesto también puede ocurrir después del último cliente;
    # ese rollout nunca entra al entrenamiento ni se devuelve como solución.
    cost = 1. + missing / max(1, problem.n) + partial / (scale * (problem.n + 1))
    return float(cost), missing, False


def _rollout(model, problem, device, prepared, prefix, first, mode, rng=None, deadline=None):
    """El inicio y cierres del prefijo son exógenos: ningún logprob para ellos."""
    encoded, order, inverse, scales = prepared
    state = _State(problem)
    for _ in range(prefix):
        state.advance(0)
    if first is not None:
        if bool(state.inputs(order, scales)[2][int(inverse[first])]):
            raise ValueError('Primer cliente POMO no admisible.')
        state.advance(first)
    terms, termination = [], 'complete'
    action_generator = None
    if mode == 'sample' and isinstance(rng, np.random.Generator):
        action_generator = torch.Generator(device=device).manual_seed(int(rng.integers(0, 2**63 - 1)))
    while not state.done:
        if _expired(deadline, device):
            termination = 'time_limit'
            break
        context, action, blocked = _tensors(state, order, scales, device)
        if bool(blocked.all()):
            termination = 'dead_end'
            break
        logits = model.next_logits(encoded, int(inverse[state.node]), context, action, blocked)
        if not bool(torch.isfinite(logits[~blocked]).all()):
            raise FloatingPointError('Logits POMO no finitos.')
        logprobs = torch.log_softmax(logits, dim=-1)
        if mode == 'greedy':
            local = int(torch.argmax(logits).item())
        elif mode == 'sample' and isinstance(rng, np.random.Generator):
            # Sample on-device: do not copy the full action distribution to CPU.
            # Its dedicated seed comes from the checkpointed NumPy RNG.
            probabilities = torch.softmax(logits, dim=-1)
            local = int(torch.multinomial(probabilities, 1, generator=action_generator).item())
        else:
            raise ValueError('Use greedy o sample con un Generator NumPy explícito.')
        terms.append(logprobs[local])
        state.advance(int(order[local]))
    cost, missing, complete = _state_cost(state, _cost_scale(problem))
    # Mantiene backward definido incluso en un grupo bloqueado antes de decidir.
    logprob = encoded.sum() * 0.
    if terms:
        logprob = logprob + torch.stack(terms).sum()
    if complete:
        termination = 'complete'
    return _Rollout(state.routes if complete else None, logprob, len(terms), cost, missing, termination)


def group_loss(model, problem, device, rng, params=None, node_order=None):
    """REINFORCE por instancia; caller controla train/eval, backward y optimizer.

    Selecciona primeros clientes sin reemplazo y muestrea sus continuaciones.
    Dropout debe ser cero: toda aleatoriedad de acciones procede de rng, cuyo
    estado debe conservarse en el checkpoint. No cambia datos ni flota.
    """
    effective = validate_pomo_params(params)
    if not isinstance(rng, np.random.Generator):
        raise ValueError('rng debe ser un Generator NumPy explícito.')
    if model.config.dropout != 0:
        raise ValueError('REINFORCE POMO requiere dropout=0 para la política de muestreo.')
    prefix, customers = _starting_customers(problem)
    starts = _select_starts(customers, effective['n_starts'], rng, duplicate_single=True)
    prepared = _prepare(model, problem, device, node_order)
    rollouts = [_rollout(model, problem, device, prepared, prefix, first, 'sample', rng)
                for first in starts]
    logprobs = torch.stack([rollout.logprob for rollout in rollouts])
    costs = torch.as_tensor([rollout.cost for rollout in rollouts], dtype=logprobs.dtype, device=device)
    baseline = costs.mean().detach()
    advantages = (costs - baseline).detach()
    loss = (advantages * logprobs).mean()
    if not bool(torch.isfinite(loss)):
        raise FloatingPointError('Loss REINFORCE no finita; detener entrenamiento.')
    complete_count = sum(rollout.routes is not None for rollout in rollouts)
    scale = _cost_scale(problem)
    return loss, {
        'decisions': sum(rollout.decisions for rollout in rollouts),
        'mean_cost': float(costs.mean().detach().cpu()),
        'mean_penalized_cost_minutes': float(costs.mean().detach().cpu()) * scale,
        'normalization_scale_minutes': scale,
        'complete_fraction': complete_count / len(rollouts),
        'complete_count': complete_count, 'rollout_count': len(rollouts),
        'forced_starts': starts, 'leading_empty_vehicles': prefix,
        'costs': [rollout.cost for rollout in rollouts],
        'advantages': advantages.cpu().tolist(),
        'zero_advantage_group': not bool((advantages != 0).any()),
        'unserved': [rollout.unserved for rollout in rollouts],
        'baseline': float(baseline.cpu()), 'penalty': PENALTY,
    }


def validation_score(model, problem, device, params=None):
    """Mejor costo de greedy multistart determinista; no usa semillas/escenarios.

    Los primeros IDs admisibles fijan el subconjunto cuando hay más que n_starts.
    Una sola alternativa se evalúa una vez; replicarla no cambiaría el greedy.
    """
    effective = validate_pomo_params(params)
    prefix, customers = _starting_customers(problem)
    starts = _select_starts(customers, effective['n_starts'])
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            prepared = _prepare(model, problem, device, None)
            rollouts = [_rollout(model, problem, device, prepared, prefix, first, 'greedy')
                        for first in starts]
        complete = sum(rollout.routes is not None for rollout in rollouts)
        return {'score': min(rollout.cost for rollout in rollouts),
                'complete_fraction': complete / len(rollouts), 'rollout_count': len(rollouts),
                'best_complete': complete > 0, 'forced_starts': starts,
                'normalization_scale_minutes': _cost_scale(problem), 'penalty': PENALTY}
    finally:
        model.train(was_training)


def solve_pomo(problem: SearchProblem, seed, config, model, device):
    """Greedy multistart + rondas de muestreo, con presupuesto único compartido.

    Sólo devuelve rutas completas. El límite incluye codificación, selección de
    inicios, construcción y validación; la carga de pesos pertenece al wrapper.
    """
    if isinstance(seed, (bool, np.bool_)) or not isinstance(seed, (int, np.integer)) or seed < 0:
        raise ValueError('La semilla debe ser un entero no negativo.')
    effective = _inference_config(config)
    device = torch.device(device)
    _sync(device)
    started = time.perf_counter()
    deadline = started + effective['seconds']
    rng = np.random.default_rng(int(seed))
    best, best_score, history = None, (math.inf, math.inf), []
    attempts = completed = failed = 0
    termination = 'round_limit'
    completion_search = {'termination': 'not_needed', 'expanded_nodes': 0, 'is_policy_sample': False}
    eligible_starts = starts_scheduled = 0
    was_training = model.training
    model.eval()
    try:
        with torch.inference_mode():
            for round_index in range(effective['sampling_rounds'] + 1):
                if _expired(deadline, device):
                    termination = 'time_limit'
                    break
                mode = 'greedy' if round_index == 0 else 'sample'
                prefix, customers = _starting_customers(problem)
                eligible_starts = len(customers)
                limit = len(customers) if effective['all_starts'] else effective['n_starts']
                starts = _select_starts(customers, limit, rng if mode == 'sample' else None)
                starts_scheduled += len(starts)
                order = (rng.permutation(np.arange(1, problem.n + 1))
                         if mode == 'sample' and effective['permute_samples'] else None)
                prepared = _prepare(model, problem, device, order)
                for first in starts:
                    if _expired(deadline, device):
                        termination = 'time_limit'
                        break
                    attempts += 1
                    rollout = _rollout(model, problem, device, prepared, prefix, first,
                                       mode, rng=rng, deadline=deadline)
                    if rollout.routes is None:
                        failed += 1
                    else:
                        completed += 1
                        score = problem.score(rollout.routes)
                        if score < best_score:
                            best, best_score = rollout.routes, score
                            _sync(device)
                            history.append({'iteration': attempts - 1,
                                            'elapsed_seconds': time.perf_counter() - started,
                                            'violation_minutes': float(score[0]), 'travel_minutes': float(score[1]),
                                            'phase': mode, 'round': round_index, 'first_customer': first})
                    if _expired(deadline, device):
                        termination = 'time_limit'
                        break
                if termination == 'time_limit':
                    break
            if best is None and effective['completion_search_nodes'] and not _expired(deadline, device):
                best, completion_search = policy_completion_search(
                    model, problem, device, deadline, effective['completion_search_nodes'])
                if best is not None:
                    best_score = problem.score(best)
                    history.append({'iteration': attempts, 'elapsed_seconds': time.perf_counter() - started,
                                    'violation_minutes': float(best_score[0]), 'travel_minutes': float(best_score[1]),
                                    'phase': 'policy_completion_search'})
    finally:
        model.train(was_training)
    if best is None and completion_search['termination'] == 'not_needed':
        completion_search['termination'] = ('no_remaining_budget' if effective['completion_search_nodes'] else 'disabled')
    if _expired(deadline, device):
        termination = 'time_limit'
    _sync(device)
    elapsed = time.perf_counter() - started
    result = problem.result(
        best, 'feasible' if best is not None else 'no_solution', elapsed, history,
        method='pomo', seed=int(seed), implementation=IMPLEMENTATION,
        model_config=model.config.to_dict(), effective_config=effective,
        termination=termination, attempts=attempts, completed_decodes=completed, failed_decodes=failed,
        raw_decode_complete_fraction=completed / attempts if attempts else None,
        completion_search=completion_search, mask_version=MASK_VERSION,
        eligible_first_customers=eligible_starts, starts_scheduled=starts_scheduled,
        all_starts_completed=(effective['all_starts'] and eligible_starts > 0
                              and attempts == starts_scheduled and termination != 'time_limit'),
        solution_origin='policy_completion_search' if completion_search['termination'] == 'feasible_found' else
                        'raw_policy_decode' if best is not None else 'none',
        inference_device=str(device), budget_includes_construction=True,
        budget_includes_encoding=True, budget_overrun_seconds=max(0., elapsed - effective['seconds']),
        time_limit_granularity='synchronized_decoder_step_or_completed_validation',
        search_objective='conservative_upper_travel_minutes',
        nominal_scope='conservative_upper_mean_matrix_not_stochastic_feasibility',
        construction=('policy_ordered_depth_first_completion' if completion_search['termination'] == 'feasible_found'
                      else 'forced_feasible_first_customer_without_certificates_or_repair'),
        no_solution_scope='decoder_dead_end_or_budget_not_infeasibility_proof',
        seed_reproducibility='same_environment_and_round_cutoff; wall_time_cutoff_may_differ',
        training_included_in_solve_seconds=False)
    _sync(device)
    elapsed = time.perf_counter() - started
    result['solve_seconds'] = elapsed
    result['metadata']['budget_overrun_seconds'] = max(0., elapsed - effective['seconds'])
    return result


Overwriting svrplab/pomo.py


In [54]:
%%writefile svrplab/reinforcement.py
"""04: POMO por refuerzo, separación por geometría y checkpoints recuperables.

Ejecución sólo Colab. Sin profesores, certificados ni escenarios de evaluación
para entrenar o seleccionar modelos. Objetivo conservador de desarrollo, no c+Q.
"""
from __future__ import annotations

import math
import random
import time
import uuid
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch

from . import experiments as e
from . import pilot as p
from .attention import AttentionConfig, AttentionPolicy, UniformMaskedPolicy, MASK_VERSION
from .pomo import group_loss, validation_score, solve_pomo, validate_pomo_params
from .search import OBJECTIVE, SearchProblem
from .supervised_data import make_geometry_folds, validate_fold, validate_holdout_fold
from .supervised import (require_colab, synchronize, choose_device, _cpu_state,
                         _load_checkpoint, _runtime_signature)


CHECKPOINT_SELECTION = 'prespecified_gate_then_validation_coverage_completion_fraction_cost_v1'
FINAL_CHECKPOINT_SELECTION = CHECKPOINT_SELECTION


def validation_candidate_is_better(score, diagnostics, best_score, best_diagnostics, min_delta):
    """Prioridad previa a completitud; ninguna media baja rescata un caso fallido.

    Desarrollo exige completitud en cada instancia. La puerta final exige alguna
    geometría completa por tamaño y cobertura train del checkpoint. Entre candidatos
    elegibles se priorizan cobertura val, fracción de trayectorias y costo.
    No consulta heldout ni escenarios.
    """
    coverage = diagnostics['instances_with_completion_fraction']
    fraction = diagnostics['complete_fraction']
    if any(not math.isfinite(value) or not 0. <= value <= 1. for value in (coverage, fraction)):
        raise ValueError('Diagnóstico de completitud fuera de [0,1].')
    if not math.isfinite(score):
        raise ValueError('Costo de validación no finito.')
    eligible = diagnostics.get('checkpoint_eligible', coverage == 1.)
    if not eligible or fraction <= 0.:
        return False
    if best_diagnostics is None:
        return True
    previous_coverage = best_diagnostics['instances_with_completion_fraction']
    if abs(coverage - previous_coverage) > 1e-12:
        return coverage > previous_coverage
    previous = best_diagnostics['complete_fraction']
    return (fraction > previous + 1e-12 or
            abs(fraction - previous) <= 1e-12 and score < best_score - min_delta)


def selected_checkpoint_path(final, summary):
    """Comprueba el checkpoint seleccionado o el último diagnóstico rechazado.

    Un resultado rechazado referencia su state.pt ya sellado, sin crear best.pt
    ni permitir cargar sus pesos para inferencia. El hash no se inventa.
    """
    final = Path(final)
    if summary.get('selection_status') == 'selected_complete_validation':
        if (summary.get('best_epoch', 0) < 1 or
                not (summary.get('best_validation_diagnostics') or {}).get('checkpoint_eligible', False)):
            raise ValueError('Checkpoint elegido sin completitud de validación.')
        path = final / 'best.pt'
    elif summary.get('selection_status') == 'validation_gate_failed':
        if (summary.get('best_epoch') != 0 or (final / 'best.pt').exists() or
                summary.get('checkpoint_selection') != 'rejected_no_complete_validation'):
            raise ValueError('Modelo rechazado presentado como seleccionado.')
        epoch = summary.get('epochs_completed')
        if type(epoch) is not int or epoch < 1:
            raise ValueError('Fallo de validación sin checkpoint diagnóstico.')
        directory = final.parent / f'epoch_{epoch:04d}'
        p.verify_bundle(directory)
        path = directory / 'state.pt'
    else:
        raise ValueError('Modelo sin regla de selección por completitud; conservar como evidencia histórica.')
    if p.file_hash(path) != summary['model_checkpoint_sha256']:
        raise ValueError('Checkpoint de entrenamiento modificado.')
    return path


def validate_training_config(config):
    if (config['schema_version'] != 'reinforcement-04-v1' or
            (config['role'], config['split_strategy']) not in {
                ('development_only', 'leave_one_geometry_out_with_inner_validation'),
                ('train_validation_for_final_holdout', 'independent_train_validation_test_v1')}):
        raise ValueError('Configuración de refuerzo incompatible.')
    if config['objective'] != OBJECTIVE or type(config['time_scale']) is not int or config['time_scale'] < 1:
        raise ValueError('Objetivo o escala temporal incompatible.')
    if not isinstance(config.get('replicate_id'), str) or not config['replicate_id'].strip():
        raise ValueError('replicate_id debe identificar una réplica con texto no vacío.')
    model = AttentionConfig.from_value(config['model'])
    if model.dropout != 0.:
        raise ValueError('POMO requiere dropout=0 para la política y su baseline compartida.')
    validate_pomo_params(config['rollout'])
    settings = config['training']
    if settings.get('checkpoint_gate', 'all_validation_instances') not in ('all_validation_instances', 'one_complete_geometry_per_size'):
        raise ValueError('Puerta de completitud desconocida.')
    if config['role'] == 'development_only' and settings.get('checkpoint_gate', 'all_validation_instances') != 'all_validation_instances':
        raise ValueError('Los folds históricos requieren completitud de cada instancia de validación.')
    if settings.get('geometry_sampling', 'iid') not in ('iid', 'shuffled_without_replacement'):
        raise ValueError('Muestreo de geometrías desconocido.')
    expected_selection = CHECKPOINT_SELECTION if config['role'] == 'development_only' else FINAL_CHECKPOINT_SELECTION
    if settings.get('checkpoint_selection', expected_selection) != expected_selection:
        raise ValueError('Regla de selección debe priorizar completitud en toda validación.')
    for name in ['epochs', 'steps_per_epoch', 'gradient_accumulation', 'patience']:
        if type(settings[name]) is not int or settings[name] < 1:
            raise ValueError(f'{name} debe ser entero positivo.')
    for name in ['learning_rate', 'gradient_clip_norm', 'seconds_per_model']:
        if isinstance(settings[name], bool) or not math.isfinite(settings[name]) or settings[name] <= 0:
            raise ValueError(f'{name} debe ser finito y positivo.')
    for name in ['weight_decay', 'min_delta']:
        if isinstance(settings[name], bool) or not math.isfinite(settings[name]) or settings[name] < 0:
            raise ValueError(f'{name} debe ser finito y no negativo.')
    for name in ['permute_nodes', 'deterministic_algorithms']:
        if type(settings[name]) is not bool:
            raise ValueError(f'{name} debe ser booleano.')
    return True


def _source_hashes():
    directory = Path(__file__).parent
    return {name: p.file_hash(directory / name) for name in
            ('reinforcement.py', 'pomo.py', 'supervised.py', 'supervised_data.py',
             'attention.py', 'search.py', 'metaheuristics.py', 'baselines.py', 'experiments.py')}


def _group_instances(instances, groups):
    grouped = {group: [] for group in groups}
    for x in instances:
        group = x.provenance['geometry_group_sha256']
        if group in grouped:
            grouped[group].append(x.sha)
    if not grouped or any(not values for values in grouped.values()):
        raise ValueError('Faltan instancias de alguna geometría autorizada.')
    return {group: sorted(shas) for group, shas in grouped.items()}


def validation_cost(model, problems, grouped, groups, device, params, return_diagnostics=False, selection_gate='all_validation_instances'):
    """Media por instancia y luego geometría del mínimo costo greedy multistart.

    Incompletas tienen costo normalizado >1; completas <=1. No usa ruido,
    etiquetas ni gradientes y mantiene el mismo panel de primeros clientes.
    """
    if set(grouped) != set(groups) or any(not values for values in grouped.values()):
        raise ValueError('Geometrías de validación ausentes o ajenas.')
    model.eval()
    losses, completeness, successes, sizes_completed = {}, {}, {}, {}
    with torch.no_grad():
        for group in sorted(groups):
            records = [validation_score(model, problems[sha], device, params) for sha in grouped[group]]
            if selection_gate == 'one_complete_geometry_per_size':
                for sha, record in zip(grouped[group], records):
                    size = str(problems[sha].n)
                    sizes_completed[size] = sizes_completed.get(size, False) or bool(record['best_complete'])
            values = [float(record['score']) for record in records]
            if not all(math.isfinite(value) for value in values):
                raise ValueError('Costo de validación no finito.')
            losses[group] = float(np.mean(values))
            completeness[group] = float(np.mean([record['complete_fraction'] for record in records]))
            successes[group] = float(np.mean([record.get('best_complete', record['score'] <= 1.) for record in records]))
    mean = float(np.mean(list(losses.values())))
    if not return_diagnostics:
        return mean, losses
    eligible = (all(sizes_completed.values()) and bool(sizes_completed) if selection_gate == 'one_complete_geometry_per_size'
                else all(value == 1. for value in successes.values()))
    return mean, losses, {'checkpoint_eligible': eligible, 'selection_gate': selection_gate,
                         'sizes_with_complete_geometry': sizes_completed, 'complete_fraction': float(np.mean(list(completeness.values()))),
                         'instances_with_completion_fraction': float(np.mean(list(successes.values()))),
                         'complete_fraction_by_geometry': completeness}


def readiness_diagnostics(initial, history, min_delta):
    """Transparent development signals, not a claim of sufficient training."""
    updates = sum(row['optimizer_updates'] for row in history)
    best = (next((row for row in history if row['epoch'] == history[-1]['best_epoch']), None)
            if history else None)
    initial_score = initial['validation_score']
    improvement = initial_score - best['validation_score'] if best else 0.
    completed = bool(best and best.get('validation_checkpoint_eligible', best['validation_instances_with_completion_fraction'] == 1.))
    benefit = improvement > min_delta
    status = ('no_optimizer_updates' if not updates else 'no_complete_validation' if not completed
              else 'no_observed_learning_benefit' if not benefit else 'development_learning_signal')
    return {'status': status, 'optimizer_updates': updates,
            'raw_training_rollouts': sum(row['rollout_count'] for row in history),
            'decisions': sum(row['decisions'] for row in history),
            'validation_improvement_from_initialization': improvement,
            'improves_initialization_by_min_delta': benefit,
            'validation_has_complete_solution': completed,
            'adequate_training_demonstrated': False,
            'statistical_superiority_demonstrated': False,
            'interpretation': 'development_diagnostic_requires_independent_geometry_replication'}


def train_fold(instances, spec, config, seed, fold, directory, remote, device, split_strategy='development_crossfit'):
    require_colab()
    validate_training_config(config)
    if type(seed) is not int or seed < 0:
        raise ValueError('La semilla de entrenamiento debe ser un entero no negativo.')
    if split_strategy == 'development_crossfit':
        validate_fold(fold, instances)
    elif split_strategy == 'independent_train_validation_test_v1':
        validate_holdout_fold(fold, instances)
    else:
        raise ValueError('Estrategia de partición no admitida.')
    directory, remote = Path(directory), Path(remote)
    device = choose_device(str(device))
    started = time.perf_counter()
    grouped = _group_instances(instances, fold['train_geometry_groups'])
    validation = _group_instances(instances, fold['validation_geometry_groups'])
    identity = {'schema_version': config['schema_version'], 'fold': fold, 'seed': seed,
                'split_strategy': split_strategy,
                'config': config, 'source_hashes': _source_hashes(), 'traffic': asdict(spec),
                'train_instances': grouped, 'validation_instances': validation,
                'runtime_signature': _runtime_signature(device),
                'evaluator_sha256': p.file_hash(p.__file__)}
    if e._recover(directory / 'meta', remote / 'meta'):
        if e.read_json(directory / 'meta/identity.json') != identity:
            raise ValueError('Checkpoint de entrenamiento incompatible con instancias, código, fold o parámetros.')
    else:
        e.write_json(directory / 'meta/identity.json', identity)
        e.write_json(directory / 'meta/environment.json', p.environment_report())
        e._commit(directory / 'meta', remote / 'meta')
    if e._recover(directory / 'final', remote / 'final'):
        recovered_summary = e.read_json(directory / 'final/summary.json')
        if recovered_summary.get('selection_status') == 'validation_gate_failed':
            rejected_epoch = f'epoch_{recovered_summary["epochs_completed"]:04d}'
            if not e._recover(directory / rejected_epoch, remote / rejected_epoch):
                raise ValueError('Falta el checkpoint diagnóstico del modelo rechazado.')
        selected_checkpoint_path(directory / 'final', recovered_summary)
        if recovered_summary['identity_sha256'] != p.digest(identity):
            raise ValueError('Modelo final de otro entrenamiento.')
        return directory / 'final'
    instances_by_sha = {x.sha: x for x in instances}
    selected = {sha for groups in (grouped, validation) for shas in groups.values() for sha in shas}
    # No se construye ni inspecciona el problema de la geometría heldout.
    problems = {sha: SearchProblem(instances_by_sha[sha], spec, config['time_scale'])
                for sha in sorted(selected)}
    settings = config['training']
    derived_seed = p.seed_for('reinforcement-04', fold['fold_id'], seed)
    random.seed(derived_seed)
    torch.manual_seed(derived_seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(derived_seed)
    torch.use_deterministic_algorithms(settings['deterministic_algorithms'])
    torch.backends.cudnn.benchmark = False
    rng = np.random.default_rng(derived_seed)
    model = AttentionPolicy(config['model']).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings['learning_rate'],
                                 weight_decay=settings['weight_decay'])
    best_state, best_loss, best_epoch, stale = None, None, 0, 0
    best_diagnostics, best_geometry_visits = None, None
    history, elapsed_total, validation_total, last_epoch, terminal = [], 0.0, 0.0, 0, None
    geometry_visits, geometry_queue = {group: 0 for group in grouped}, []
    checkpoint_numbers = set()
    for location in (directory, remote):
        if location.exists():
            checkpoint_numbers.update(int(child.name[6:]) for child in location.glob('epoch_*')
                                      if child.name[6:].isdigit() and (child / 'CHECKSUMS.json').exists())
    if checkpoint_numbers and sorted(checkpoint_numbers) != list(range(1, max(checkpoint_numbers) + 1)):
        raise ValueError('Faltan épocas selladas en la secuencia de checkpoints.')
    for epoch in sorted(checkpoint_numbers):
        if not e._recover(directory / f'epoch_{epoch:04d}', remote / f'epoch_{epoch:04d}'):
            raise ValueError('Checkpoint de época no recuperable.')
    if checkpoint_numbers:
        last_epoch = max(checkpoint_numbers)
        latest = directory / f'epoch_{last_epoch:04d}'
        progress = e.read_json(latest / 'progress.json')
        if (progress['identity_sha256'] != p.digest(identity)
                or progress['runtime_signature'] != _runtime_signature(device)):
            raise ValueError('No se reanuda entrenamiento parcial con otro entorno/fold. Use el entorno original o una configuración nueva.')
        state = _load_checkpoint(latest / 'state.pt')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        best_state = state['best_model']
        best_loss, best_epoch, stale = progress['best_validation_score'], progress['best_epoch'], progress['stale_epochs']
        best_diagnostics = progress['best_validation_diagnostics']
        best_geometry_visits = progress['selected_checkpoint_training_geometry_visits']
        geometry_visits, geometry_queue = progress['training_geometry_visits'], progress['geometry_queue']
        history = progress['history']
        elapsed_total, validation_total = progress['training_seconds'], progress['validation_seconds']
        rng.bit_generator.state = progress['numpy_generator_state']
        random.setstate((progress['python_rng_state'][0], tuple(progress['python_rng_state'][1]), progress['python_rng_state'][2]))
        torch.set_rng_state(state['torch_rng'])
        if device.type == 'cuda':
            torch.cuda.set_rng_state_all(state['cuda_rng'])
        terminal = progress['stop_reason']
    synchronize(device)
    elapsed_total += time.perf_counter() - started
    initial = None
    if e._recover(directory / 'initial', remote / 'initial'):
        initial = e.read_json(directory / 'initial/diagnostics.json')
        if initial['identity_sha256'] != p.digest(identity):
            raise ValueError('Diagnóstico de inicialización de otro entrenamiento.')
        if not checkpoint_numbers:
            elapsed_total += initial['validation_seconds']
            validation_total += initial['validation_seconds']
    elif elapsed_total < settings['seconds_per_model']:
        if checkpoint_numbers:
            raise ValueError('Falta diagnóstico inicial del entrenamiento recuperado.')
        baseline_started = time.perf_counter()
        score, by_geometry, diagnostics = validation_cost(
            model, problems, validation, fold['validation_geometry_groups'], device,
            config['rollout'], return_diagnostics=True, selection_gate=settings.get('checkpoint_gate', 'all_validation_instances'))
        synchronize(device)
        seconds = time.perf_counter() - baseline_started
        initial = {'identity_sha256': p.digest(identity), 'validation_score': score,
                   'validation_by_geometry': by_geometry, **diagnostics,
                   'validation_seconds': seconds, 'model_state': 'untrained_initialization',
                   'heldout_used': False, 'mask_version': MASK_VERSION}
        elapsed_total += seconds
        validation_total += seconds
        e.write_json(directory / 'initial/diagnostics.json', initial)
        e._commit(directory / 'initial', remote / 'initial')
    groups = sorted(grouped)
    for epoch in range(last_epoch + 1, settings['epochs'] + 1):
        if terminal:
            break
        if elapsed_total >= settings['seconds_per_model']:
            terminal = 'time_limit'
            break
        epoch_started = time.perf_counter()
        model.train()
        values, costs, fractions, rollouts, updates, decisions = [], [], [], 0, 0, 0
        zero_advantage_groups = 0
        for offset in range(0, settings['steps_per_epoch'], settings['gradient_accumulation']):
            if elapsed_total + time.perf_counter() - epoch_started >= settings['seconds_per_model']:
                break
            batch_size = min(settings['gradient_accumulation'], settings['steps_per_epoch'] - offset)
            optimizer.zero_grad(set_to_none=True)
            for _ in range(batch_size):
                if settings.get('geometry_sampling', 'iid') == 'shuffled_without_replacement':
                    if not geometry_queue:
                        geometry_queue = [str(group) for group in rng.permutation(groups)]
                    group = geometry_queue.pop()
                else:
                    group = groups[int(rng.integers(len(groups)))]
                items = grouped[group]
                sha = items[int(rng.integers(len(items)))]
                problem = problems[sha]
                order = rng.permutation(np.arange(1, problem.n + 1)).tolist() if settings['permute_nodes'] else None
                loss, stats = group_loss(model, problem, device, rng, config['rollout'], node_order=order)
                if not bool(torch.isfinite(loss).item()):
                    raise ValueError('Pérdida de entrenamiento no finita.')
                (loss / batch_size).backward()
                geometry_visits[group] += 1
                values.append(float(loss.detach().cpu()))
                decisions += stats['decisions']
                costs.append(stats['mean_cost'])
                fractions.append(stats['complete_fraction'])
                rollouts += stats['rollout_count']
                zero_advantage_groups += int(stats.get('zero_advantage_group', False))
            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), settings['gradient_clip_norm'], error_if_nonfinite=True)
            if not math.isfinite(float(norm)):
                raise ValueError('Gradientes no finitos.')
            optimizer.step(); updates += 1
        if not values:
            elapsed_total += time.perf_counter() - epoch_started
            terminal = 'time_limit'
            break
        synchronize(device)
        val_started = time.perf_counter()
        val_loss, group_losses, val_diagnostics = validation_cost(
            model, problems, validation, fold['validation_geometry_groups'], device,
            config['rollout'], return_diagnostics=True, selection_gate=settings.get('checkpoint_gate', 'all_validation_instances'))
        synchronize(device)
        val_seconds = time.perf_counter() - val_started
        if split_strategy == 'independent_train_validation_test_v1':
            val_diagnostics['training_geometry_coverage'] = all(value >= 1 for value in geometry_visits.values())
            val_diagnostics['checkpoint_eligible'] &= val_diagnostics['training_geometry_coverage']
        if validation_candidate_is_better(val_loss, val_diagnostics, best_loss, best_diagnostics, settings['min_delta']):
            best_loss, best_epoch, best_state, stale = val_loss, epoch, _cpu_state(model), 0
            best_diagnostics = val_diagnostics
            best_geometry_visits = dict(geometry_visits)
        else:
            stale += 1
        elapsed_total += time.perf_counter() - epoch_started
        validation_total += val_seconds
        terminal = ('patience' if stale >= settings['patience'] else
                    'time_limit' if elapsed_total >= settings['seconds_per_model'] else
                    'epochs' if epoch == settings['epochs'] else None)
        history.append({'epoch': epoch, 'train_loss': float(np.mean(values)), 'train_cost': float(np.mean(costs)),
                        'train_complete_fraction': float(np.mean(fractions)), 'rollout_count': rollouts,
                        'validation_score': val_loss,
                        'validation_complete_fraction': val_diagnostics['complete_fraction'],
                        'validation_instances_with_completion_fraction': val_diagnostics['instances_with_completion_fraction'],
                        'validation_checkpoint_eligible': val_diagnostics['checkpoint_eligible'],
                        'best_validation_score': best_loss, 'best_epoch': best_epoch,
                        'instance_groups_sampled': len(values), 'decisions': decisions, 'optimizer_updates': updates,
                        'zero_advantage_groups': zero_advantage_groups,
                        'epoch_compute_seconds': time.perf_counter() - epoch_started,
                        'epoch_decisions_per_second': decisions / max(1e-12, time.perf_counter() - epoch_started - val_seconds),
                        'training_seconds': elapsed_total, 'validation_seconds': validation_total})
        target = directory / f'epoch_{epoch:04d}'
        # Preservar una escritura interrumpida de esta época antes de repetirla.
        if e._recover(target, remote / target.name):
            raise ValueError('La época nueva ya existe; no sobrescribir un checkpoint.')
        target.mkdir(parents=True, exist_ok=True)
        torch.save({'model': _cpu_state(model), 'optimizer': optimizer.state_dict(), 'best_model': best_state,
                    'torch_rng': torch.get_rng_state(),
                    'cuda_rng': torch.cuda.get_rng_state_all() if device.type == 'cuda' else []}, target / 'state.pt')
        e.write_json(target / 'progress.json', {'epoch': epoch, 'identity_sha256': p.digest(identity),
                     'runtime_signature': _runtime_signature(device), 'best_validation_score': best_loss,
                     'best_epoch': best_epoch, 'best_validation_diagnostics': best_diagnostics,
                     'stale_epochs': stale, 'history': history,
                     'training_geometry_visits': geometry_visits, 'geometry_queue': geometry_queue,
                     'selected_checkpoint_training_geometry_visits': best_geometry_visits,
                     'training_seconds': elapsed_total, 'validation_seconds': validation_total,
                     'numpy_generator_state': rng.bit_generator.state, 'python_rng_state': random.getstate(),
                     'validation_by_geometry': group_losses, 'stop_reason': terminal})
        e.write_json(target / 'environment.json', p.environment_report())
        e._commit(target, remote / target.name)
        print(f'{fold["fold_id"]} seed={seed} epoch={epoch}: train={history[-1]["train_loss"]:.5f}, '
              f'val={val_loss:.5f}, best={best_epoch}, '
              f'completos train={history[-1]["train_complete_fraction"]:.1%}/val={val_diagnostics["complete_fraction"]:.1%}, '
              f'updates={sum(row["optimizer_updates"] for row in history)}', flush=True)
    if not history:
        raise RuntimeError('No hay checkpoint entrenado; no evaluar pesos aleatorios.')
    final = directory / 'final'
    final.mkdir(parents=True, exist_ok=True)
    selection_status = 'selected_complete_validation' if best_state is not None else 'validation_gate_failed'
    checkpoint = final / 'best.pt' if best_state is not None else directory / f'epoch_{len(history):04d}/state.pt'
    if best_state is not None:
        torch.save(best_state, checkpoint)
    summary = {'identity_sha256': p.digest(identity), 'fold': fold, 'training_seed': seed,
               'training_seconds': elapsed_total, 'validation_seconds': validation_total,
               'training_budget_seconds': settings['seconds_per_model'],
               'training_budget_overrun_seconds': max(0., elapsed_total - settings['seconds_per_model']),
               'training_time_scope': 'setup_and_epoch_compute_validation_excludes_epoch_and_final_checkpoint_persistence',
               'teacher_seconds': 0.0,
               'selection_status': selection_status, 'best_validation_diagnostics': best_diagnostics,
               'split_strategy': split_strategy, 'training_geometry_visits': geometry_visits,
               'selected_checkpoint_training_geometry_visits': best_geometry_visits,
               'best_epoch': best_epoch, 'best_validation_score': best_loss, 'epochs_completed': len(history),
               'initial_validation_score': initial['validation_score'],
               'initial_validation_complete_fraction': initial['complete_fraction'],
               'readiness': readiness_diagnostics(initial, history, settings['min_delta']),
               'readiness_status': readiness_diagnostics(initial, history, settings['min_delta'])['status'],
               'optimizer_updates': sum(row['optimizer_updates'] for row in history),
               'mask_version': MASK_VERSION, 'decoder_execution': 'sequential_starts_on_device_sampling',
               'stop_reason': terminal, 'model_config': config['model'], 'traffic': asdict(spec),
               'time_scale': config['time_scale'], 'objective': OBJECTIVE,
               'checkpoint_selection': settings.get('checkpoint_selection', CHECKPOINT_SELECTION) if best_state is not None else 'rejected_no_complete_validation',
               'checkpoint_artifact_kind': 'selected_weights' if best_state is not None else 'unselected_epoch_state_for_audit_only',
               'training_scope': ('development_geometry_crossfit' if split_strategy == 'development_crossfit' else 'independent_train_validation_for_final_holdout') + '_conservative_pomo',
               'zero_advantage_groups': sum(row['zero_advantage_groups'] for row in history),
               'instance_groups_sampled': sum(row['instance_groups_sampled'] for row in history),
               'train_instances': sum(map(len, grouped.values())),
               'validation_instances': sum(map(len, validation.values())),
               'train_geometries': len(grouped), 'heldout_used_for_training_or_selection': False,
               'model_checkpoint_sha256': p.file_hash(checkpoint)}
    e.write_json(final / 'summary.json', summary)
    e.write_json(final / 'history.json', history)
    e.write_json(final / 'initial_diagnostics.json', initial)
    e.write_csv(final / 'history.csv', history)
    save_learning_plot(history, final / 'learning.png', fold['fold_id'], seed, initial)
    e._commit(final, remote / 'final')
    del model, optimizer
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return final


def save_learning_plot(history, path, fold_id, seed, initial=None):
    import matplotlib.pyplot as plt
    fig, (ax, completion) = plt.subplots(1, 2, figsize=(12, 4))
    for metric in ['train_cost', 'validation_score']:
        ax.plot([row['epoch'] for row in history], [row[metric] for row in history], label=metric)
    ax.set(title=f'{fold_id} · seed={seed} · Costo conservador penalizado', xlabel='Época', ylabel='Costo normalizado')
    if initial is not None:
        ax.axhline(initial['validation_score'], color='gray', linestyle='--', label='Sin entrenar (validación)')
    for metric, label in [('train_complete_fraction', 'Entrenamiento'),
                          ('validation_complete_fraction', 'Validación greedy')]:
        completion.plot([row['epoch'] for row in history], [row[metric] for row in history], label=label)
    completion.set(title='Trayectorias completas antes de búsqueda adicional', xlabel='Época',
                   ylabel='Fracción completa', ylim=(-.02, 1.02))
    ax.legend(); completion.legend()
    fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def train_crossfit(instances, spec, config, seeds, work_root, drive_root, bank_id,
                   bank_sha256, evaluator_sha256, device='auto'):
    require_colab()
    validate_training_config(config)
    if (not seeds or any(type(seed) is not int or seed < 0 for seed in seeds)
            or len(set(seeds)) != len(seeds)):
        raise ValueError('Semillas de entrenamiento vacías, repetidas o inválidas.')
    if evaluator_sha256 != p.file_hash(p.__file__):
        raise ValueError('Evaluador de entrenamiento distinto del módulo sellado activo.')
    device = choose_device(device)
    folds = make_geometry_folds(instances)
    identity = {'config': config, 'seeds': seeds, 'folds': folds,
                'bank_sha256': bank_sha256,
                'runtime_signature': _runtime_signature(device),
                'instance_sha256': sorted(x.sha for x in instances),
                'evaluator_sha256': evaluator_sha256, 'source_hashes': _source_hashes()}
    training_id = 'pomo04_' + p.digest(identity)[:20]
    root = Path(work_root) / 'rl_training' / bank_id / training_id
    remote = Path(drive_root) / 'rl_training' / bank_id / training_id
    print('TRAINING_ID =', training_id, '| dispositivo:', device, flush=True)
    if e._recover(root / 'meta', remote / 'meta'):
        if e.read_json(root / 'meta/identity.json') != identity:
            raise ValueError('Identidad de entrenamiento inconsistente.')
    else:
        e.write_json(root / 'meta/identity.json', identity)
        e.write_json(root / 'meta/folds.json', folds)
        for name in _source_hashes():
            target = root / 'meta/code' / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(Path(__file__).with_name(name).read_bytes())
        e._commit(root / 'meta', remote / 'meta')
    models = []
    for fold in folds:
        for seed in seeds:
            relative = Path(fold['fold_id']) / f'seed_{seed}'
            attempt_started = time.perf_counter()
            try:
                final = train_fold(instances, spec, config, seed, fold,
                                   root / relative, remote / relative, device)
            except Exception as exc:
                failure = root / 'failures' / (fold['fold_id'] + f'_seed_{seed}_' + uuid.uuid4().hex[:8])
                e.write_json(failure / 'failure.json', {'fold': fold, 'seed': seed, 'training_id': training_id,
                             'error_type': type(exc).__name__, 'error': str(exc),
                             'observed_call_wall_seconds_including_io': time.perf_counter() - attempt_started,
                             'completed_checkpoints_preserved': True})
                e.write_json(failure / 'environment.json', p.environment_report())
                e._commit(failure, remote / 'failures' / failure.name)
                raise
            models.append({**e.read_json(final / 'summary.json'), 'relative_directory': str(relative / 'final'),
                           'bundle_sha256': p.verify_bundle(final)})
    # Toda selección terminó antes de que el notebook invoque evaluate sobre heldout.
    if not e._recover(root / 'index', remote / 'index'):
        e.write_json(root / 'index/models.json', models)
        e.write_json(root / 'index/offline_costs.json', {'offline_cost_id': training_id,
                     'teacher_seconds_unique': 0.0,
                     'training_seconds_unique': sum(row['training_seconds'] for row in models),
                     'validation_seconds_included_in_training': sum(row['validation_seconds'] for row in models),
                     'models_trained': len(models),
                     'models_selected': sum(row['selection_status'] == 'selected_complete_validation' for row in models),
                     'validation_gate_failures': sum(row['selection_status'] == 'validation_gate_failed' for row in models),
                     'geometry_count': len(folds),
                     'summary_times_must_not_be_summed_across_policies_or_scenarios': True})
        e.write_csv(root / 'index/training_summary.csv', [{k: v for k, v in row.items()
                                                         if not isinstance(v, (dict, list))} for row in models])
        e._commit(root / 'index', remote / 'index')
    if e.read_json(root / 'index/models.json') != models:
        raise ValueError('Inventario de modelos distinto de los checkpoints.')
    return root


def make_solver(training_root, instances, spec, device='auto', expected_bank_sha256=None):
    """Carga sólo el modelo que dejó fuera la geometría de la instancia solicitada."""
    root, device = Path(training_root), choose_device(device)
    p.verify_bundle(root / 'meta')
    identity = e.read_json(root / 'meta/identity.json')
    if expected_bank_sha256 is not None and identity['bank_sha256'] != expected_bank_sha256:
        raise ValueError('Entrenamiento vinculado a otro banco.')
    validate_training_config(identity['config'])
    if (identity['evaluator_sha256'] != p.file_hash(p.__file__)
            or identity['source_hashes'] != _source_hashes()):
        raise ValueError('Código o evaluador distintos de los usados para entrenar.')
    if identity['instance_sha256'] != sorted(x.sha for x in instances):
        raise ValueError('Instancias distintas del registro de entrenamiento.')
    if identity['folds'] != make_geometry_folds(instances):
        raise ValueError('Folds distintos de los prefijados por geometría.')
    p.verify_bundle(root / 'index')
    index = e.read_json(root / 'index/models.json')
    registry, provenance = {}, {}
    expected_pairs = {(fold['fold_id'], seed) for fold in identity['folds'] for seed in identity['seeds']}
    seen_pairs = set()
    for model in index:
        relative = Path(model['relative_directory'])
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Ruta de checkpoint fuera del entrenamiento.')
        final = root / relative
        if p.verify_bundle(final) != model['bundle_sha256']:
            raise ValueError('Checkpoint de modelo modificado.')
        summary = e.read_json(final / 'summary.json')
        selected_checkpoint_path(final, summary)
        if model != {**summary, 'relative_directory': str(relative), 'bundle_sha256': model['bundle_sha256']}:
            raise ValueError('Inventario de modelos no coincide con el resumen sellado.')
        if model['traffic'] != asdict(spec):
            raise ValueError('Modelo entrenado con otro tráfico conocido.')
        fold = model['fold']
        validate_fold(fold, instances)
        pair = (fold['fold_id'], model['training_seed'])
        if pair not in expected_pairs or pair in seen_pairs or fold not in identity['folds']:
            raise ValueError('Fold o semilla fuera del plan de entrenamiento, o repetidos.')
        seen_pairs.add(pair)
        if (model['time_scale'] != identity['config']['time_scale'] or model['objective'] != OBJECTIVE
                or model['model_config'] != identity['config']['model']):
            raise ValueError('Modelo y configuración de entrenamiento incompatibles.')
        p.verify_bundle(final.parent / 'meta')
        fold_identity = e.read_json(final.parent / 'meta/identity.json')
        if (p.digest(fold_identity) != model['identity_sha256']
                or fold_identity['fold'] != fold or fold_identity['seed'] != model['training_seed']
                or fold_identity['config'] != identity['config']
                or fold_identity['source_hashes'] != identity['source_hashes']
                or fold_identity['runtime_signature'] != identity['runtime_signature']
                or fold_identity['traffic'] != asdict(spec)
                or fold_identity['evaluator_sha256'] != identity['evaluator_sha256']
                or fold_identity['train_instances'] != _group_instances(instances, fold['train_geometry_groups'])
                or fold_identity['validation_instances'] != _group_instances(instances, fold['validation_geometry_groups'])):
            raise ValueError('Identidad del modelo no corresponde a su fold y datos autorizados.')
        for x in instances:
            if x.provenance['geometry_group_sha256'] not in fold['heldout_geometry_groups']:
                continue
            key = f'{x.sha}:{model["training_seed"]}'
            if key in registry:
                raise ValueError('Dos modelos compiten por una observación heldout.')
            registry[key] = (final, model)
            provenance[key] = {key_: model[key_] for key_ in ['training_seed', 'training_seconds',
                'validation_seconds', 'teacher_seconds', 'model_checkpoint_sha256', 'checkpoint_selection', 'training_scope']}
            provenance[key].update(training_id=root.name, fold_id=fold['fold_id'],
                train_geometry_groups=fold['train_geometry_groups'],
                validation_geometry_groups=fold['validation_geometry_groups'],
                heldout_geometry_groups=fold['heldout_geometry_groups'], offline_cost_id=root.name,
                offline_manifest_sha256=p.verify_bundle(root / 'index'))
    if seen_pairs != expected_pairs:
        raise ValueError('Faltan modelos para completar el plan de entrenamiento.')

    def solver(problem, seed, config, output_dir):
        started = time.perf_counter()
        key = f'{problem.x.sha}:{seed}'
        if key not in registry:
            raise ValueError('No existe modelo fuera de geometría para esta instancia/semilla.')
        final, record = registry[key]
        if problem.time_scale != record['time_scale']:
            raise ValueError('Escala temporal de inferencia distinta del entrenamiento.')
        selected_checkpoint_path(final, record)
        if record['selection_status'] == 'validation_gate_failed':
            return problem.result(None, 'validation_gate_failed', time.perf_counter() - started, [],
                method='pomo', learned=False, validation_gate_failed=True, heldout_inference_executed=False,
                model_checkpoint_sha256=record['model_checkpoint_sha256'],
                checkpoint_artifact_kind='unselected_epoch_state_for_audit_only',
                no_solution_scope='no_checkpoint_met_prespecified_validation_completion_gate')
        model = AttentionPolicy(record['model_config']).to(device)
        model.load_state_dict(_load_checkpoint(final / 'best.pt'))
        model.eval(); synchronize(device)
        loading = time.perf_counter() - started
        remaining = config['seconds'] - loading
        try:
            if remaining <= 0:
                result = problem.result(None, 'time_limit_model_load', loading, [])
            else:
                result = solve_pomo(problem, seed, {**config, 'seconds': remaining}, model, device)
            result['metadata'].update(model_load_seconds=loading, inference_device=str(device),
                                      model_config=record['model_config'], mask_version=MASK_VERSION,
                                      initialization_seed=p.seed_for('reinforcement-04', record['fold']['fold_id'], seed),
                                      model_checkpoint_sha256=record['model_checkpoint_sha256'],
                                      fold_id=record['fold']['fold_id'])
            synchronize(device)
            result['solve_seconds'] = time.perf_counter() - started
            return result
        finally:
            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
    return solver, {'pomo': provenance}


def make_control_solvers(instances, config, seeds, device='auto'):
    """Explicit untrained-policy and uniform-mask comparators, with zero fitting.

    The untrained model uses precisely the initialization seed of its matching
    trained fold. Neither comparator opens a checkpoint or teacher corpus.
    All construction, initialization, search and validation count in the same
    inference deadline. Uniform-mask completion is a decoder diagnostic, not
    evidence that a network learned anything.
    """
    require_colab()
    validate_training_config(config)
    if not seeds or any(type(seed) is not int or seed < 0 for seed in seeds) or len(set(seeds)) != len(seeds):
        raise ValueError('Semillas de control inválidas.')
    device = choose_device(device)
    folds = make_geometry_folds(instances)
    registry = {x.sha: fold for fold in folds for x in instances
                if x.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups']}
    provenance = {}

    def build_solver(method):
        def solver(problem, seed, settings, output_dir):
            if problem.x.sha not in registry or seed not in seeds:
                raise ValueError('Instancia/semilla fuera del plan de controles.')
            if problem.time_scale != config['time_scale']:
                raise ValueError('Escala del control distinta del entrenamiento.')
            started = time.perf_counter()
            fold = registry[problem.x.sha]
            initialization_seed = p.seed_for('reinforcement-04', fold['fold_id'], seed)
            if method == 'random_masked':
                model = UniformMaskedPolicy(config['model']).to(device)
            else:
                cuda_devices = list(range(torch.cuda.device_count())) if torch.cuda.is_available() else []
                with torch.random.fork_rng(devices=cuda_devices):
                    torch.manual_seed(initialization_seed)
                    model = AttentionPolicy(config['model']).to(device)
            model.eval()
            synchronize(device)
            initialization_seconds = time.perf_counter() - started
            remaining = settings['seconds'] - initialization_seconds
            if remaining <= 0:
                result = problem.result(None, 'time_limit_model_load', initialization_seconds, [])
            else:
                result = solve_pomo(problem, seed, {**settings, 'seconds': remaining}, model, device)
            result['metadata'].update(method=method, control_type=method, inference_device=str(device), model_config=config['model'],
                                      initialization_seed=initialization_seed,
                                      initialization_seconds=initialization_seconds,
                                      training_seconds=0., optimizer_updates=0,
                                      training_scope='untrained_control_no_fitting',
                                      fold_id=fold['fold_id'],
                                      mask_version=MASK_VERSION,
                                      initialization_matches_training=(method == 'pomo_untrained'))
            synchronize(device)
            result['solve_seconds'] = time.perf_counter() - started
            del model
            return result
        return solver

    for method in ('pomo_untrained', 'random_masked'):
        provenance[method] = {f'{x.sha}:{seed}': {
            'fold_id': registry[x.sha]['fold_id'], 'training_seed': seed,
            'training_seconds': 0., 'validation_seconds': 0., 'teacher_seconds': 0.,
            'checkpoint_selection': 'none_untrained_control',
            'training_scope': 'untrained_control_no_fitting',
        } for x in instances for seed in seeds}
    return {method: build_solver(method) for method in provenance}, provenance


Overwriting svrplab/reinforcement.py


In [55]:
%%writefile svrplab/research_design.py
"""Protocolo final, auditoría y población independiente. Cómputo sólo en Colab.

La población nueva usa el generador oficial fijado y el evaluador sellado; no cambia
el banco de desarrollo. Una reserva, un certificado y un estudio completado son
estados distintos. Los runners actuales de desarrollo no abren el test nuevo.
"""
from __future__ import annotations

from collections import defaultdict
from copy import deepcopy
import hashlib
import json
import math
from pathlib import Path


def _hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     ensure_ascii=False, allow_nan=False).encode()).hexdigest()


def _valid_hash(value):
    return isinstance(value, str) and len(value) == 64 and all(c in '0123456789abcdef' for c in value)


def _validate_config(config):
    if config['schema_version'] not in ('research-design-v1', 'research-design-v2'):
        raise ValueError('Diseño desconocido.')
    quotas = config['geometry_quotas_per_size']
    if set(quotas) != {'train', 'validation', 'test'} or any(type(v) is not int or v < 1 for v in quotas.values()):
        raise ValueError('Cuotas por geometría inválidas.')
    sizes = config['sizes']
    if len(set(sizes)) != len(sizes) or any(type(n) is not int or n < 1 for n in sizes):
        raise ValueError('Tamaños inválidos/repetidos.')
    return quotas


def plan_geometry_split(records, excluded_groups, config):
    """Inventario por geometría; hashes distintos no prueban independencia de origen."""
    quotas = _validate_config(config)
    excluded = set(excluded_groups)
    if any(not _valid_hash(group) for group in excluded):
        raise ValueError('Exclusión sin hash de geometría válido.')
    groups, sizes = defaultdict(list), {}
    for row in records:
        group = row['geometry_group_sha256']
        if not _valid_hash(group):
            raise ValueError('Hash de geometría inválido.')
        n = row['n_customers']
        if type(n) is not int or n <= 0:
            raise ValueError('Tamaño inválido.')
        if group in sizes and sizes[group] != n:
            raise ValueError('Una geometría aparece con tamaños inconsistentes.')
        sizes[group] = n
        groups[group].append(row['source_id'])
    planned, shortages = [], []
    for n in config['sizes']:
        available = sorted((g for g in groups if sizes[g] == n and g not in excluded),
                           key=lambda g: (_hash([config['split_seed'], g]), g))
        offset = 0
        for role in ('train', 'validation', 'test'):
            selected = available[offset:offset + quotas[role]]
            if len(selected) < quotas[role]:
                shortages.append({'n_customers': n, 'role': role, 'required': quotas[role],
                                  'available': len(selected)})
            for group in selected:
                planned.append({'geometry_group_sha256': group, 'n_customers': n,
                                'role': role, 'source_ids': sorted(set(groups[group]))})
            offset += quotas[role]
    return {'schema_version': config['schema_version'], 'config': config,
            'assignment': planned, 'excluded_development_groups': sorted(excluded),
            'shortages': shortages, 'status': 'insufficient_population' if shortages else 'planned_not_validated',
            'ready_for_confirmatory': False, 'independence_requires_source_audit': True,
            'plan_sha256': _hash({'config': config, 'assignment': planned, 'excluded': sorted(excluded)})}


def scenario_precision_plan(config):
    """Hoeffding para FR de un plan fijo; cola esperada no garantiza precisión CVaR."""
    epsilon, error = config['fr_absolute_error'], config['fr_error_probability']
    alpha, tail = config['cvar_alpha'], config['minimum_expected_tail_observations']
    if not 0 < epsilon < 1 or not 0 < error < 1 or not 0 < alpha < 1 or type(tail) is not int or tail < 1:
        raise ValueError('Precisión/cola fuera del dominio.')
    fr_n = math.ceil(math.log(2. / error) / (2. * epsilon**2))
    tail_n = math.ceil(tail / (1. - alpha))
    return {'scenarios_per_plan': max(fr_n, tail_n), 'fr_hoeffding_count': fr_n,
            'cvar_tail_count_floor': tail_n,
            'fr_scope': 'pointwise_per_fixed_plan_iid_scenarios_not_simultaneous_all_methods',
            'cvar_precision_guaranteed': False,
            'scenario_count_is_not_number_of_independent_geometries': True,
            'note': 'Congelar M antes de abrir test. No detener al obtener significancia.'}


def geometry_precision_sensitivity(config):
    """Escenarios analíticos de planificación, no varianzas estimadas ni potencia."""
    quotas = _validate_config(config)
    n = quotas['test']
    strata = len(config['sizes'])
    return {'status': 'sensitivity_not_power_calculation', 'n_test_per_size': n,
            'n_total_test_geometries': n * strata,
            'rows': [{'assumed_between_geometry_paired_score_sd': sd,
                      'approx_normal_95_halfwidth_per_size': 1.96 * sd / math.sqrt(n),
                      'approx_normal_95_halfwidth_equal_weight_pooled': 1.96 * sd / math.sqrt(n * strata)}
                     for sd in config.get('planning_paired_score_sd_sensitivity', [.25, .5, 1.])],
            'assumptions': 'independent_geometries_equal_variance_fixed_checkpoints_normal_approximation',
            'limitations': 'Not simultaneous Holm intervals; no guarantee with sparse/discrete outcomes. '
                           'Training and validation quotas are computational planning targets.'}


def paired_win_score(left, right, config):
    """Éxito conservador antes de costo; jamás convierte un archivo ausente en fallo.

    Cada entrada declara artifact_status='observed', conservative_valid y, si es
    válida, conservative_cost. Fallos observados también requieren artefacto.
    """
    for row in (left, right):
        if row.get('artifact_status') != 'observed' or type(row.get('conservative_valid')) is not bool:
            raise ValueError('Falta resultado observado; no imputar fallo del método.')
        if row['conservative_valid'] and (not isinstance(row.get('conservative_cost'), (int, float))
                or not math.isfinite(row['conservative_cost']) or row['conservative_cost'] < 0):
            raise ValueError('Plan válido sin costo conservador finito no negativo.')
    a, b = left['conservative_valid'], right['conservative_valid']
    if a != b:
        return 1 if a else -1
    if not a:
        return 0
    rules = config['win_score']
    if math.isclose(left['conservative_cost'], right['conservative_cost'],
                    rel_tol=rules['cost_tie_relative'], abs_tol=rules['cost_tie_absolute_minutes']):
        return 0
    return 1 if left['conservative_cost'] < right['conservative_cost'] else -1


def traffic_density_audit():
    """Momentos analíticos de ambas fuentes; ×60 es diagnóstico, no calibración.

    Accidentes implementados: N~Poisson(lambda_por_consulta), D~U(30,120)
    independiente por accidente, Y=sum(D_1..D_N). E[Y]=75 lambda y
    Var[Y]=6300 lambda. No es una tasa integrada por duración recorrida.
    """
    def pdf(t, mu, sigma):
        return math.exp(-.5 * ((t - mu) / sigma)**2) / (sigma * math.sqrt(2. * math.pi))
    rows, components = [], []
    for t in (0, 360, 480, 780, 1020, 1260, 1440):
        minute = pdf(t, 480, 90) + pdf(t, 1020, 90)
        hour = pdf(t / 60., 8, 1.5) + pdf(t / 60., 17, 1.5)
        rows.append({'time_minutes': t, 'density_per_minute': minute,
                     'density_per_hour': hour, 'hour_to_minute_density_ratio': hour / minute,
                     'accident_lambda_implemented_per_arc_query': .05 * pdf(t, 1260, 120),
                     'accident_lambda_naive_hour_density_per_arc_query': .05 * pdf(t / 60., 21, 2)})
        for distance in (50., 250., 1000.):
            for label, factor in (('sealed_minute_model', 1.), ('naive_density_x60_NOT_calibrated', 60.)):
                peak = factor * minute
                base = .25 * (.5 + 2 * peak) * (1 - math.exp(-distance / 50))
                mu, sigma = .1 * peak, .3 + .2 * peak
                rate = factor * .05 * pdf(t, 1260, 120)
                mean_ln = base * math.exp(mu + sigma * sigma / 2)
                var_ln = base**2 * math.expm1(sigma * sigma) * math.exp(2 * mu + sigma * sigma)
                components.append({'time_minutes': t, 'distance_source_units': distance, 'interpretation': label,
                    'free_flow_minutes_velocity_1': distance, 'mean_lognormal_delay_minutes': mean_ln,
                    'sd_lognormal_delay_minutes': math.sqrt(var_ln),
                    'lambda_accidents_per_arc_query': rate, 'probability_accident_this_arc_query': -math.expm1(-rate),
                    'mean_accident_delay_minutes': 75 * rate, 'sd_accident_delay_minutes': math.sqrt(6300 * rate),
                    'accident_variance_share': (6300 * rate) / (var_ln + 6300 * rate),
                    'probability_any_accident_300_fixed_same_time_queries': -math.expm1(-300 * rate)})
    return {'status': 'audited_retain_sealed_model_no_empirical_calibration', 'rows': rows,
            'component_rows': components, 'sealed_evaluator_modified': False,
            'decision': 'retain_original_minute_implementation_for_final_primary_population',
            'reason': 'La densidad tiene unidades inversas al tiempo; amplitudes y tasa por intervalo también '
                      'necesitan unidades. Cambiar sólo sigma/t aumenta amplitudes y NO es una conversión física completa.',
            'accident_scope': 'Compound Poisson per arc query; not blocked arcs or rate integrated over travel duration.',
            'probability_scope': 'The 300-query column assumes fixed identical departure times and independent queries; '
                                 'it is not an observed route probability with endogenous departure times.',
            'paper': 'https://arxiv.org/html/2505.21887v1#S2.SS1'}


def population_ticket_plan(config, excluded_groups):
    """Reserva semillas/roles antes de generar geometrías o consultar solucionadores."""
    quotas = _validate_config(config)
    population = config['population']
    multiplier = population['candidate_multiplier']
    if population.get('certification_seconds_per_strategy') != [120, 600, 1800]:
        raise ValueError('La continuación final prefijada requiere 120/600/1800 s por estrategia.')
    if type(multiplier) is not int or multiplier < 1 or any(not _valid_hash(g) for g in excluded_groups):
        raise ValueError('Presupuesto/exclusiones inválidos.')
    if (population['map_size'] != [1000, 1000] or population['demand_range'] != [0, 100]
            or population['num_depots'] != 1 or population['dynamic_customers'] is not False):
        raise ValueError('La familia final implementada conserva convenciones oficiales de 00.')
    identity = _hash({'config': config, 'excluded': sorted(set(excluded_groups))})
    tickets, used_seeds = [], set()
    for n in config['sizes']:
        for role in ('train', 'validation', 'test'):
            for index in range(quotas[role] * multiplier):
                key = [population['master_seed'], n, role, index]
                seeds = {part + '_seed': int(_hash(key + [part])[:16], 16) for part in ('geometry', 'windows')}
                # El generador upstream usa numpy legacy: se controla su semilla efectiva de 32 bits.
                effective = seeds['geometry_seed'] % 2**32
                if effective in used_seeds:
                    raise ValueError('Colisión de semilla espacial efectiva; revisar plan antes de generar datos.')
                used_seeds.add(effective)
                tickets.append({'ticket_id': f'n{n:04d}_{role}_{index:04d}', 'n_customers': n,
                                'role': role, 'candidate_index': index, **seeds})
    payload = {'schema_version': 'population-reservation-v1', 'design_sha256': _hash(config),
               'population_id': 'final07_' + identity[:20], 'excluded_development_groups': sorted(set(excluded_groups)),
               'quotas': quotas, 'tickets': tickets, 'generated': False, 'ready_for_confirmatory': False}
    return dict(payload, reservation_sha256=_hash(payload))


def population_resource_bound(config, admission_config):
    """Cota administrativa extrema; no predice duración ni bytes de los archivos."""
    quotas = _validate_config(config)
    tickets = len(config['sizes']) * sum(quotas.values()) * config['population']['candidate_multiplier']
    candidates = admission_config['windows']['max_joint_candidates']
    budgets = config['population']['certification_seconds_per_strategy']
    seconds_per_candidate = 2 * sum(budgets)  # hasta dos estrategias en cada fase
    files_per_ticket = candidates * (4 * len(budgets) + 2) + 3 + 6
    return {'max_reserved_geometry_tickets': tickets,
            'max_window_candidates_per_ticket': candidates,
            'search_seconds_per_strategy_stages': budgets,
            'max_solver_search_seconds_per_window_candidate': seconds_per_candidate,
            'max_solver_search_hours_all_tickets': tickets * candidates * seconds_per_candidate / 3600,
            'max_phase_search_bundles': tickets * candidates * len(budgets),
            'max_fixed_artifact_files_excluding_report_snapshots_and_interruption_archives': tickets * files_per_ticket + 2,
            'storage_bytes_bounded': False,
            'interpretation': 'Extremely loose administrative bound, not an expected runtime. '
                              'Free-flow proofs, accepted quotas and early certificates stop work sooner. '
                              'Generation, solver setup, IO, verification and reports add overhead; '
                              'preserve archive/checkpoint storage and inspect Drive capacity.'}


def _require_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Generación/admisión exclusivamente en Google Colab; no ejecutar localmente.') from exc


def generate_official_parent(ticket, upstream_root, config):
    """Geometría nueva por ticket; nunca recorta otro mapa ni reutiliza catálogo.

    Preserva la regla de flota calculada sobre las ventanas originales, antes de
    cualquier condicionamiento. La familia es derivada del generador, no las filas HF.
    """
    _require_colab()
    import random
    import numpy as np
    from threadpoolctl import threadpool_limits
    from . import pilot as p
    common, tw = p.upstream_modules(upstream_root)
    population, n = config['population'], ticket['n_customers']
    with p.legacy_seed(ticket['geometry_seed']), threadpool_limits(limits=1):
        base = common.generate_base_instance(n, tuple(population['map_size']), max(1, n // 50),
                                             1, tuple(population['demand_range']), False)
    appears = base['appear_time'].tolist()
    with p.legacy_seed(ticket['windows_seed']):
        windows = [tw.sample_time_window(random.randint(0, 1), a) for a in appears]
    windows[0] = (0, 1440)
    windows = np.asarray(windows, dtype=np.uint16)
    fleet = max(1, int(np.histogram(windows.ravel(), bins=24, range=(0, 1440))[0].max()))
    demands = base['demands'].tolist()
    payload = {'instance_id': 'final_' + ticket['ticket_id'], 'depot_indices': [0],
               'locations': base['locations'].tolist(), 'demands': demands,
               'capacities': [float(sum(demands))] * fleet, 'windows': windows.tolist(), 'appear_times': appears,
               'provenance': {'kind': 'derived_official_base_generator', 'upstream_commit': p.UPSTREAM_SHA,
                   'generator': 'common.generate_base_instance', 'num_cities': max(1, n // 50),
                   'geometry_seed': ticket['geometry_seed'], 'windows_seed': ticket['windows_seed'],
                   'population_family': population['family'], 'ticket': ticket,
                   'geometry_group_sha256': p.digest({'locations': base['locations'].astype(float).tolist(), 'depot_indices': [0]}),
                   'windows': 'derived_NOT_original_HF_windows', 'time_unit': 'minute',
                   'coordinate_unit': 'source_map_unit', 'service_minutes': 0,
                   'customer_ids': '1..n_including_zero_demands', 'depot_ids': [0], 'capacity_modified': False}}
    return p.freeze_instance(payload)


def _sync_phase_bundle(source, destination):
    """Recupera copia .partial íntegra; conserva toda copia incompleta como archivo."""
    from . import pilot as p
    import uuid
    source, destination = Path(source), Path(destination)
    partial = destination.with_name(destination.name + '.partial')
    if not destination.exists() and partial.exists():
        try:
            matches = p.verify_bundle(partial) == p.verify_bundle(source)
        except (ValueError, OSError, json.JSONDecodeError):
            matches = False
        if matches:
            partial.rename(destination)
        else:
            partial.rename(partial.with_name(partial.name + '.interrupted_' + uuid.uuid4().hex[:8]))
    return p.sync_verified(source, destination)


def _recover_phase_bundle(local, remote):
    """Sólo una fase sellada se reutiliza; conservar escritura local interrumpida."""
    from . import pilot as p
    import uuid
    local, remote = Path(local), Path(remote)
    if (local / 'CHECKSUMS.json').exists():
        p.verify_bundle(local)
        _sync_phase_bundle(local, remote)
        return True
    if local.exists():
        local.rename(local.with_name(local.name + '.interrupted_' + uuid.uuid4().hex[:8]))
    if remote.exists():
        _sync_phase_bundle(remote, local)
        return True
    return False


def _ticket_admission(parent, upstream_root, admission_config, attempts_dir, remote_dir, search_budgets=None):
    """Secuencia de ventanas de 00 con continuación de búsqueda del MISMO candidato.

    Las fases 120/600/1800 son presupuestos por estrategia CP-SAT, no candidatos
    nuevos. Sólo UNKNOWN permite otra fase; error de modelo/infraestructura bloquea.
    El record final conserva la forma que verifica validate_admission_history y
    añade evidencia de todas las fases en certification_searches.
    """
    from . import pilot as p
    spec = p.Traffic(**admission_config['traffic'])
    scale = admission_config['certification']['time_scale']
    limit = admission_config['windows']['max_joint_candidates']
    budgets = list(search_budgets or [admission_config['certification']['seconds_per_strategy']])
    if (not budgets or budgets[0] != admission_config['certification']['seconds_per_strategy']
            or any(type(v) is not int or v < 1 for v in budgets)
            or budgets != sorted(set(budgets))):
        raise ValueError('Fases de certificación inválidas o primera fase distinta de 00.')
    history = {'policy': p.JOINT_WINDOW_POLICY, 'max_joint_candidates': limit, 'attempts': []}
    for index in range(limit):
        candidate, ledger = p.make_conditioned_candidate(parent, upstream_root, spec,
            admission_config['windows']['max_draws_per_customer'], scale, joint_attempt=index)
        checkpoint = attempts_dir / f'attempt{index:03d}'
        remote = remote_dir / checkpoint.name
        searches = []
        certificate = report = None
        for stage, seconds in enumerate(budgets):
            phase_name = f'search{stage:02d}_{seconds:05d}s'
            phase, phase_remote = checkpoint / phase_name, remote / phase_name
            phase_config = {'admission_config': admission_config, 'search_stage': stage,
                            'seconds_per_strategy': seconds, 'prefixed_search_budgets': budgets}
            if _recover_phase_bundle(phase, phase_remote):
                p.verify_bundle(phase)
                search = json.loads((phase / 'record.json').read_text())
                if (search['candidate'] != candidate.payload() or search['ledger'] != ledger
                        or search['joint_attempt'] != index or search['search_stage'] != stage
                        or search['seconds_per_strategy'] != seconds
                        or json.loads((phase / 'config.json').read_text()) != phase_config):
                    raise ValueError('Fase recuperada distinta del mismo candidato/presupuesto prefijado.')
                certificate, report = search['certificate'], search['certification']
            else:
                certificate, report = p.certify_candidate(candidate, spec, seconds, scale, progress=True)
                search = {'joint_attempt': index, 'candidate': candidate.payload(), 'ledger': ledger,
                          'search_stage': stage, 'seconds_per_strategy': seconds,
                          'certificate': certificate, 'certification': report}
                p.write_json(phase / 'record.json', search)
                p.write_json(phase / 'config.json', phase_config)
                p.write_json(phase / 'environment.json', p.environment_report())
                p.seal_bundle(phase)
            _sync_phase_bundle(phase, phase_remote)
            searches.append({'search_stage': stage, 'seconds_per_strategy': seconds,
                             'phase_bundle_sha256': p.verify_bundle(phase),
                             'certificate': certificate, 'certification': report})
            if certificate is not None or p.proven_rejection(candidate, report, spec, scale):
                break
            if report.get('status') != 'unknown_no_certificate':
                raise RuntimeError(f'{parent.instance_id}: modelo/estado no válido ({report.get("status")}); '
                                   f'conservar {phase_remote}. No continuar ni sustituir el candidato.')
            if stage == len(budgets) - 1:
                raise RuntimeError(f'{parent.instance_id}: UNKNOWN tras fases {budgets}; conservar {phase_remote}. '
                                   'No sustituir geometría ni saltar este intento.')
            print(f'UNKNOWN: continuar MISMO candidato, fase {stage + 2}/{len(budgets)}; '
                  f'{budgets[stage + 1]} s por estrategia.', flush=True)
        record = {'joint_attempt': index, 'candidate': candidate.payload(), 'ledger': ledger,
                  'certificate': certificate, 'certification': report, 'certification_searches': searches}
        decision, decision_remote = checkpoint / 'decision', remote / 'decision'
        if _recover_phase_bundle(decision, decision_remote):
            p.verify_bundle(decision)
            if json.loads((decision / 'record.json').read_text()) != record:
                raise ValueError('Decisión recuperada distinta de las fases selladas.')
        else:
            p.write_json(decision / 'record.json', record)
            p.seal_bundle(decision)
        _sync_phase_bundle(decision, decision_remote)
        history['attempts'].append(record)
        if certificate is not None:
            p.validate_admission_history(candidate, history, spec, scale)
            p.verify_certificate(candidate, certificate, spec)
            return 'certified_nominal', candidate, ledger, certificate, history
    return 'window_sequence_exhausted_proven_candidates', None, None, None, history


def build_population(config, reservation, admission_config, upstream_root, work_root, drive_root):
    """Construye/acepta tickets en Colab; conserva test sin evaluar por métodos.

    La construcción no entrena ni ejecuta comparadores. La inspección de metadatos
    de aceptación no es exposición de resultados de test. Un fallo interrumpe la
    secuencia y el informe apunta al mismo ticket para reanudarlo.
    """
    _require_colab()
    from . import pilot as p
    if reservation != population_ticket_plan(config, reservation['excluded_development_groups']):
        raise ValueError('Reserva modificada o configuración distinta.')
    if (admission_config['traffic'] != {'velocity': 1.0, 'noise_scale': 1.0, 'accident_scale': 1.0,
                                       'bucket_minutes': 1.0, 'mode': 'stochastic'}
            or admission_config['protocol_version'] != p.PROTOCOL_VERSION
            or admission_config['windows']['policy'] != p.WINDOW_POLICY
            or admission_config['windows']['joint_policy'] != p.JOINT_WINDOW_POLICY):
        raise ValueError('La población final conserva tráfico y admisión de 00.')
    local = Path(work_root) / reservation['population_id']
    remote = Path(drive_root) / reservation['population_id']
    p.fetch_upstream(upstream_root)
    source_hashes = {name: p.file_hash(Path(upstream_root) / 'vrp_bench' / name) for name in p.SOURCE_FILES}
    plan_dir = local / 'reservation'
    plan_recovered = _recover_phase_bundle(plan_dir, remote / 'reservation')
    expected_plan = {'reservation': reservation, 'admission_config': admission_config,
                     'source_hashes': source_hashes, 'evaluator_sha256': p.file_hash(p.__file__),
                     'research_design_sha256': p.file_hash(__file__)}
    if plan_recovered:
        p.verify_bundle(plan_dir)
        if json.loads((plan_dir / 'plan.json').read_text()) != expected_plan:
            raise ValueError('No cambiar código/configuración de una población empezada. Conservar e investigar.')
    else:
        p.write_json(plan_dir / 'plan.json', expected_plan)
        p.seal_bundle(plan_dir)
    _sync_phase_bundle(plan_dir, remote / 'reservation')
    counts, seen, records = defaultdict(int), set(reservation['excluded_development_groups']), []
    blocked = None
    for ticket in reservation['tickets']:
        key = (ticket['n_customers'], ticket['role'])
        if counts[key] >= reservation['quotas'][ticket['role']]:
            continue
        ticket_dir = local / 'tickets' / ticket['ticket_id']
        ticket_remote = remote / 'tickets' / ticket['ticket_id']
        try:
            parent = generate_official_parent(ticket, upstream_root, config)
            parent_dir = local / 'parents' / ticket['ticket_id']
            parent_remote = remote / 'parents' / ticket['ticket_id']
            if _recover_phase_bundle(parent_dir, parent_remote):
                p.verify_bundle(parent_dir)
                if json.loads((parent_dir / 'parent.json').read_text()) != parent.payload():
                    raise ValueError('Fuente regenerada diferente del checkpoint; no sustituir datos.')
            else:
                p.write_json(parent_dir / 'parent.json', parent.payload())
                p.write_json(parent_dir / 'environment.json', p.environment_report())
                p.seal_bundle(parent_dir)
            _sync_phase_bundle(parent_dir, parent_remote)
            group = parent.provenance['geometry_group_sha256']
            if _recover_phase_bundle(ticket_dir, ticket_remote):
                p.verify_bundle(ticket_dir)
                record = json.loads((ticket_dir / 'record.json').read_text())
                if record['parent_sha256'] != parent.sha or record['ticket'] != ticket:
                    raise ValueError('Ticket almacenado distinto de su reserva/fuente.')
                if record['status'] == 'certified_nominal':
                    x = p.freeze_instance(json.loads((ticket_dir / 'instance.json').read_text()))
                    history = json.loads((ticket_dir / 'history.json').read_text())
                    p.validate_admission_history(x, history, p.Traffic(**admission_config['traffic']),
                                                admission_config['certification']['time_scale'])
            else:
                record = {'ticket': ticket, 'geometry_group_sha256': group, 'parent_sha256': parent.sha}
                if group in seen:
                    record['status'] = 'duplicate_or_excluded_geometry'
                else:
                    # Sólo esta cota física demuestra imposibilidad para toda ventana/historia.
                    distances = p.distances(parent)
                    margins = (2 * distances[0] / admission_config['traffic']['velocity'] - 1440).tolist()
                    violations = [{'customer': i, 'excess_minutes': float(m)} for i, m in enumerate(margins) if i > 0 and m > 1e-8]
                    if violations:
                        record.update(status='proven_geometry_horizon_infeasible', violations=violations)
                    else:
                        status, x, ledger, certificate, history = _ticket_admission(parent, upstream_root,
                            admission_config, local / 'attempts' / ticket['ticket_id'],
                            remote / 'attempts' / ticket['ticket_id'],
                            config['population']['certification_seconds_per_strategy'])
                        record['status'] = status
                        p.write_json(ticket_dir / 'history.json', history)
                        if x is not None:
                            p.write_json(ticket_dir / 'instance.json', x.payload())
                            p.write_json(ticket_dir / 'ledger.json', ledger)
                            p.write_json(ticket_dir / 'certificate.json', certificate)
                            record['instance_sha256'] = x.sha
                p.write_json(ticket_dir / 'record.json', record)
                p.seal_bundle(ticket_dir)
            _sync_phase_bundle(ticket_dir, ticket_remote)
            if record['status'] == 'certified_nominal':
                if group in seen:
                    raise ValueError('Una geometría aceptada colisiona con otro rol/desarrollo; test bloqueado.')
                counts[key] += 1
            seen.add(group)
            records.append(dict(record, bundle_sha256=p.verify_bundle(ticket_dir)))
            print(ticket['ticket_id'], record['status'], f'aceptadas={counts[key]}/{reservation["quotas"][ticket["role"]]}', flush=True)
        except Exception as exc:
            blocked = {'ticket': ticket, 'error_type': type(exc).__name__, 'message': str(exc),
                       'action': 'preserve_same_ticket_no_substitution'}
            break
    shortages = [{'n_customers': n, 'role': role, 'required': quota, 'accepted': counts[(n, role)]}
                 for n in config['sizes'] for role, quota in reservation['quotas'].items() if counts[(n, role)] < quota]
    report = {'schema_version': 'independent-population-v1', 'population_id': reservation['population_id'],
              'reservation_sha256': reservation['reservation_sha256'], 'design_sha256': _hash(config),
              'status': 'blocked' if blocked else ('insufficient_population' if shortages else 'nominal_population_accepted'),
              'records': records, 'shortages': shortages, 'blocked': blocked,
              'all_required_nominal_certificates': not shortages and blocked is None,
              'stochastic_feasibility_validated': False, 'ready_for_confirmatory': False,
              'remaining_gate': 'validated_holdout_runner_and_frozen_methods_models_scenarios_analysis',
              'estimand': 'official_generator_conditioned_on_prefixed_finite_window_admission_protocol',
              'not_the_unconditional_official_catalogue': True}
    report['population_sha256'] = _hash(report)
    report_dir = local / 'reports' / report['population_sha256']
    p.write_json(report_dir / 'population.json', report)
    p.seal_bundle(report_dir)
    _sync_phase_bundle(report_dir, remote / 'reports' / report['population_sha256'])
    return report


def final_freeze_checklist(config, population=None):
    """Requisitos concretos; un plan/acta nominal nunca autoriza ejecutar test."""
    return {'ready_for_confirmatory': False,
            'population_nominally_complete': bool(population and population.get('all_required_nominal_certificates')),
            'required_frozen_artifacts': ['population_manifest_sha256', 'geometry_role_manifest_sha256',
                'design_config_sha256', 'all_method_configs_sha256', 'all_method_source_sha256',
                'sealed_evaluator_sha256', 'search_module_sha256', 'checkpoint_hashes_by_method_and_seed',
                'validation_selection_records_sha256', 'scenario_seed_manifest_sha256',
                'R_analysis_source_sha256', 'environment_and_hardware_manifest_sha256',
                'holdout_adapter_validation_in_colab_sha256'],
            'required_checks': ['no_development_geometry_in_test', 'no_test_geometry_in_training_or_selection',
                'all_planned_methods_and_failures_registered', 'primary_ablations_share_EDD_LS_and_budget',
                'all_three_seed_ids_preserved', 'no_certificate_available_to_solver',
                'test_not_opened_by_current_leave_one_geometry_out_runners',
                'R_unit_is_geometry_not_seed_or_scenario', 'complete_results_inventory_before_analysis'],
            'current_blocker': config.get('confirmatory_runner_status', 'holdout_adapter_not_yet_validated'),
            'endpoint_is_fixed_before_test': config['primary_endpoint'],
            'test_opening_policy': 'A validated separate holdout runner must verify all artifact hashes before opening test.'}


def load_population(population_root, population_report_sha256, design):
    """Carga final estricta en Colab; devuelve instancias/registro, nunca certificados.

    El runner debe separar roles y congelar modelos antes de usar test. Esta carga
    valida población nominal y su historial; no habilita por sí misma inferencia.
    """
    _require_colab()
    from . import pilot as p
    import numpy as np
    if not _valid_hash(population_report_sha256):
        raise ValueError('Se requiere el hash completo del acta final de población.')
    root = Path(population_root)
    report_dir = root / 'reports' / population_report_sha256
    p.verify_bundle(report_dir)
    report = json.loads((report_dir / 'population.json').read_text())
    report_payload = {key: value for key, value in report.items() if key != 'population_sha256'}
    if (report.get('population_sha256') != population_report_sha256 or _hash(report_payload) != population_report_sha256
            or report.get('status') != 'nominal_population_accepted' or report.get('blocked') is not None
            or report.get('shortages') or report.get('all_required_nominal_certificates') is not True):
        raise ValueError('Acta incompleta/modificada; no autoriza población final.')
    p.verify_bundle(root / 'reservation')
    plan = json.loads((root / 'reservation' / 'plan.json').read_text())
    reservation = plan['reservation']
    if (reservation != population_ticket_plan(design, reservation['excluded_development_groups'])
            or reservation['reservation_sha256'] != report['reservation_sha256']
            or report['design_sha256'] != _hash(design) or report['population_id'] != root.name
            or reservation['population_id'] != root.name):
        raise ValueError('Reserva, rol, configuración o ruta diferente del acta.')
    if plan['evaluator_sha256'] != p.file_hash(p.__file__) or plan['source_hashes'] != p.SOURCE_HASHES:
        raise ValueError('Evaluador/generador distinto del usado en la admisión.')
    admission_config = plan['admission_config']
    if (admission_config['protocol_version'] != p.PROTOCOL_VERSION
            or admission_config['windows']['policy'] != p.WINDOW_POLICY
            or admission_config['windows']['joint_policy'] != p.JOINT_WINDOW_POLICY
            or admission_config['traffic'] != {'velocity': 1.0, 'noise_scale': 1.0, 'accident_scale': 1.0,
                                              'bucket_minutes': 1.0, 'mode': 'stochastic'}):
        raise ValueError('Tráfico/admisión de otra familia.')
    spec = p.Traffic(**admission_config['traffic'])
    scale = admission_config['certification']['time_scale']
    limit = admission_config['windows']['max_joint_candidates']
    counts, seen, registry, instances = defaultdict(int), set(reservation['excluded_development_groups']), [], []
    records = iter(report['records'])

    def validate_derivative(x, parent):
        if x.provenance['transformation']['parent_instance_sha256'] != parent.sha:
            raise ValueError('Instancia derivada de otro padre.')
        for field in ('locations', 'demands', 'capacities', 'appear_times'):
            if not np.array_equal(getattr(x, field), getattr(parent, field)):
                raise ValueError('Derivado alteró geometría, flota, demanda o apariciones.')
        if (not np.array_equal(x.windows[:, 1] - x.windows[:, 0], parent.windows[:, 1] - parent.windows[:, 0])
                or not np.array_equal(x.windows[0], parent.windows[0])):
            raise ValueError('Derivado alteró duración de ventanas o depósito.')

    for ticket in reservation['tickets']:
        key = (ticket['n_customers'], ticket['role'])
        if counts[key] >= reservation['quotas'][ticket['role']]:
            continue
        record = next(records, None)
        if record is None or record['ticket'] != ticket:
            raise ValueError('Ticket omitido/reordenado antes de llenar una cuota.')
        directory = root / 'tickets' / ticket['ticket_id']
        if p.verify_bundle(directory) != record['bundle_sha256']:
            raise ValueError('Ticket modificado después del acta.')
        stored = json.loads((directory / 'record.json').read_text())
        if stored != {k: v for k, v in record.items() if k != 'bundle_sha256'}:
            raise ValueError('Acta y registro de ticket discrepan.')
        parent_dir = root / 'parents' / ticket['ticket_id']
        p.verify_bundle(parent_dir)
        parent = p.freeze_instance(json.loads((parent_dir / 'parent.json').read_text()))
        group = p.digest({'locations': parent.locations.tolist(), 'depot_indices': [0]})
        if (parent.sha != record['parent_sha256'] or parent.n != ticket['n_customers']
                or parent.provenance.get('ticket') != ticket or parent.provenance.get('upstream_commit') != p.UPSTREAM_SHA
                or parent.provenance.get('geometry_seed') != ticket['geometry_seed']
                or parent.provenance.get('windows_seed') != ticket['windows_seed']
                or parent.provenance.get('geometry_group_sha256') != group
                or group != record['geometry_group_sha256']):
            raise ValueError('Fuente, semilla o geometría distinta de reserva/acta.')
        status = record['status']
        if status == 'duplicate_or_excluded_geometry':
            if group not in seen:
                raise ValueError('Exclusión por duplicado sin geometría anterior.')
        else:
            if group in seen:
                raise ValueError('Geometría de desarrollo/otro ticket reaparece como independiente.')
            if status == 'proven_geometry_horizon_infeasible':
                margins = (2 * p.distances(parent)[0] / spec.velocity - 1440).tolist()
                expected = [{'customer': i, 'excess_minutes': float(m)} for i, m in enumerate(margins)
                            if i > 0 and m > 1e-8]
                if not expected or expected != record['violations']:
                    raise ValueError('Rechazo espacial sin prueba física válida.')
            elif status in ('certified_nominal', 'window_sequence_exhausted_proven_candidates'):
                history = json.loads((directory / 'history.json').read_text())
                if history.get('policy') != p.JOINT_WINDOW_POLICY or history.get('max_joint_candidates') != limit:
                    raise ValueError('Política/presupuesto de ventanas modificado.')
                budgets = design['population']['certification_seconds_per_strategy']
                for attempt_index, attempt in enumerate(history['attempts']):
                    searches = attempt.get('certification_searches', [])
                    if not searches or len(searches) > len(budgets):
                        raise ValueError('Fases de certificación ausentes o fuera del presupuesto.')
                    attempt_root = root / 'attempts' / ticket['ticket_id'] / f'attempt{attempt_index:03d}'
                    for stage, search in enumerate(searches):
                        seconds = budgets[stage]
                        phase = attempt_root / f'search{stage:02d}_{seconds:05d}s'
                        if (search['search_stage'] != stage or search['seconds_per_strategy'] != seconds
                                or p.verify_bundle(phase) != search['phase_bundle_sha256']):
                            raise ValueError('Se omitió/cambió una fase del mismo candidato.')
                        expected_phase_config = {'admission_config': admission_config, 'search_stage': stage,
                            'seconds_per_strategy': seconds, 'prefixed_search_budgets': budgets}
                        phase_record = json.loads((phase / 'record.json').read_text())
                        if (json.loads((phase / 'config.json').read_text()) != expected_phase_config
                                or phase_record != {'joint_attempt': attempt_index, 'candidate': attempt['candidate'],
                                    'ledger': attempt['ledger'], 'search_stage': stage, 'seconds_per_strategy': seconds,
                                    'certificate': search['certificate'], 'certification': search['certification']}):
                            raise ValueError('La continuación cambió candidato, fuente, ventana o configuración.')
                        if stage < len(searches) - 1 and (search['certificate'] is not None
                                or search['certification'].get('status') != 'unknown_no_certificate'):
                            raise ValueError('Sólo UNKNOWN permite continuar; se omitió un resultado decisivo/error.')
                    if (searches[-1]['certificate'] != attempt['certificate']
                            or searches[-1]['certification'] != attempt['certification']):
                        raise ValueError('Decisión de candidato distinta de su última fase de búsqueda.')
                    p.verify_bundle(attempt_root / 'decision')
                    if json.loads((attempt_root / 'decision' / 'record.json').read_text()) != attempt:
                        raise ValueError('Historial distinto de la decisión sellada del candidato.')
                if status == 'window_sequence_exhausted_proven_candidates':
                    if len(history['attempts']) != limit:
                        raise ValueError('Secuencia no agotada: faltan intentos.')
                    for index, attempt in enumerate(history['attempts']):
                        candidate = p.freeze_instance(attempt['candidate'])
                        validate_derivative(candidate, parent)
                        if (attempt['joint_attempt'] != index
                                or candidate.provenance['transformation'].get('joint_attempt', 0) != index
                                or attempt['certificate'] is not None
                                or not p.proven_rejection(candidate, attempt['certification'], spec, scale)):
                            raise ValueError('Se excluyó un ticket sin pruebas completas en todos los intentos.')
                        p.validate_sampling_ledger(candidate, attempt['ledger'])
                else:
                    x = p.freeze_instance(json.loads((directory / 'instance.json').read_text()))
                    validate_derivative(x, parent)
                    if x.sha != record['instance_sha256'] or x.provenance['geometry_group_sha256'] != group:
                        raise ValueError('Instancia aceptada diferente del registro.')
                    ledger = json.loads((directory / 'ledger.json').read_text())
                    certificate = json.loads((directory / 'certificate.json').read_text())
                    if (history['attempts'][-1]['certificate'] != certificate
                            or history['attempts'][-1]['ledger'] != ledger):
                        raise ValueError('Certificado/ledger distintos del intento aceptado.')
                    p.validate_sampling_ledger(x, ledger)
                    p.validate_admission_history(x, history, spec, scale)
                    p.verify_certificate(x, certificate, spec)
                    counts[key] += 1
                    instances.append(x)
                    registry.append({'instance_sha256': x.sha, 'instance_id': x.instance_id,
                                     'geometry_group_sha256': group, 'n_customers': x.n,
                                     'role': ticket['role'], 'ticket_id': ticket['ticket_id']})
            else:
                raise ValueError('Estado no admitido en población final: ' + str(status))
        seen.add(group)
    if next(records, None) is not None:
        raise ValueError('Registros extra después de llenar cuotas.')
    for n in design['sizes']:
        for role, quota in reservation['quotas'].items():
            if counts[(n, role)] != quota:
                raise ValueError('La población no llena todas las cuotas por tamaño/rol.')
    manifest = {'population_id': report['population_id'], 'population_sha256': population_report_sha256,
                'population_report_sha256': population_report_sha256,
                'reservation_sha256': reservation['reservation_sha256'], 'design_sha256': _hash(design),
                'traffic': admission_config['traffic'], 'evaluator_sha256': plan['evaluator_sha256'],
                'admission_config': admission_config, 'source_hashes': plan['source_hashes'],
                'population_builder_sha256': plan['research_design_sha256'],
                'excluded_development_groups': reservation['excluded_development_groups'],
                'registry_sha256': _hash(registry), 'report': report, 'reservation': reservation,
                'all_required_nominal_certificates': True,
                'stochastic_feasibility_validated': False, 'ready_for_confirmatory': False}
    return instances, registry, manifest


Overwriting svrplab/research_design.py


In [56]:
%%writefile svrplab/search.py
"""Contrato de búsqueda determinista; evaluación estocástica en pilot, sin certificados.

El objetivo es el viaje con una cota uniforme de medias por arco. No representa
el costo óptimo del SVRP ni el costo completo de recurso todavía pendiente.
"""
from __future__ import annotations

import math
import numpy as np

from . import pilot as p

OBJECTIVE = 'conservative_upper_travel_minutes_v1'


class SearchProblem:
    def __init__(self, x, spec, time_scale=1000):
        self.x = x
        self.n = x.n
        self.vehicles = len(x.capacities)
        self.capacities = x.capacities
        self.demands = x.demands
        self.open = np.maximum(x.windows[:, 0], x.appear_times)
        self.close = x.windows[:, 1]
        self.horizon = float(x.windows[0, 1])
        self.distances = p.distances(x)
        self.travel = p.nominal_upper_matrix(x, spec, time_scale) / time_scale
        self.time_scale = time_scale

    def route_stats(self, route, vehicle):
        clock = travel = late = 0.0
        node = 0
        starts = []
        for customer in route:
            leg = float(self.travel[node, customer])
            travel += leg
            clock = max(clock + leg, float(self.open[customer]))
            late += max(0.0, clock - float(self.close[customer]))
            starts.append(clock)
            node = customer
        if route:
            leg = float(self.travel[node, 0])
            clock += leg
            travel += leg
        return {'travel_minutes': travel, 'lateness_minutes': late,
                'depot_overtime_minutes': max(0.0, clock - self.horizon),
                'capacity_excess': max(0.0, float(sum(self.demands[j] for j in route)
                                                - self.capacities[vehicle])),
                'service_starts': starts, 'return_minutes': clock}

    def score(self, routes):
        p.validate_routes(self.x, routes)
        stats = [self.route_stats(route, v) for v, route in enumerate(routes)]
        return (sum(r['lateness_minutes'] + r['depot_overtime_minutes'] for r in stats),
                sum(r['travel_minutes'] for r in stats))

    def result(self, routes, status, elapsed, history, **metadata):
        if not math.isfinite(elapsed) or elapsed < 0:
            raise ValueError('Duración de búsqueda inválida.')
        if routes is not None:
            routes = [[int(j) for j in route] for route in routes]
            violation, cost = self.score(routes)
            metadata.update(search_violation_minutes=violation,
                            search_objective_minutes=cost,
                            conservative_feasible=violation <= 1e-9)
        metadata['objective_scope'] = OBJECTIVE
        return {'routes': routes, 'status': status, 'solve_seconds': float(elapsed),
                'history': history, 'metadata': metadata}


Overwriting svrplab/search.py


In [57]:
%%writefile svrplab/supervised.py
"""03: imitación supervisada por geometría, checkpoints y adaptador del runner.

Sólo Colab. Ningún escenario evaluation interviene en etiquetas, gradientes o
selección de checkpoint. El piloto no es una prueba final de generalización.
"""
from __future__ import annotations

import math
import random
import time
import uuid
from dataclasses import asdict
from pathlib import Path

import numpy as np
import torch

from . import experiments as e
from . import pilot as p
from .attention import AttentionPolicy, teacher_loss, solve_attention
from .search import SearchProblem
from .supervised_data import make_geometry_folds, validate_fold, load_fold_examples


def require_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Entrenamiento y experimentación exclusivamente en Google Colab.') from exc


def validate_training_config(config):
    if (config['schema_version'] != 'supervised-03-v1' or config['role'] != 'development_only'
            or config['split_strategy'] != 'leave_one_geometry_out_with_inner_validation'):
        raise ValueError('Configuración supervisada incompatible.')
    settings = config['training']
    for name in ['epochs', 'steps_per_epoch', 'gradient_accumulation', 'patience']:
        if type(settings[name]) is not int or settings[name] < 1:
            raise ValueError(f'{name} debe ser entero positivo.')
    for name in ['learning_rate', 'gradient_clip_norm', 'seconds_per_model']:
        if isinstance(settings[name], bool) or not math.isfinite(settings[name]) or settings[name] <= 0:
            raise ValueError(f'{name} debe ser finito y positivo.')
    for name in ['weight_decay', 'min_delta']:
        if isinstance(settings[name], bool) or not math.isfinite(settings[name]) or settings[name] < 0:
            raise ValueError(f'{name} debe ser finito y no negativo.')
    for name in ['permute_nodes', 'deterministic_algorithms']:
        if type(settings[name]) is not bool:
            raise ValueError(f'{name} debe ser booleano.')
    return True


def synchronize(device):
    if torch.device(device).type == 'cuda':
        torch.cuda.synchronize(device)


def choose_device(requested='auto'):
    if requested == 'auto':
        requested = 'cuda' if torch.cuda.is_available() else 'cpu'
    device = torch.device(requested)
    if device.type not in ('cpu', 'cuda') or (device.type == 'cuda' and not torch.cuda.is_available()):
        raise ValueError('Dispositivo no disponible: usar cpu, cuda o auto.')
    return device


def _source_hashes():
    directory = Path(__file__).parent
    return {name: p.file_hash(directory / name) for name in
            ('supervised.py', 'supervised_data.py', 'attention.py', 'search.py', 'metaheuristics.py', 'baselines.py')}


def _group_examples(examples, allowed):
    groups = {group: [] for group in allowed}
    for example in examples:
        group = example['geometry_group_sha256']
        if group not in groups:
            raise ValueError('Una etiqueta pertenece a una geometría no autorizada para este split.')
        groups[group].append(example)
    if not groups or any(not values for values in groups.values()):
        raise ValueError('Faltan etiquetas para alguna geometría del split.')
    return groups


def validation_nll(model, problems, examples, groups, device):
    """Promedio de NLL por decisión, luego etiqueta y geometría; sin heldout."""
    grouped = _group_examples(examples, groups)
    model.eval()
    losses = {}
    with torch.no_grad():
        for group, items in grouped.items():
            values = []
            for example in items:
                loss, _ = teacher_loss(model, problems[example['instance_sha256']],
                                       example['routes'], device)
                value = float(loss.detach().cpu())
                if not math.isfinite(value):
                    raise ValueError('NLL de validación no finita.')
                values.append(value)
            losses[group] = float(np.mean(values))
    return float(np.mean(list(losses.values()))), losses


def _cpu_state(model):
    return {key: tensor.detach().cpu().clone() for key, tensor in model.state_dict().items()}


def _load_checkpoint(path):
    # Sólo bundles propios, previamente verificados; evita pickle arbitrario.
    return torch.load(path, map_location='cpu', weights_only=True)


def _runtime_signature(device):
    return {'torch': torch.__version__, 'numpy': np.__version__, 'device_type': torch.device(device).type,
            'cuda': torch.version.cuda,
            'device_name': torch.cuda.get_device_name(device) if torch.device(device).type == 'cuda' else 'cpu'}


def train_fold(instances, spec, config, seed, fold, corpus, directory, remote, device):
    require_colab()
    validate_training_config(config)
    if type(seed) is not int or seed < 0:
        raise ValueError('La semilla de entrenamiento debe ser un entero no negativo.')
    validate_fold(fold, instances)
    directory, remote = Path(directory), Path(remote)
    device = choose_device(str(device))
    started = time.perf_counter()
    train, validation, provenance = load_fold_examples(corpus, instances, fold)
    grouped = _group_examples(train, fold['train_geometry_groups'])
    _group_examples(validation, fold['validation_geometry_groups'])
    identity = {'schema_version': config['schema_version'], 'fold': fold, 'seed': seed,
                'config': config, 'source_hashes': _source_hashes(),
                'corpus_id': provenance['corpus_id'], 'traffic': asdict(spec),
                'train_examples_sha256': p.digest(train), 'validation_examples_sha256': p.digest(validation)}
    if e._recover(directory / 'meta', remote / 'meta'):
        if e.read_json(directory / 'meta/identity.json') != identity:
            raise ValueError('Checkpoint de entrenamiento incompatible con corpus, código, fold o parámetros.')
    else:
        e.write_json(directory / 'meta/identity.json', identity)
        e.write_json(directory / 'meta/provenance.json', provenance)
        e.write_json(directory / 'meta/environment.json', p.environment_report())
        e._commit(directory / 'meta', remote / 'meta')
    if e._recover(directory / 'final', remote / 'final'):
        if e.read_json(directory / 'final/summary.json')['identity_sha256'] != p.digest(identity):
            raise ValueError('Modelo final de otro entrenamiento.')
        return directory / 'final'
    instances_by_sha = {x.sha: x for x in instances}
    for role, examples in [('train_geometry_groups', train), ('validation_geometry_groups', validation)]:
        for example in examples:
            instance = instances_by_sha.get(example['instance_sha256'])
            if (instance is None or instance.provenance['geometry_group_sha256'] != example['geometry_group_sha256']
                    or instance.provenance['geometry_group_sha256'] not in fold[role]):
                raise ValueError('Etiqueta fuera del registro o del rol de geometría autorizado.')
    selected = {ex['instance_sha256'] for ex in train + validation}
    problems = {sha: SearchProblem(instances_by_sha[sha], spec, config['teacher']['time_scale']) for sha in selected}
    for example in train + validation:
        if problems[example['instance_sha256']].score(example['routes'])[0] > 1e-9:
            raise ValueError('Etiqueta sin factibilidad conservadora.')
    settings = config['training']
    derived_seed = p.seed_for('supervised-03', fold['fold_id'], seed)
    random.seed(derived_seed)
    torch.manual_seed(derived_seed)
    if device.type == 'cuda':
        torch.cuda.manual_seed_all(derived_seed)
    torch.use_deterministic_algorithms(settings['deterministic_algorithms'])
    torch.backends.cudnn.benchmark = False
    rng = np.random.default_rng(derived_seed)
    model = AttentionPolicy(config['model']).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=settings['learning_rate'],
                                 weight_decay=settings['weight_decay'])
    best_state, best_loss, best_epoch, stale = None, math.inf, 0, 0
    history, elapsed_total, validation_total, last_epoch, terminal = [], 0.0, 0.0, 0, None
    checkpoint_numbers = set()
    for location in (directory, remote):
        if location.exists():
            checkpoint_numbers.update(int(child.name[6:]) for child in location.glob('epoch_*')
                                      if child.name[6:].isdigit() and (child / 'CHECKSUMS.json').exists())
    if checkpoint_numbers and sorted(checkpoint_numbers) != list(range(1, max(checkpoint_numbers) + 1)):
        raise ValueError('Faltan épocas selladas en la secuencia de checkpoints.')
    for epoch in sorted(checkpoint_numbers):
        if not e._recover(directory / f'epoch_{epoch:04d}', remote / f'epoch_{epoch:04d}'):
            raise ValueError('Checkpoint de época no recuperable.')
    if checkpoint_numbers:
        last_epoch = max(checkpoint_numbers)
        latest = directory / f'epoch_{last_epoch:04d}'
        progress = e.read_json(latest / 'progress.json')
        if (progress['identity_sha256'] != p.digest(identity)
                or progress['runtime_signature'] != _runtime_signature(device)):
            raise ValueError('No se reanuda entrenamiento parcial con otro entorno/fold. Use el entorno original o una configuración nueva.')
        state = _load_checkpoint(latest / 'state.pt')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        best_state = state['best_model']
        best_loss, best_epoch, stale = progress['best_validation_nll'], progress['best_epoch'], progress['stale_epochs']
        history = progress['history']
        elapsed_total, validation_total = progress['training_seconds'], progress['validation_seconds']
        rng.bit_generator.state = progress['numpy_generator_state']
        random.setstate((progress['python_rng_state'][0], tuple(progress['python_rng_state'][1]), progress['python_rng_state'][2]))
        torch.set_rng_state(state['torch_rng'])
        if device.type == 'cuda':
            torch.cuda.set_rng_state_all(state['cuda_rng'])
        terminal = progress['stop_reason']
    synchronize(device)
    elapsed_total += time.perf_counter() - started
    groups = sorted(grouped)
    for epoch in range(last_epoch + 1, settings['epochs'] + 1):
        if terminal:
            break
        if elapsed_total >= settings['seconds_per_model']:
            terminal = 'time_limit'
            break
        epoch_started = time.perf_counter()
        model.train()
        values, updates, decisions = [], 0, 0
        for offset in range(0, settings['steps_per_epoch'], settings['gradient_accumulation']):
            if elapsed_total + time.perf_counter() - epoch_started >= settings['seconds_per_model']:
                break
            batch_size = min(settings['gradient_accumulation'], settings['steps_per_epoch'] - offset)
            optimizer.zero_grad(set_to_none=True)
            for _ in range(batch_size):
                group = groups[int(rng.integers(len(groups)))]
                items = grouped[group]
                example = items[int(rng.integers(len(items)))]
                problem = problems[example['instance_sha256']]
                order = rng.permutation(np.arange(1, problem.n + 1)).tolist() if settings['permute_nodes'] else None
                loss, stats = teacher_loss(model, problem, example['routes'], device, node_order=order)
                if not bool(torch.isfinite(loss).item()):
                    raise ValueError('Pérdida de entrenamiento no finita.')
                (loss / batch_size).backward()
                values.append(float(loss.detach().cpu()))
                decisions += stats['decisions']
            norm = torch.nn.utils.clip_grad_norm_(model.parameters(), settings['gradient_clip_norm'], error_if_nonfinite=True)
            if not math.isfinite(float(norm)):
                raise ValueError('Gradientes no finitos.')
            optimizer.step(); updates += 1
        if not values:
            elapsed_total += time.perf_counter() - epoch_started
            terminal = 'time_limit'
            break
        synchronize(device)
        val_started = time.perf_counter()
        val_loss, group_losses = validation_nll(model, problems, validation, fold['validation_geometry_groups'], device)
        synchronize(device)
        val_seconds = time.perf_counter() - val_started
        if val_loss < best_loss - settings['min_delta']:
            best_loss, best_epoch, best_state, stale = val_loss, epoch, _cpu_state(model), 0
        else:
            stale += 1
        elapsed_total += time.perf_counter() - epoch_started
        validation_total += val_seconds
        terminal = ('patience' if stale >= settings['patience'] else
                    'time_limit' if elapsed_total >= settings['seconds_per_model'] else
                    'epochs' if epoch == settings['epochs'] else None)
        history.append({'epoch': epoch, 'train_nll': float(np.mean(values)), 'validation_nll': val_loss,
                        'best_validation_nll': best_loss, 'best_epoch': best_epoch,
                        'examples_sampled': len(values), 'decisions': decisions, 'optimizer_updates': updates,
                        'training_seconds': elapsed_total, 'validation_seconds': validation_total})
        target = directory / f'epoch_{epoch:04d}'
        # Preservar una escritura interrumpida de esta época antes de repetirla.
        if e._recover(target, remote / target.name):
            raise ValueError('La época nueva ya existe; no sobrescribir un checkpoint.')
        target.mkdir(parents=True, exist_ok=True)
        torch.save({'model': _cpu_state(model), 'optimizer': optimizer.state_dict(), 'best_model': best_state,
                    'torch_rng': torch.get_rng_state(),
                    'cuda_rng': torch.cuda.get_rng_state_all() if device.type == 'cuda' else []}, target / 'state.pt')
        e.write_json(target / 'progress.json', {'epoch': epoch, 'identity_sha256': p.digest(identity),
                     'runtime_signature': _runtime_signature(device), 'best_validation_nll': best_loss,
                     'best_epoch': best_epoch, 'stale_epochs': stale, 'history': history,
                     'training_seconds': elapsed_total, 'validation_seconds': validation_total,
                     'numpy_generator_state': rng.bit_generator.state, 'python_rng_state': random.getstate(),
                     'validation_by_geometry': group_losses, 'stop_reason': terminal})
        e.write_json(target / 'environment.json', p.environment_report())
        e._commit(target, remote / target.name)
        print(f'{fold["fold_id"]} seed={seed} epoch={epoch}: train={history[-1]["train_nll"]:.5f}, '
              f'val={val_loss:.5f}, best={best_epoch}', flush=True)
    if best_state is None:
        raise RuntimeError('No hay checkpoint entrenado y seleccionado; no evaluar pesos aleatorios.')
    final = directory / 'final'
    final.mkdir(parents=True, exist_ok=True)
    torch.save(best_state, final / 'best.pt')
    summary = {'identity_sha256': p.digest(identity), 'fold': fold, 'training_seed': seed,
               'training_seconds': elapsed_total, 'validation_seconds': validation_total,
               'training_budget_seconds': settings['seconds_per_model'],
               'training_budget_overrun_seconds': max(0., elapsed_total - settings['seconds_per_model']),
               'training_time_scope': 'setup_and_epoch_compute_validation_excludes_epoch_and_final_checkpoint_persistence',
               'teacher_seconds': provenance['selected_teacher_costs']['teacher_wall_seconds'],
               'best_epoch': best_epoch, 'best_validation_nll': best_loss, 'epochs_completed': len(history),
               'stop_reason': terminal, 'model_config': config['model'], 'traffic': asdict(spec),
               'corpus_id': provenance['corpus_id'], 'checkpoint_selection': 'inner_validation_teacher_nll',
               'training_scope': 'development_geometry_crossfit_surrogate_imitation',
               'train_examples': len(train), 'validation_examples': len(validation),
               'train_geometries': len(grouped), 'heldout_used_for_training_or_selection': False,
               'model_checkpoint_sha256': p.file_hash(final / 'best.pt')}
    e.write_json(final / 'summary.json', summary)
    e.write_json(final / 'history.json', history)
    e.write_csv(final / 'history.csv', history)
    save_learning_plot(history, final / 'learning.png', fold['fold_id'], seed)
    e._commit(final, remote / 'final')
    del model, optimizer
    if device.type == 'cuda':
        torch.cuda.empty_cache()
    return final


def save_learning_plot(history, path, fold_id, seed):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8, 4))
    for metric in ['train_nll', 'validation_nll']:
        ax.plot([row['epoch'] for row in history], [row[metric] for row in history], label=metric)
    ax.set(title=f'{fold_id} · seed={seed} · NLL de imitación', xlabel='Época', ylabel='NLL por decisión')
    ax.legend(); fig.tight_layout(); fig.savefig(path, dpi=150); plt.close(fig)


def train_crossfit(instances, spec, config, seeds, corpus, work_root, drive_root, bank_id,
                   bank_sha256, evaluator_sha256, device='auto'):
    require_colab()
    validate_training_config(config)
    if (not seeds or any(type(seed) is not int or seed < 0 for seed in seeds)
            or len(set(seeds)) != len(seeds)):
        raise ValueError('Semillas de entrenamiento vacías, repetidas o inválidas.')
    if evaluator_sha256 != p.file_hash(p.__file__):
        raise ValueError('Evaluador de entrenamiento distinto del módulo sellado activo.')
    device = choose_device(device)
    folds = make_geometry_folds(instances)
    identity = {'config': config, 'seeds': seeds, 'folds': folds,
                'corpus_id': Path(corpus).name, 'bank_sha256': bank_sha256,
                'evaluator_sha256': evaluator_sha256, 'source_hashes': _source_hashes()}
    training_id = 'attention03_' + p.digest(identity)[:20]
    root = Path(work_root) / 'supervised_training' / bank_id / training_id
    remote = Path(drive_root) / 'supervised_training' / bank_id / training_id
    print('TRAINING_ID =', training_id, '| dispositivo:', device, flush=True)
    if e._recover(root / 'meta', remote / 'meta'):
        if e.read_json(root / 'meta/identity.json') != identity:
            raise ValueError('Identidad de entrenamiento inconsistente.')
    else:
        e.write_json(root / 'meta/identity.json', identity)
        e.write_json(root / 'meta/folds.json', folds)
        for name in _source_hashes():
            target = root / 'meta/code' / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(Path(__file__).with_name(name).read_bytes())
        e._commit(root / 'meta', remote / 'meta')
    models = []
    for fold in folds:
        for seed in seeds:
            relative = Path(fold['fold_id']) / f'seed_{seed}'
            attempt_started = time.perf_counter()
            try:
                final = train_fold(instances, spec, config, seed, fold, corpus,
                                   root / relative, remote / relative, device)
            except Exception as exc:
                failure = root / 'failures' / (fold['fold_id'] + f'_seed_{seed}_' + uuid.uuid4().hex[:8])
                e.write_json(failure / 'failure.json', {'fold': fold, 'seed': seed, 'training_id': training_id,
                             'error_type': type(exc).__name__, 'error': str(exc),
                             'observed_call_wall_seconds_including_io': time.perf_counter() - attempt_started,
                             'completed_checkpoints_preserved': True})
                e.write_json(failure / 'environment.json', p.environment_report())
                e._commit(failure, remote / 'failures' / failure.name)
                raise
            models.append({**e.read_json(final / 'summary.json'), 'relative_directory': str(relative / 'final'),
                           'bundle_sha256': p.verify_bundle(final)})
    # Toda selección terminó antes de que el notebook invoque evaluate sobre heldout.
    if not e._recover(root / 'index', remote / 'index'):
        _, _, provenance = load_fold_examples(corpus, instances, folds[0])
        e.write_json(root / 'index/models.json', models)
        e.write_json(root / 'index/offline_costs.json', {'offline_cost_id': training_id,
                     'teacher_corpus_id': provenance['corpus_id'],
                     'teacher_costs_unique': provenance['all_teacher_costs'],
                     'training_seconds_unique': sum(row['training_seconds'] for row in models),
                     'validation_seconds_included_in_training': sum(row['validation_seconds'] for row in models),
                     'models_trained': len(models), 'geometry_count': len(folds),
                     'summary_times_must_not_be_summed_across_policies_or_scenarios': True})
        e.write_csv(root / 'index/training_summary.csv', [{k: v for k, v in row.items()
                                                         if not isinstance(v, (dict, list))} for row in models])
        e._commit(root / 'index', remote / 'index')
    if e.read_json(root / 'index/models.json') != models:
        raise ValueError('Inventario de modelos distinto de los checkpoints.')
    return root


def make_solver(training_root, instances, spec, device='auto', expected_bank_sha256=None):
    """Carga sólo el modelo que dejó fuera la geometría de la instancia solicitada."""
    root, device = Path(training_root), choose_device(device)
    p.verify_bundle(root / 'meta')
    identity = e.read_json(root / 'meta/identity.json')
    if expected_bank_sha256 is not None and identity['bank_sha256'] != expected_bank_sha256:
        raise ValueError('Entrenamiento vinculado a otro banco.')
    p.verify_bundle(root / 'index')
    index = e.read_json(root / 'index/models.json')
    registry, provenance = {}, {}
    for model in index:
        final = root / model['relative_directory']
        if p.verify_bundle(final) != model['bundle_sha256'] or p.file_hash(final / 'best.pt') != model['model_checkpoint_sha256']:
            raise ValueError('Checkpoint de modelo modificado.')
        if model['traffic'] != asdict(spec):
            raise ValueError('Modelo entrenado con otro tráfico conocido.')
        fold = model['fold']
        validate_fold(fold, instances)
        for x in instances:
            if x.provenance['geometry_group_sha256'] not in fold['heldout_geometry_groups']:
                continue
            key = f'{x.sha}:{model["training_seed"]}'
            if key in registry:
                raise ValueError('Dos modelos compiten por una observación heldout.')
            registry[key] = (final, model)
            provenance[key] = {key_: model[key_] for key_ in ['training_seed', 'training_seconds',
                'validation_seconds', 'teacher_seconds', 'model_checkpoint_sha256', 'checkpoint_selection', 'training_scope']}
            provenance[key].update(training_id=root.name, fold_id=fold['fold_id'],
                train_geometry_groups=fold['train_geometry_groups'],
                validation_geometry_groups=fold['validation_geometry_groups'],
                heldout_geometry_groups=fold['heldout_geometry_groups'], offline_cost_id=root.name,
                offline_manifest_sha256=p.verify_bundle(root / 'index'))

    def solver(problem, seed, config, output_dir):
        started = time.perf_counter()
        key = f'{problem.x.sha}:{seed}'
        if key not in registry:
            raise ValueError('No existe modelo fuera de geometría para esta instancia/semilla.')
        final, record = registry[key]
        if p.file_hash(final / 'best.pt') != record['model_checkpoint_sha256']:
            raise ValueError('Checkpoint cambió antes de inferencia.')
        model = AttentionPolicy(record['model_config']).to(device)
        model.load_state_dict(_load_checkpoint(final / 'best.pt'))
        model.eval(); synchronize(device)
        loading = time.perf_counter() - started
        remaining = config['seconds'] - loading
        try:
            if remaining <= 0:
                result = problem.result(None, 'time_limit_model_load', loading, [])
            else:
                result = solve_attention(problem, seed, {**config, 'seconds': remaining}, model, device)
            result['metadata'].update(model_load_seconds=loading,
                                      model_checkpoint_sha256=record['model_checkpoint_sha256'],
                                      fold_id=record['fold']['fold_id'])
            synchronize(device)
            result['solve_seconds'] = time.perf_counter() - started
            return result
        finally:
            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
    return solver, {'am_supervised': provenance}


Overwriting svrplab/supervised.py


In [58]:
%%writefile svrplab/supervised_data.py
"""Corpus heurístico auditado y particiones por geometría para el método 03.

La construcción se ejecuta exclusivamente en Colab. ACO/Tabú y el respaldo
CP-SAT parten de cero: nunca leen certificados ni escenarios de evaluación.
Los cuatro folds del piloto son desarrollo cruzado, no un test confirmatorio.
"""
from __future__ import annotations

import math
import shutil
import time
from collections.abc import Mapping
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

from . import experiments as exp
from . import metaheuristics as mh
from . import pilot as p
from .search import OBJECTIVE, SearchProblem

SCHEMA = 'supervised-teachers-v2'
TEACHER_DEFAULTS = {
    'seeds': [101, 211, 307], 'seconds_per_method': 120., 'time_scale': 1000,
    'aco': {'max_iterations': 200, 'ants': 20, 'alpha': 1., 'beta': 2.,
            'evaporation': .1, 'q0': .2},
    'tabu': {'max_iterations': 200, 'neighborhood_samples': 80, 'tenure': 15,
             'initial_restarts': 20},
    'fallback': {'enabled': False, 'seeds': [401, 503, 607],
                 'seconds_per_attempt': 600., 'workers': 1},
}
_ROLES = ('train_geometry_groups', 'validation_geometry_groups',
          'heldout_geometry_groups')
_COSTS = ('teacher_search_seconds', 'teacher_preparation_seconds',
          'teacher_validation_seconds', 'teacher_wall_seconds')


def _require_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError as exc:
        raise RuntimeError('Experimentación exclusivamente en Google Colab.') from exc


def _registry(instances):
    rows = []
    for x in instances:
        group = x.provenance.get('geometry_group_sha256')
        if (not isinstance(group, str) or len(group) != 64
                or any(c not in '0123456789abcdef' for c in group)):
            raise ValueError('Cada instancia requiere geometry_group_sha256 válido.')
        rows.append({'instance_id': x.instance_id, 'instance_sha256': x.sha,
                     'geometry_group_sha256': group, 'n_customers': x.n})
    if not rows or len({r['instance_id'] for r in rows}) != len(rows):
        raise ValueError('Registro de instancias vacío o con identificadores repetidos.')
    if len({r['instance_sha256'] for r in rows}) != len(rows):
        raise ValueError('Registro de instancias con hashes repetidos.')
    return sorted(rows, key=lambda r: (r['geometry_group_sha256'], r['instance_sha256']))


def make_geometry_folds(instances):
    """Cada grupo es heldout una vez; validación es el siguiente hash circular.

    Con las cuatro geometrías del piloto quedan dos grupos de entrenamiento,
    uno de validación y uno excluido de ambos. Ningún derivado cambia de rol.
    """
    instances = list(instances)
    groups = sorted({r['geometry_group_sha256'] for r in _registry(instances)})
    if len(groups) < 3:
        raise ValueError('Se requieren al menos tres geometrías independientes para los folds.')
    folds = []
    for i, heldout in enumerate(groups):
        validation = groups[(i + 1) % len(groups)]
        fold = {'fold_id': f'geometry_{i:02d}',
                'train_geometry_groups': [g for g in groups if g not in (heldout, validation)],
                'validation_geometry_groups': [validation],
                'heldout_geometry_groups': [heldout]}
        validate_fold(fold, instances)
        folds.append(fold)
    return folds


def validate_fold(fold, instances):
    """Rechaza cualquier mezcla de roles, grupo ajeno u omisión del registro."""
    known = {r['geometry_group_sha256'] for r in _registry(instances)}
    if (not isinstance(fold, Mapping) or not isinstance(fold.get('fold_id'), str)
            or not fold['fold_id'] or Path(fold['fold_id']).name != fold['fold_id']
            or fold['fold_id'] in ('.', '..')):
        raise ValueError('fold_id inválido.')
    used = set()
    for role in _ROLES:
        values = fold.get(role)
        if (not isinstance(values, (list, tuple)) or not values
                or any(not isinstance(v, str) for v in values)
                or len(set(values)) != len(values)):
            raise ValueError(f'Rol vacío, repetido o inválido: {role}')
        group_set = set(values)
        if not group_set <= known or used & group_set:
            raise ValueError('Grupos desconocidos o solapados entre train/validation/heldout.')
        used.update(group_set)
    if used != known or len(fold['heldout_geometry_groups']) != 1:
        raise ValueError('Todos los grupos deben tener exactamente un rol y un solo heldout.')
    return True



def validate_holdout_fold(fold, instances):
    """Partición final explícita: varias geometrías test, ninguna reutilizada.

    El runner final verifica además la reserva sellada y aporta su hash. Este
    validador comprueba exhaustividad/disjunción sin relajar los folds históricos.
    """
    known = {r['geometry_group_sha256'] for r in _registry(instances)}
    if (not isinstance(fold, Mapping) or fold.get('split_strategy') != 'independent_train_validation_test_v1'
            or not isinstance(fold.get('fold_id'), str) or not fold['fold_id']
            or Path(fold['fold_id']).name != fold['fold_id'] or fold['fold_id'] in ('.', '..')):
        raise ValueError('Partición holdout final inválida.')
    _check_hash(fold.get('split_registry_sha256'), 'split_registry_sha256')
    used = set()
    for role in _ROLES:
        values = fold.get(role)
        if (not isinstance(values, list) or not values or len(set(values)) != len(values)
                or any(not isinstance(value, str) for value in values)):
            raise ValueError('Rol holdout vacío o repetido.')
        group_set = set(values)
        if not group_set <= known or used & group_set:
            raise ValueError('Geometría ajena o compartida entre train/validation/test.')
        used.update(group_set)
    if used != known:
        raise ValueError('Partición holdout omite geometrías de la población.')
    return True

def _configuration(config):
    if not isinstance(config, Mapping) or set(config) - set(TEACHER_DEFAULTS):
        raise ValueError('Configuración de profesores inválida o con campos desconocidos.')
    cfg = {**TEACHER_DEFAULTS, **dict(config)}
    seeds = cfg['seeds']
    if (not isinstance(seeds, (list, tuple)) or not seeds
            or any(type(s) is not int or s < 0 for s in seeds)
            or len(set(seeds)) != len(seeds)):
        raise ValueError('Semillas de profesores inválidas o repetidas.')
    seconds = cfg['seconds_per_method']
    if (type(seconds) not in (int, float) or not math.isfinite(seconds) or seconds <= 0
            or type(cfg['time_scale']) is not int or cfg['time_scale'] <= 0):
        raise ValueError('Presupuesto y escala temporal deben ser positivos y finitos.')
    cfg['seeds'] = list(seeds)
    cfg['seconds_per_method'] = float(seconds)
    for name, defaults in [('aco', mh.ACO_DEFAULTS), ('tabu', mh.TABU_DEFAULTS)]:
        if not isinstance(cfg[name], Mapping) or 'seconds' in cfg[name]:
            raise ValueError('Use seconds_per_method para el presupuesto común de profesores.')
        cfg[name] = mh._configuration(defaults, {**cfg[name], 'seconds': float(seconds)})
        cfg[name].pop('seconds')
    fallback = cfg['fallback']
    if (not isinstance(fallback, Mapping)
            or set(fallback) - set(TEACHER_DEFAULTS['fallback'])):
        raise ValueError('Configuración de respaldo inválida.')
    fallback = {**TEACHER_DEFAULTS['fallback'], **dict(fallback)}
    if type(fallback['enabled']) is not bool:
        raise ValueError('fallback.enabled debe ser booleano.')
    seeds = fallback['seeds']
    if (not isinstance(seeds, (list, tuple)) or not seeds
            or any(type(seed) is not int or seed < 0 for seed in seeds)
            or len(set(seeds)) != len(seeds)):
        raise ValueError('Semillas de respaldo inválidas.')
    seconds = fallback['seconds_per_attempt']
    if (type(seconds) not in (int, float) or not math.isfinite(seconds) or seconds <= 0
            or type(fallback['workers']) is not int or fallback['workers'] != 1):
        raise ValueError('Respaldo requiere presupuesto positivo y un worker.')
    cfg['fallback'] = {**fallback, 'seeds': list(seeds), 'seconds_per_attempt': float(seconds)}
    return cfg


def solve_feasibility(problem, seed, config):
    """Dependencia diferida: los folds de 04/05 no necesitan cargar un solver."""
    from .teacher_feasibility import solve_feasibility as solve
    return solve(problem, seed, config)


def _check_hash(value, name):
    if (not isinstance(value, str) or len(value) != 64
            or any(c not in '0123456789abcdef' for c in value)):
        raise ValueError(f'Hash inválido: {name}')


def _check_result(problem, result, spec):
    """Validez de etiqueta: estructura, cota conservadora y simulador nominal."""
    validation = {'eligible_label': False, 'structural_valid': False,
                  'conservative_feasible': False, 'nominal_feasible': False,
                  'reason': 'no_solution', 'objective_scope': OBJECTIVE}
    nominal = None
    try:
        routes = result.get('routes')
        if routes is None:
            return validation, nominal
        p.validate_routes(problem.x, routes)
        validation['structural_valid'] = True
        violation, cost = problem.score(routes)
        if not math.isfinite(violation) or not math.isfinite(cost):
            raise ValueError('Puntuación de profesor no finita.')
        validation.update(search_violation_minutes=float(violation),
                          search_objective_minutes=float(cost),
                          conservative_feasible=violation <= 1e-9)
        nominal_spec = p.Traffic(**(asdict(spec) | {'mode': 'nominal_mean'}))
        # El 0 es una identidad formal: nominal_mean no muestrea ruido.
        nominal = p.simulate(problem.x, routes, 0, nominal_spec, 'fixed_order', trace=True)
        validation['nominal_feasible'] = bool(nominal['strict_feasible'])
        validation['eligible_label'] = validation['conservative_feasible'] and validation['nominal_feasible']
        validation['reason'] = ('accepted_heuristic_label' if validation['eligible_label'] else
                                'conservative_infeasible' if not validation['conservative_feasible'] else
                                'nominal_infeasible')
    except Exception as exc:
        validation.update(reason='validation_error', error_type=type(exc).__name__, error=str(exc))
    return validation, nominal


def _run_attempt(x, spec, cfg, method, seed, identity, directory):
    """Persiste incluso soluciones rechazadas; retorna KeyboardInterrupt diferido."""
    directory.mkdir(parents=True)
    started = time.perf_counter()
    preparation = search_seconds = validation_seconds = 0.
    result = {'routes': None, 'history': [], 'metadata': {}, 'status': 'solver_error',
              'solve_seconds': 0.}
    validation = {'eligible_label': False, 'structural_valid': False,
                  'conservative_feasible': False, 'nominal_feasible': False,
                  'reason': 'solver_error'}
    nominal, interrupt = None, None
    problem, search_started = None, None
    try:
        problem = SearchProblem(x, spec, cfg['time_scale'])
        preparation = time.perf_counter() - started
        search_started = time.perf_counter()
        budget = max(0., identity['search_budget_seconds'] - preparation)
        if method == 'cp_sat_feasibility':
            solver = solve_feasibility
            settings = {'workers': cfg['fallback']['workers'], 'seconds': budget}
        else:
            solver = {'aco': mh.solve_aco, 'tabu': mh.solve_tabu}[method]
            settings = {**cfg[method], 'seconds': budget}
        result = solver(problem, seed, settings)
        if not isinstance(result, dict):
            raise TypeError('El profesor debe devolver un diccionario de resultado.')
        search_seconds = time.perf_counter() - search_started
        search_started = None
        validation_started = time.perf_counter()
        validation, nominal = _check_result(problem, result, spec)
        validation_seconds = time.perf_counter() - validation_started
    except (Exception, KeyboardInterrupt) as exc:
        if search_started is not None:
            search_seconds = time.perf_counter() - search_started
        elif problem is None:
            preparation = time.perf_counter() - started
        if not isinstance(result, dict):
            result = {'routes': None, 'history': [], 'metadata': {}}
        result.update(status='interrupted' if isinstance(exc, KeyboardInterrupt) else 'solver_error')
        result['metadata'] = {**result.get('metadata', {}),
                              'error_type': type(exc).__name__, 'error': str(exc)}
        validation.update(eligible_label=False, reason=result['status'])
        if isinstance(exc, KeyboardInterrupt):
            interrupt = exc
    if x.sha != identity['instance_sha256']:
        raise AssertionError('El profesor modificó la instancia.')
    result.setdefault('history', [])
    result.setdefault('metadata', {})
    result.setdefault('routes', None)
    result.setdefault('solve_seconds', search_seconds)
    # JSON conserva rutas de intentos inválidos; el CSV sólo requiere estructura válida.
    exp.write_json(directory / 'result.json', result)
    exp.write_json(directory / 'nominal.json', nominal)
    exp.write_json(directory / 'validation.json', validation)
    exp.write_csv(directory / 'search_history.csv', result['history'],
                  None if result['history'] else ['iteration', 'elapsed_seconds'])
    rows = exp.route_rows(x, result['routes'], identity) if validation['structural_valid'] else []
    exp.write_csv(directory / 'planned_routes.csv', rows,
                  None if rows else ['instance_id', 'teacher_method', 'teacher_seed', 'vehicle', 'position', 'node'])
    if nominal is not None:
        exp.write_csv(directory / 'nominal_events.csv', nominal['events'])
    costs = {'teacher_preparation_seconds': float(preparation),
             'teacher_search_seconds': float(search_seconds),
             'teacher_validation_seconds': float(validation_seconds),
             'teacher_wall_seconds': time.perf_counter() - started,
             'wall_scope': 'preparation_search_validation_local_artifacts_before_final_record; excludes_drive_sync'}
    exp.write_json(directory / 'attempt.json', {**identity, **costs,
                   'status': result['status'], 'label_validation': validation,
                   'solver_termination': result['metadata'].get('termination'),
                   'solver_iterations': result['metadata'].get('iterations'),
                   'solver_error_type': result['metadata'].get('error_type'),
                   'solver_error': result['metadata'].get('error'),
                   'routes_sha256': p.digest(result['routes']) if result['routes'] is not None else None,
                   'construction': 'independent_from_scratch_without_certificates',
                   'label_kind': 'heuristic_not_proven_optimal'})
    return interrupt


def _cost_summary(attempts, groups):
    costs = []
    for group in sorted(groups):
        subset = [a for a in attempts if a['geometry_group_sha256'] == group]
        costs.append({'geometry_group_sha256': group, 'attempts': len(subset),
                      'accepted_attempts': sum(a['label_validation']['eligible_label'] for a in subset),
                      'rejected_attempts': sum(not a['label_validation']['eligible_label'] for a in subset),
                      **{key: sum(a[key] for a in subset) for key in _COSTS}})
    return costs


def build_teacher_corpus(instances, spec, config, work_root, drive_root, bank_id,
                         bank_sha256, evaluator_sha256):
    """Genera o recupera cada intento inmutable y retorna la raíz del corpus.

    El llamador debe cargar antes el banco aceptado mediante p.load_bank.
    Identidad: banco, registro íntegro, tráfico, código y configuración efectiva.
    Sólo se sella index/ al terminar: nunca la raíz que recibe nuevos intentos.
    """
    _require_colab()
    instances = list(instances)
    registry, cfg = _registry(instances), _configuration(config)
    folds = make_geometry_folds(instances)
    if not isinstance(bank_id, str) or Path(bank_id).name != bank_id or bank_id in ('', '.', '..'):
        raise ValueError('bank_id debe ser un nombre de carpeta simple.')
    _check_hash(bank_sha256, 'bank_sha256')
    _check_hash(evaluator_sha256, 'evaluator_sha256')
    if p.file_hash(p.__file__) != evaluator_sha256:
        raise ValueError('El evaluador activo no coincide con el banco sellado.')
    package = Path(__file__).parent
    source_hashes = {name: p.file_hash(package / name) for name in
                     ('supervised_data.py', 'metaheuristics.py', 'baselines.py', 'search.py', 'experiments.py')}
    source_hashes['pilot.py'] = evaluator_sha256
    if cfg['fallback']['enabled']:
        source_hashes['teacher_feasibility.py'] = p.file_hash(package / 'teacher_feasibility.py')
    protocol = {'schema_version': SCHEMA, 'role': 'development_geometry_crossfit',
                'bank_id': bank_id, 'bank_manifest_sha256': bank_sha256,
                'evaluator_sha256': evaluator_sha256, 'instances': registry,
                'traffic': asdict(spec), 'config': cfg, 'source_hashes': source_hashes,
                'objective_scope': OBJECTIVE, 'folds': folds,
                'labels': 'all_distinct_eligible_heuristic_final_incumbents',
                'fallback_trigger': 'zero_eligible_base_labels_for_instance_after_all_aco_tabu_attempts',
                'fallback_seed_policy': 'all_predeclared_seeds_even_after_success',
                'fallback_objective': 'feasibility_only_not_cost_optimality',
                'independent_unit': 'geometry_group_sha256'}
    corpus_id = p.digest(protocol)
    root = Path(work_root) / 'teacher_corpora' / bank_id / corpus_id
    remote = Path(drive_root) / 'teacher_corpora' / bank_id / corpus_id
    if root.resolve() == remote.resolve():
        raise ValueError('Trabajo local y Drive requieren directorios distintos.')
    print(f'TEACHER_CORPUS_ID = {corpus_id}\nDrive: {remote}', flush=True)
    if exp._recover(root / 'meta', remote / 'meta'):
        if exp.read_json(root / 'meta/protocol.json') != protocol:
            raise ValueError('Protocolo de profesores distinto del checkpoint.')
    else:
        exp.write_json(root / 'meta/protocol.json', protocol)
        exp.write_json(root / 'meta/identity.json', {'corpus_id': corpus_id,
                       'created_utc': datetime.now(timezone.utc).isoformat()})
        exp.write_json(root / 'meta/environment.json', p.environment_report())
        code_dir = root / 'meta/code'
        code_dir.mkdir()
        for name in source_hashes:
            shutil.copy2(Path(p.__file__) if name == 'pilot.py' else package / name, code_dir / name)
        exp._commit(root / 'meta', remote / 'meta')
    attempts, labels, seen, inventory = [], [], set(), {'meta': p.verify_bundle(root / 'meta')}
    by_sha = {x.sha: x for x in instances}
    def collect(row, method, seed, budget):
        x = by_sha[row['instance_sha256']]
        key = f'attempts/{x.sha}/{method}/seed_{seed}'
        identity = {**row, 'corpus_id': corpus_id, 'bank_id': bank_id,
                    'bank_manifest_sha256': bank_sha256,
                    'evaluator_sha256': evaluator_sha256,
                    'teacher_method': method, 'teacher_seed': seed,
                    'search_budget_seconds': budget, 'attempt_key': key}
        directory = root / key
        recovered = exp._recover(directory, remote / key)
        if not recovered:
            print(f'Profesor {method} · {x.instance_id} · seed={seed} · presupuesto={budget:g}s', flush=True)
            interrupt = _run_attempt(x, spec, cfg, method, seed, identity, directory)
            exp._commit(directory, remote / key)
            if interrupt is not None:
                raise interrupt
        attempt = exp.read_json(directory / 'attempt.json')
        if any(attempt.get(k) != v for k, v in identity.items()):
            raise ValueError('Identidad de intento incompatible con el corpus.')
        attempts.append(attempt)
        inventory[key] = p.verify_bundle(directory)
        if p.verify_bundle(remote / key) != inventory[key]:
            raise ValueError('Intento en Drive distinto del local.')
        validation = attempt['label_validation']
        print(f"  {'recuperado' if recovered else 'terminado'} · n={x.n} · {method} · seed={seed} · "
              f"estado={attempt['status']} · {validation['reason']} · "
              f"violación={validation.get('search_violation_minutes')} · "
              f"búsqueda={attempt['teacher_search_seconds']:.2f}s · "
              f"terminación={attempt.get('solver_termination')}", flush=True)
        if attempt.get('solver_error') or validation.get('error'):
            print('  error:', attempt.get('solver_error') or validation.get('error'), flush=True)
        label_key = (x.sha, attempt['routes_sha256'])
        if validation['eligible_label'] and label_key not in seen:
            seen.add(label_key)
            labels.append({**row, 'routes_sha256': attempt['routes_sha256'],
                           'attempt_key': key, 'teacher_method': method, 'teacher_seed': seed,
                           'objective_minutes': validation['search_objective_minutes']})

    for row in registry:
        for method in ('aco', 'tabu'):
            for seed in cfg['seeds']:
                collect(row, method, seed, cfg['seconds_per_method'])
        if cfg['fallback']['enabled'] and not any(
                label['instance_sha256'] == row['instance_sha256'] for label in labels):
            print(f"Respaldo independiente CP-SAT · n={row['n_customers']} · "
                  'sin etiquetas ACO/Tabú; no se leen certificados.', flush=True)
            for seed in cfg['fallback']['seeds']:
                collect(row, 'cp_sat_feasibility', seed, cfg['fallback']['seconds_per_attempt'])
    groups = {r['geometry_group_sha256'] for r in registry}
    labelled_instances = {label['instance_sha256'] for label in labels}
    missing_instances = [r for r in registry if r['instance_sha256'] not in labelled_instances]
    missing = sorted({r['geometry_group_sha256'] for r in missing_instances})
    costs = _cost_summary(attempts, groups)
    index = {'corpus_id': corpus_id, 'schema_version': SCHEMA,
             'status': 'no_labels_for_geometry' if missing else 'complete',
             'missing_geometry_groups': missing, 'protocol_sha256': p.digest(protocol),
             'missing_instance_sha256': [r['instance_sha256'] for r in missing_instances],
             'instances': registry, 'labels': labels, 'attempts': len(attempts),
             'teacher_costs_by_geometry': costs, 'bundles': inventory,
             'label_kind': 'heuristic_not_proven_optimal',
             'ready_for_confirmatory_comparison': False}
    if exp._recover(root / 'index', remote / 'index'):
        if exp.read_json(root / 'index/corpus.json') != index:
            raise ValueError('Índice de corpus distinto de los intentos recuperados.')
    else:
        exp.write_json(root / 'index/corpus.json', index)
        # Una fila por intento, incluidos errores, duplicados y rutas no elegibles.
        export_rows = [{**{k: v for k, v in a.items() if k != 'label_validation'},
                        **a['label_validation']} for a in attempts]
        exp.write_csv(root / 'index/teacher_attempts.csv', export_rows)
        exp.write_csv(root / 'index/teacher_costs_by_geometry.csv', costs)
        exp.write_csv(root / 'index/labels.csv', labels,
                      None if labels else ['instance_sha256', 'geometry_group_sha256', 'attempt_key'])
        exp.write_json(root / 'index/folds.json', folds)
        exp._commit(root / 'index', remote / 'index')
    if _registry(instances) != registry:
        raise AssertionError('El registro del banco cambió durante los profesores.')
    if missing:
        raise RuntimeError('Corpus sin etiquetas válidas para geometrías: ' + ', '.join(missing)
                           + '. Instancias: ' + ', '.join(str(r['n_customers']) for r in missing_instances)
                           + f'. Intentos y costos conservados en {remote}. '
                             'Revisar index/teacher_attempts.csv (reason, solver_termination, solver_error). '
                             'No se remuestrea ni se reutilizan certificados.')
    return root


def load_fold_examples(corpus_dir, instances, fold):
    """Retorna (train, validation, provenance), sin devolver rutas heldout.

    La verificación de integridad contempla todos los paquetes, pero sólo se
    abren result.json de grupos elegibles para entrenamiento o validación.
    """
    root = Path(corpus_dir)
    instances = list(instances)
    validate_fold(fold, instances)
    p.verify_bundle(root / 'index')
    meta_hash = p.verify_bundle(root / 'meta')
    index = exp.read_json(root / 'index/corpus.json')
    protocol = exp.read_json(root / 'meta/protocol.json')
    registry = _registry(instances)
    if (index.get('status') != 'complete' or index.get('instances') != registry
            or protocol.get('instances') != registry
            or index.get('corpus_id') != p.digest(protocol)
            or index.get('protocol_sha256') != p.digest(protocol)
            or index.get('bundles', {}).get('meta') != meta_hash
            or fold not in protocol.get('folds', [])):
        raise ValueError('Corpus incompleto, registro incompatible o fold ajeno al protocolo.')
    if protocol['evaluator_sha256'] != p.file_hash(p.__file__):
        raise ValueError('Evaluador del corpus distinto del módulo activo.')
    for key, expected in index['bundles'].items():
        relative = Path(key)
        if relative.is_absolute() or '..' in relative.parts:
            raise ValueError('Ruta de paquete inválida en corpus.')
        if p.verify_bundle(root / relative) != expected:
            raise ValueError('Paquete de profesores modificado.')
    by_sha = {x.sha: x for x in instances}
    groups = {'train': set(fold['train_geometry_groups']),
              'validation': set(fold['validation_geometry_groups'])}
    examples = {'train': [], 'validation': []}
    for label in index['labels']:
        role = next((role for role, selected in groups.items()
                     if label['geometry_group_sha256'] in selected), None)
        if role is None:
            continue
        key = label['attempt_key']
        if key not in index['bundles']:
            raise ValueError('Etiqueta sin intento sellado.')
        attempt = exp.read_json(root / key / 'attempt.json')
        result = exp.read_json(root / key / 'result.json')
        x = by_sha.get(label['instance_sha256'])
        if (x is None or label['instance_id'] != x.instance_id
                or label['geometry_group_sha256'] != x.provenance['geometry_group_sha256']
                or any(attempt.get(k) != label[k] for k in
                       ('instance_sha256', 'geometry_group_sha256', 'attempt_key',
                        'teacher_method', 'teacher_seed', 'routes_sha256'))
                or attempt['label_validation']['eligible_label'] is not True
                or p.digest(result['routes']) != label['routes_sha256']):
            raise ValueError('Identidad o validación de etiqueta incompatible.')
        p.validate_routes(x, result['routes'])
        examples[role].append({'instance_id': x.instance_id, 'instance_sha256': x.sha,
                               'geometry_group_sha256': label['geometry_group_sha256'],
                               'routes': result['routes'], 'teacher': {
                                   'method': label['teacher_method'], 'seed': label['teacher_seed'],
                                   'attempt_key': key, 'objective_minutes': label['objective_minutes'],
                                   'label_kind': 'heuristic_not_proven_optimal'}})
    for role in groups:
        if {e['geometry_group_sha256'] for e in examples[role]} != groups[role]:
            raise ValueError(f'Faltan etiquetas de una geometría del rol {role}.')
        expected_instances = {x.sha for x in instances
                              if x.provenance['geometry_group_sha256'] in groups[role]}
        if {e['instance_sha256'] for e in examples[role]} != expected_instances:
            raise ValueError(f'Faltan etiquetas de una instancia del rol {role}.')
    costs = index['teacher_costs_by_geometry']
    selected = groups['train'] | groups['validation']
    provenance = {'corpus_id': index['corpus_id'], 'fold_id': fold['fold_id'],
                  'bank_id': protocol['bank_id'],
                  'bank_manifest_sha256': protocol['bank_manifest_sha256'],
                  'evaluator_sha256': protocol['evaluator_sha256'],
                  'teacher_costs_by_geometry': costs,
                  'selected_teacher_costs': {key: sum(c[key] for c in costs
                                                     if c['geometry_group_sha256'] in selected)
                                             for key in _COSTS},
                  **{f'{role}_teacher_costs': {
                      key: sum(c[key] for c in costs if c['geometry_group_sha256'] in group_set)
                      for key in _COSTS} for role, group_set in groups.items()},
                  'all_teacher_costs': {key: sum(c[key] for c in costs) for key in _COSTS},
                  'train_examples': len(examples['train']),
                  'validation_examples': len(examples['validation']),
                  'heldout_labels_exposed': False,
                  'independent_unit': 'geometry_group_sha256',
                  'role': 'development_geometry_crossfit'}
    return examples['train'], examples['validation'], provenance


Overwriting svrplab/supervised_data.py


In [59]:
%%writefile svrplab/teacher_feasibility.py
"""Profesor de factibilidad independiente para 03; ejecutar sólo en Colab.

CP-SAT construye rutas desde cero sobre el mismo SearchProblem conservador.
No consulta certificados, solvers del certificador, escenarios ni rutas/hints
externos. No optimiza costo: OPTIMAL de CP-SAT en un modelo sin objetivo sólo
indica factibilidad. El caller registra tiempo y procedencia como costo docente.

La formulación sin índices de vehículo requiere que CADA capacidad pueda
transportar toda la demanda. Si no se cumple, devuelve unsupported_capacity;
no relaja ni agrega vehículos. AddMultipleCircuit exige que cada circuito pase
por el depósito, incluso cuando hay tiempos cero. Clientes de demanda cero son
obligatorios mediante self-loops falsos. Sobran vehículos: se conservan vacíos.

API primaria OR-Tools: https://or-tools.github.io/docs/pdoc/ortools/sat/python/cp_model.html
"""
from __future__ import annotations

from collections.abc import Mapping
import importlib.metadata
import math
from numbers import Integral, Real
import time


IMPLEMENTATION = 'project_independent_cp_sat_teacher_v1'
DEFAULTS = {'seconds': 600., 'workers': 1}
_TOL = 1e-9
_MAX_SAFE_INTEGER = 2**60


class _BudgetExpired(Exception):
    pass


def _deadline(deadline):
    if time.perf_counter() >= deadline:
        raise _BudgetExpired


def validate_config(config=None):
    if config is not None and not isinstance(config, Mapping):
        raise ValueError('La configuración del profesor CP-SAT debe ser un mapping.')
    unknown = set(config or {}) - set(DEFAULTS)
    if unknown:
        raise ValueError(f'Parámetros del profesor desconocidos: {sorted(unknown)}')
    result = DEFAULTS | dict(config or {})
    seconds = result['seconds']
    if (isinstance(seconds, bool) or not isinstance(seconds, Real)
            or not math.isfinite(float(seconds)) or seconds < 0):
        raise ValueError('seconds debe ser finito y no negativo.')
    workers = result['workers']
    if isinstance(workers, bool) or not isinstance(workers, Integral) or workers != 1:
        raise ValueError('El profesor reproducible requiere workers=1.')
    return {'seconds': float(seconds), 'workers': 1}


def _cp_sat():
    # Importar este módulo no carga OR-Tools ni inicia ningún solver.
    try:
        from ortools.sat.python import cp_model
    except ImportError as exc:
        raise RuntimeError('El profesor requiere OR-Tools instalado en Google Colab.') from exc
    return cp_model


def _integer_problem(problem, deadline):
    """Reconstruye la rejilla ya fijada; nunca redondea una matriz distinta."""
    _deadline(deadline)
    scale = problem.time_scale
    if isinstance(scale, bool) or not isinstance(scale, Integral) or scale < 1:
        raise ValueError('SearchProblem.time_scale debe ser entero positivo.')
    scale = int(scale)
    matrix = []
    for i in range(problem.n + 1):
        _deadline(deadline)
        row = []
        for j in range(problem.n + 1):
            value = float(problem.travel[i, j]) * scale
            if not math.isfinite(value) or value < 0 or value > _MAX_SAFE_INTEGER:
                raise ValueError('La matriz de búsqueda tiene tiempos inválidos.')
            integer = round(value)
            if not math.isclose(value, integer, rel_tol=0., abs_tol=1e-6):
                raise ValueError('La matriz de búsqueda no pertenece a su time_scale; no redondearla de nuevo.')
            if i == j and integer != 0:
                raise ValueError('La diagonal de viajes debe ser cero.')
            row.append(int(integer))
        matrix.append(row)
    lower, upper = [0], [0]
    horizon_value = float(problem.horizon) * scale
    if not math.isfinite(horizon_value) or not 0 <= horizon_value <= _MAX_SAFE_INTEGER:
        raise ValueError('Horizonte inválido para CP-SAT.')
    horizon = math.floor(horizon_value)
    upper[0] = horizon
    for customer in range(1, problem.n + 1):
        _deadline(deadline)
        start = float(problem.open[customer]) * scale
        end = float(problem.close[customer]) * scale
        if not all(math.isfinite(value) and 0 <= value <= _MAX_SAFE_INTEGER for value in (start, end)):
            raise ValueError('Ventanas inválidas para CP-SAT.')
        lower.append(math.ceil(start))
        upper.append(min(math.floor(end), horizon))
    return {'matrix': matrix, 'lower': lower, 'upper': upper, 'horizon': horizon, 'time_scale': scale}


def _build_model(problem, data, cp, deadline):
    model = cp.CpModel()
    times = {}
    for customer in range(1, problem.n + 1):
        _deadline(deadline)
        lo, hi = data['lower'][customer], data['upper'][customer]
        if lo > hi:
            model.add(False)
        times[customer] = model.new_int_var(min(lo, hi), hi, f'start_{customer}')
    edges, variables = [], {}
    # Registrar incluso clientes aislados. El literal falso impide omitirlos.
    for customer in range(1, problem.n + 1):
        edges.append((customer, customer, model.new_constant(0)))
    departures = []
    for origin in range(problem.n + 1):
        _deadline(deadline)
        for target in range(problem.n + 1):
            if origin == target:
                continue
            transit = data['matrix'][origin][target]
            earliest = 0 if origin == 0 else data['lower'][origin]
            latest = data['horizon'] if target == 0 else data['upper'][target]
            if earliest + transit > latest:
                continue
            selected = model.new_bool_var(f'edge_{origin}_{target}')
            edges.append((origin, target, selected))
            variables[origin, target] = selected
            if origin == 0:
                departures.append(selected)
                model.add(times[target] >= transit).only_enforce_if(selected)
            elif target == 0:
                model.add(times[origin] + transit <= data['horizon']).only_enforce_if(selected)
            else:
                model.add(times[target] >= times[origin] + transit).only_enforce_if(selected)
    # No objective and no hints. The circuit constraint excludes subtours even
    # when all their arc durations are zero; time inequalities alone would not.
    model.add_multiple_circuit(edges)
    model.add(sum(departures) <= problem.vehicles)
    _deadline(deadline)
    return model, variables


def _extract_routes(problem, selected):
    departures = sorted(target for origin, target in selected if origin == 0)
    if len(departures) > problem.vehicles:
        raise ValueError('CP-SAT excedió la flota fija.')
    successors = {}
    for origin, target in selected:
        if origin:
            if origin in successors:
                raise ValueError('CP-SAT devolvió dos sucesores para un cliente.')
            successors[origin] = target
    seen, routes = set(), []
    for first in departures:
        route, customer = [], first
        while customer:
            if customer in seen or customer not in successors or not 1 <= customer <= problem.n:
                raise ValueError('La extracción encontró ciclo desconectado o cliente inválido.')
            seen.add(customer)
            route.append(customer)
            customer = successors[customer]
        routes.append(route)
    if seen != set(range(1, problem.n + 1)):
        raise ValueError('La solución CP-SAT no cubre todos los clientes.')
    return routes + [[] for _ in range(problem.vehicles - len(routes))]


def _finish(problem, routes, status, started, config, metadata, history):
    validation_started = time.perf_counter()
    result = problem.result(routes, status, validation_started - started, history, **metadata)
    finished = time.perf_counter()
    result['solve_seconds'] = finished - started
    result['metadata']['validation_seconds'] += finished - validation_started
    result['metadata']['budget_overrun_seconds'] = max(0., finished - started - config['seconds'])
    return result


def solve_feasibility(problem, seed, config=None):
    """Obtiene una etiqueta independiente, sin optimizar costo ni usar certificados.

    El presupuesto único incluye importación, preparación, modelo, solver,
    extracción y validación. No interrumpe la operación nativa en curso; deja
    constancia de cualquier exceso. UNKNOWN no prueba infactibilidad.
    """
    started = time.perf_counter()
    effective = validate_config(config)
    if isinstance(seed, bool) or not isinstance(seed, Integral) or seed < 0:
        raise ValueError('seed debe ser entero no negativo.')
    seed = int(seed)
    deadline = started + effective['seconds']
    metadata = {
        'method': 'cp_sat_feasibility', 'seed': seed, 'cp_sat_seed': seed % (2**31 - 1),
        'implementation': IMPLEMENTATION, 'effective_config': effective,
        'construction': 'independent_from_scratch_without_certificates_or_hints',
        'feasibility_only': True, 'surrogate_optimal': False, 'stochastic_optimal': False,
        'cost_optimization_performed': False, 'best_bound': None,
        'cp_sat_status': None, 'cp_sat_status_name': 'NOT_RUN', 'solver_version': None,
        'termination': 'not_started', 'search_solver_seconds': 0., 'model_seconds': 0.,
        'import_seconds': 0., 'validation_seconds': 0., 'response_stats': None,
        'time_scale': int(problem.time_scale), 'capacity_scope': 'nonbinding_for_each_vehicle',
        'conservative_integer_infeasible': False,
        'infeasibility_scope': 'conservative_integer_model_only_not_original_stochastic_problem',
        'optimal_status_scope': 'satisfaction_of_feasibility_constraints_not_minimum_route_cost',
        'budget_includes_imports': True, 'budget_includes_construction': True,
        'budget_includes_extraction_and_validation': True,
        'time_limit_granularity': 'model_row_or_native_solver_call_or_validation',
        'seed_reproducibility': 'same_ortools_environment_and_one_worker; wall_time_cutoff_may_differ',
    }
    routes, history, status = None, [], 'no_solution'
    try:
        _deadline(deadline)
        total_demand = math.fsum(float(value) for value in problem.demands[1:])
        if (not math.isfinite(total_demand) or total_demand < 0
                or any(not math.isfinite(float(value)) or float(value) < 0 for value in problem.demands)):
            raise ValueError('Demandas inválidas.')
        if any(not math.isfinite(float(capacity)) or float(capacity) < 0 for capacity in problem.capacities):
            raise ValueError('Capacidades inválidas.')
        if problem.vehicles < 1 or any(float(capacity) + _TOL < total_demand for capacity in problem.capacities):
            metadata['termination'] = 'unsupported_capacity'
            return _finish(problem, None, 'unsupported_capacity', started, effective, metadata, history)
        model_started = time.perf_counter()
        try:
            data = _integer_problem(problem, deadline)
        finally:
            metadata['model_seconds'] += time.perf_counter() - model_started
        if problem.n == 0:
            routes = [[] for _ in range(problem.vehicles)]
            metadata['termination'] = 'empty_instance_feasible'
            return _finish(problem, routes, 'feasible', started, effective, metadata, history)
        import_started = time.perf_counter()
        cp = _cp_sat()
        metadata['solver_version'] = importlib.metadata.version('ortools')
        metadata['import_seconds'] = time.perf_counter() - import_started
        _deadline(deadline)
        model_started = time.perf_counter()
        try:
            model, variables = _build_model(problem, data, cp, deadline)
            model_error = model.validate()
        finally:
            metadata['model_seconds'] += time.perf_counter() - model_started
        if model_error:
            metadata.update(termination='model_invalid', model_validation_error=model_error,
                            cp_sat_status_name='MODEL_INVALID')
            return _finish(problem, None, 'model_invalid', started, effective, metadata, history)
        solver = cp.CpSolver()
        solver.parameters.num_search_workers = effective['workers']
        solver.parameters.random_seed = metadata['cp_sat_seed']
        solver.parameters.stop_after_first_solution = True
        _deadline(deadline)
        solver.parameters.max_time_in_seconds = max(0., deadline - time.perf_counter())
        # A zero native limit can mean no limit in solver APIs: never invoke it.
        if solver.parameters.max_time_in_seconds <= 0:
            raise _BudgetExpired
        solver_started = time.perf_counter()
        result_code = solver.solve(model)
        metadata.update(search_solver_seconds=time.perf_counter() - solver_started,
                        cp_sat_status=int(result_code), cp_sat_status_name=solver.status_name(result_code),
                        response_stats=solver.response_stats(), model_arc_count=len(variables))
        if result_code in (cp.FEASIBLE, cp.OPTIMAL):
            validation_started = time.perf_counter()
            selected = [arc for arc, variable in variables.items() if solver.boolean_value(variable)]
            candidate = _extract_routes(problem, selected)
            violation, cost = problem.score(candidate)
            if not math.isfinite(violation) or not math.isfinite(cost) or violation > _TOL:
                raise ValueError('La etiqueta CP-SAT no pasó la validación conservadora común.')
            metadata['validation_seconds'] += time.perf_counter() - validation_started
            routes, status = candidate, 'feasible'
            metadata['termination'] = 'feasible_found'
            history.append({'phase': 'independent_feasibility_teacher',
                            'elapsed_seconds': time.perf_counter() - started,
                            'violation_minutes': float(violation), 'travel_minutes': float(cost)})
        elif result_code == cp.INFEASIBLE:
            metadata.update(termination='conservative_integer_infeasible', conservative_integer_infeasible=True)
        elif result_code == cp.MODEL_INVALID:
            status, metadata['termination'] = 'model_invalid', 'model_invalid'
        else:
            metadata['termination'] = 'unknown_no_feasible_solution'
    except _BudgetExpired:
        metadata['termination'] = 'time_limit_before_search'
    return _finish(problem, routes, status, started, effective, metadata, history)


Overwriting svrplab/teacher_feasibility.py


In [60]:
%%writefile tests/test_final_experiment.py
"""Contratos de 10. Ejecutar exclusivamente en Google Colab; localmente sólo AST."""
from copy import deepcopy
import csv
import json
from pathlib import Path
import sys

import pytest
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import final_experiment as f
from svrplab import pilot as p
from svrplab import experiments as e
from svrplab import reinforcement as rl


def test_test_is_closed_before_any_model_or_solver_import(monkeypatch, tmp_path):
    monkeypatch.setattr(f, '_colab', lambda: None)
    with pytest.raises(ValueError, match='Test cerrado'):
        f.run_test(tmp_path, tmp_path / 'remote', [], tmp_path / 'claims', open_test=False)
    assert not list(tmp_path.rglob('exposure.json'))


def test_resource_plan_counts_geometries_seeds_methods_and_scenarios():
    registry = [{'role': 'test', 'n_customers': 50}, {'role': 'test', 'n_customers': 100},
                {'role': 'train', 'n_customers': 50}]
    config = {'methods': {'a': {}, 'b': {}}, 'solver_seeds': [11, 29, 47], 'policies': ['fixed_order', 'causal_edd'],
              'search_seconds': 120., 'scenarios': {'evaluation_count': 1000},
              'training': {'a': {'training': {'seconds_per_model': 3600}}},
              'resource_limits': {'maximum_planned_search_hours': 10, 'maximum_planned_training_hours': 10,
                                  'maximum_scenario_rows': 30000}}
    result = f.resource_plan(registry, config)
    assert result['solver_attempts'] == 12
    assert result['scenario_metric_rows_upper_bound'] == 24000
    assert result['planned_training_budget_hours'] == 3
    assert result['passes_declared_resource_limits'] is True


def test_final_gate_allows_imperfect_coverage_but_requires_all_size_strata():
    no_stratum = {'instances_with_completion_fraction': .5, 'complete_fraction': .2, 'checkpoint_eligible': False}
    assert not rl.validation_candidate_is_better(.5, no_stratum, None, None, .001)
    all_sizes = {**no_stratum, 'checkpoint_eligible': True}
    assert rl.validation_candidate_is_better(.5, all_sizes, None, None, .001)
    more_coverage = {**all_sizes, 'instances_with_completion_fraction': .6, 'complete_fraction': .1}
    assert rl.validation_candidate_is_better(.9, more_coverage, .5, all_sizes, .001)


def _export_fixture(tmp_path):
    """Datos manuales de contratos; ningún solver, entrenamiento o escenario."""
    root, remote = tmp_path / 'final10_manual', tmp_path / 'remote'
    registry = [{'instance_sha256': p.digest([role, 'instance']), 'instance_id': role,
                 'geometry_group_sha256': p.digest([role, 'geometry']), 'n_customers': 2,
                 'role': role, 'ticket_id': role + '_ticket'} for role in ['train', 'validation', 'test']]
    config = {'methods': {'hbg_facs': {}, 'hbg_untrained': {}, 'hbg_no_pheromone': {}},
              'solver_seeds': [11], 'policies': ['fixed_order'], 'training': {'hbg_facs': {}},
              'training_geometry_minimum_observations': 1,
              'scenarios': {'export': 'scenario_metrics_csv_gzip_all_scenarios_with_fixed_trace_audit'}}
    population = {'status': 'nominal_population_accepted'}
    population['population_sha256'] = p.digest(population)
    protocol = {'config': config, 'design': {'seed_ids': [11], 'sizes': [2]},
                'registry': registry, 'registry_sha256': p.digest(registry),
                'population_sha256': population['population_sha256'],
                'population_manifest': {'excluded_development_groups': []},
                'source_sha256': {'manual.py': 'a' * 64}, 'analysis_source_sha256': 'b' * 64,
                'analysis_r_sha256': 'b' * 64, 'analysis_config_sha256': 'c' * 64,
                'sealed_at': '2026-01-01T00:00:00+00:00'}
    e.write_json(root / 'protocol/protocol.json', protocol)
    e.write_json(root / 'protocol/population.json', population)
    fold = f._holdout_fold(protocol)
    summary = {'fold': fold, 'heldout_used_for_training_or_selection': False,
               'selection_status': 'selected_complete_validation',
               'selected_checkpoint_training_geometry_visits': {fold['train_geometry_groups'][0]: 1},
               'training_geometry_visits': {fold['train_geometry_groups'][0]: 1},
               'best_validation_diagnostics': {'checkpoint_eligible': True, 'sizes_with_complete_geometry': {'2': True}}}
    freeze = {'frozen_at': '2026-01-01T01:00:00+00:00',
              'protocol_sha256': p.file_hash(root / 'protocol/protocol.json'),
              'population_sha256': population['population_sha256'], 'source_sha256': protocol['source_sha256'],
              'analysis_source_sha256': 'b' * 64, 'analysis_r_sha256': 'b' * 64,
              'all_training_geometries_observed': True, 'heldout_in_training_or_selection': False,
              'models': [{'method': 'hbg_facs', 'seed': 11, 'summary': summary}]}
    e.write_json(root / 'freeze/freeze.json', freeze)
    exposure = {'opened_at': '2026-01-01T02:00:00+00:00',
                'population_sha256': population['population_sha256'],
                'protocol_sha256': p.file_hash(root / 'protocol/protocol.json'),
                'freeze_sha256': p.file_hash(root / 'freeze/freeze.json')}
    e.write_json(root / 'test/exposure.json', exposure)
    rows, artifacts = [], {}
    entry = registry[-1]
    for method in config['methods']:
        relative = Path('attempts') / entry['instance_sha256'] / method / 'seed_11'
        solution = {'routes': None, 'status': 'no_solution', 'structural_valid': False, 'conservative_feasible': False}
        e.write_json(root / relative / 'solution/solution.json', solution)
        e.write_json(root / relative / 'solution/environment.json', {'manual_fixture': True})
        artifacts[str(relative / 'solution')] = p.seal_bundle(root / relative / 'solution')
        row = {**{name: entry[name] for name in ['instance_sha256', 'geometry_group_sha256', 'n_customers']},
               'method': method, 'solver_seed': 11, 'policy': 'fixed_order', 'artifact_status': 'observed',
               'structural_valid': False, 'conservative_feasible': False, 'solver_status': 'no_solution', 'evaluation_status': 'no_solution',
               'scenario_success_per_scheduled_attempt': 0.}
        rows.append(row)
        e.write_json(root / relative / 'fixed_order/summary.json', row)
        e.write_json(root / relative / 'fixed_order/environment.json', {'manual_fixture': True})
        artifacts[str(relative / 'fixed_order')] = p.seal_bundle(root / relative / 'fixed_order')
    report = f._export(root, remote, protocol, freeze, exposure, rows, artifacts)
    return root, report


def test_verified_export_keeps_observed_failures_and_frozen_analysis_identity(tmp_path):
    root, report = _export_fixture(tmp_path)
    verified = f.verify_analysis_export(report, p.file_hash(report / 'CHECKSUMS.json'))
    assert verified['analysis_r_sha256'] == 'b' * 64
    assert len(verified['registry']) == 1
    assert verified['source_chain']['test']['observed_policy_rows'] == 3


@pytest.mark.parametrize('damage', ['csv_flag', 'missing_bundle', 'wrong_external_hash'])
def test_export_verifier_rejects_unbacked_flags_missing_artifacts_and_changed_identity(tmp_path, damage):
    root, report = _export_fixture(tmp_path)
    expected = p.file_hash(report / 'CHECKSUMS.json')
    if damage == 'csv_flag':
        rows = f._csv(report / 'attempts.csv')
        rows[0]['conservative_feasible'] = 'True'
        e.write_csv(report / 'attempts.csv', rows)
        p.seal_bundle(report)
        expected = p.file_hash(report / 'CHECKSUMS.json')
    elif damage == 'missing_bundle':
        target = next((root / 'attempts').glob('*/hbg_facs/seed_11/solution/solution.json'))
        target.unlink()
    else:
        expected = 'f' * 64
    with pytest.raises((ValueError, FileNotFoundError)):
        f.verify_analysis_export(report, expected)


@pytest.mark.parametrize('rejected', [False, True])
def test_fresh_runtime_restores_model_metadata_and_rejected_epoch_before_freeze_check(tmp_path, monkeypatch, rejected):
    from types import SimpleNamespace
    from svrplab import hybrid, reinforcement
    local, remote = tmp_path / 'fresh', tmp_path / 'drive'
    relative = Path('training/hbg_facs/seed_11/final')
    checkpoint = relative.parent / 'epoch_0001/state.pt' if rejected else relative / 'best.pt'
    e.write_json(remote / relative / 'summary.json', {'manual_fixture': True})
    if not rejected:
        (remote / checkpoint).write_bytes(b'manual checkpoint bytes for recovery contract')
    p.seal_bundle(remote / relative)
    e.write_json(remote / relative.parent / 'meta/identity.json', {'manual_identity': True})
    p.seal_bundle(remote / relative.parent / 'meta')
    if rejected:
        (remote / checkpoint).parent.mkdir(parents=True)
        (remote / checkpoint).write_bytes(b'manual rejected epoch bytes')
        p.seal_bundle((remote / checkpoint).parent)
    e.write_json(remote / 'freeze/freeze.json', {'models': [{'relative_directory': str(relative),
                                                           'checkpoint_relative_path': str(checkpoint)}]})
    p.seal_bundle(remote / 'freeze')
    registry = [{'instance_sha256': role, 'geometry_group_sha256': p.digest(role), 'role': role}
                for role in ['train', 'validation', 'test']]
    protocol = {'config': {'require_cuda': False}, 'registry': registry}
    monkeypatch.setattr(f, '_colab', lambda: None)
    monkeypatch.setattr(f, '_protocol', lambda root: protocol)

    def verify_recovered(root, settings):
        assert p.verify_bundle(root / relative)
        assert p.verify_bundle(root / relative.parent / 'meta')
        assert f._read(root / relative.parent / 'meta/identity.json') == {'manual_identity': True}
        assert (root / checkpoint).read_bytes() == (remote / checkpoint).read_bytes()

    def forbidden(*args, **kwargs):
        raise AssertionError('Un freeze recuperado no debe volver a entrenar.')

    monkeypatch.setattr(f, '_verify_freeze', verify_recovered)
    monkeypatch.setattr(hybrid, 'train_fold', forbidden)
    monkeypatch.setattr(reinforcement, 'train_fold', forbidden)
    result = f.train_and_freeze(local, remote, [SimpleNamespace(sha=row['instance_sha256']) for row in registry], device='cpu')
    assert result == local / 'freeze'


def test_finished_report_resume_verifies_remote_siblings_without_copying_attempts(tmp_path, monkeypatch):
    from types import SimpleNamespace
    original, report = _export_fixture(tmp_path)
    fresh = tmp_path / 'clean_runtime'
    protocol = f._read(original / 'protocol/protocol.json')
    protocol['config']['require_cuda'] = False
    freeze = f._read(original / 'freeze/freeze.json')
    monkeypatch.setattr(f, '_colab', lambda: None)
    monkeypatch.setattr(f, '_protocol', lambda root: protocol)
    monkeypatch.setattr(f, '_verify_freeze', lambda *args: freeze)
    monkeypatch.setattr(f, '_claim_test', lambda *args: f._read(original / 'test/exposure.json'))
    monkeypatch.setattr(f, '_make_solvers', lambda *args: pytest.fail('No reconstruir solvers de un reporte terminado.'))
    result = f.run_test(fresh, original, [SimpleNamespace(sha=row['instance_sha256']) for row in protocol['registry']],
                        tmp_path / 'claims', device='cpu', gurobi_env=object(), open_test=True)
    assert result == report
    assert not (fresh / 'attempts').exists()
    assert (fresh / 'report/CHECKSUMS.json').is_file()


@pytest.mark.parametrize('status,returncode', [('passed', 0), ('failed', 1)])
def test_contract_validation_resume_preserves_original_bytes_and_failure(tmp_path, status, returncode):
    local, remote = tmp_path / 'fresh/checks', tmp_path / 'drive/checks'
    sources = {'manual.py': 'a' * 64}
    e.write_json(remote / 'validation.json', {'status': status, 'returncode': returncode, 'source_sha256': sources})
    (remote / 'pytest.txt').write_text('original pytest report with original duration\n')
    original = p.seal_bundle(remote)
    if status == 'failed':
        with pytest.raises(ValueError, match='previa falló'):
            f.recover_contract_validation(local, remote, sources)
    else:
        assert f.recover_contract_validation(local, remote, sources) is True
    assert p.verify_bundle(local) == p.verify_bundle(remote) == original
    assert (local / 'pytest.txt').read_bytes() == (remote / 'pytest.txt').read_bytes()


def test_null_validity_flags_rejected_even_when_csv_and_all_bundles_agree(tmp_path):
    root, report = _export_fixture(tmp_path)
    rows = f._csv(report / 'attempts.csv')
    row = rows[0]
    relative = Path('attempts') / row['instance_sha256'] / row['method'] / f'seed_{row["solver_seed"]}'
    test = f._read(report / 'source_manifests/test.json')
    for part, filename in [('solution', 'solution.json'), (row['policy'], 'summary.json')]:
        directory = root / relative / part
        payload = f._read(directory / filename)
        payload['structural_valid'] = None
        e.write_json(directory / filename, payload)
        test['artifact_bundle_sha256'][str(relative / part)] = p.seal_bundle(directory)
    rows[0]['structural_valid'] = ''
    e.write_csv(report / 'attempts.csv', rows)
    e.write_json(report / 'source_manifests/test.json', test)
    for filename in ['analysis_export_contract.json', 'final_release.json']:
        payload = f._read(report / filename)
        payload['source_manifest_sha256']['test'] = p.file_hash(report / 'source_manifests/test.json')
        e.write_json(report / filename, payload)
    p.seal_bundle(report)
    with pytest.raises(ValueError, match='Flags de validez'):
        f.verify_analysis_export(report, p.file_hash(report / 'CHECKSUMS.json'))


Overwriting tests/test_final_experiment.py


In [61]:
%%writefile tests/test_hybrid.py
"""Pruebas del entrenamiento 05: ejecutar exclusivamente en Google Colab.

Las geometrías son fixtures manuales. Los dobles diferenciables aíslan el
protocolo de entrenamiento y consumen RNG de Python, NumPy y Torch para probar
la reanudación. Las trayectorias y gradientes reales se prueban en test_hbg.py.
No ejecutar estas pruebas ni sus fixtures localmente.
"""
from copy import deepcopy
from dataclasses import asdict
from pathlib import Path
import random
import sys

import numpy as np
import pytest

torch = pytest.importorskip('torch')
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import experiments as e
from svrplab import pilot as p
from svrplab import hybrid as r
from svrplab import supervised_data as sd
from svrplab.search import OBJECTIVE, SearchProblem


def manual_instances():
    instances = []
    for i in range(3):
        xy = [[0, 0], [i + 1, 0], [0, i + 1]]
        instances.append(p.freeze_instance({
            'instance_id': f'hybrid_manual_{i}', 'depot_indices': [0],
            'locations': xy, 'demands': [0, 0, 1], 'capacities': [2, 2],
            'windows': [[0, 100], [0, 100], [0, 100]], 'appear_times': [0, 0, 0],
            'provenance': {'kind': 'manual_test',
                           'geometry_group_sha256': p.digest({'locations': xy})},
        }))
    return instances


def config():
    return {
        'schema_version': 'hybrid-05-v1', 'role': 'development_only',
        'replicate_id': 'manual_test',
        'split_strategy': 'leave_one_geometry_out_with_inner_validation',
        'time_scale': 1000, 'objective': OBJECTIVE,
        'model': {'model_dim': 16, 'num_heads': 2, 'num_layers': 1,
                  'dropout': 0., 'tanh_clip': 10.},
        'hbg': {'ants': 4, 'iterations': 2, 'alpha': 1., 'evaporation': .1,
                'temperature': 1., 'lambda_db': .5},
        'replay': {'capacity_per_instance': 32, 'samples_per_step': 2},
        'training': {'epochs': 2, 'steps_per_epoch': 2, 'gradient_accumulation': 1,
                     'learning_rate': .001, 'weight_decay': 0., 'gradient_clip_norm': 1.,
                     'patience': 5, 'min_delta': 0., 'seconds_per_model': 120.,
                     'permute_nodes': True, 'deterministic_algorithms': True},
    }


def colony_stub(model, problem, device, rng, params, deadline=None):
    # Trayectorias completas del propio generador de prueba; no etiquetas externas.
    scale = max(1., (problem.n + problem.vehicles) * float(np.max(problem.travel)))
    records = []
    for _ in range(params['ants'] * params['iterations']):
        route = [1, 2] if rng.random() < .5 else [2, 1]
        routes = [route, []]
        cost = problem.score(routes)[1] / scale
        records.append({'format_version': 1, 'instance_sha': problem.x.sha,
                        'actions': route + [0, 0], 'termination': 'complete',
                        'routes': routes, 'cost': cost, 'log_reward': -cost / params['temperature'],
                        'unserved': 0, 'temperature': params['temperature'],
                        'normalization_scale_minutes': scale})
    return records, {'mean_cost': float(np.mean([item['cost'] for item in records])),
                     'complete_fraction': 1., 'rollout_count': len(records),
                     'decisions': sum(len(item['actions']) for item in records)}


def trajectory_loss_stub(model, problem, device, record, params, node_order=None):
    assert model.training and torch.is_grad_enabled()
    assert record['instance_sha'] == problem.x.sha
    parameter = next(model.parameters())
    noise = torch.rand((), device=device) + random.random() + record['cost']
    loss = (parameter + noise).square().mean()
    value = float(loss.detach().cpu())
    return loss, {'mean_cost': record['cost'], 'complete': True,
                  'decisions': len(record['actions']), 'tb_loss': value, 'db_loss': value,
                  'penalized_cost_minutes': record['cost'] * record['normalization_scale_minutes'],
                  'logZ': 0., 'logPF': -1., 'log_reward': record['log_reward']}


def validation_stub(model, problem, device, params, seed=0):
    assert not model.training and not torch.is_grad_enabled()
    score = .5 + float(next(model.parameters()).square().mean().detach().cpu())
    return {'score': score, 'complete_fraction': 1., 'best_complete': True, 'rollout_count': params['ants'] * params['iterations']}


@pytest.fixture
def setup(monkeypatch):
    # El notebook que invoca pytest ya exige Colab; se aísla la guardia interna.
    monkeypatch.setattr(r, 'require_colab', lambda: None)
    monkeypatch.setattr(p, 'environment_report', lambda: {'test_fixture': True})
    monkeypatch.setattr(r, 'save_learning_plot', lambda *args: None)
    monkeypatch.setattr(r, 'collect_colony', colony_stub)
    monkeypatch.setattr(r, 'trajectory_loss', trajectory_loss_stub)
    monkeypatch.setattr(r, 'validation_score', validation_stub)

    def forbidden(*args, **kwargs):
        raise AssertionError('05 no debe abrir profesores, etiquetas ni corpus de 03.')

    monkeypatch.setattr(sd, 'load_fold_examples', forbidden)
    monkeypatch.setattr(sd, 'build_teacher_corpus', forbidden)
    instances = manual_instances()
    return instances, sd.make_geometry_folds(instances)[0]


def train(tmp_path, instances, fold, label='run', settings=None):
    return r.train_fold(instances, p.Traffic(mode='no_noise'), settings or config(),
                        11, fold, tmp_path / label / 'local',
                        tmp_path / label / 'drive', 'cpu')


def crossfit(tmp_path, instances, settings=None, seeds=None):
    return r.train_crossfit(instances, p.Traffic(mode='no_noise'), settings or config(),
                            [11] if seeds is None else seeds,
                            tmp_path / 'work', tmp_path / 'drive', 'manual_bank',
                            'a' * 64, p.file_hash(p.__file__), device='cpu')


@pytest.mark.parametrize('key,value', [
    ('epochs', 0), ('steps_per_epoch', True), ('gradient_accumulation', 0),
    ('patience', -1), ('learning_rate', 0), ('gradient_clip_norm', float('nan')),
    ('seconds_per_model', float('inf')), ('weight_decay', -1),
    ('min_delta', True), ('permute_nodes', 1), ('deterministic_algorithms', 'yes'),
])
def test_training_configuration_rejects_invalid_settings(key, value):
    settings = config()
    settings['training'][key] = value
    with pytest.raises(ValueError):
        r.validate_training_config(settings)


@pytest.mark.parametrize('key,value', [
    ('schema_version', 'other'), ('role', 'final_test'),
    ('split_strategy', 'random_rows'), ('objective', 'mean_evaluation_cost'),
    ('time_scale', 0), ('time_scale', True), ('replicate_id', ''),
])
def test_training_configuration_rejects_other_protocols(key, value):
    with pytest.raises(ValueError):
        r.validate_training_config(config() | {key: value})


@pytest.mark.parametrize('section,key,value', [
    ('hbg', 'ants', 0), ('hbg', 'iterations', True),
    ('hbg', 'temperature', 0), ('hbg', 'lambda_db', -1),
    ('replay', 'capacity_per_instance', 0), ('replay', 'samples_per_step', True),
    ('replay', 'samples_per_step', -1), ('replay', 'samples_per_step', 33),
    ('model', 'dropout', .1), ('model', 'model_dim', 0),
])
def test_training_configuration_rejects_invalid_policy_or_rollouts(section, key, value):
    settings = config()
    settings[section][key] = value
    with pytest.raises(ValueError):
        r.validate_training_config(settings)


@pytest.mark.parametrize('seeds', [[], [11, 11], [-1], [True]])
def test_crossfit_rejects_invalid_seeds_before_training(tmp_path, monkeypatch, setup, seeds):
    instances, _ = setup

    def forbidden(*args, **kwargs):
        raise AssertionError('Semillas inválidas deben rechazarse antes de entrenar.')

    monkeypatch.setattr(r, 'train_fold', forbidden)
    with pytest.raises(ValueError):
        crossfit(tmp_path, instances, seeds=seeds)


def test_validation_weights_geometry_equally_and_excludes_gradient_graphs(monkeypatch):
    model = torch.nn.Linear(1, 1).train()
    grouped = {'a': ['a1'], 'b': ['b1', 'b2', 'b3']}
    calls = []

    def score_stub(model, problem, device, params, seed=0):
        assert not model.training and not torch.is_grad_enabled()
        calls.append(problem)
        return {'score': float(problem), 'complete_fraction': 1., 'best_complete': True, 'rollout_count': 8}

    monkeypatch.setattr(r, 'validation_score', score_stub)
    mean, by_group = r.validation_cost(model, {'a1': 2, 'b1': 4, 'b2': 6, 'b3': 8},
                                       grouped, ['a', 'b'], 'cpu', config()['hbg'])
    assert by_group == {'a': 2., 'b': 6.}
    assert mean == 4. and mean != np.mean(calls)


def test_train_fold_uses_only_authorized_geometries_and_final_resume_is_immutable(tmp_path, monkeypatch, setup):
    instances, fold = setup
    real_problem = r.SearchProblem
    seen = set()
    real_sample = r.sample_replay
    replay_sizes_before_sampling = []

    def guarded(instance, *args, **kwargs):
        assert instance.provenance['geometry_group_sha256'] not in fold['heldout_geometry_groups']
        seen.add(instance.sha)
        return real_problem(instance, *args, **kwargs)

    monkeypatch.setattr(r, 'SearchProblem', guarded)

    def observe_replay(records, count, rng):
        replay_sizes_before_sampling.append(len(records))
        return real_sample(records, count, rng)

    monkeypatch.setattr(r, 'sample_replay', observe_replay)
    final = train(tmp_path, instances, fold)
    before = p.verify_bundle(final)
    summary = e.read_json(final / 'summary.json')
    authorized = {x.sha for x in instances
                  if x.provenance['geometry_group_sha256'] not in fold['heldout_geometry_groups']}
    assert seen == authorized
    assert summary['heldout_used_for_training_or_selection'] is False
    assert summary['epochs_completed'] == 2 and summary['teacher_seconds'] == 0
    assert 'corpus_id' not in summary
    assert summary['training_seconds'] >= summary['validation_seconds'] >= 0
    assert summary['model_checkpoint_sha256'] == p.file_hash(final / 'best.pt')
    assert p.verify_bundle(tmp_path / 'run/drive/final') == before
    history = e.read_json(final / 'history.json')
    assert all({'train_loss', 'train_cost', 'train_complete_fraction', 'validation_score',
                'train_tb_loss', 'train_db_loss', 'replay_samples', 'replay_entries'} <= set(row)
               for row in history)
    assert replay_sizes_before_sampling[0] == 0
    assert any(size > 0 for size in replay_sizes_before_sampling[1:])
    assert summary['replay_samples'] == sum(row['replay_samples'] for row in history) > 0
    assert summary['best_validation_score'] == min(row['validation_score'] for row in history)
    latest = r._load_checkpoint(final.parent / 'epoch_0002/state.pt')
    assert latest['optimizer']['state'] and latest['cuda_rng'] == []
    assert latest['torch_rng'].dtype == torch.uint8
    replay = e.read_json(final.parent / 'epoch_0002/progress.json')['replay']
    assert e.read_json(final / 'replay.json') == replay
    train_shas = {x.sha for x in instances
                  if x.provenance['geometry_group_sha256'] in fold['train_geometry_groups']}
    assert set(replay) == train_shas
    for sha, records in replay.items():
        assert 0 < len(records) <= config()['replay']['capacity_per_instance']
        assert all(record['instance_sha'] == sha and record['termination'] == 'complete'
                   for record in records)

    def forbidden(*args, **kwargs):
        raise AssertionError('Un final recuperado no debe construir problemas ni volver a entrenar.')

    monkeypatch.setattr(r, 'SearchProblem', forbidden)
    assert train(tmp_path, instances, fold) == final
    assert p.verify_bundle(final) == before
    with (final / 'best.pt').open('ab') as stream:
        stream.write(b'corruption')
    with pytest.raises(ValueError):
        train(tmp_path, instances, fold)


def test_overlapping_fold_is_rejected_before_constructing_problems(tmp_path, monkeypatch, setup):
    instances, fold = setup
    damaged = deepcopy(fold)
    damaged['train_geometry_groups'].append(damaged['heldout_geometry_groups'][0])

    def forbidden(*args, **kwargs):
        raise AssertionError('Rechazar roles superpuestos antes de construir problemas.')

    monkeypatch.setattr(r, 'SearchProblem', forbidden)
    with pytest.raises(ValueError):
        train(tmp_path, instances, damaged)


def test_training_budget_exhausted_by_setup_never_exports_random_weights(tmp_path, monkeypatch, setup):
    instances, fold = setup
    settings = config()
    settings['training']['seconds_per_model'] = 1e-12

    def forbidden(*args, **kwargs):
        raise AssertionError('Sin presupuesto no se inicia un batch ni validación.')

    monkeypatch.setattr(r, 'collect_colony', forbidden)
    monkeypatch.setattr(r, 'trajectory_loss', forbidden)
    monkeypatch.setattr(r, 'validation_score', forbidden)
    with pytest.raises(RuntimeError, match='No hay checkpoint entrenado'):
        train(tmp_path, instances, fold, settings=settings)
    assert not list(tmp_path.glob('**/final/best.pt'))


def test_changing_heldout_windows_cannot_change_training_inputs_or_weights(tmp_path, setup):
    instances, fold = setup
    first = train(tmp_path, instances, fold, 'first')
    changed = []
    for instance in instances:
        if instance.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups']:
            payload = instance.payload()
            payload['windows'][1][0] = 20
            changed.append(p.freeze_instance(payload))
        else:
            changed.append(instance)
    second = train(tmp_path, changed, fold, 'second')
    expected = r._load_checkpoint(first / 'best.pt')
    actual = r._load_checkpoint(second / 'best.pt')
    assert expected.keys() == actual.keys()
    assert all(torch.equal(expected[key], actual[key]) for key in expected)
    assert e.read_json(first / 'summary.json')['identity_sha256'] == e.read_json(second / 'summary.json')['identity_sha256']


def test_partial_resume_restores_optimizer_and_all_rng_without_repeating_epoch(tmp_path, monkeypatch, setup):
    instances, fold = setup
    reference = train(tmp_path, instances, fold, 'reference')
    real_commit = e._commit

    def interrupt_after_epoch_one(local, remote):
        real_commit(local, remote)
        if Path(local).name == 'epoch_0001':
            raise KeyboardInterrupt('Desconexión simulada después del checkpoint completo.')

    monkeypatch.setattr(e, '_commit', interrupt_after_epoch_one)
    with pytest.raises(KeyboardInterrupt):
        train(tmp_path, instances, fold, 'interrupted')
    first_epoch = tmp_path / 'interrupted/local/epoch_0001'
    original_checksum = p.verify_bundle(first_epoch)
    monkeypatch.setattr(e, '_commit', real_commit)
    resumed = train(tmp_path, instances, fold, 'interrupted')
    assert p.verify_bundle(first_epoch) == original_checksum
    assert e.read_json(resumed / 'summary.json')['epochs_completed'] == 2
    expected = r._load_checkpoint(reference.parent / 'epoch_0002/state.pt')
    actual = r._load_checkpoint(resumed.parent / 'epoch_0002/state.pt')
    for name in ['model', 'best_model']:
        assert all(torch.equal(expected[name][key], actual[name][key]) for key in expected[name])
    assert torch.equal(expected['torch_rng'], actual['torch_rng'])
    for key, values in expected['optimizer']['state'].items():
        for field, value in values.items():
            if isinstance(value, torch.Tensor):
                assert torch.equal(value, actual['optimizer']['state'][key][field])
            else:
                assert value == actual['optimizer']['state'][key][field]
    expected_progress = e.read_json(reference.parent / 'epoch_0002/progress.json')
    actual_progress = e.read_json(resumed.parent / 'epoch_0002/progress.json')
    for key in ['numpy_generator_state', 'python_rng_state', 'best_validation_score', 'best_epoch', 'replay']:
        assert expected_progress[key] == actual_progress[key]


def test_partial_resume_rejects_heldout_record_in_sealed_replay_before_training(tmp_path, monkeypatch, setup):
    instances, fold = setup
    real_commit = e._commit

    def interrupt_after_epoch_one(local, remote):
        real_commit(local, remote)
        if Path(local).name == 'epoch_0001':
            raise KeyboardInterrupt('Desconexión simulada después del replay persistido.')

    monkeypatch.setattr(e, '_commit', interrupt_after_epoch_one)
    with pytest.raises(KeyboardInterrupt):
        train(tmp_path, instances, fold)
    monkeypatch.setattr(e, '_commit', real_commit)
    heldout = next(x for x in instances
                   if x.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups'])
    for location in ['local', 'drive']:
        checkpoint = tmp_path / 'run' / location / 'epoch_0001'
        progress = e.read_json(checkpoint / 'progress.json')
        records = next(records for records in progress['replay'].values() if records)
        records[0]['instance_sha'] = heldout.sha
        e.write_json(checkpoint / 'progress.json', progress)
        p.seal_bundle(checkpoint)

    def forbidden(*args, **kwargs):
        raise AssertionError('Rechazar replay de otra geometría antes de pérdidas o colonias.')

    monkeypatch.setattr(r, 'collect_colony', forbidden)
    monkeypatch.setattr(r, 'trajectory_loss', forbidden)
    with pytest.raises(ValueError):
        train(tmp_path, instances, fold)
    assert not list(tmp_path.glob('**/final/best.pt'))


def test_solver_selects_matching_heldout_model_and_preserves_zero_teacher_cost(tmp_path, monkeypatch, setup):
    instances, _ = setup
    root = crossfit(tmp_path, instances)
    spec = p.Traffic(mode='no_noise')
    solver, provenance = r.make_solver(root, instances, spec, 'cpu', expected_bank_sha256='a' * 64)
    records = provenance['hbg_facs']
    assert len(records) == len(instances)

    def solve_stub(problem, seed, settings, model, device):
        assert not model.training
        assert settings['seconds'] > 0
        return problem.result([[1, 2], []], 'manual_test', 0., [])

    monkeypatch.setattr(r, 'solve_hbg', solve_stub)
    for instance in instances:
        record = records[f'{instance.sha}:11']
        group = instance.provenance['geometry_group_sha256']
        assert group in record['heldout_geometry_groups']
        assert group not in record['train_geometry_groups'] + record['validation_geometry_groups']
        assert record['teacher_seconds'] == 0
        result = solver(SearchProblem(instance, spec, 1000), 11,
                        {'seconds': 120., **config()['hbg']}, tmp_path / 'inference')
        assert result['metadata']['fold_id'] == record['fold_id']
        assert result['metadata']['model_checkpoint_sha256'] == record['model_checkpoint_sha256']
    with pytest.raises(ValueError):
        solver(SearchProblem(instances[0], spec, 1000), 12,
               {'seconds': 120., **config()['hbg']}, tmp_path / 'missing_seed')
    costs = e.read_json(root / 'index/offline_costs.json')
    assert costs['models_trained'] == len(instances)
    assert costs['teacher_seconds_unique'] == 0


def test_solver_rejects_other_bank_scale_code_and_corrupt_weights(tmp_path, monkeypatch, setup):
    instances, _ = setup
    root = crossfit(tmp_path, instances)
    spec = p.Traffic(mode='no_noise')
    with pytest.raises(ValueError):
        r.make_solver(root, instances, spec, 'cpu', expected_bank_sha256='b' * 64)
    solver, _ = r.make_solver(root, instances, spec, 'cpu', expected_bank_sha256='a' * 64)
    with pytest.raises(ValueError):
        solver(SearchProblem(instances[0], spec, 100), 11,
               {'seconds': 120., **config()['hbg']}, tmp_path / 'wrong_scale')
    for name, value in [('temperature', 2.), ('lambda_db', .8)]:
        with pytest.raises(ValueError):
            solver(SearchProblem(instances[0], spec, 1000), 11,
                   {'seconds': 120., **config()['hbg'], name: value}, tmp_path / 'wrong_reward')
    original_hashes = r._source_hashes()
    with monkeypatch.context() as altered_code:
        changed = dict(original_hashes)
        changed[next(iter(changed))] = 'f' * 64
        altered_code.setattr(r, '_source_hashes', lambda: changed)
        with pytest.raises(ValueError):
            r.make_solver(root, instances, spec, 'cpu', expected_bank_sha256='a' * 64)
    records = e.read_json(root / 'index/models.json')
    damaged = root / records[0]['relative_directory'] / 'best.pt'
    with damaged.open('ab') as stream:
        stream.write(b'corruption')
    with pytest.raises(ValueError):
        r.make_solver(root, instances, spec, 'cpu', expected_bank_sha256='a' * 64)


@pytest.mark.parametrize('damage', ['missing', 'duplicate', 'forged_fold'])
def test_solver_rejects_incomplete_or_forged_fold_inventory(tmp_path, setup, damage):
    instances, _ = setup
    root = crossfit(tmp_path, instances)
    models = e.read_json(root / 'index/models.json')
    if damage == 'missing':
        models.pop()
    elif damage == 'duplicate':
        models.append(deepcopy(models[0]))
    else:
        models[0]['fold']['heldout_geometry_groups'] = models[0]['fold']['train_geometry_groups']
    e.write_json(root / 'index/models.json', models)
    p.seal_bundle(root / 'index')
    with pytest.raises(ValueError):
        r.make_solver(root, instances, p.Traffic(mode='no_noise'), 'cpu',
                      expected_bank_sha256='a' * 64)


def runner_fixture(tmp_path, monkeypatch, instances, fold):
    instance = next(item for item in instances
                    if item.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups'])
    bank = tmp_path / 'manual_bank'
    bank_config = {'traffic': asdict(p.Traffic(mode='no_noise')),
                   'policies': ['fixed_order', 'causal_edd'], 'alpha_cvar': .5,
                   'scenarios': {'train': 1, 'selection': 1, 'evaluation': 1}}
    e.write_json(bank / 'config.json', bank_config)
    e.write_json(bank / f'scenarios/n{instance.n:04d}.json',
                 {'train': [1], 'selection': [2], 'evaluation': [3]})
    p.seal_bundle(bank)
    monkeypatch.setattr(p, 'load_bank', lambda path: [instance])
    runner_config = {'schema_version': e.SCHEMA, 'role': 'development_only',
                     'objective': OBJECTIVE,
                     'time_scale': 1000, 'seeds': [11], 'search_seconds': 120.,
                     'scenario_split': 'evaluation', 'export_all_scenario_events': True,
                     'methods': {'hbg_facs': config()['hbg']}}
    provenance = {'training_id': 'manual_training', 'fold_id': fold['fold_id'],
                  'training_seed': 11, 'training_seconds': 8., 'validation_seconds': 2.,
                  'teacher_seconds': 0., 'model_checkpoint_sha256': 'a' * 64,
                  'train_geometry_groups': fold['train_geometry_groups'],
                  'validation_geometry_groups': fold['validation_geometry_groups'],
                  'heldout_geometry_groups': fold['heldout_geometry_groups']}
    return bank, runner_config, instance, provenance


def test_runner_preserves_training_provenance_when_inference_fails(tmp_path, monkeypatch, setup):
    bank, runner_config, instance, provenance = runner_fixture(tmp_path, monkeypatch, *setup)

    def failing_solver(*args):
        raise RuntimeError('Fallo manual de inferencia.')

    root = e.run_methods(bank, runner_config, {'hbg_facs': failing_solver},
                         tmp_path / 'work', tmp_path / 'drive', run_id='failure_test',
                         method_provenance={'hbg_facs': {f'{instance.sha}:11': provenance}})
    rows = e.read_json(root / 'report/summary.json')
    assert len(rows) == 2
    for row in rows:
        assert row['solver_status'] == 'solver_error' and row['evaluation_status'] == 'no_solution'
        assert row['mean_travel_minutes'] is None
        for key in ['training_id', 'fold_id', 'training_seed', 'training_seconds',
                    'validation_seconds', 'teacher_seconds', 'model_checkpoint_sha256']:
            assert row[key] == provenance[key]


def test_runner_requires_hybrid_training_provenance_before_inference(tmp_path, monkeypatch, setup):
    bank, runner_config, _, _ = runner_fixture(tmp_path, monkeypatch, *setup)

    def forbidden(*args):
        raise AssertionError('Sin procedencia híbrida no debe comenzar inferencia.')

    with pytest.raises(ValueError, match='procedencia'):
        e.run_methods(bank, runner_config, {'hbg_facs': forbidden},
                      tmp_path / 'work', tmp_path / 'drive', run_id='missing_provenance')
    assert not list(tmp_path.glob('**/solution/solution.json'))


def test_resume_diagnostic_distinguishes_identity_and_runtime(monkeypatch):
    identity = {'fold': 'manual'}
    monkeypatch.setattr(r, '_runtime_signature', lambda device: {'device_name': 'GPU B'})
    progress = {'identity_sha256': p.digest(identity),
                'runtime_signature': {'device_name': 'GPU A'}}
    with pytest.raises(ValueError, match='GPU A.*GPU B'):
        r._check_resume(progress, identity, 'cpu')
    progress['identity_sha256'] = 'wrong'
    with pytest.raises(ValueError, match='otro fold'):
        r._check_resume(progress, identity, 'cpu')
    progress.update(identity_sha256=p.digest(identity),
                    runtime_signature={'device_name': 'GPU B'})
    r._check_resume(progress, identity, 'cpu')


def test_crossfit_isolates_runtime_and_recovers_original(tmp_path, monkeypatch, setup):
    instances, _ = setup
    signature = dict(r._runtime_signature('cpu'))
    monkeypatch.setattr(r, '_runtime_signature', lambda device: dict(signature))
    original = crossfit(tmp_path, instances)
    original_index_hash = p.verify_bundle(original / 'index')
    signature['torch'] += '-different-runtime'
    changed = crossfit(tmp_path, instances)
    assert changed != original
    assert p.verify_bundle(original / 'index') == original_index_hash
    assert e.read_json(changed / 'meta/identity.json')['runtime_signature'] == signature
    signature['torch'] = signature['torch'].removesuffix('-different-runtime')

    def forbidden(*args, **kwargs):
        raise AssertionError('El entorno original debe recuperar sus modelos sin entrenar.')

    monkeypatch.setattr(r, 'collect_colony', forbidden)
    assert crossfit(tmp_path, instances) == original
    assert p.verify_bundle(original / 'index') == original_index_hash


def test_validation_disables_paid_seed_and_exposes_neural_completion(monkeypatch):
    model = torch.nn.Linear(1, 1)
    calls = []

    def score(model, problem, device, params, seed=0):
        calls.append(params)
        assert params['constructive_seed'] is False and params['local_search_moves'] == 0
        return {'score': 1.2, 'complete_fraction': 0., 'best_complete': False, 'rollout_count': 8}

    monkeypatch.setattr(r, 'validation_score', score)
    original = {**config()['hbg'], 'constructive_seed': True, 'local_search_moves': 50}
    _, _, diagnostics = r.validation_cost(model, {'sha': object()}, {'group': ['sha']}, ['group'],
                                          'cpu', original, return_diagnostics=True)
    assert calls and original['constructive_seed'] is True and original['local_search_moves'] == 50
    assert diagnostics['instances_with_completion_fraction'] == 0.
    assert diagnostics['scope'] == 'neural_colony_without_edd_seed_or_local_search'


def test_no_neural_validation_completion_preserves_full_inventory_without_best_weights(tmp_path, monkeypatch, setup):
    instances, _ = setup
    monkeypatch.setattr(r, 'validation_score', lambda *args, **kwargs:
                        {'score': 1.2, 'complete_fraction': 0., 'best_complete': False, 'rollout_count': 8})
    root = crossfit(tmp_path, instances)
    models = e.read_json(root / 'index/models.json')
    assert len(models) == len(instances)
    assert not list(root.glob('geometry_*/seed_*/final/best.pt'))
    assert all(row['selection_status'] == 'validation_gate_failed' and row['best_epoch'] == 0 for row in models)
    assert e.read_json(root / 'index/offline_costs.json')['validation_gate_failures'] == len(instances)
    solver, provenance = r.make_solver(root, instances, p.Traffic(mode='no_noise'), 'cpu')

    def forbidden(*args, **kwargs):
        raise AssertionError('Los pesos rechazados no se cargan ni se infiere en heldout.')

    monkeypatch.setattr(r, '_load_checkpoint', forbidden)
    monkeypatch.setattr(r, 'solve_hbg', forbidden)
    for instance in instances:
        result = solver(SearchProblem(instance, p.Traffic(mode='no_noise'), 1000), 11,
                        {'seconds': 120., **config()['hbg']}, tmp_path / 'inference')
        assert result['routes'] is None
        assert result['metadata']['validation_gate_failed'] is True
        assert result['metadata']['heldout_inference_executed'] is False
        e.validate_learned_provenance(provenance['hbg_facs'][f'{instance.sha}:11'], instance, 11)


def test_gpu_gate_precedes_training_or_fold_artifacts(tmp_path, monkeypatch, setup):
    instances, fold = setup
    settings = config()
    settings['training']['require_gpu'] = True
    with pytest.raises(RuntimeError, match='requiere GPU'):
        train(tmp_path, instances, fold, settings=settings)
    assert not list(tmp_path.glob('**/identity.json'))


def test_inference_controls_reject_hidden_search_or_budget_advantages():
    base = {**config()['hbg'], 'constructive_seed': True, 'local_search_moves': 100}
    common = {'methods': {'hbg_facs': base, 'hbg_no_pheromone': {**base, 'alpha': 0.},
                          'hbg_untrained': dict(base)}}
    assert r.validate_inference_controls(common)
    common['methods']['hbg_untrained']['local_search_moves'] = 0
    with pytest.raises(ValueError, match='toda la configuración'):
        r.validate_inference_controls(common)
    common['methods']['hbg_untrained'] = dict(base)
    common['methods']['hbg_no_pheromone']['ants'] += 1
    with pytest.raises(ValueError, match='sólo puede cambiar alpha'):
        r.validate_inference_controls(common)


def test_untrained_control_checks_actual_initial_tensors(tmp_path, monkeypatch, setup):
    instances, _ = setup
    root = crossfit(tmp_path, instances)
    _, provenance = r.make_solver(root, instances, p.Traffic(mode='no_noise'), 'cpu')
    expected = {key: value['initialization_state_sha256'] for key, value in provenance['hbg_facs'].items()}
    control = r.make_untrained_solver(instances, config(), 'cpu', expected_initializations=expected)
    monkeypatch.setattr(r, 'solve_hbg', lambda problem, *args:
                        problem.result([[1, 2], []], 'manual_test', 0., []))
    instance = instances[0]
    result = control(SearchProblem(instance, p.Traffic(mode='no_noise'), 1000), 11,
                     {'seconds': 120., **config()['hbg']}, tmp_path / 'control')
    assert result['metadata']['initialization_state_sha256'] == expected[f'{instance.sha}:11']
    expected[f'{instance.sha}:11'] = 'f' * 64
    with pytest.raises(ValueError, match='inicialización'):
        control(SearchProblem(instance, p.Traffic(mode='no_noise'), 1000), 11,
                {'seconds': 120., **config()['hbg']}, tmp_path / 'bad_control')


Overwriting tests/test_hybrid.py


In [62]:
%%writefile tests/test_reinforcement.py
"""Pruebas del entrenamiento 04: ejecutar exclusivamente en Google Colab.

Las geometrías son fixtures manuales. Los dobles diferenciables aíslan el
protocolo de entrenamiento y consumen RNG de Python, NumPy y Torch para probar
la reanudación. Las trayectorias y gradientes reales se prueban en test_pomo.py.
No ejecutar estas pruebas ni sus fixtures localmente.
"""
from copy import deepcopy
from dataclasses import asdict
from pathlib import Path
import random
import sys

import numpy as np
import pytest

torch = pytest.importorskip('torch')
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import experiments as e
from svrplab import pilot as p
from svrplab import reinforcement as r
from svrplab import supervised_data as sd
from svrplab.search import OBJECTIVE, SearchProblem


def manual_instances():
    instances = []
    for i in range(3):
        xy = [[0, 0], [i + 1, 0], [0, i + 1]]
        instances.append(p.freeze_instance({
            'instance_id': f'reinforcement_manual_{i}', 'depot_indices': [0],
            'locations': xy, 'demands': [0, 0, 1], 'capacities': [2, 2],
            'windows': [[0, 100], [0, 100], [0, 100]], 'appear_times': [0, 0, 0],
            'provenance': {'kind': 'manual_test',
                           'geometry_group_sha256': p.digest({'locations': xy})},
        }))
    return instances


def config():
    return {
        'schema_version': 'reinforcement-04-v1', 'role': 'development_only',
        'replicate_id': 'manual_test',
        'split_strategy': 'leave_one_geometry_out_with_inner_validation',
        'time_scale': 1000, 'objective': OBJECTIVE,
        'model': {'model_dim': 16, 'num_heads': 2, 'num_layers': 1,
                  'dropout': 0., 'tanh_clip': 10.},
        'rollout': {'n_starts': 8},
        'training': {'epochs': 2, 'steps_per_epoch': 2, 'gradient_accumulation': 1,
                     'learning_rate': .001, 'weight_decay': 0., 'gradient_clip_norm': 1.,
                     'patience': 5, 'min_delta': 0., 'seconds_per_model': 120.,
                     'permute_nodes': True, 'deterministic_algorithms': True},
    }


def training_loss_stub(model, problem, device, rng, params, node_order=None):
    assert model.training and torch.is_grad_enabled()
    parameter = next(model.parameters())
    noise = torch.rand((), device=device) + float(rng.random()) + random.random()
    loss = (parameter + noise).square().mean()
    return loss, {'mean_cost': float(noise.detach().cpu()) + 1.,
                  'complete_fraction': .5, 'rollout_count': params['n_starts'],
                  'decisions': params['n_starts'] * (problem.n + problem.vehicles)}


def validation_stub(model, problem, device, params):
    assert not model.training and not torch.is_grad_enabled()
    score = .5 + float(next(model.parameters()).square().mean().detach().cpu())
    return {'score': score, 'complete_fraction': 1., 'best_complete': True, 'rollout_count': params['n_starts']}


@pytest.fixture
def setup(monkeypatch):
    # El notebook que invoca pytest ya exige Colab; se aísla la guardia interna.
    monkeypatch.setattr(r, 'require_colab', lambda: None)
    monkeypatch.setattr(p, 'environment_report', lambda: {'test_fixture': True})
    monkeypatch.setattr(r, 'save_learning_plot', lambda *args: None)
    monkeypatch.setattr(r, 'group_loss', training_loss_stub)
    monkeypatch.setattr(r, 'validation_score', validation_stub)

    def forbidden(*args, **kwargs):
        raise AssertionError('04 no debe abrir profesores, etiquetas ni corpus de 03.')

    monkeypatch.setattr(sd, 'load_fold_examples', forbidden)
    instances = manual_instances()
    return instances, sd.make_geometry_folds(instances)[0]


def train(tmp_path, instances, fold, label='run', settings=None):
    return r.train_fold(instances, p.Traffic(mode='no_noise'), settings or config(),
                        11, fold, tmp_path / label / 'local',
                        tmp_path / label / 'drive', 'cpu')


def crossfit(tmp_path, instances, settings=None, seeds=None):
    return r.train_crossfit(instances, p.Traffic(mode='no_noise'), settings or config(),
                            [11] if seeds is None else seeds,
                            tmp_path / 'work', tmp_path / 'drive', 'manual_bank',
                            'a' * 64, p.file_hash(p.__file__), device='cpu')


@pytest.mark.parametrize('key,value', [
    ('epochs', 0), ('steps_per_epoch', True), ('gradient_accumulation', 0),
    ('patience', -1), ('learning_rate', 0), ('gradient_clip_norm', float('nan')),
    ('seconds_per_model', float('inf')), ('weight_decay', -1),
    ('min_delta', True), ('permute_nodes', 1), ('deterministic_algorithms', 'yes'),
])
def test_training_configuration_rejects_invalid_settings(key, value):
    settings = config()
    settings['training'][key] = value
    with pytest.raises(ValueError):
        r.validate_training_config(settings)


@pytest.mark.parametrize('key,value', [
    ('schema_version', 'other'), ('role', 'final_test'),
    ('split_strategy', 'random_rows'), ('objective', 'mean_evaluation_cost'),
    ('time_scale', 0), ('time_scale', True), ('replicate_id', ''),
])
def test_training_configuration_rejects_other_protocols(key, value):
    with pytest.raises(ValueError):
        r.validate_training_config(config() | {key: value})


@pytest.mark.parametrize('section,key,value', [
    ('rollout', 'n_starts', 0), ('rollout', 'n_starts', True),
    ('model', 'dropout', .1), ('model', 'model_dim', 0),
])
def test_training_configuration_rejects_invalid_policy_or_rollouts(section, key, value):
    settings = config()
    settings[section][key] = value
    with pytest.raises(ValueError):
        r.validate_training_config(settings)


@pytest.mark.parametrize('seeds', [[], [11, 11], [-1], [True]])
def test_crossfit_rejects_invalid_seeds_before_training(tmp_path, monkeypatch, setup, seeds):
    instances, _ = setup

    def forbidden(*args, **kwargs):
        raise AssertionError('Semillas inválidas deben rechazarse antes de entrenar.')

    monkeypatch.setattr(r, 'train_fold', forbidden)
    with pytest.raises(ValueError):
        crossfit(tmp_path, instances, seeds=seeds)


def test_validation_weights_geometry_equally_and_excludes_gradient_graphs(monkeypatch):
    model = torch.nn.Linear(1, 1).train()
    grouped = {'a': ['a1'], 'b': ['b1', 'b2', 'b3']}
    calls = []

    def score_stub(model, problem, device, params):
        assert not model.training and not torch.is_grad_enabled()
        calls.append(problem)
        return {'score': float(problem), 'complete_fraction': 1., 'best_complete': True, 'rollout_count': 8}

    monkeypatch.setattr(r, 'validation_score', score_stub)
    mean, by_group = r.validation_cost(model, {'a1': 2, 'b1': 4, 'b2': 6, 'b3': 8},
                                       grouped, ['a', 'b'], 'cpu', {'n_starts': 8})
    assert by_group == {'a': 2., 'b': 6.}
    assert mean == 4. and mean != np.mean(calls)


def test_train_fold_uses_only_authorized_geometries_and_final_resume_is_immutable(tmp_path, monkeypatch, setup):
    instances, fold = setup
    real_problem = r.SearchProblem
    seen = set()

    def guarded(instance, *args, **kwargs):
        assert instance.provenance['geometry_group_sha256'] not in fold['heldout_geometry_groups']
        seen.add(instance.sha)
        return real_problem(instance, *args, **kwargs)

    monkeypatch.setattr(r, 'SearchProblem', guarded)
    final = train(tmp_path, instances, fold)
    before = p.verify_bundle(final)
    summary = e.read_json(final / 'summary.json')
    authorized = {x.sha for x in instances
                  if x.provenance['geometry_group_sha256'] not in fold['heldout_geometry_groups']}
    assert seen == authorized
    assert summary['heldout_used_for_training_or_selection'] is False
    assert summary['epochs_completed'] == 2 and summary['teacher_seconds'] == 0
    assert 'corpus_id' not in summary
    assert summary['training_seconds'] >= summary['validation_seconds'] >= 0
    assert summary['model_checkpoint_sha256'] == p.file_hash(final / 'best.pt')
    assert p.verify_bundle(tmp_path / 'run/drive/final') == before
    history = e.read_json(final / 'history.json')
    assert all({'train_loss', 'train_cost', 'train_complete_fraction', 'validation_score'} <= set(row)
               for row in history)
    assert summary['best_validation_score'] == min(row['validation_score'] for row in history)
    latest = r._load_checkpoint(final.parent / 'epoch_0002/state.pt')
    assert latest['optimizer']['state'] and latest['cuda_rng'] == []
    assert latest['torch_rng'].dtype == torch.uint8

    def forbidden(*args, **kwargs):
        raise AssertionError('Un final recuperado no debe construir problemas ni volver a entrenar.')

    monkeypatch.setattr(r, 'SearchProblem', forbidden)
    assert train(tmp_path, instances, fold) == final
    assert p.verify_bundle(final) == before
    with (final / 'best.pt').open('ab') as stream:
        stream.write(b'corruption')
    with pytest.raises(ValueError):
        train(tmp_path, instances, fold)


def test_overlapping_fold_is_rejected_before_constructing_problems(tmp_path, monkeypatch, setup):
    instances, fold = setup
    damaged = deepcopy(fold)
    damaged['train_geometry_groups'].append(damaged['heldout_geometry_groups'][0])

    def forbidden(*args, **kwargs):
        raise AssertionError('Rechazar roles superpuestos antes de construir problemas.')

    monkeypatch.setattr(r, 'SearchProblem', forbidden)
    with pytest.raises(ValueError):
        train(tmp_path, instances, damaged)


def test_training_budget_exhausted_by_setup_never_exports_random_weights(tmp_path, monkeypatch, setup):
    instances, fold = setup
    settings = config()
    settings['training']['seconds_per_model'] = 1e-12

    def forbidden(*args, **kwargs):
        raise AssertionError('Sin presupuesto no se inicia un batch ni validación.')

    monkeypatch.setattr(r, 'group_loss', forbidden)
    monkeypatch.setattr(r, 'validation_score', forbidden)
    with pytest.raises(RuntimeError, match='No hay checkpoint entrenado'):
        train(tmp_path, instances, fold, settings=settings)
    assert not list(tmp_path.glob('**/final/best.pt'))


def test_changing_heldout_windows_cannot_change_training_inputs_or_weights(tmp_path, setup):
    instances, fold = setup
    first = train(tmp_path, instances, fold, 'first')
    changed = []
    for instance in instances:
        if instance.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups']:
            payload = instance.payload()
            payload['windows'][1][0] = 20
            changed.append(p.freeze_instance(payload))
        else:
            changed.append(instance)
    second = train(tmp_path, changed, fold, 'second')
    expected = r._load_checkpoint(first / 'best.pt')
    actual = r._load_checkpoint(second / 'best.pt')
    assert expected.keys() == actual.keys()
    assert all(torch.equal(expected[key], actual[key]) for key in expected)
    assert e.read_json(first / 'summary.json')['identity_sha256'] == e.read_json(second / 'summary.json')['identity_sha256']


def test_partial_resume_restores_optimizer_and_all_rng_without_repeating_epoch(tmp_path, monkeypatch, setup):
    instances, fold = setup
    reference = train(tmp_path, instances, fold, 'reference')
    real_commit = e._commit

    def interrupt_after_epoch_one(local, remote):
        real_commit(local, remote)
        if Path(local).name == 'epoch_0001':
            raise KeyboardInterrupt('Desconexión simulada después del checkpoint completo.')

    monkeypatch.setattr(e, '_commit', interrupt_after_epoch_one)
    with pytest.raises(KeyboardInterrupt):
        train(tmp_path, instances, fold, 'interrupted')
    first_epoch = tmp_path / 'interrupted/local/epoch_0001'
    original_checksum = p.verify_bundle(first_epoch)
    monkeypatch.setattr(e, '_commit', real_commit)
    resumed = train(tmp_path, instances, fold, 'interrupted')
    assert p.verify_bundle(first_epoch) == original_checksum
    assert e.read_json(resumed / 'summary.json')['epochs_completed'] == 2
    expected = r._load_checkpoint(reference.parent / 'epoch_0002/state.pt')
    actual = r._load_checkpoint(resumed.parent / 'epoch_0002/state.pt')
    for name in ['model', 'best_model']:
        assert all(torch.equal(expected[name][key], actual[name][key]) for key in expected[name])
    assert torch.equal(expected['torch_rng'], actual['torch_rng'])
    for key, values in expected['optimizer']['state'].items():
        for field, value in values.items():
            if isinstance(value, torch.Tensor):
                assert torch.equal(value, actual['optimizer']['state'][key][field])
            else:
                assert value == actual['optimizer']['state'][key][field]
    expected_progress = e.read_json(reference.parent / 'epoch_0002/progress.json')
    actual_progress = e.read_json(resumed.parent / 'epoch_0002/progress.json')
    for key in ['numpy_generator_state', 'python_rng_state', 'best_validation_score', 'best_epoch']:
        assert expected_progress[key] == actual_progress[key]


def test_solver_selects_matching_heldout_model_and_preserves_zero_teacher_cost(tmp_path, monkeypatch, setup):
    instances, _ = setup
    root = crossfit(tmp_path, instances)
    spec = p.Traffic(mode='no_noise')
    solver, provenance = r.make_solver(root, instances, spec, 'cpu', expected_bank_sha256='a' * 64)
    records = provenance['pomo']
    assert len(records) == len(instances)

    def solve_stub(problem, seed, settings, model, device):
        assert not model.training
        assert settings['seconds'] > 0
        return problem.result([[1, 2], []], 'manual_test', 0., [])

    monkeypatch.setattr(r, 'solve_pomo', solve_stub)
    for instance in instances:
        record = records[f'{instance.sha}:11']
        group = instance.provenance['geometry_group_sha256']
        assert group in record['heldout_geometry_groups']
        assert group not in record['train_geometry_groups'] + record['validation_geometry_groups']
        assert record['teacher_seconds'] == 0
        result = solver(SearchProblem(instance, spec, 1000), 11,
                        {'seconds': 120., 'n_starts': 8}, tmp_path / 'inference')
        assert result['metadata']['fold_id'] == record['fold_id']
        assert result['metadata']['model_checkpoint_sha256'] == record['model_checkpoint_sha256']
    with pytest.raises(ValueError):
        solver(SearchProblem(instances[0], spec, 1000), 12,
               {'seconds': 120., 'n_starts': 8}, tmp_path / 'missing_seed')
    costs = e.read_json(root / 'index/offline_costs.json')
    assert costs['models_trained'] == len(instances)
    assert costs['teacher_seconds_unique'] == 0


def test_solver_rejects_other_bank_scale_code_and_corrupt_weights(tmp_path, monkeypatch, setup):
    instances, _ = setup
    root = crossfit(tmp_path, instances)
    spec = p.Traffic(mode='no_noise')
    with pytest.raises(ValueError):
        r.make_solver(root, instances, spec, 'cpu', expected_bank_sha256='b' * 64)
    solver, _ = r.make_solver(root, instances, spec, 'cpu', expected_bank_sha256='a' * 64)
    with pytest.raises(ValueError):
        solver(SearchProblem(instances[0], spec, 100), 11,
               {'seconds': 120., 'n_starts': 8}, tmp_path / 'wrong_scale')
    original_hashes = r._source_hashes()
    with monkeypatch.context() as altered_code:
        changed = dict(original_hashes)
        changed[next(iter(changed))] = 'f' * 64
        altered_code.setattr(r, '_source_hashes', lambda: changed)
        with pytest.raises(ValueError):
            r.make_solver(root, instances, spec, 'cpu', expected_bank_sha256='a' * 64)
    records = e.read_json(root / 'index/models.json')
    damaged = root / records[0]['relative_directory'] / 'best.pt'
    with damaged.open('ab') as stream:
        stream.write(b'corruption')
    with pytest.raises(ValueError):
        r.make_solver(root, instances, spec, 'cpu', expected_bank_sha256='a' * 64)


@pytest.mark.parametrize('damage', ['missing', 'duplicate', 'forged_fold'])
def test_solver_rejects_incomplete_or_forged_fold_inventory(tmp_path, setup, damage):
    instances, _ = setup
    root = crossfit(tmp_path, instances)
    models = e.read_json(root / 'index/models.json')
    if damage == 'missing':
        models.pop()
    elif damage == 'duplicate':
        models.append(deepcopy(models[0]))
    else:
        models[0]['fold']['heldout_geometry_groups'] = models[0]['fold']['train_geometry_groups']
    e.write_json(root / 'index/models.json', models)
    p.seal_bundle(root / 'index')
    with pytest.raises(ValueError):
        r.make_solver(root, instances, p.Traffic(mode='no_noise'), 'cpu',
                      expected_bank_sha256='a' * 64)


def test_runner_preserves_training_provenance_when_inference_fails(tmp_path, monkeypatch, setup):
    instances, fold = setup
    instance = next(item for item in instances
                    if item.provenance['geometry_group_sha256'] in fold['heldout_geometry_groups'])
    bank = tmp_path / 'manual_bank'
    bank_config = {'traffic': asdict(p.Traffic(mode='no_noise')),
                   'policies': ['fixed_order', 'causal_edd'], 'alpha_cvar': .5,
                   'scenarios': {'train': 1, 'selection': 1, 'evaluation': 1}}
    e.write_json(bank / 'config.json', bank_config)
    e.write_json(bank / f'scenarios/n{instance.n:04d}.json',
                 {'train': [1], 'selection': [2], 'evaluation': [3]})
    p.seal_bundle(bank)
    monkeypatch.setattr(p, 'load_bank', lambda path: [instance])
    runner_config = {'schema_version': e.SCHEMA, 'role': 'development_only',
                     'objective': OBJECTIVE,
                     'time_scale': 1000, 'seeds': [11], 'search_seconds': 120.,
                     'scenario_split': 'evaluation', 'export_all_scenario_events': True,
                     'methods': {'pomo': {'n_starts': 8, 'sampling_rounds': 1, 'permute_samples': False}}}
    provenance = {'training_id': 'manual_training', 'fold_id': fold['fold_id'],
                  'training_seed': 11, 'training_seconds': 8., 'validation_seconds': 2.,
                  'teacher_seconds': 0., 'model_checkpoint_sha256': 'a' * 64,
                  'train_geometry_groups': fold['train_geometry_groups'],
                  'validation_geometry_groups': fold['validation_geometry_groups'],
                  'heldout_geometry_groups': fold['heldout_geometry_groups']}

    def failing_solver(*args):
        raise RuntimeError('Fallo manual de inferencia.')

    root = e.run_methods(bank, runner_config, {'pomo': failing_solver},
                         tmp_path / 'work', tmp_path / 'drive', run_id='failure_test',
                         method_provenance={'pomo': {f'{instance.sha}:11': provenance}})
    rows = e.read_json(root / 'report/summary.json')
    assert len(rows) == 2
    for row in rows:
        assert row['solver_status'] == 'solver_error' and row['evaluation_status'] == 'no_solution'
        assert row['mean_travel_minutes'] is None
        for key in ['training_id', 'fold_id', 'training_seed', 'training_seconds',
                    'validation_seconds', 'teacher_seconds', 'model_checkpoint_sha256']:
            assert row[key] == provenance[key]


def test_readiness_never_calls_pilot_sufficient_or_statistically_superior():
    initial = {'validation_score': .5}
    history = [{'epoch': 1, 'best_epoch': 1, 'validation_score': .6,
                'validation_instances_with_completion_fraction': 1.,
                'optimizer_updates': 2, 'rollout_count': 16, 'decisions': 80}]
    diagnostics = r.readiness_diagnostics(initial, history, .001)
    assert diagnostics['status'] == 'no_observed_learning_benefit'
    assert diagnostics['optimizer_updates'] == 2
    assert diagnostics['adequate_training_demonstrated'] is False
    assert diagnostics['statistical_superiority_demonstrated'] is False
    history[0]['validation_score'] = .4
    assert r.readiness_diagnostics(initial, history, .001)['status'] == 'development_learning_signal'
    history[0]['validation_instances_with_completion_fraction'] = 0.
    assert r.readiness_diagnostics(initial, history, .001)['status'] == 'no_complete_validation'


def test_initial_diagnostic_and_readiness_are_sealed_and_exported(tmp_path, setup):
    instances, fold = setup
    final = train(tmp_path, instances, fold)
    initial = e.read_json(final / 'initial_diagnostics.json')
    summary = e.read_json(final / 'summary.json')
    assert initial['model_state'] == 'untrained_initialization' and not initial['heldout_used']
    assert summary['initial_validation_score'] == initial['validation_score']
    assert summary['readiness']['optimizer_updates'] == summary['optimizer_updates'] == 4
    assert summary['decoder_execution'] == 'sequential_starts_on_device_sampling'
    assert e.read_json(final.parent / 'initial/diagnostics.json') == initial
    assert p.verify_bundle(final.parent / 'initial')
    assert all('validation_complete_fraction' in row and 'epoch_decisions_per_second' in row
               for row in e.read_json(final / 'history.json'))


def test_explicit_controls_do_not_load_weights_or_teachers_and_preserve_rng(tmp_path, monkeypatch, setup):
    instances, _ = setup
    solvers, provenance = r.make_control_solvers(instances, config(), [11], 'cpu')
    assert set(solvers) == {'pomo_untrained', 'random_masked'}
    def forbidden(*args, **kwargs):
        raise AssertionError('Los controles no leen checkpoints.')
    monkeypatch.setattr(r, '_load_checkpoint', forbidden)
    observed = []
    def solve_stub(problem, seed, settings, model, device):
        observed.append((type(model).__name__, sum(parameter.numel() for parameter in model.parameters())))
        return problem.result([[1, 2], []], 'feasible', 0., [])
    monkeypatch.setattr(r, 'solve_pomo', solve_stub)
    before = torch.get_rng_state().clone()
    for method, solver in solvers.items():
        result = solver(SearchProblem(instances[0], p.Traffic(mode='no_noise')), 11,
                        {'seconds': 60., 'n_starts': 2}, tmp_path / method)
        assert result['metadata']['method'] == method
        assert result['metadata']['training_seconds'] == result['metadata']['optimizer_updates'] == 0
        assert provenance[method][f'{instances[0].sha}:11']['training_scope'] == 'untrained_control_no_fitting'
    assert torch.equal(before, torch.get_rng_state())
    assert observed[0][0] == 'AttentionPolicy' and observed[0][1] > 0
    assert observed[1] == ('UniformMaskedPolicy', 0)


def test_crossfit_runtime_is_part_of_training_identity(tmp_path, monkeypatch, setup):
    instances, _ = setup
    first = crossfit(tmp_path, instances)
    original = r._runtime_signature('cpu')
    monkeypatch.setattr(r, '_runtime_signature', lambda device: original | {'device_name': 'other_test_environment'})
    second = crossfit(tmp_path, instances)
    assert first != second
    assert e.read_json(first / 'meta/identity.json')['runtime_signature'] != e.read_json(second / 'meta/identity.json')['runtime_signature']


def test_selection_gate_never_trades_an_uncompleted_instance_for_low_mean_cost():
    eligible = {'instances_with_completion_fraction': 1., 'complete_fraction': .5}
    incomplete = {'instances_with_completion_fraction': .5, 'complete_fraction': .9}
    assert not r.validation_candidate_is_better(.1, incomplete, None, None, 0.)
    assert r.validation_candidate_is_better(.8, eligible, None, None, 0.)
    collapsed = {'instances_with_completion_fraction': 1., 'complete_fraction': .25}
    assert not r.validation_candidate_is_better(.1, collapsed, .8, eligible, 0.)
    improved = {'instances_with_completion_fraction': 1., 'complete_fraction': .75}
    assert r.validation_candidate_is_better(.9, improved, .8, eligible, 0.)
    assert r.validation_candidate_is_better(.7, eligible, .8, eligible, .01)


def test_no_complete_validation_is_an_outcome_without_selected_weights(tmp_path, monkeypatch, setup):
    instances, fold = setup
    monkeypatch.setattr(r, 'validation_score', lambda *args, **kwargs:
                        {'score': 1.2, 'complete_fraction': 0., 'best_complete': False, 'rollout_count': 8})
    final = train(tmp_path, instances, fold)
    summary = e.read_json(final / 'summary.json')
    assert summary['selection_status'] == 'validation_gate_failed'
    assert summary['best_epoch'] == 0 and summary['best_validation_score'] is None
    assert not (final / 'best.pt').exists()
    checkpoint = r.selected_checkpoint_path(final, summary)
    assert checkpoint.name == 'state.pt'
    assert summary['model_checkpoint_sha256'] == p.file_hash(checkpoint)
    assert summary['readiness_status'] == 'no_complete_validation'


def test_validation_failure_keeps_all_folds_and_never_decodes_heldout(tmp_path, monkeypatch, setup):
    instances, _ = setup
    monkeypatch.setattr(r, 'validation_score', lambda *args, **kwargs:
                        {'score': 1.2, 'complete_fraction': 0., 'best_complete': False, 'rollout_count': 8})
    root = crossfit(tmp_path, instances)
    models = e.read_json(root / 'index/models.json')
    assert len(models) == len(instances)
    assert all(model['selection_status'] == 'validation_gate_failed' for model in models)
    solver, provenance = r.make_solver(root, instances, p.Traffic(mode='no_noise'), 'cpu')

    def forbidden(*args, **kwargs):
        raise AssertionError('No cargar ni inferir con un checkpoint que no completó validación.')

    monkeypatch.setattr(r, '_load_checkpoint', forbidden)
    monkeypatch.setattr(r, 'solve_pomo', forbidden)
    for instance in instances:
        result = solver(SearchProblem(instance, p.Traffic(mode='no_noise'), 1000), 11,
                        {'seconds': 120., **config()['rollout']}, tmp_path / 'inference')
        assert result['routes'] is None
        assert result['metadata']['validation_gate_failed'] is True
        assert result['metadata']['heldout_inference_executed'] is False
        e.validate_learned_provenance(provenance['pomo'][f'{instance.sha}:11'], instance, 11)


Overwriting tests/test_reinforcement.py


In [63]:
%%writefile tests/test_hbg.py
"""Controles funcionales de 05: ejecutar exclusivamente en Google Colab.

Localmente sólo se permite validar AST; no importar ni ejecutar este archivo.
"""
import copy
import json
import math
from pathlib import Path
import sys

import numpy as np
import pytest

torch = pytest.importorskip('torch')
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import hbg
from svrplab.attention import _State
from svrplab.pilot import Traffic, freeze_instance, validate_routes
from svrplab.search import SearchProblem


def problem(*, capacities=(10,), demands=(0, 1, 1, 1), windows=None):
    n = len(demands) - 1
    instance = freeze_instance({
        'instance_id': 'hbg-manual-fixture', 'depot_indices': [0],
        'locations': [[j, 0] for j in range(n + 1)], 'demands': list(demands),
        'capacities': list(capacities), 'windows': windows or [[0, 100]] * (n + 1),
        'appear_times': [0] * (n + 1), 'provenance': {'kind': 'test_fixture'},
    })
    return SearchProblem(instance, Traffic(mode='no_noise'))


def policy():
    torch.manual_seed(43)
    return hbg.HBGPolicy({'model_dim': 16, 'num_heads': 4, 'num_layers': 1, 'dropout': 0.})


def record_for(p, actions, temperature=1.):
    state = _State(p)
    for customer in actions:
        state.advance(customer)
    return hbg._terminal_record(state, actions, temperature)


def test_balance_matches_explicit_equations_and_telescopes(monkeypatch):
    model, p = policy(), problem()
    record = record_for(p, [1, 2, 3, 0])
    flows = torch.tensor([.7, .4, .1, .2], requires_grad=True)
    next_flow = iter(flows)
    monkeypatch.setattr(model, 'log_flow', lambda *args: next(next_flow))

    def uniform(encoded, current_local, state_features, action_features, blocked):
        return encoded.new_zeros(len(blocked)).masked_fill(blocked, -torch.inf)

    monkeypatch.setattr(model, 'next_logits', uniform)
    weight = .3
    loss, stats = hbg.trajectory_loss(model, p, 'cpu', record, {'lambda_db': weight})
    r = record['log_reward']
    tb = (.7 - math.log(6) - r) ** 2
    db = np.mean([(.7 - math.log(3) - .4) ** 2,
                  (.4 - math.log(2) - .1) ** 2, (.1 - .2) ** 2, (.2 - r) ** 2])
    assert stats['tb_loss'] == pytest.approx(tb)
    assert stats['db_loss'] == pytest.approx(db)
    assert stats['db_residual_sum'] == pytest.approx(stats['tb_residual'])
    assert stats['logPF'] == pytest.approx(-math.log(6))
    assert loss.item() == pytest.approx((1 - weight) * tb + weight * db)
    assert stats['decisions'] == stats['balance_edges'] == p.n + p.vehicles
    loss.backward()
    assert flows.grad is not None and torch.isfinite(flows.grad).all()


def test_single_path_constant_reward_flow_has_zero_tb_and_db(monkeypatch):
    model, p = policy(), problem(demands=(0, 1))
    record = record_for(p, [1, 0])
    flow = torch.tensor(record['log_reward'], requires_grad=True)
    monkeypatch.setattr(model, 'log_flow', lambda *args: flow)
    loss, stats = hbg.trajectory_loss(model, p, 'cpu', record)
    assert stats['logPF'] == 0.
    assert stats['tb_loss'] == stats['db_loss'] == loss.item() == 0.
    loss.backward()
    assert flow.grad.item() == 0.


def test_colony_includes_first_customer_is_detached_and_has_finite_gradients(monkeypatch):
    model, p = policy().train(), problem()
    original = model.next_logits
    observed = []

    def observe(encoded, current_local, context, actions, blocked):
        observed.append((current_local, torch.is_grad_enabled()))
        return original(encoded, current_local, context, actions, blocked)

    monkeypatch.setattr(model, 'next_logits', observe)
    sha = p.x.sha
    params = {'ants': 2, 'iterations': 2}
    records, stats = hbg.collect_colony(model, p, 'cpu', np.random.default_rng(7), params)
    assert observed[0] == (0, False)
    assert all(not grad for _, grad in observed)
    assert model.training and p.x.sha == sha
    assert stats['rollout_count'] == stats['complete_count'] == 4
    assert stats['decisions'] == 4 * (p.n + p.vehicles)
    assert json.loads(json.dumps(records)) == records
    loss, terms = hbg.trajectory_loss(model, p, 'cpu', records[0], params)
    assert torch.isfinite(loss) and loss.requires_grad
    assert terms['complete'] and terms['decisions'] == p.n + p.vehicles
    loss.backward()
    for layer in (model.prefix_encoder, model.flow_head, model.pointer_key):
        gradients = [param.grad for param in layer.parameters() if param.grad is not None]
        assert gradients and all(torch.isfinite(grad).all() for grad in gradients)


def test_root_dead_end_has_deterministic_sink_and_never_softmax(monkeypatch):
    model, p = policy(), problem(capacities=(1,), demands=(0, 2))

    def forbidden(*args, **kwargs):
        raise AssertionError('Un bloqueo no admite softmax.')

    monkeypatch.setattr(model, 'next_logits', forbidden)
    records, stats = hbg.collect_colony(model, p, 'cpu', np.random.default_rng(3),
                                      {'ants': 1, 'iterations': 1})
    assert stats['complete_fraction'] == stats['decisions'] == 0
    assert records[0]['actions'] == [] and records[0]['routes'] is None
    assert records[0]['cost'] == 2. and math.exp(records[0]['log_reward']) > 0
    loss, terms = hbg.trajectory_loss(model, p, 'cpu', records[0])
    assert terms['balance_edges'] == 1 and terms['logPF'] == 0
    assert terms['tb_loss'] == pytest.approx(terms['db_loss'])
    loss.backward()
    assert all(param.grad is None or torch.isfinite(param.grad).all() for param in model.parameters())


def test_partial_dead_end_is_worse_than_any_complete_and_has_sink_edge():
    model = policy()
    # Packing impossibility not detected by relaxed individual/total checks:
    # after customer 3, residual capacities are 2 and 4 for two demands of 3.
    blocked = problem(capacities=(4, 4), demands=(0, 3, 3, 2))
    record = record_for(blocked, [3])
    hbg.validate_record(blocked, record)
    assert record['termination'] == 'dead_end' and record['unserved'] == 2
    assert 1 < record['cost'] < 3 and record['routes'] is None
    _, stats = hbg.trajectory_loss(model, blocked, 'cpu', record)
    assert stats['balance_edges'] == stats['decisions'] + 1 == 2
    complete = record_for(problem(), [1, 2, 3, 0])
    assert 0 <= complete['cost'] <= 1 < record['cost']
    assert 0 < math.exp(record['log_reward']) < math.exp(complete['log_reward'])


def test_every_vehicle_and_zero_demand_customer_are_preserved():
    model = policy()
    p = problem(capacities=(1, 2, 4), demands=(0, 0, 2, 4))
    result = hbg.solve_hbg(p, 31, {'seconds': 60, 'ants': 1, 'iterations': 1}, model, 'cpu')
    # Empty leading vehicles are legal; capacities fix only clients 2 and 3.
    assert len(result['routes']) == 3
    assert 2 in result['routes'][1] and 3 in result['routes'][2]
    assert sum(route.count(1) for route in result['routes']) == 1
    assert validate_routes(p.x, result['routes'])
    assert result['metadata']['method'] == 'hbg_facs'
    assert result['metadata']['search_violation_minutes'] <= 1e-9
    p2 = problem(capacities=(1, 4), demands=(0, 2, 2))
    record = record_for(p2, [0, 1, 2, 0])
    _, stats = hbg.trajectory_loss(model, p2, 'cpu', record)
    assert stats['decisions'] == p2.n + p2.vehicles
    assert record['routes'][0] == []


@pytest.mark.parametrize('field,value', [
    ('instance_sha', 'other'), ('cost', 0.), ('log_reward', 2.), ('temperature', .5),
    ('routes', [[3, 2, 1]]), ('unserved', 1), ('termination', 'dead_end'),
    ('normalization_scale_minutes', .01), ('format_version', 9),
    ('actions', [1, 1, 3, 0]), ('actions', [1, 2, 3]), ('actions', [True, 2, 3, 0]),
])
def test_replay_rejects_tampered_rewards_routes_and_nonterminal_prefixes(field, value):
    p = problem()
    record = record_for(p, [1, 2, 3, 0])
    record[field] = value
    with pytest.raises((ValueError, FloatingPointError)):
        hbg.validate_record(p, record)


def test_permuted_node_layout_keeps_actions_and_balance():
    model, p = policy().eval(), problem()
    record = record_for(p, [1, 2, 3, 0])
    first, first_stats = hbg.trajectory_loss(model, p, 'cpu', record)
    second, second_stats = hbg.trajectory_loss(model, p, 'cpu', record, node_order=[3, 1, 2])
    assert first.item() == pytest.approx(second.item(), rel=1e-5, abs=1e-6)
    assert first_stats['logPF'] == pytest.approx(second_stats['logPF'], rel=1e-5, abs=1e-6)
    assert hbg.validate_record(p, record)['actions'] == [1, 2, 3, 0]


def test_flow_consumes_prefix_in_original_sequence_even_when_nodes_permuted(monkeypatch):
    model, p = policy(), problem()
    original = model.advance_prefix
    observed = []

    def observe(encoded, local_action, hidden):
        observed.append(local_action)
        return original(encoded, local_action, hidden)

    monkeypatch.setattr(model, 'advance_prefix', observe)
    hbg.trajectory_loss(model, p, 'cpu', record_for(p, [1, 2, 3, 0]), node_order=[3, 1, 2])
    assert observed == [2, 3, 1, 0]


def test_replay_deduplicates_keeps_best_complete_and_returns_copies():
    p = problem()
    first = record_for(p, [1, 2, 3, 0])
    other = record_for(p, [1, 3, 2, 0])
    blocked = record_for(problem(capacities=(1,), demands=(0, 2)), [])
    replay = hbg.insert_elite([], [first, other, first, blocked], capacity=1)
    assert len(replay) == 1 and replay[0]['cost'] == min(first['cost'], other['cost'])
    sample = hbg.sample_replay(replay, 9, np.random.default_rng(11))
    assert sample == replay and sample is not replay and sample[0] is not replay[0]
    sample[0]['actions'].clear()
    assert replay[0]['actions']
    assert hbg.sample_replay([], 2, np.random.default_rng(0)) == []
    incompatible = copy.deepcopy(first)
    incompatible['temperature'] = .5
    with pytest.raises(ValueError, match='incompatibles'):
        hbg.insert_elite([first], [incompatible], 3)


def test_pheromone_only_reinforces_complete_directed_vehicle_edges():
    p = problem(capacities=(2, 2))
    record = record_for(p, [1, 2, 0, 3, 0])
    tau = np.ones((2, 4, 4))
    hbg._reinforce(tau, [record], .1)
    delta = 1 / (1 + record['cost'])
    assert tau[0, 0, 1] == pytest.approx(.9 + delta)
    assert tau[0, 1, 2] == pytest.approx(.9 + delta)
    assert tau[1, 0, 3] == pytest.approx(.9 + delta)
    assert tau[1, 3, 0] == pytest.approx(.9 + delta)
    assert tau[0, 1, 0] == pytest.approx(.9)
    assert tau[1, 0, 1] == pytest.approx(.9)
    empty = np.ones_like(tau)
    hbg._reinforce(empty, [], .1)
    assert np.allclose(empty, .9)


def test_tb_db_use_pure_policy_and_do_not_depend_on_colony_hyperparameters():
    model, p = policy().eval(), problem()
    record = record_for(p, [2, 1, 3, 0])
    first, _ = hbg.trajectory_loss(model, p, 'cpu', record, {'alpha': 0., 'evaporation': .2})
    second, _ = hbg.trajectory_loss(model, p, 'cpu', record, {'alpha': 99., 'evaporation': .7})
    assert first.item() == second.item()


def test_validation_reproducible_without_mutating_mode_or_parameters():
    model, p = policy().train(), problem()
    params = {'ants': 2, 'iterations': 2}
    first = hbg.validation_score(model, p, 'cpu', params, seed=4)
    second = hbg.validation_score(model, p, 'cpu', params, seed=4)
    assert first == second and model.training
    assert first['best_complete'] and first['complete_fraction'] == 1.
    assert first['rollout_count'] == 4
    one, _ = hbg.collect_colony(model, p, 'cpu', np.random.default_rng(5), params)
    two, _ = hbg.collect_colony(model, p, 'cpu', np.random.default_rng(5), params)
    assert one == two


def test_zero_budget_skips_encoding_and_search(monkeypatch):
    def forbidden(*args, **kwargs):
        raise AssertionError('No codificar ni construir sin presupuesto.')

    monkeypatch.setattr(hbg, '_prepare', forbidden)
    monkeypatch.setattr(hbg, '_sample_ant', forbidden)
    result = hbg.solve_hbg(problem(), 0, {'seconds': 0}, policy(), 'cpu')
    assert result['routes'] is None and result['metadata']['attempts'] == 0
    assert result['metadata']['termination'] == 'time_limit'


def test_completed_incumbent_survives_shared_deadline(monkeypatch):
    clock, deadlines = [0.], []
    p, model = problem(), policy()
    record = record_for(p, [1, 2, 3, 0])

    def completed(model, problem, device, prepared, tau, rng, params, deadline):
        deadlines.append(deadline)
        clock[0] = 11.
        return record

    monkeypatch.setattr(hbg.time, 'perf_counter', lambda: clock[0])
    monkeypatch.setattr(hbg, '_sample_ant', completed)
    result = hbg.solve_hbg(p, 1, {'seconds': 10}, model, 'cpu')
    assert deadlines == [10.] and result['routes'] == [[1, 2, 3]]
    assert result['solve_seconds'] == 11. and result['metadata']['budget_overrun_seconds'] == 1.
    assert result['metadata']['completed_decodes'] == 1
    assert result['metadata']['termination'] == 'time_limit'


def test_interrupted_ant_never_enters_records_or_elite(monkeypatch):
    monkeypatch.setattr(hbg, '_sample_ant', lambda *args: None)
    records, stats = hbg.collect_colony(policy(), problem(), 'cpu', np.random.default_rng(4),
                                      {'ants': 3, 'iterations': 4})
    assert records == [] and stats['rollout_count'] == 0 and stats['mean_cost'] is None
    assert stats['attempts'] == stats['interrupted_count'] == 1
    assert stats['termination'] == 'time_limit'


@pytest.mark.parametrize('params', [[], {'unknown': 1}, {'ants': True}, {'ants': 0},
                                    {'iterations': 1.5}, {'alpha': -1}, {'evaporation': 0},
                                    {'evaporation': 1}, {'temperature': 0}, {'temperature': math.nan},
                                    {'lambda_db': -1}, {'lambda_db': 1.1}, {'constructive_seconds': 0},
                                    {'constructive_seconds': math.nan}, {'constructive_seconds': True}])
def test_invalid_hbg_configuration_rejected(params):
    with pytest.raises(ValueError):
        hbg.validate_hbg_params(params)


def test_dropout_is_rejected_for_off_policy_recomputation():
    with pytest.raises(ValueError, match='dropout=0'):
        hbg.HBGPolicy({'model_dim': 16, 'num_heads': 4, 'num_layers': 1, 'dropout': .1})


def test_homogeneous_fleet_shares_pheromone_across_vehicles(monkeypatch):
    model, p = policy(), problem(capacities=(2, 2))
    record = record_for(p, [1, 2, 0, 3, 0])
    observed = []

    def completed(model, problem, device, prepared, tau, rng, params, deadline):
        observed.append(tau.copy())
        return copy.deepcopy(record)

    monkeypatch.setattr(hbg, '_sample_ant', completed)
    _, stats = hbg.collect_colony(model, p, 'cpu', np.random.default_rng(11),
                                  {'ants': 1, 'iterations': 2})
    assert len(observed) == 2 and observed[0].shape == (1, 4, 4)
    assert observed[1][0, 0, 1] > .9 and observed[1][0, 0, 3] > .9
    assert stats['neural_complete_count'] == 2
    assert stats['constructive_seed_count'] == 0


def test_shared_pheromone_does_not_reinforce_artificial_empty_vehicle_arc():
    p = problem(capacities=(4, 4, 4))
    record = record_for(p, [1, 2, 3, 0, 0, 0])
    tau = np.ones((1, 4, 4))
    hbg._reinforce(tau, [record], .1)
    assert tau[0, 0, 0] == pytest.approx(.9)
    assert tau[0, 0, 1] > .9 and tau[0, 3, 0] > .9


def test_identical_seed_and_neural_trajectory_deduplicate_without_rejecting_provenance():
    p = problem()
    neural = record_for(p, [1, 2, 3, 0])
    seeded = copy.deepcopy(neural)
    seeded['generation_source'] = 'paid_edd_constructive_seed'
    replay = hbg.insert_elite([], [seeded, neural], capacity=4)
    assert len(replay) == 1
    assert set(replay[0]['generation_sources']) == {'paid_edd_constructive_seed', 'neural_pheromone_colony'}
    assert hbg.validate_record(p, replay[0])['actions'] == neural['actions']
    # It must still reject altered substantive reward/temperature fields.
    changed = copy.deepcopy(neural)
    changed['log_reward'] -= .5
    with pytest.raises(ValueError, match='incompatibles'):
        hbg.insert_elite(replay, [changed], capacity=4)


def test_paid_seed_survives_timeout_without_starting_encoder(monkeypatch):
    from svrplab import baselines

    p, model, clock, budgets = problem(), policy(), [0.], []
    monkeypatch.setattr(hbg.time, 'perf_counter', lambda: clock[0])

    def paid_seed(problem, seed, config):
        budgets.append(config['seconds'])
        clock[0] = 11.
        return {'routes': [[1, 2, 3]]}

    monkeypatch.setattr(baselines, 'solve_edd', paid_seed)
    monkeypatch.setattr(hbg, '_prepare', lambda *a, **k: pytest.fail('Do not encode after shared deadline.'))
    monkeypatch.setattr(hbg, '_sample_ant', lambda *a, **k: pytest.fail('Do not sample after shared deadline.'))
    records, stats = hbg.collect_colony(model, p, 'cpu', np.random.default_rng(11),
                                       {'constructive_seed': True, 'ants': 1, 'iterations': 1}, deadline=10.)
    assert budgets and 0 < budgets[0] <= 10.
    assert len(records) == stats['constructive_seed_count'] == 1
    assert stats['neural_complete_count'] == 0 and stats['neural_complete_fraction'] == 0.
    assert stats['termination'] == 'time_limit' and stats['elapsed_seconds'] == 11.


def test_depot_guided_sampling_uses_greedy_at_customers_only():
    model, p = policy().eval(), problem()

    class FirstAllowed:
        def __init__(self):
            self.calls = 0

        def choice(self, count, p):
            self.calls += 1
            return int(np.flatnonzero(p > 0)[0])

    rng = FirstAllowed()
    params = hbg.validate_hbg_params({'depot_guided': True})
    with torch.no_grad():
        prepared = hbg._prepare(model, p, torch.device('cpu'), None)
        record = hbg._sample_ant(model, p, torch.device('cpu'), prepared,
                                 np.ones((1, 4, 4)), rng, params, deadline=None)
    assert record['termination'] == 'complete'
    assert rng.calls == 1  # Departure only; remaining choices are greedy.
    assert validate_routes(p.x, record['routes'])


def test_lower_temperature_separates_sink_reward_without_zero_or_nan():
    blocked = problem(capacities=(1,), demands=(0, 2))
    state = _State(blocked)
    old = hbg._terminal_record(state, [], 1.)
    sharp = hbg._terminal_record(state, [], .05)
    assert sharp['cost'] == old['cost']
    assert sharp['log_reward'] == pytest.approx(old['log_reward'] / .05)
    assert 0 < math.exp(sharp['log_reward']) < math.exp(old['log_reward'])
    hbg.validate_record(blocked, sharp, {'temperature': .05})


@pytest.mark.parametrize('alpha,expected_guided', [(1., 2), (0., 0)])
def test_pheromone_diagnostics_count_only_attempts_after_actual_deposit(monkeypatch, alpha, expected_guided):
    model, p = policy(), problem()
    complete = record_for(p, [1, 2, 3, 0])
    monkeypatch.setattr(hbg, '_sample_ant', lambda *args: copy.deepcopy(complete))
    _, stats = hbg.collect_colony(model, p, 'cpu', np.random.default_rng(3),
                                  {'ants': 2, 'iterations': 2, 'alpha': alpha})
    assert stats['pheromone_deposit_count'] == 2
    assert stats['pheromone_guided_attempts'] == expected_guided
    assert stats['pheromone_behavior_active'] is (expected_guided > 0)


def test_colony_without_complete_trajectories_reports_inactive_pheromone(monkeypatch):
    model = policy()
    p = problem(capacities=(1,), demands=(0, 2))
    dead_end = record_for(p, [])
    monkeypatch.setattr(hbg, '_sample_ant', lambda *args: copy.deepcopy(dead_end))
    records, stats = hbg.collect_colony(model, p, 'cpu', np.random.default_rng(3),
                                        {'ants': 2, 'iterations': 2})
    assert len(records) == 4 and stats['complete_count'] == 0
    assert stats['pheromone_deposit_count'] == stats['pheromone_guided_attempts'] == 0
    assert stats['pheromone_behavior_active'] is False


Overwriting tests/test_hbg.py


In [64]:
%%writefile tests/test_pomo.py
"""Controles funcionales de 04: ejecutar exclusivamente en Google Colab.

Localmente sólo se permite analizar AST; no importar ni ejecutar este archivo.
"""
import math
from pathlib import Path
import sys

import numpy as np
import pytest

torch = pytest.importorskip('torch')
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import pomo
from svrplab.attention import AttentionPolicy, _State
from svrplab.pilot import Traffic, freeze_instance, validate_routes
from svrplab.search import SearchProblem


def problem(*, capacities=(10,), demands=(0, 1, 1, 1), windows=None, appear_times=None):
    n = len(demands) - 1
    instance = freeze_instance({
        'instance_id': 'pomo-manual-fixture', 'depot_indices': [0],
        'locations': [[j, 0] for j in range(n + 1)], 'demands': list(demands),
        'capacities': list(capacities), 'windows': windows or [[0, 100]] * (n + 1),
        'appear_times': appear_times or [0] * (n + 1),
        'provenance': {'kind': 'test_fixture'},
    })
    return SearchProblem(instance, Traffic(mode='no_noise'))


def policy():
    torch.manual_seed(29)
    return AttentionPolicy({'model_dim': 16, 'num_heads': 4, 'num_layers': 1, 'dropout': 0.})


def test_group_has_distinct_starts_finite_gradients_and_excludes_forced_action(monkeypatch):
    p, model = problem(), policy()
    original = model.next_logits
    observed_nodes = []

    def observe(encoded, current_local, state_features, action_features, blocked):
        observed_nodes.append(current_local)
        return original(encoded, current_local, state_features, action_features, blocked)

    monkeypatch.setattr(model, 'next_logits', observe)
    original_sha = p.x.sha
    loss, stats = pomo.group_loss(model, p, 'cpu', np.random.default_rng(4), {'n_starts': 3})
    assert sorted(stats['forced_starts']) == [1, 2, 3]
    assert stats['rollout_count'] == stats['complete_count'] == 3
    assert stats['complete_fraction'] == 1.
    assert stats['decisions'] == 3 * (p.n + p.vehicles - 1)
    # En una única ruta, ninguna decisión del modelo puede ocurrir en depósito:
    # el primer cliente ya fue impuesto antes de invocar al decoder.
    assert 0 not in observed_nodes
    assert abs(sum(stats['advantages'])) < 1e-6
    assert torch.isfinite(loss) and loss.ndim == 0
    loss.backward()
    gradients = [parameter.grad for parameter in model.parameters() if parameter.grad is not None]
    assert gradients and all(bool(torch.isfinite(gradient).all()) for gradient in gradients)
    assert p.x.sha == original_sha


def test_cost_minimization_sign_and_shared_detached_baseline(monkeypatch):
    model, p = policy(), problem()
    parameter = torch.tensor(2., requires_grad=True)
    rows = iter([(0.1, 1.), (0.7, 3.)])

    def controlled(*args, **kwargs):
        cost, coefficient = next(rows)
        return pomo._Rollout([[1, 2, 3]], coefficient * parameter, 2, cost, 0, 'complete')

    monkeypatch.setattr(pomo, '_rollout', controlled)
    loss, stats = pomo.group_loss(model, p, 'cpu', np.random.default_rng(1), {'n_starts': 2})
    assert stats['baseline'] == pytest.approx(.4)
    assert stats['advantages'] == pytest.approx([-.3, .3])
    loss.backward()
    # Gradiente de ((.1-.4)*theta + (.7-.4)*3*theta)/2.
    assert parameter.grad.item() == pytest.approx(.3)


def test_single_feasible_start_uses_two_sampled_continuations():
    p = problem(capacities=(1, 4), demands=(0, 1, 2, 2))
    _, stats = pomo.group_loss(policy(), p, 'cpu', np.random.default_rng(7), {'n_starts': 8})
    assert stats['forced_starts'] == [1, 1]
    assert stats['rollout_count'] == 2 and stats['complete_fraction'] == 1.
    assert stats['decisions'] == 2 * (p.n + p.vehicles - 1)


def test_leading_empty_vehicle_retains_heterogeneous_fleet_and_zero_demand():
    # El cliente de demanda cero también debe atenderse; no puede saltarse.
    p = problem(capacities=(1, 2, 4), demands=(0, 0, 2, 4))
    result = pomo.solve_pomo(p, 31, {'seconds': 60, 'n_starts': 3, 'sampling_rounds': 0}, policy(), 'cpu')
    assert result['status'] == 'feasible' and validate_routes(p.x, result['routes'])
    assert len(result['routes']) == 3
    assert result['routes'][0] == [1]
    assert result['routes'][1] == [2]
    assert result['routes'][2] == [3]
    assert result['metadata']['search_violation_minutes'] <= 1e-9


def test_vehicle_without_any_admissible_customer_can_be_closed_legally():
    p = problem(capacities=(1, 4), demands=(0, 2, 2))
    _, stats = pomo.group_loss(policy(), p, 'cpu', np.random.default_rng(7), {'n_starts': 8})
    assert stats['leading_empty_vehicles'] == 1
    assert stats['complete_fraction'] == 1.
    assert stats['decisions'] == 2 * (p.n + p.vehicles - 2)
    result = pomo.solve_pomo(p, 0, {'seconds': 60, 'n_starts': 2, 'sampling_rounds': 0}, policy(), 'cpu')
    assert result['routes'][0] == []
    assert validate_routes(p.x, result['routes'])


def test_masks_reject_appearance_window_and_return_dead_ends():
    p = problem(windows=[[0, 10], [0, .5], [0, 10], [0, 10]], appear_times=[0, 0, 9, 0])
    _, starts = pomo._starting_customers(p)
    assert starts.tolist() == []  # La anticipación detecta al cliente 1 imposible.
    result = pomo.solve_pomo(p, 17, {'seconds': 60, 'n_starts': 2, 'sampling_rounds': 1}, policy(), 'cpu')
    assert result['routes'] is None and result['status'] == 'no_solution'
    assert result['metadata']['completed_decodes'] == 0
    assert result['metadata']['no_solution_scope'] == 'decoder_dead_end_or_budget_not_infeasibility_proof'


def test_fully_blocked_group_is_bounded_differentiable_and_does_not_decode(monkeypatch):
    p, model = problem(capacities=(1,), demands=(0, 2)), policy()

    def forbidden(*args, **kwargs):
        raise AssertionError('No llamar softmax cuando no hay acciones.')

    monkeypatch.setattr(model, 'next_logits', forbidden)
    loss, stats = pomo.group_loss(model, p, 'cpu', np.random.default_rng(9), {'n_starts': 2})
    assert stats['forced_starts'] == [None, None]
    assert stats['complete_fraction'] == 0 and stats['decisions'] == 0
    assert stats['costs'] == [2., 2.]
    assert loss.item() == 0 and loss.requires_grad
    loss.backward()
    assert all(parameter.grad is None or bool(torch.isfinite(parameter.grad).all())
               for parameter in model.parameters())


def test_blocked_penalty_dominates_every_complete_route_and_rewards_coverage():
    p = problem()
    scale = pomo._cost_scale(p)
    empty, partial, complete = _State(p), _State(p), _State(p)
    partial.advance(3)
    for customer in (1, 2, 3, 0):
        complete.advance(customer)
    empty_cost, _, _ = pomo._state_cost(empty, scale)
    partial_cost, _, _ = pomo._state_cost(partial, scale)
    complete_cost, missing, done = pomo._state_cost(complete, scale)
    assert done and missing == 0
    assert 0 <= complete_cost <= 1 < partial_cost < empty_cost < 3
    assert scale >= p.score([[2, 1, 3]])[1]


def test_explicit_rng_reproduces_group_and_permuted_nodes_remain_original_ids():
    model, p = policy().eval(), problem()
    _, first = pomo.group_loss(model, p, 'cpu', np.random.default_rng(52), {'n_starts': 3})
    _, second = pomo.group_loss(model, p, 'cpu', np.random.default_rng(52), {'n_starts': 3})
    assert first == second
    result = pomo.solve_pomo(p, 42, {'seconds': 60, 'n_starts': 3,
                                   'sampling_rounds': 1, 'permute_samples': True}, model, 'cpu')
    assert validate_routes(p.x, result['routes'])
    assert result['metadata']['attempts'] == 6


def test_validation_is_deterministic_without_changing_model_mode():
    p, model = problem(), policy().train()
    first = pomo.validation_score(model, p, 'cpu', {'n_starts': 3})
    second = pomo.validation_score(model, p, 'cpu', {'n_starts': 3})
    assert first == second and model.training
    assert first['best_complete'] and first['complete_fraction'] == 1.
    assert first['rollout_count'] == 3 and math.isfinite(first['score'])
    assert first['forced_starts'] == [1, 2, 3]


def test_inference_zero_budget_does_not_encode_or_search(monkeypatch):
    def forbidden(*args, **kwargs):
        raise AssertionError('No iniciar construcción sin presupuesto.')

    monkeypatch.setattr(pomo, '_prepare', forbidden)
    monkeypatch.setattr(pomo, '_starting_customers', forbidden)
    result = pomo.solve_pomo(problem(), 1, {'seconds': 0}, policy(), 'cpu')
    assert result['routes'] is None and result['metadata']['attempts'] == 0
    assert result['metadata']['termination'] == 'time_limit'


def test_completed_incumbent_survives_shared_deadline(monkeypatch):
    clock, deadlines = [0.], []

    def completed(model, problem, device, prepared, prefix, first, mode, rng=None, deadline=None):
        deadlines.append(deadline)
        clock[0] = 11.
        return pomo._Rollout([[1, 2, 3]], torch.tensor(0.), 3, .5, 0, 'complete')

    monkeypatch.setattr(pomo.time, 'perf_counter', lambda: clock[0])
    monkeypatch.setattr(pomo, '_rollout', completed)
    result = pomo.solve_pomo(problem(), 1, {'seconds': 10}, policy(), 'cpu')
    assert deadlines == [10.]
    assert result['routes'] == [[1, 2, 3]]
    assert result['solve_seconds'] == 11. and result['metadata']['budget_overrun_seconds'] == 1.
    assert result['metadata']['termination'] == 'time_limit'


@pytest.mark.parametrize('params', [{'n_starts': 0}, {'n_starts': 1}, {'n_starts': True},
                                    {'n_starts': 2.5}, {'unknown': 2}, []])
def test_invalid_group_configuration_rejected(params):
    with pytest.raises(ValueError):
        pomo.validate_pomo_params(params)


def test_dropout_rejected_for_reinforce_policy():
    model = AttentionPolicy({'model_dim': 16, 'num_heads': 4, 'num_layers': 1, 'dropout': .1})
    with pytest.raises(ValueError, match='dropout=0'):
        pomo.group_loss(model, problem(), 'cpu', np.random.default_rng(2), {'n_starts': 2})


def test_all_starts_means_all_admissible_customers_when_budget_allows():
    p = problem()
    result = pomo.solve_pomo(p, 3, {'seconds': 60, 'n_starts': 2, 'all_starts': True,
                                   'sampling_rounds': 0}, policy(), 'cpu')
    assert result['metadata']['attempts'] == 3
    assert result['metadata']['eligible_first_customers'] == 3
    assert result['metadata']['all_starts_completed'] is True
    assert result['metadata']['raw_decode_complete_fraction'] == 1.


@pytest.mark.parametrize('config', [{'all_starts': 1}, {'completion_search_nodes': -1},
                                   {'completion_search_nodes': True}])
def test_new_inference_contract_rejects_invalid_settings(config):
    with pytest.raises(ValueError):
        pomo.solve_pomo(problem(), 0, config, policy(), 'cpu')


def test_sampling_does_not_consume_global_torch_rng():
    model, p = policy(), problem()
    before = torch.get_rng_state().clone()
    pomo.group_loss(model, p, 'cpu', np.random.default_rng(8), {'n_starts': 3})
    assert torch.equal(before, torch.get_rng_state())


def test_pomo_completion_search_is_identified_separately_from_raw_rollouts(monkeypatch):
    p, model = problem(), policy()
    def blocked(model, problem, device, prepared, *args, **kwargs):
        return pomo._Rollout(None, prepared[0].sum() * 0., 1, 2., 2, 'dead_end')
    monkeypatch.setattr(pomo, '_rollout', blocked)
    result = pomo.solve_pomo(p, 2, {'seconds': 60., 'n_starts': 2, 'sampling_rounds': 0,
                                   'completion_search_nodes': 20}, model, 'cpu')
    assert validate_routes(p.x, result['routes'])
    assert result['metadata']['solution_origin'] == 'policy_completion_search'
    assert result['metadata']['completed_decodes'] == 0
    assert result['metadata']['raw_decode_complete_fraction'] == 0.


Overwriting tests/test_pomo.py


In [65]:
%%writefile tests/test_attention.py
"""Pruebas funcionales de 03: ejecutar exclusivamente en Google Colab."""
import math
from pathlib import Path
import sys

import numpy as np
import pytest

torch = pytest.importorskip('torch')
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import attention as am
from svrplab.pilot import Traffic, freeze_instance, validate_routes
from svrplab.search import SearchProblem


def problem(*, capacities=(1, 2, 4), demands=(0, 0, 2, 4),
            windows=None, appear_times=None):
    n = len(demands) - 1
    x = freeze_instance({
        'instance_id': 'attention-manual-fixture', 'depot_indices': [0],
        'locations': [[j, 0] for j in range(n + 1)], 'demands': list(demands),
        'capacities': list(capacities),
        'windows': windows or [[0, 100]] * (n + 1),
        'appear_times': appear_times or [0] * (n + 1),
        'provenance': {'kind': 'test_fixture'},
    })
    return SearchProblem(x, Traffic(mode='no_noise'))


def policy():
    torch.manual_seed(19)
    return am.AttentionPolicy({'model_dim': 16, 'num_heads': 4,
                               'num_layers': 2, 'dropout': 0.})


def test_teacher_loss_finite_gradients_zero_demand_and_empty_vehicles():
    p = problem()
    model = policy()
    before = p.x.sha
    loss, stats = am.teacher_loss(model, p, [[], [1, 2], [3]], 'cpu')
    assert stats['decisions'] == p.n + p.vehicles
    assert math.isfinite(stats['nll_sum'])
    assert loss.ndim == 0 and torch.isfinite(loss)
    loss.backward()
    gradients = [parameter.grad for parameter in model.parameters() if parameter.grad is not None]
    assert gradients and all(torch.isfinite(gradient).all() for gradient in gradients)
    assert any(bool((gradient != 0).any()) for gradient in gradients)
    assert p.x.sha == before


def test_teacher_loss_is_equivariant_to_input_customer_order_in_eval():
    p = problem()
    model = policy().eval()
    routes = [[], [1, 2], [3]]
    first, first_stats = am.teacher_loss(model, p, routes, 'cpu')
    permuted, permuted_stats = am.teacher_loss(model, p, routes, 'cpu', node_order=[3, 1, 2])
    assert first_stats['decisions'] == permuted_stats['decisions']
    assert torch.allclose(first, permuted, atol=2e-5, rtol=2e-5)


@pytest.mark.parametrize('routes', [
    [[], [2], [3]],             # Falta cliente de demanda cero.
    [[1], [2, 2], [3]],         # Repetido.
    [[2], [1], [3]],            # Capacidad del primer vehículo.
    [[0], [1, 2], [3]],         # Depósito no es cliente.
])
def test_invalid_teacher_is_rejected(routes):
    with pytest.raises(ValueError):
        am.teacher_loss(policy(), problem(), routes, 'cpu')


def test_teacher_that_misses_window_is_rejected():
    p = problem(windows=[[0, 100], [0, .5], [0, 100], [0, 100]])
    with pytest.raises(ValueError, match='conservadora'):
        am.teacher_loss(policy(), p, [[], [1, 2], [3]], 'cpu')


def test_masks_enforce_appearance_window_and_depot_return():
    p = problem(capacities=(10,), demands=(0, 1, 1, 1),
                windows=[[0, 10], [0, .5], [0, 10], [0, 10]],
                appear_times=[0, 0, 9, 0])
    state = am._State(p)
    _, _, blocked = state.inputs(np.arange(4), am._scales(p))
    assert blocked.tolist() == [True, True, True, True]
    # Cliente 1 llega después del cierre; cliente 2 espera hasta 9 y regresaría a 11.
    state.advance(3)
    _, _, after = state.inputs(np.arange(4), am._scales(p))
    assert after[3]                         # No revisitar.
    assert after[0]                         # Clientes pendientes en último vehículo.
    assert am.decode(policy(), p, 'cpu') is None


def test_empty_routes_allowed_and_last_vehicle_must_serve_zero_demand():
    p = problem(capacities=(1, 1), demands=(0, 0))
    state = am._State(p)
    order, scales = np.arange(2), am._scales(p)
    assert not state.inputs(order, scales)[2][0]
    state.advance(0)
    assert state.inputs(order, scales)[2].tolist() == [True, False]
    state.advance(1)
    assert state.inputs(order, scales)[2].tolist() == [False, True]
    state.advance(0)
    assert state.done and state.routes == [[], [1]]


def test_heterogeneous_capacity_masks_and_future_capacity_reservation():
    p = problem(capacities=(4, 2), demands=(0, 4, 2))
    state = am._State(p)
    order, scales = np.arange(3), am._scales(p)
    _, _, blocked = state.inputs(order, scales)
    assert blocked.tolist() == [True, False, True]
    state.advance(1)
    assert state.inputs(order, scales)[2].tolist() == [False, True, True]
    state.advance(0)
    assert state.inputs(order, scales)[2].tolist() == [True, True, False]


def test_greedy_preserves_fleet_coverage_and_original_node_ids(monkeypatch):
    p = problem()
    before = p.x.sha
    model = policy()

    def ranked(encoded, current_local, state_features, action_features, blocked):
        return torch.arange(len(blocked), dtype=encoded.dtype).masked_fill(blocked, -torch.inf)

    monkeypatch.setattr(model, 'next_logits', ranked)
    routes = am.decode(model, p, 'cpu', node_order=[3, 1, 2])
    assert routes is not None and validate_routes(p.x, routes)
    assert p.score(routes)[0] <= 1e-9
    assert p.x.sha == before
    assert model.training                   # Restaurar modo previo tras inferencia.


def test_sampling_reproducible_with_explicit_generator():
    p = problem(capacities=(10,))
    model = policy()
    first = am.decode(model, p, 'cpu', mode='sample', generator=torch.Generator().manual_seed(7))
    second = am.decode(model, p, 'cpu', mode='sample', generator=torch.Generator().manual_seed(7))
    assert first is not None and first == second
    assert validate_routes(p.x, first)


def test_dead_end_never_uses_teacher_or_repair():
    p = problem(capacities=(1,), demands=(0, 2))
    result = am.solve_attention(p, 21, {'seconds': 60, 'n_samples': 2}, policy(), 'cpu')
    assert result['status'] == 'no_solution' and result['routes'] is None
    assert result['metadata']['completed_decodes'] == 0
    assert result['metadata']['failed_decodes'] == 3
    assert result['metadata']['no_solution_scope'] == 'decoder_dead_end_or_budget_not_infeasibility_proof'


def test_zero_budget_does_not_start_decode(monkeypatch):
    def forbidden(*args, **kwargs):
        raise AssertionError('No decodificar sin presupuesto.')
    monkeypatch.setattr(am, 'decode', forbidden)
    result = am.solve_attention(problem(), 2, {'seconds': 0, 'n_samples': 2}, policy(), 'cpu')
    assert result['routes'] is None
    assert result['metadata']['attempts'] == 0
    assert result['metadata']['termination'] == 'time_limit'


def test_completed_incumbent_survives_time_limit_without_extra_budget(monkeypatch):
    clock = [0.]
    deadlines = []
    p = problem()

    def completed(model, problem, device, **kwargs):
        deadlines.append(kwargs['deadline'])
        clock[0] = 11.
        return [[], [1, 2], [3]]

    monkeypatch.setattr(am.time, 'perf_counter', lambda: clock[0])
    monkeypatch.setattr(am, 'decode', completed)
    result = am.solve_attention(p, 7, {'seconds': 10, 'n_samples': 16}, policy(), 'cpu')
    assert deadlines == [10.]
    assert result['routes'] == [[], [1, 2], [3]]
    assert result['metadata']['termination'] == 'time_limit'
    assert result['metadata']['budget_includes_encoding'] is True
    assert result['solve_seconds'] == 11.
    assert result['metadata']['budget_overrun_seconds'] == 1.


def test_best_sample_is_selected_using_common_conservative_cost(monkeypatch):
    p = problem(capacities=(10,))
    candidates = iter([[[2, 1, 3]], [[1, 3, 2]], [[1, 2, 3]]])
    monkeypatch.setattr(am, 'decode', lambda *args, **kwargs: next(candidates))
    result = am.solve_attention(p, 11, {'seconds': 60, 'n_samples': 2}, policy(), 'cpu')
    assert result['routes'] == [[1, 3, 2]]     # Empata al mínimo; conserva el primero.
    assert result['metadata']['attempts'] == 3
    assert result['metadata']['completed_decodes'] == 3
    assert result['metadata']['search_objective_minutes'] == 6.
    assert [row['travel_minutes'] for row in result['history']] == [8., 6.]


@pytest.mark.parametrize('config', [
    {'model_dim': 15, 'num_heads': 4}, {'num_layers': 0}, {'dropout': 1.},
    {'tanh_clip': float('nan')}, {'unknown': 1}, {'dim': 16, 'model_dim': 16},
])
def test_model_configuration_rejects_invalid_values(config):
    with pytest.raises(ValueError):
        am.AttentionPolicy(config)


@pytest.mark.parametrize('order', [[0, 1, 2], [1, 1, 3], [1., 2., 3.], [True, 2, 3]])
def test_invalid_customer_permutation_rejected(order):
    with pytest.raises(ValueError, match='permutación'):
        am.decode(policy(), problem(), 'cpu', node_order=order)


def test_lookahead_prevents_serving_late_customer_before_urgent_one():
    p = problem(capacities=(10,), demands=(0, 1, 1),
                windows=[[0, 100], [0, 1.5], [0, 100]])
    blocked = am._State(p).inputs(np.arange(3), am._scales(p))[2]
    assert blocked.tolist() == [True, False, True]
    assert am.decode(policy(), p, 'cpu') == [[1, 2]]


def test_lookahead_never_masks_prefix_of_a_feasible_completion_nonmetric_or_metric():
    # Exhaustive tiny contract belongs in Colab only. No search on the bank.
    from itertools import permutations
    p = problem(capacities=(3, 3), demands=(0, 0, 2, 1),
                windows=[[0, 20], [0, 4], [0, 8], [0, 12]])
    for nonmetric in (False, True):
        if nonmetric:
            p.travel[1, 0] = 30.  # Returning through another customer can be shorter.
            p.travel[0, 3] = 15.
            p.__dict__.pop('_attention_travel_bound', None)
        checked = 0
        for permutation in permutations(range(1, p.n + 1)):
            for split in range(p.n + 1):
                routes = [list(permutation[:split]), list(permutation[split:])]
                try:
                    violation, _ = p.score(routes)
                except ValueError:
                    continue
                if violation > 1e-9:
                    continue
                state = am._State(p)
                for route in routes:
                    for customer in [*route, 0]:
                        assert not state.inputs(np.arange(p.n + 1), am._scales(p))[2][customer]
                        state.advance(customer)
                checked += 1
        assert checked > 0


def test_policy_completion_search_is_bounded_and_reports_non_sample_origin():
    p, model = problem(capacities=(10,)), policy()
    routes, stats = am.policy_completion_search(model, p, 'cpu', None, max_nodes=100)
    assert validate_routes(p.x, routes) and p.score(routes)[0] == 0.
    assert stats['expanded_nodes'] <= 100 and stats['termination'] == 'feasible_found'
    assert stats['is_policy_sample'] is False
    assert model.training
    routes, stats = am.policy_completion_search(model, p, 'cpu', None, max_nodes=1)
    assert routes is None and stats['termination'] == 'node_limit'


def test_completion_search_shares_inference_budget_and_is_not_counted_as_raw_decode(monkeypatch):
    p = problem(capacities=(10,))
    observed = []
    monkeypatch.setattr(am, 'decode', lambda *args, **kwargs: None)
    def search(model, problem, device, deadline, max_nodes):
        observed.append((deadline, max_nodes))
        return [[1, 2, 3]], {'termination': 'feasible_found', 'expanded_nodes': 4,
                             'is_policy_sample': False}
    monkeypatch.setattr(am, 'policy_completion_search', search)
    monkeypatch.setattr(am.time, 'perf_counter', lambda: 10.)
    result = am.solve_attention(p, 1, {'seconds': 30., 'n_samples': 2,
                                     'completion_search_nodes': 10}, policy(), 'cpu')
    assert observed == [(40., 10)]
    assert result['metadata']['solution_origin'] == 'policy_completion_search'
    assert result['metadata']['completed_decodes'] == 0
    assert result['metadata']['raw_decode_complete_fraction'] == 0.


def test_uniform_masked_policy_has_no_learned_parameters_and_equal_allowed_logits():
    model = am.UniformMaskedPolicy()
    assert list(model.parameters()) == []
    logits = model.next_logits(torch.zeros(3, 1), 0, None, None,
                               torch.tensor([True, False, False]))
    assert logits[0] == -torch.inf and logits[1] == logits[2] == 0.


Overwriting tests/test_attention.py


In [66]:
%%writefile tests/test_baselines.py
"""Conventional baselines: run exclusively in Colab, never locally."""
from pathlib import Path
import sys
import time

import numpy as np
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import baselines as b
from svrplab.pilot import Traffic, freeze_instance, validate_routes
from svrplab.search import SearchProblem


def problem(*, capacities=(3, 3, 3), demands=(0, 0, 1, 1), windows=None):
    n = len(demands) - 1
    x = freeze_instance({
        'instance_id': 'baseline-manual-fixture', 'depot_indices': [0],
        'locations': [[j, 0] for j in range(n + 1)],
        'demands': list(demands), 'capacities': list(capacities),
        'windows': windows or [[0, 100]] * (n + 1),
        'appear_times': [0] * (n + 1), 'provenance': {'kind': 'manual_test_fixture'},
    })
    return SearchProblem(x, Traffic(mode='no_noise'))


def test_edd_fixed_fleet_zero_demand_and_determinism():
    p = problem()
    before = p.x.sha
    a, c = b.solve_edd(p, 11), b.solve_edd(p, 47)
    assert a['status'] == c['status'] == 'feasible'
    assert a['routes'] == c['routes']
    assert validate_routes(p.x, a['routes'])
    assert len(a['routes']) == len(p.capacities)
    assert p.x.sha == before
    assert a['metadata']['seed_affects_search'] is False
    assert a['metadata']['reference_is_bound'] is False


def test_edd_inserts_before_existing_customer_when_append_would_be_late():
    p = problem(capacities=(5,), demands=(0, 1, 1), windows=[[0, 100], [5, 6], [0, 8]])
    # Customer 1 has the earlier closing time, but its opening time imposes
    # waiting. Insert customer 2 before it instead of appending after waiting.
    p.travel = np.array([[0., 1., 1.], [1., 0., 4.], [1., 4., 0.]])
    result = b.solve_edd(p, 11)
    assert result['routes'] == [[2, 1]]
    assert result['status'] == 'feasible'
    assert p.score([[1, 2]])[0] > 0


def test_impossible_window_returns_complete_diagnostic_without_feasibility_claim():
    p = problem(windows=[[0, 100]] + [[0, .01]] * 3)
    result = b.solve_edd(p, 11)
    assert result['status'] == 'conservative_infeasible'
    assert validate_routes(p.x, result['routes'])
    assert result['metadata']['search_violation_minutes'] > 0
    assert result['metadata']['surrogate_optimal'] is False


def test_capacity_dead_end_never_drops_customers_or_adds_vehicles():
    p = problem(capacities=(1,), demands=(0, 0, 1, 1))
    result = b.solve_edd(p, 11)
    assert result['routes'] is None and result['status'] == 'no_solution'
    assert result['metadata']['termination'] == 'capacity_dead_end'


def test_deadline_stops_incomplete_construction_without_returning_partial_routes(monkeypatch):
    clock = [0.]

    def now():
        clock[0] += .1
        return clock[0]

    monkeypatch.setattr(b.time, 'perf_counter', now)
    routes, reason = b.construct_insertion(problem(), np.random.default_rng(11), .25)
    assert routes is None and reason == 'time_limit'


def test_local_search_repairs_lateness_without_extra_vehicle_or_customer_loss():
    p = problem(capacities=(5,), demands=(0, 1, 1), windows=[[0, 100], [5, 6], [0, 8]])
    p.travel = np.array([[0., 1., 1.], [1., 0., 4.], [1., 4., 0.]])
    before = [[1, 2]]
    routes, stats = b.improve_feasible_routes(p, before, np.random.default_rng(11), time.perf_counter() + 30)
    assert before == [[1, 2]]
    assert p.score(routes)[0] == 0.
    assert validate_routes(p.x, routes)
    assert stats['accepted_moves'] >= 1


def test_local_search_improves_travel_and_never_replaces_feasible_with_late():
    p = problem(capacities=(3, 3, 3))
    before = [[1], [2], [3]]
    routes, stats = b.improve_feasible_routes(p, before, np.random.default_rng(11), time.perf_counter() + 30)
    assert p.score(routes) <= p.score(before)
    assert p.score(routes)[0] <= 1e-9
    assert validate_routes(p.x, routes)
    assert stats['evaluated_neighbors'] > 0


def test_local_search_zero_budget_retains_input_incumbent():
    p = problem()
    before = [[1], [2], [3]]
    routes, stats = b.improve_feasible_routes(p, before, np.random.default_rng(11), -1.)
    assert routes == before
    assert stats['accepted_moves'] == stats['evaluated_neighbors'] == 0
    assert stats['termination'] == 'time_limit'


def test_ortools_fixed_fleet_capacity_and_zero_demand_customer():
    pytest.importorskip('ortools')
    p = problem(capacities=(1, 1, 1))
    before = p.x.sha
    result = b.solve_ortools_routing(p, 29, {'seconds': 2.})
    assert result['status'] == 'feasible'
    assert validate_routes(p.x, result['routes'])
    assert len(result['routes']) == 3 and p.x.sha == before
    meta = result['metadata']
    assert not meta['surrogate_optimal'] and not meta['stochastic_optimal']
    assert not meta['reference_is_bound']
    assert meta['budget_includes_construction'] and meta['seed_affects_search'] is False
    assert meta['bound_scope'] == 'no_optimality_bound_from_routing_search'


def test_ortools_tight_windows_use_same_conservative_contract():
    pytest.importorskip('ortools')
    p = problem(capacities=(5,), demands=(0, 1, 1), windows=[[0, 100], [5, 6], [0, 8]])
    p.travel = np.array([[0., 1., 1.], [1., 0., 4.], [1., 4., 0.]])
    result = b.solve_ortools_routing(p, 11, {'seconds': 2.})
    assert result['routes'] == [[2, 1]]
    assert p.score(result['routes'])[0] == 0.


def test_ortools_zero_budget_does_not_import_solver(monkeypatch):
    monkeypatch.setattr(b, '_ortools', lambda: pytest.fail('No solver import without budget.'))
    result = b.solve_ortools_routing(problem(), 11, {'seconds': 0.})
    assert result['routes'] is None and result['metadata']['termination'] == 'time_limit'


def test_solver_import_is_part_of_budget(monkeypatch):
    clock = [0.]
    monkeypatch.setattr(b.time, 'perf_counter', lambda: clock[0])

    def delayed_import():
        clock[0] = 11.
        return object(), object()

    monkeypatch.setattr(b, '_ortools', delayed_import)
    result = b.solve_ortools_routing(problem(), 11, {'seconds': 10.})
    assert result['routes'] is None
    assert result['solve_seconds'] == 11.
    assert result['metadata']['budget_overrun_seconds'] == 1.


def test_integer_grid_cannot_be_loosened_to_find_solution():
    assert b._integer(1.25, 1000) == 1250
    with pytest.raises(ValueError, match='rejilla'):
        b._integer(1.2505, 1000)


def test_ortools_never_relabels_infeasible_edd_as_solver_solution():
    pytest.importorskip('ortools')
    p = problem(windows=[[0, 100]] + [[0, .01]] * 3)
    result = b.solve_ortools_routing(p, 11, {'seconds': 2.})
    assert result['routes'] is None and result['status'] == 'no_solution'
    assert result['metadata']['solution_origin'] == 'none'
    assert result['metadata']['edd_diagnostic_score'][0] > 0
    assert not result['metadata']['returned_paid_edd_fallback']


@pytest.mark.parametrize('solver,config', [
    (b.solve_edd, {'unknown': 1}), (b.solve_edd, {'seconds': float('nan')}),
    (b.solve_ortools_routing, {'seconds': -1}),
    (b.solve_ortools_routing, {'capacity_scale': 0}),
    (b.solve_ortools_routing, {'capacity_scale': True}),
])
def test_invalid_baseline_configuration_is_rejected(solver, config):
    with pytest.raises(ValueError):
        solver(problem(), 11, config)


Overwriting tests/test_baselines.py


In [67]:
%%writefile tests/test_metaheuristics.py
"""Pruebas para ejecutar exclusivamente en Colab; no ejecutarlas localmente."""
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import metaheuristics as mh
from svrplab.pilot import Traffic, freeze_instance, validate_routes
from svrplab.search import SearchProblem


def problem(*, impossible_window=False, impossible_capacity=False):
    x = freeze_instance({
        'instance_id': 'metaheuristic-manual-fixture', 'depot_indices': [0],
        'locations': [[0, 0], [1, 0], [2, 0], [3, 0], [-2, 0], [-3, 0]],
        'demands': [0, 0, 2, 2, 2, 2],
        'capacities': [3, 3] if impossible_capacity else [4, 4],
        'windows': [[0, 100]] + [[0, .1 if impossible_window else 80]] * 5,
        'appear_times': [0] * 6, 'provenance': {'kind': 'test_fixture'},
    })
    return SearchProblem(x, Traffic())


def small_config(solver, **overrides):
    config = {'seconds': 60, 'max_iterations': 4, 'local_search_seconds': 10., 'local_search_moves': 4}
    config.update({'ants': 4} if solver is mh.solve_aco else
                  {'initial_restarts': 4, 'neighborhood_samples': 30, 'tenure': 3})
    return config | overrides


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_complete_routes_capacity_and_zero_demand_customer(solver):
    p = problem()
    before = p.x.sha
    result = solver(p, 31, small_config(solver))
    assert result['status'] == 'feasible'
    assert validate_routes(p.x, result['routes'])
    assert sorted(j for route in result['routes'] for j in route) == [1, 2, 3, 4, 5]
    assert len(result['routes']) == 2
    assert p.score(result['routes'])[0] <= 1e-9
    assert p.x.sha == before
    assert result['solve_seconds'] >= 0
    assert result['metadata']['budget_includes_construction'] is True
    progress = [(row['violation_minutes'], row['travel_minutes']) for row in result['history']]
    assert progress and all(b <= a for a, b in zip(progress, progress[1:]))


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_impossible_windows_report_violation_without_losing_clients(solver):
    p = problem(impossible_window=True)
    result = solver(p, 19, small_config(solver))
    assert result['status'] == 'conservative_infeasible'
    assert validate_routes(p.x, result['routes'])
    assert p.score(result['routes'])[0] > 0
    # Es el estado de las rutas; no una prueba de infactibilidad del problema.
    assert result['metadata']['nominal_scope'] == 'conservative_upper_mean_matrix_not_stochastic_feasibility'


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_impossible_capacity_returns_explicit_no_solution(solver):
    result = solver(problem(impossible_capacity=True), 3, small_config(solver))
    assert result['routes'] is None
    assert result['status'] == 'no_solution'
    assert result['metadata']['termination'] == 'capacity_screen_failed'


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_iteration_limited_seed_is_reproducible(solver):
    p = problem()
    first = solver(p, 41, small_config(solver))
    second = solver(p, 41, small_config(solver))
    assert first['metadata']['termination'] == second['metadata']['termination'] == 'iteration_limit'
    assert first['routes'] == second['routes']
    assert first['status'] == second['status']
    assert [(r['violation_minutes'], r['travel_minutes']) for r in first['history']] == [
        (r['violation_minutes'], r['travel_minutes']) for r in second['history']]


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_zero_budget_does_not_construct(monkeypatch, solver):
    def forbidden(*args, **kwargs):
        raise AssertionError('La construcción no debe iniciarse sin presupuesto.')
    monkeypatch.setattr(mh, '_construct', forbidden)
    result = solver(problem(), 5, small_config(solver, seconds=0))
    assert result['status'] == 'no_solution' and result['routes'] is None
    assert result['metadata']['termination'] == 'time_limit'


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_construction_consumes_same_budget_and_preserves_completed_incumbent(monkeypatch, solver):
    clock = [0.]
    calls = []

    def construction(*args, **kwargs):
        calls.append(True)
        clock[0] = 11.
        return [[1, 2, 3], [4, 5]], 'complete'

    monkeypatch.setattr(mh.time, 'perf_counter', lambda: clock[0])
    monkeypatch.setattr(mh, '_construct', construction)
    result = solver(problem(), 7, small_config(solver, seconds=10))
    assert len(calls) == 1
    assert result['routes'] == [[1, 2, 3], [4, 5]]
    assert result['metadata']['termination'] == 'time_limit'
    assert result['solve_seconds'] == 11.
    assert result['metadata']['budget_overrun_seconds'] == 1.


@pytest.mark.parametrize('kind,first,second,inverse_first,inverse_second', [
    ('relocate', (0, 1), (1, 1), (1, 1), (0, 1)),
    ('swap', (0, 0), (1, 1), (0, 0), (1, 1)),
    ('2opt', (0, 0), (0, 2), (0, 0), (0, 2)),
    ('2opt_star', (0, 1), (1, 1), (0, 1), (1, 1)),
])
def test_tabu_blocks_actual_inverse_and_respects_aspiration_and_expiry(
        kind, first, second, inverse_first, inverse_second):
    before = [[1, 2, 3], [4, 5]]
    after = mh._apply_move(before, kind, first, second)
    restored = mh._apply_move(after, kind, inverse_first, inverse_second)
    assert restored == before
    removed = mh._arcs(before) - mh._arcs(after)
    inverse_added = mh._arcs(restored) - mh._arcs(after)
    assert removed and removed == inverse_added
    memory = {arc: 8 for arc in removed}
    assert not mh._tabu_admissible(inverse_added, memory, 6, (0., 10.), (0., 10.))
    assert mh._tabu_admissible(inverse_added, memory, 6, (0., 9.), (0., 10.))
    # Menor viaje nunca compensa mayor violación en la regla de aspiración.
    assert not mh._tabu_admissible(inverse_added, memory, 6, (1., 1.), (0., 10.))
    assert mh._tabu_admissible(inverse_added, memory, 8, (0., 10.), (0., 10.))


@pytest.mark.parametrize('solver,config', [
    (mh.solve_aco, {'q0': 1.1}), (mh.solve_aco, {'evaporation': 0}),
    (mh.solve_aco, {'ants': 1.2}), (mh.solve_aco, {'seconds': float('nan')}),
    (mh.solve_tabu, {'tenure': 0}), (mh.solve_tabu, {'initial_restarts': 0}),
    (mh.solve_tabu, {'neighborhood_samples': True}), (mh.solve_tabu, {'unknown': 1}),
])
def test_invalid_configuration_is_rejected(solver, config):
    with pytest.raises(ValueError):
        solver(problem(), 1, config)


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_time_only_mode_does_not_stop_at_legacy_200_iterations(monkeypatch, solver):
    clock = [0.]
    monkeypatch.setattr(mh.time, 'perf_counter', lambda: clock[0])

    def construction(*args, **kwargs):
        clock[0] += 1.
        return [[1, 2, 3], [4, 5]], 'complete'

    def neighbor(*args, **kwargs):
        clock[0] += 1.
        return None

    monkeypatch.setattr(mh, '_construct', construction)
    monkeypatch.setattr(mh, '_random_neighbor', neighbor)
    config = small_config(solver, seconds=205., max_iterations=None, local_search_seconds=0.)
    if solver is mh.solve_aco:
        config['ants'] = 1
    else:
        config.update(initial_restarts=1, neighborhood_samples=1)
    result = solver(problem(), 11, config)
    assert result['metadata']['iterations'] > 200
    assert result['metadata']['termination'] == 'time_limit'
    assert result['metadata']['effective_config']['max_iterations'] is None


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_paid_local_search_is_called_and_cannot_use_extra_budget(monkeypatch, solver):
    clock, calls = [0.], []
    monkeypatch.setattr(mh.time, 'perf_counter', lambda: clock[0])
    monkeypatch.setattr(mh, '_construct', lambda *a, **k: ([[1, 2, 3], [4, 5]], 'complete'))

    def improve(problem, routes, rng, deadline, max_moves):
        calls.append((deadline, max_moves))
        assert deadline <= 10.
        clock[0] = 11.
        return routes, {'evaluated_neighbors': 3, 'accepted_moves': 1}

    monkeypatch.setattr(mh, 'improve_feasible_routes', improve)
    result = solver(problem(), 11, small_config(solver, seconds=10., local_search_seconds=20.))
    assert len(calls) == 1
    assert result['metadata']['budget_includes_repair']
    assert result['metadata']['local_search_evaluated_neighbors'] == 3
    assert result['metadata']['local_search_accepted_moves'] == 1
    assert result['metadata']['termination'] == 'time_limit'
    assert result['solve_seconds'] == 11.


@pytest.mark.parametrize('solver', [mh.solve_aco, mh.solve_tabu])
def test_both_methods_keep_ordinary_edd_incumbent_when_later_construction_fails(monkeypatch, solver):
    calls = [0]

    def construct(*args, **kwargs):
        calls[0] += 1
        return ([[1, 2, 3], [4, 5]], 'complete') if calls[0] == 1 else (None, 'capacity_dead_end')

    monkeypatch.setattr(mh, '_construct', construct)
    result = solver(problem(), 11, small_config(solver, local_search_seconds=0.))
    assert result['status'] == 'feasible'
    assert result['metadata']['initialization'] == 'paid_edd_feasible_cheapest_insertion'
    assert result['metadata']['capacity_dead_ends'] > 0
    assert result['metadata']['budget_includes_construction']


Overwriting tests/test_metaheuristics.py


In [68]:
%%writefile tests/test_experiments.py
"""Controles del protocolo 01/02. Ejecutar exclusivamente desde Google Colab.

Los casos son manuales: no generan el banco, no buscan certificados y no usan
ninguna ruta testigo del banco como solución de un método.
"""
import copy
import csv
import json
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import pilot as p
from svrplab import experiments as e
from svrplab.search import SearchProblem


def manual_instance(windows=None, capacities=None):
    return p.freeze_instance({
        'instance_id': 'manual_runner', 'depot_indices': [0],
        'locations': [[0, 0], [3, 0], [6, 0]],
        'demands': [0, 0, 1], 'capacities': capacities or [5, 5],
        'windows': windows or [[0, 100], [5, 10], [0, 100]],
        'appear_times': [0, 0, 0], 'provenance': {'kind': 'manual_test'},
    })


def manual_solution(x, routes=None, spec=None):
    routes = [[1, 2], []] if routes is None else routes
    spec = p.Traffic(mode='no_noise') if spec is None else spec
    return {'routes': routes, 'nominal': p.simulate(x, routes, 0, spec, trace=True)}


def csv_rows(path):
    with Path(path).open(newline='', encoding='utf-8') as stream:
        return list(csv.DictReader(stream))


@pytest.mark.parametrize('scenarios', [
    {'train': [1, 2], 'selection': [2], 'evaluation': [4, 5]},
    {'train': [1, 2], 'selection': [3], 'evaluation': [4, 4]},
    {'train': [1, 2], 'selection': [3], 'evaluation': [4]},
    {'train': [1, 2], 'selection': [3], 'evaluation': [4, True]},
    {'train': [1, 2], 'selection': [3], 'evaluation': [4, 5.0]},
    {'train': [1, 2], 'selection': [3], 'evaluation': [4, -1]},
    {'train': [1, 2], 'evaluation': [4, 5]},
])
def test_scenario_contract_rejects_leakage_or_malformed_splits(scenarios):
    with pytest.raises(ValueError):
        e.validate_scenarios(scenarios, {'scenarios': {'train': 2, 'selection': 1, 'evaluation': 2}})


def test_scenario_contract_preserves_large_seeds_and_original_order():
    scenarios = {'train': [2**64 - 1, 7], 'selection': [3], 'evaluation': [2**53 + 1, 1]}
    before = copy.deepcopy(scenarios)
    e.validate_scenarios(scenarios, {'scenarios': {'train': 2, 'selection': 1, 'evaluation': 2}})
    assert scenarios == before


def test_csv_seeds_have_lossless_decimal_roundtrip(tmp_path):
    seeds = [2**53 + 1, 2**63 + 17, 2**64 - 1]
    path = tmp_path / 'seeds.csv'
    e.write_csv(path, [{'scenario_seed': str(seed), 'metric': 1.25} for seed in seeds])
    rows = csv_rows(path)
    assert [row['scenario_seed'] for row in rows] == [str(seed) for seed in seeds]
    assert [int(row['scenario_seed']) for row in rows] == seeds


def test_search_objective_and_clock_have_manual_values():
    x = manual_instance()
    problem = SearchProblem(x, p.Traffic(mode='no_noise'))
    before = x.sha
    stats = problem.route_stats([1, 2], 0)
    # 0->1: 3, espera 2; 1->2: 3; 2->0: 6. Viaje 12, retorno 14.
    assert stats['travel_minutes'] == 12
    assert stats['service_starts'] == [5, 8]
    assert stats['return_minutes'] == 14
    assert problem.score([[1, 2], []]) == (0, 12)
    assert x.sha == before


def test_search_objective_records_lateness_and_return_separately_from_distance():
    x = manual_instance([[0, 8], [0, 2], [0, 4]])
    problem = SearchProblem(x, p.Traffic(mode='no_noise'))
    stats = problem.route_stats([1, 2], 0)
    assert stats['lateness_minutes'] == 3  # (3-2) + (6-4).
    assert stats['depot_overtime_minutes'] == 4  # 12-8.
    assert problem.score([[1, 2], []]) == (7, 12)
    result = problem.result([[1, 2], []], 'time_limit', .1, [])
    assert result['metadata']['conservative_feasible'] is False
    assert result['metadata']['search_violation_minutes'] == 7
    assert result['metadata']['search_objective_minutes'] == 12


def test_search_never_accepts_missing_zero_demand_customer_or_excess_capacity():
    problem = SearchProblem(manual_instance(), p.Traffic(mode='no_noise'))
    with pytest.raises(ValueError):
        problem.score([[2], []])
    problem = SearchProblem(manual_instance(capacities=[.5, .5]), p.Traffic(mode='no_noise'))
    with pytest.raises(ValueError):
        problem.score([[1, 2], []])


@pytest.mark.parametrize('damage', ['metric', 'missing_event', 'invalid_customer', 'invalid_vehicle',
                                    'broken_arc', 'broken_clock'])
def test_trace_qc_rejects_corruption_even_when_aggregate_totals_match(damage):
    x = manual_instance()
    routes = [[1, 2], []]
    spec = p.Traffic(mode='no_noise')
    raw = p.simulate(x, routes, 1, spec)
    trace = p.simulate(x, routes, 1, spec, trace=True)
    if damage == 'metric':
        trace['travel_minutes'] += 1
    elif damage == 'missing_event':
        trace['events'].pop()
    elif damage == 'invalid_customer':
        trace['events'][0]['to'] = 99
    elif damage == 'invalid_vehicle':
        trace['events'][0]['vehicle'] = 99
    elif damage == 'broken_arc':
        trace['events'][1]['from'] = 0
    else:
        # Conserva duración, espera y tardanza agregadas, pero rompe continuidad.
        for name in ('departure', 'arrival', 'service_start'):
            trace['events'][1][name] += 1
    with pytest.raises((AssertionError, ValueError)):
        e.validate_trace(x, routes, trace, raw)


def test_trace_qc_rejects_vehicle_reassignment_with_same_metrics():
    x = manual_instance([[0, 100], [0, 100], [0, 100]])
    routes = [[1], [2]]
    spec = p.Traffic(mode='no_noise')
    raw = p.simulate(x, routes, 1, spec)
    trace = p.simulate(x, routes, 1, spec, trace=True)
    trace['realized_routes'] = [[2], [1]]
    with pytest.raises(AssertionError, match='asignación'):
        e.validate_trace(x, routes, trace, raw)


@pytest.mark.parametrize('policy', ['fixed_order', 'causal_edd'])
def test_evaluation_exports_common_metrics_and_every_route_event(tmp_path, policy):
    x = manual_instance()
    before = x.sha
    solution = manual_solution(x)
    seeds = [2**53 + 1, 2**63 + 17, 2**64 - 1]
    spec = p.Traffic(mode='no_noise')
    context = {'method': 'manual', 'instance_sha256': x.sha, 'solver_seed': 11}
    expected, raw = p.evaluate(x, solution['routes'], seeds, spec, policy, .5)
    actual = e.evaluate_solution(x, solution, seeds, spec, policy, .5, tmp_path, context)
    for name, value in expected.items():
        if name != 'evaluation_seconds':
            assert actual[name] == value
    assert actual['mean_travel_minutes'] == actual['cvar_travel_minutes'] == 12
    assert actual['mean_waiting_minutes'] == 2
    assert actual['scenario_strict_feasibility'] == 1
    assert actual['mean_net_execution_delta_minutes'] == 0
    assert actual['std_travel_minutes'] == 0
    assert x.sha == before
    exported = csv_rows(tmp_path / 'scenario_metrics.csv')
    assert [int(row['scenario_seed']) for row in exported] == seeds
    assert [float(row['travel_minutes']) for row in exported] == [row['travel_minutes'] for row in raw]
    events = csv_rows(tmp_path / 'events.csv')
    routes = csv_rows(tmp_path / 'realized_routes.csv')
    assert len(events) == 9  # Dos clientes y un retorno por escenario.
    assert len(routes) == 18  # Dos clientes y dos depósitos por vehículo/escenario.
    for seed in seeds:
        subset = [row for row in routes if int(row['scenario_seed']) == seed]
        assert [int(row['node']) for row in subset if row['vehicle'] == '0'] == [0, 1, 2, 0]
        empty = [row for row in subset if row['vehicle'] == '1']
        assert [row['node'] for row in empty] == ['0', '0']
        assert all(row['empty_vehicle'] == 'True' for row in empty)
    qc = json.loads((tmp_path / 'validation.json').read_text())
    assert qc['status'] == 'passed' and qc['nominal_feasible'] is True
    assert qc['event_rows'] == qc['expected_event_rows'] == 9
    assert qc['realized_route_rows'] == qc['expected_realized_route_rows'] == 18


def test_structurally_valid_but_temporally_infeasible_routes_remain_in_results(tmp_path):
    x = manual_instance([[0, 8], [0, 2], [0, 4]])
    spec = p.Traffic(mode='no_noise')
    actual = e.evaluate_solution(x, manual_solution(x), [1, 2], spec, 'fixed_order', .5,
                                 tmp_path, {'method': 'manual'})
    assert actual['scenario_strict_feasibility'] == 0
    assert actual['mean_lateness_minutes'] == 3
    assert actual['mean_depot_overtime_minutes'] == 4
    assert actual['mean_customer_violation_fraction'] == 1
    assert actual['mean_travel_minutes'] == 12
    qc = json.loads((tmp_path / 'validation.json').read_text())
    assert qc['status'] == 'passed' and qc['nominal_feasible'] is False
    assert len(csv_rows(tmp_path / 'scenario_metrics.csv')) == 2


def test_depot_only_failure_remains_exportable_without_numpy_boolean_errors(tmp_path):
    # No hay clientes tardíos; sólo falla el retorno. El evaluador sellado puede
    # producir numpy.bool_ en strict_feasible: la exportación debe normalizarlo.
    x = manual_instance([[0, 8], [0, 100], [0, 100]])
    summary = e.evaluate_solution(x, manual_solution(x), [1, 2], p.Traffic(mode='no_noise'),
                                 'fixed_order', .5, tmp_path, {'method': 'manual'})
    assert summary['scenario_strict_feasibility'] == 0
    assert summary['mean_customer_violation_fraction'] == 0
    assert summary['mean_lateness_minutes'] == 0
    assert summary['mean_depot_overtime_minutes'] == 4
    qc = json.loads((tmp_path / 'validation.json').read_text())
    assert qc['status'] == 'passed' and qc['nominal_feasible'] is False


def test_invalid_routes_fail_before_creating_a_validated_result(tmp_path):
    x = manual_instance()
    with pytest.raises(ValueError):
        e.evaluate_solution(x, {'routes': [[2], []], 'nominal': {}}, [1],
                            p.Traffic(mode='no_noise'), 'fixed_order', .95, tmp_path, {})
    assert not (tmp_path / 'validation.json').exists()


def test_stochastic_exports_match_shared_evaluator_and_fractional_cvar(tmp_path):
    x = manual_instance([[0, 100], [5, 10], [0, 100]])
    routes = [[1, 2], []]
    spec = p.Traffic()
    seeds = [13, 17, 19, 23]
    solution = manual_solution(x, routes, p.Traffic(mode='nominal_mean'))
    expected, records = p.evaluate(x, routes, seeds, spec, 'fixed_order', .625)
    summary = e.evaluate_solution(x, solution, seeds, spec, 'fixed_order', .625,
                                  tmp_path, {'method': 'manual'})
    for name, value in expected.items():
        if name != 'evaluation_seconds':
            assert summary[name] == value
    # Cola de masa 1.5: peor costo más media observación del segundo peor.
    costs = sorted((record['travel_minutes'] for record in records), reverse=True)
    assert summary['cvar_travel_minutes'] == pytest.approx((costs[0] + .5 * costs[1]) / 1.5)
    rows = csv_rows(tmp_path / 'scenario_metrics.csv')
    assert [int(row['scenario_seed']) for row in rows] == seeds
    assert [float(row['travel_minutes']) for row in rows] == [r['travel_minutes'] for r in records]


def test_recourse_route_export_records_actual_order_without_hiding_failure(tmp_path):
    x = manual_instance([[0, 100], [0, 1], [0, 20]])
    summary = e.evaluate_solution(x, manual_solution(x), [1], p.Traffic(mode='no_noise'),
                                 'causal_edd', .95, tmp_path, {'method': 'manual'})
    rows = csv_rows(tmp_path / 'realized_routes.csv')
    assert [int(row['node']) for row in rows if row['vehicle'] == '0'] == [0, 2, 1, 0]
    assert summary['mean_recourse_reorders'] == 1
    assert summary['scenario_strict_feasibility'] == 0
    assert summary['mean_late_customers'] == 1


def test_recovery_preserves_incomplete_local_files_and_restores_verified_remote(tmp_path):
    local, remote = tmp_path / 'local', tmp_path / 'remote'
    p.write_json(remote / 'result.json', {'status': 'completed'})
    expected = p.seal_bundle(remote)
    p.write_json(local / 'uncommitted.json', {'status': 'interrupted'})
    assert e._recover(local, remote) is True
    assert p.verify_bundle(local) == expected
    preserved = list(tmp_path.glob('local.interrupted_*'))
    assert len(preserved) == 1
    assert json.loads((preserved[0] / 'uncommitted.json').read_text())['status'] == 'interrupted'
    assert not (local / 'uncommitted.json').exists()


def test_recovery_detects_modified_checkpoint_instead_of_reusing_it(tmp_path):
    local, remote = tmp_path / 'local', tmp_path / 'remote'
    p.write_json(local / 'result.json', {'metric': 12})
    p.seal_bundle(local)
    p.write_json(local / 'result.json', {'metric': 0})
    with pytest.raises(ValueError, match='modificado'):
        e._recover(local, remote)
    assert not remote.exists()


def test_comparison_contract_allows_other_algorithms_but_not_other_budgets_or_data():
    config = {'objective': 'conservative_upper_travel_minutes_v1', 'time_scale': 1000,
              'role': 'development_only', 'seeds': [11], 'search_seconds': 120.,
              'scenario_split': 'evaluation', 'export_all_scenario_events': True,
              'methods': {'aco': {'ants': 20}}}
    bank_config = {'policies': ['fixed_order', 'causal_edd'], 'alpha_cvar': .95}
    first = e.comparison_protocol('a' * 64, 'b' * 64, bank_config, config)
    learned = copy.deepcopy(config)
    learned['methods'] = {'am_supervised': {'n_samples': 16}}
    assert e.comparison_protocol('a' * 64, 'b' * 64, bank_config, learned) == first
    assert e.comparison_protocol('c' * 64, 'b' * 64, bank_config, config) != first
    assert e.comparison_protocol('a' * 64, 'b' * 64, bank_config, config | {'search_seconds': 60.}) != first


def test_learned_provenance_rejects_training_geometry_in_evaluation_and_false_costs():
    x = manual_instance()
    payload = x.payload()
    payload['provenance']['geometry_group_sha256'] = 'a' * 64
    x = p.freeze_instance(payload)
    provenance = {'training_id': 'trained', 'training_seed': 11, 'model_checkpoint_sha256': 'b' * 64,
                  'train_geometry_groups': ['c' * 64], 'validation_geometry_groups': ['d' * 64],
                  'heldout_geometry_groups': ['a' * 64]}
    e.validate_learned_provenance(provenance, x, 11)
    with pytest.raises(ValueError, match='fuga'):
        e.validate_learned_provenance(provenance | {'train_geometry_groups': ['a' * 64]}, x, 11)
    with pytest.raises(ValueError, match='semilla'):
        e.validate_learned_provenance(provenance, x, 29)
    with pytest.raises(ValueError, match='Costo offline'):
        e.provenance_columns(provenance | {'training_seconds': True})


Overwriting tests/test_experiments.py


In [69]:
%%writefile tests/test_exact.py
"""Pruebas para ejecutar en Colab; no ejecutar localmente (AGENTS.md).

Las pruebas de índices/extracción no importan Gurobi. Las de optimización usan
el entorno prestado por Notebook 01 o SVRPLAB_TEST_GUROBI=1 dentro de Colab.
"""
import os
import sys
from pathlib import Path

import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import exact
from svrplab.pilot import Traffic, freeze_instance, validate_routes
from svrplab.search import SearchProblem


def tiny(*, locations=None, demands=None, capacities=None, windows=None):
    locations = [[0, 0], [3, 0], [6, 0]] if locations is None else locations
    count = len(locations)
    x = freeze_instance({
        'instance_id': 'exact-manual-fixture', 'depot_indices': [0],
        'locations': locations,
        'demands': [0] + [1] * (count - 1) if demands is None else demands,
        'capacities': [5] if capacities is None else capacities,
        'windows': [[0, 100]] * count if windows is None else windows,
        'appear_times': [0] * count, 'provenance': {'kind': 'manual_test_fixture'},
    })
    return SearchProblem(x, Traffic(mode='no_noise'))


def test_pending_license_never_imports_or_contacts_gurobi(monkeypatch):
    def forbidden():
        raise AssertionError('No debe iniciarse Gurobi sin un entorno explícito.')
    monkeypatch.setattr(exact, '_gurobi', forbidden)
    result = exact.solve_exact(tiny(), 7, {'seconds': 1}, env=None)
    assert result['status'] == 'license_pending'
    assert result['routes'] is None
    assert result['metadata']['relative_gap'] is None
    assert result['metadata']['incumbent_objective'] is None
    assert not result['metadata']['surrogate_optimal']


def test_homogeneous_model_has_no_vehicle_expansion():
    data = exact.prepare_exact_data(tiny(capacities=[5, 5, 5]))
    assert data['homogeneous']
    assert data['vehicle_arcs'] is None
    assert len(data['arcs']) == 6
    assert data['formulation'] == 'homogeneous_two_index_mtz'


def test_heterogeneous_arcs_exclude_incapable_vehicle_only():
    data = exact.prepare_exact_data(tiny(demands=[0, 1, 2], capacities=[1, 2]))
    assert not data['homogeneous']
    assert (1, 0, 2) in data['vehicle_arcs']
    assert all(2 not in (i, j) for k, i, j in data['vehicle_arcs'] if k == 0)
    assert (0, 0, 1) in data['vehicle_arcs']


def test_arc_pruning_respects_departure_zero_and_return_horizon():
    problem = tiny(windows=[[0, 8], [0, 2], [5, 7]])
    data = exact.prepare_exact_data(problem)
    assert (0, 1) not in data['arcs']  # 3 > cierre 2.
    assert (2, 0) not in data['arcs']  # apertura 5 + regreso 6 > 8.
    assert (0, 2) in data['arcs']


def test_zero_demand_and_zero_time_nodes_are_retained():
    problem = tiny(locations=[[0, 0]] * 4, demands=[0, 0, 0, 0])
    data = exact.prepare_exact_data(problem)
    assert len(data['arcs']) == 12
    routes = exact._extract_routes([(0, 1), (1, 2), (2, 3), (3, 0)], data)
    assert routes == [[1, 2, 3]]


def test_extraction_rejects_disconnected_subtour():
    problem = tiny(locations=[[0, 0]] * 4, demands=[0, 0, 0, 0])
    data = exact.prepare_exact_data(problem)
    with pytest.raises(ValueError):
        exact._extract_routes([(0, 1), (1, 0), (2, 3), (3, 2)], data)


def test_extraction_preserves_vehicle_capacity_assignment():
    problem = tiny(demands=[0, 1, 2], capacities=[1, 2])
    data = exact.prepare_exact_data(problem)
    routes = exact._extract_routes([(0, 0, 1), (0, 1, 0), (1, 0, 2), (1, 2, 0)], data)
    assert routes == [[1], [2]]
    assert validate_routes(problem.x, routes)


def test_unavailable_bound_gap_and_infinity_remain_missing():
    class Unsolved:
        @property
        def ObjVal(self):
            raise RuntimeError('No incumbent')
    assert exact._safe_attr(Unsolved(), 'ObjVal') is None
    assert exact._finite(float('inf')) is None
    assert exact._finite(1e100) is None
    assert exact._finite(float('nan')) is None
    assert exact._finite(0) == 0


@pytest.mark.parametrize('line', [
    'Set parameter WLSAccessID', 'WLSSecret = never-export',
    'LicenseID 123', 'token refresh', 'Username example',
])
def test_log_filter_omits_license_lines(line):
    assert not exact._safe_log_line(line)
    assert exact._safe_log_line('Explored 10 nodes in 0.10 seconds\n')


@pytest.mark.parametrize('config', [
    {'seconds': 0}, {'seconds': float('nan')}, {'threads': 0},
    {'threads': True}, {'mip_gap': -1},
])
def test_invalid_budgets_fail_before_license_creation(config):
    with pytest.raises(ValueError):
        exact.solve_exact(tiny(), 0, config, env=None)


@pytest.fixture(scope='module')
def gurobi_env(request):
    borrowed = getattr(request.config, 'svrplab_gurobi_env', None)
    if borrowed is not None:
        # El notebook crea/cierra el único entorno; el fixture sólo lo utiliza.
        yield borrowed
        return
    if os.environ.get('SVRPLAB_TEST_GUROBI') != '1':
        pytest.skip('Pruebas Gurobi opt-in: requieren WLS en Colab.')
    try:
        from google.colab import userdata
    except ImportError:
        pytest.skip('La experimentación está autorizada exclusivamente en Colab.')
    # Al optar por estas pruebas, una licencia fallida sí falla: no se oculta.
    credentials = {}
    try:
        for name in ('WLSACCESSID', 'WLSSECRET', 'LICENSEID'):
            credentials[name] = userdata.get(name)
        env = exact.create_wls_env(credentials)
    except Exception:
        raise RuntimeError('No se inició WLS para las pruebas. Ejecute pytest.main en el kernel '
                           'Colab o proporcione svrplab_gurobi_env desde el notebook.') from None
    finally:
        credentials.clear()
    try:
        yield env
    finally:
        env.dispose()


def test_gurobi_model_structure_includes_independent_subtour_orders(gurobi_env):
    problem = tiny(locations=[[0, 0]] * 4, demands=[0, 0, 0, 0])
    model, _, data = exact.build_exact_model(problem, gurobi_env)
    try:
        assert model.NumBinVars == len(data['arcs'])
        assert model.getConstrByName('no_subtour[1,2]') is not None
        assert model.getConstrByName('no_subtour[2,1]') is not None
        assert model.getConstrByName('return[1]') is not None
    finally:
        model.dispose()


def test_gurobi_tiny_known_optimum_and_artifacts(gurobi_env, tmp_path):
    result = exact.solve_exact(tiny(), 7, {'seconds': 10}, gurobi_env, tmp_path)
    assert result['routes'] is not None
    assert result['status'] == 'optimal'
    assert result['metadata']['surrogate_optimal']
    assert result['metadata']['incumbent_objective'] == pytest.approx(12)
    assert result['metadata']['best_bound'] == pytest.approx(12)
    assert (tmp_path / 'model.lp.gz').is_file()
    assert (tmp_path / 'incumbent.sol').is_file()
    assert (tmp_path / 'gurobi.log').is_file()
    assert not list(tmp_path.glob('*.prm'))


def test_gurobi_zero_time_zero_demand_cannot_form_subtour(gurobi_env):
    problem = tiny(locations=[[0, 0]] * 4, demands=[0, 0, 0, 0])
    result = exact.solve_exact(problem, 7, {'seconds': 10}, gurobi_env)
    assert result['status'] == 'optimal'
    assert result['metadata']['incumbent_objective'] == 0
    assert validate_routes(problem.x, result['routes'])
    assert len(result['routes'][0]) == 3


def test_gurobi_heterogeneous_assignments_are_valid(gurobi_env):
    problem = tiny(demands=[0, 1, 2], capacities=[1, 2])
    result = exact.solve_exact(problem, 7, {'seconds': 10}, gurobi_env)
    assert result['status'] == 'optimal'
    assert validate_routes(problem.x, result['routes'])
    assert result['routes'][0] == [1]
    assert result['routes'][1] == [2]
    assert result['metadata']['incumbent_objective'] == pytest.approx(18)


def test_gurobi_infeasibility_has_no_fictitious_routes_or_gap(gurobi_env):
    problem = tiny(windows=[[0, 100], [0, 2], [0, 100]])
    result = exact.solve_exact(problem, 7, {'seconds': 10}, gurobi_env)
    assert result['status'] == 'infeasible'
    assert result['routes'] is None
    assert result['metadata']['incumbent_objective'] is None
    assert result['metadata']['relative_gap'] is None
    assert not result['metadata']['surrogate_optimal']


Overwriting tests/test_exact.py


In [70]:
%%writefile tests/test_research_design.py
"""Pruebas del protocolo: ejecutar exclusivamente en Colab, nunca localmente."""
import copy
import math
import sys
from pathlib import Path

import pytest
sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))
from svrplab import research_design as r


def config():
    return {'schema_version': 'research-design-v2', 'sizes': [50], 'split_seed': 7,
            'geometry_quotas_per_size': {'train': 1, 'validation': 1, 'test': 1},
            'population': {'master_seed': 91, 'candidate_multiplier': 2,
                           'map_size': [1000, 1000], 'demand_range': [0, 100],
                           'num_depots': 1, 'dynamic_customers': False,
                           'certification_seconds_per_strategy': [120, 600, 1800]},
            'win_score': {'cost_tie_relative': 1e-9, 'cost_tie_absolute_minutes': 1e-6},
            'primary_endpoint': 'geometry_mean_feasibility_first_conservative_cost_win_score'}


def test_splits_are_grouped_exclude_development_and_order_invariant():
    rows = [{'geometry_group_sha256': f'{i:064x}', 'n_customers': 50, 'source_id': str(i)} for i in range(5)]
    rows.append(dict(rows[1], source_id='derivative'))
    excluded = [rows[0]['geometry_group_sha256']]
    first = r.plan_geometry_split(rows, excluded, config())
    second = r.plan_geometry_split(list(reversed(rows)), excluded, config())
    assert first == second
    assert not first['shortages'] and not first['ready_for_confirmatory']
    assert not set(excluded) & {row['geometry_group_sha256'] for row in first['assignment']}
    assert len({row['geometry_group_sha256'] for row in first['assignment']}) == 3


def test_replicas_cannot_fill_geometry_quotas():
    row = {'geometry_group_sha256': 'a' * 64, 'n_customers': 50, 'source_id': 'one'}
    result = r.plan_geometry_split([copy.deepcopy(row) for _ in range(100)], [], config())
    assert result['status'] == 'insufficient_population' and len(result['assignment']) == 1


def test_density_audit_includes_poisson_compound_not_just_peak_densities():
    result = r.traffic_density_audit()
    assert not result['sealed_evaluator_modified']
    assert all(abs(row['hour_to_minute_density_ratio'] - 60.) < 1e-8 for row in result['rows'])
    assert result['component_rows']
    for row in result['component_rows']:
        rate = row['lambda_accidents_per_arc_query']
        assert math.isclose(row['mean_accident_delay_minutes'], 75 * rate)
        assert math.isclose(row['sd_accident_delay_minutes'] ** 2, 6300 * rate)
        assert math.isclose(row['probability_accident_this_arc_query'], -math.expm1(-rate))
        assert 0 <= row['accident_variance_share'] <= 1
    original = [row for row in result['component_rows'] if row['interpretation'] == 'sealed_minute_model']
    alternate = [row for row in result['component_rows'] if row['interpretation'] != 'sealed_minute_model']
    assert len(original) == len(alternate)
    for a, b in zip(original, alternate):
        assert math.isclose(b['lambda_accidents_per_arc_query'], 60 * a['lambda_accidents_per_arc_query'])


def test_precision_is_pointwise_does_not_guarantee_cvar_or_geometry_power():
    precision = r.scenario_precision_plan({'fr_absolute_error': .05, 'fr_error_probability': .05,
                                         'cvar_alpha': .95, 'minimum_expected_tail_observations': 50})
    assert precision['scenarios_per_plan'] >= 1000
    assert not precision['cvar_precision_guaranteed']
    geometry = r.geometry_precision_sensitivity(config())
    assert geometry['status'] == 'sensitivity_not_power_calculation'
    assert geometry['n_total_test_geometries'] == 1


def observed(valid, cost=None):
    return {'artifact_status': 'observed', 'conservative_valid': valid, 'conservative_cost': cost}


@pytest.mark.parametrize('left,right,expected', [
    (observed(True, 1000.), observed(False), 1),
    (observed(False), observed(True, 1000.), -1),
    (observed(False), observed(False), 0),
    (observed(True, 10.), observed(True, 11.), 1),
    (observed(True, 11.), observed(True, 10.), -1),
    (observed(True, 10.), observed(True, 10. + 1e-8), 0),
])
def test_win_score_preserves_failures_and_paired_cost_order(left, right, expected):
    assert r.paired_win_score(left, right, config()) == expected
    assert r.paired_win_score(right, left, config()) == -expected


@pytest.mark.parametrize('invalid', [{}, {'artifact_status': 'missing', 'conservative_valid': False},
                                    observed(True), observed(True, float('nan')), observed(True, -1)])
def test_missing_or_invalid_results_never_become_solver_failures(invalid):
    with pytest.raises(ValueError):
        r.paired_win_score(invalid, observed(False), config())


def test_reservation_preassigns_roles_seeds_and_is_immutable():
    design = config()
    before = copy.deepcopy(design)
    excluded = ['a' * 64]
    one = r.population_ticket_plan(design, excluded)
    two = r.population_ticket_plan(copy.deepcopy(design), excluded)
    assert design == before and one == two
    assert one['generated'] is False and not one['ready_for_confirmatory']
    assert len(one['tickets']) == 6
    assert len({t['geometry_seed'] % 2**32 for t in one['tickets']}) == 6
    assert {t['role'] for t in one['tickets']} == {'train', 'validation', 'test'}
    for role in ('train', 'validation', 'test'):
        assert [t['candidate_index'] for t in one['tickets'] if t['role'] == role] == [0, 1]
    changed = copy.deepcopy(design)
    changed['geometry_quotas_per_size']['test'] = 2
    assert r.population_ticket_plan(changed, excluded)['population_id'] != one['population_id']


def test_nominal_population_alone_cannot_open_test():
    check = r.final_freeze_checklist(config(), {'all_required_nominal_certificates': True})
    assert check['population_nominally_complete'] is True
    assert check['ready_for_confirmatory'] is False
    assert 'holdout_adapter_validation_in_colab_sha256' in check['required_frozen_artifacts']
    assert 'all_method_source_sha256' in check['required_frozen_artifacts']


@pytest.mark.parametrize('field,value', [('map_size', [500, 500]), ('demand_range', [1, 100]),
                                        ('num_depots', 2), ('dynamic_customers', True)])
def test_population_does_not_change_source_conventions_silently(field, value):
    design = config()
    design['population'][field] = value
    with pytest.raises(ValueError):
        r.population_ticket_plan(design, [])


def test_unknown_admission_is_not_skipped_or_retried_as_another_geometry(tmp_path, monkeypatch):
    from svrplab import pilot as p
    calls = []
    class Candidate:
        def payload(self):
            return {'fixture': 'same_candidate'}
    class Parent:
        instance_id = 'fixture_parent'
    monkeypatch.setattr(p, 'make_conditioned_candidate', lambda *a, **kw: (Candidate(), [{'attempt': kw['joint_attempt']}]))
    def unknown(*args, **kwargs):
        calls.append('solver')
        return None, {'status': 'unknown_no_certificate'}
    monkeypatch.setattr(p, 'certify_candidate', unknown)
    monkeypatch.setattr(p, 'proven_rejection', lambda *a, **kw: False)
    monkeypatch.setattr(p, 'environment_report', lambda: {})
    admission = {'traffic': {}, 'windows': {'max_joint_candidates': 2, 'max_draws_per_customer': 1},
                 'certification': {'time_scale': 1000, 'seconds_per_strategy': 1}}
    for _ in range(2):
        with pytest.raises(RuntimeError, match='UNKNOWN'):
            r._ticket_admission(Parent(), tmp_path, admission, tmp_path / 'attempts', tmp_path / 'remote')
    assert calls == ['solver']  # reanudación conserva UNKNOWN, no lo salta ni vuelve a resolverlo
    assert not (tmp_path / 'attempts' / 'attempt001').exists()


def test_finite_proven_window_exhaustion_is_distinct_from_geometry_impossibility(tmp_path, monkeypatch):
    from svrplab import pilot as p
    calls = []
    class Candidate:
        def payload(self):
            return {'fixture': 'candidate'}
    class Parent:
        instance_id = 'fixture_parent'
    def candidate(*args, **kwargs):
        calls.append(kwargs['joint_attempt'])
        return Candidate(), []
    monkeypatch.setattr(p, 'make_conditioned_candidate', candidate)
    monkeypatch.setattr(p, 'certify_candidate', lambda *a, **kw: (None, {'status': 'proven_infeasible_optimistic_model'}))
    monkeypatch.setattr(p, 'proven_rejection', lambda *a, **kw: True)
    monkeypatch.setattr(p, 'environment_report', lambda: {})
    admission = {'traffic': {}, 'windows': {'max_joint_candidates': 2, 'max_draws_per_customer': 1},
                 'certification': {'time_scale': 1000, 'seconds_per_strategy': 1}}
    status, instance, ledger, certificate, history = r._ticket_admission(
        Parent(), tmp_path, admission, tmp_path / 'attempts', tmp_path / 'remote')
    assert status == 'window_sequence_exhausted_proven_candidates'
    assert instance is None and certificate is None and ledger is None
    assert calls == [0, 1] and len(history['attempts']) == 2


def test_unknown_search_stages_resume_same_candidate_after_interruption(tmp_path, monkeypatch):
    from svrplab import pilot as p
    calls, interrupted = [], [False]
    class Candidate:
        def payload(self):
            return {'fixture': 'same_geometry_and_windows'}
    class Parent:
        instance_id = 'fixture_parent'
    monkeypatch.setattr(p, 'make_conditioned_candidate', lambda *a, **kw: (Candidate(), [{'same': True}]))
    def search(candidate, spec, seconds, scale, progress):
        calls.append(seconds)
        if seconds < 3:
            return None, {'status': 'unknown_no_certificate'}
        if not interrupted[0]:
            interrupted[0] = True
            raise OSError('interrupted_same_candidate')
        return None, {'status': 'proven_infeasible_optimistic_model'}
    monkeypatch.setattr(p, 'certify_candidate', search)
    monkeypatch.setattr(p, 'proven_rejection', lambda x, report, *a: report['status'].startswith('proven_'))
    monkeypatch.setattr(p, 'environment_report', lambda: {})
    admission = {'traffic': {}, 'windows': {'max_joint_candidates': 1, 'max_draws_per_customer': 1},
                 'certification': {'time_scale': 1000, 'seconds_per_strategy': 1}}
    with pytest.raises(OSError, match='interrupted_same_candidate'):
        r._ticket_admission(Parent(), tmp_path, admission, tmp_path / 'attempts', tmp_path / 'remote', [1, 2, 3])
    status, _, _, _, history = r._ticket_admission(
        Parent(), tmp_path, admission, tmp_path / 'attempts', tmp_path / 'remote', [1, 2, 3])
    assert calls == [1, 2, 3, 3]
    assert status == 'window_sequence_exhausted_proven_candidates'
    assert [s['seconds_per_strategy'] for s in history['attempts'][0]['certification_searches']] == [1, 2, 3]
    assert len(history['attempts']) == 1  # fases de búsqueda nunca se cuentan como nuevos candidatos


def test_invalid_model_does_not_authorize_larger_search_or_new_candidate(tmp_path, monkeypatch):
    from svrplab import pilot as p
    calls = []
    class Candidate:
        def payload(self):
            return {'fixture': 'same_candidate'}
    class Parent:
        instance_id = 'fixture_parent'
    monkeypatch.setattr(p, 'make_conditioned_candidate', lambda *a, **kw: (Candidate(), []))
    def invalid(candidate, spec, seconds, scale, progress):
        calls.append(seconds)
        return None, {'status': 'certifier_model_invalid'}
    monkeypatch.setattr(p, 'certify_candidate', invalid)
    monkeypatch.setattr(p, 'proven_rejection', lambda *a: False)
    monkeypatch.setattr(p, 'environment_report', lambda: {})
    admission = {'traffic': {}, 'windows': {'max_joint_candidates': 2, 'max_draws_per_customer': 1},
                 'certification': {'time_scale': 1000, 'seconds_per_strategy': 1}}
    with pytest.raises(RuntimeError, match='modelo/estado no válido'):
        r._ticket_admission(Parent(), tmp_path, admission, tmp_path / 'attempts', tmp_path / 'remote', [1, 2, 3])
    assert calls == [1]
    assert not (tmp_path / 'attempts' / 'attempt001').exists()
    assert not (tmp_path / 'attempts' / 'attempt000' / 'search01_00002s').exists()


def test_unsealed_phase_is_archived_and_sealed_remote_phase_recovers(tmp_path):
    from svrplab import pilot as p
    local, remote = tmp_path / 'local_phase', tmp_path / 'remote_phase'
    local.mkdir()
    (local / 'record.json').write_text('{"incomplete": true}')
    assert r._recover_phase_bundle(local, remote) is False
    archives = list(tmp_path.glob('local_phase.interrupted_*'))
    assert len(archives) == 1 and (archives[0] / 'record.json').exists()
    p.write_json(remote / 'record.json', {'completed': True})
    p.seal_bundle(remote)
    assert r._recover_phase_bundle(local, remote) is True
    assert p.verify_bundle(local) == p.verify_bundle(remote)


Overwriting tests/test_research_design.py


In [71]:
%%writefile confirmatory_results_08.R
# Confirmación 08 sobre exporte final10 íntegro; ejecución sólo en Colab.
args <- commandArgs(trailingOnly = TRUE)
if (Sys.getenv("SVRPLAB_COLAB_ANALYSIS") != "1" || length(args) != 2L ||
    !file.exists("/content/.svrplab_final08_verified")) stop("Usar notebook 08 en Colab con verificación final10.")
input <- normalizePath(args[[1]], mustWork = TRUE); output <- args[[2]]
if (dir.exists(output)) stop("No sobrescribir evidencia de análisis.")
dir.create(output, recursive = TRUE)
options(stringsAsFactors = FALSE, warn = 2, OutDec = "."); Sys.setenv(TZ = "UTC")
read <- function(name) read.csv(file.path(input, name), colClasses = "character", check.names = FALSE,
                               na.strings = "", fileEncoding = "UTF-8")
write <- function(x, name) write.csv(x, file.path(output, name), row.names = FALSE, na = "", fileEncoding = "UTF-8")
need <- function(x, fields) if (!all(fields %in% names(x))) stop(paste("Columnas faltantes:", paste(setdiff(fields, names(x)), collapse = ",")))
num <- function(x) {
  value <- suppressWarnings(as.numeric(x))
  if (any(!is.na(x) & !is.finite(value))) stop("Métrica no finita.")
  value
}
truth <- function(x) {
  if (any(!is.na(x) & !x %in% c("True", "False", "true", "false"))) stop("Booleano inválido.")
  !is.na(x) & x %in% c("True", "true")
}
key <- function(x, fields) do.call(paste, c(x[fields], sep = "|"))
mean_available <- function(x) if (all(is.na(x))) NA_real_ else mean(x[!is.na(x)])
bridge <- read("analysis_bridge.csv"); meta <- setNames(bridge$value, bridge$key)
if (meta[["schema_version"]] != "final-analysis-export-v1" || meta[["role"]] != "confirmatory_holdout" ||
    meta[["analysis_mode"]] != "confirmatory" || meta[["primary_inference"]] != "stratified_geometry_weighted_hoeffding_mean_bound") stop("Contrato final distinto del análisis preespecificado.")
methods <- read("expected_methods.csv")$method; seeds <- read("expected_seeds.csv")$solver_seed
policies <- read("expected_policies.csv")$policy; sizes <- read("expected_sizes.csv")$n_customers
contrasts <- read("primary_contrasts.csv")
if (nrow(contrasts) != 2L || !identical(contrasts$method, c("hbg_facs", "hbg_facs")) ||
    !setequal(contrasts$reference_method, c("hbg_untrained", "hbg_no_pheromone"))) stop("Familia coprimaria alterada.")
alpha <- num(meta[["familywise_alpha"]]); tie_abs <- num(meta[["cost_tie_absolute_minutes"]]); tie_rel <- num(meta[["cost_tie_relative"]])
if (!identical(alpha, .05) || tie_abs < 0 || tie_rel < 0) stop("Parámetros estadísticos inválidos.")
hashes <- read("input_file_hashes.csv")
verify <- function() {
  for (i in seq_len(nrow(hashes))) {
    actual <- system2("sha256sum", shQuote(file.path(input, hashes$relative_path[[i]])), stdout = TRUE)
    if (length(actual) != 1L || substr(actual, 1, 64) != hashes$sha256[[i]]) stop("Fuente final cambió.")
  }
}
verify()
registry <- read("registry.csv")
need(registry, c("instance_sha256", "geometry_group_sha256", "n_customers"))
if (anyNA(registry[c("instance_sha256", "geometry_group_sha256", "n_customers")]) || anyDuplicated(registry$instance_sha256) || anyDuplicated(registry$geometry_group_sha256)) stop("Test debe contener una instancia por geometría independiente.")
if (!setequal(registry$n_customers, sizes)) stop("Estratos de tamaños incompletos.")
quota <- num(meta[["test_geometries_per_size"]])
if (any(table(factor(registry$n_customers, levels = sizes)) != quota)) stop("Cuota final diferente del sello.")
attempts <- read("attempts.csv")
keys <- c("instance_sha256", "policy", "method", "solver_seed")
need(attempts, c(keys, "geometry_group_sha256", "n_customers", "evaluation_status", "solver_status",
  "structural_valid", "conservative_feasible", "search_objective_minutes", "scenario_strict_feasibility",
  "mean_travel_minutes", "cvar_travel_minutes", "mean_waiting_minutes", "mean_recourse_reorders", "search_total_seconds",
  "artifact_status", "scenario_success_per_scheduled_attempt"))
# Ausencia de un indicador de validez es un error de datos, nunca FALSE.
required_status <- c("artifact_status", "solver_status", "evaluation_status", "structural_valid", "conservative_feasible")
if (anyNA(attempts[required_status]) || any(attempts$artifact_status != "observed")) stop("Artefacto/estado/flags ausentes o no observados: detener análisis.")
if (any(!attempts$evaluation_status %in% c("validated", "no_solution"))) stop("Estado de evaluación no admitido.")
structural <- truth(attempts$structural_valid)
conservative <- truth(attempts$conservative_feasible)
evaluated <- attempts$evaluation_status == "validated"
if (any(conservative & !structural) || any(evaluated & !structural) || any(!evaluated & structural)) stop("Incoherencia entre evaluación, estructura y factibilidad conservadora.")
if (any(attempts$solver_status %in% c("solver_error", "evaluation_error", "environment_error", "license_pending", "not_available", "unavailable"))) stop("Error técnico/disponibilidad: no convertir en fallo estadístico del método.")
expected <- expand.grid(instance_sha256 = registry$instance_sha256, policy = policies,
                        method = methods, solver_seed = seeds, stringsAsFactors = FALSE)
if (anyNA(attempts[keys]) || anyDuplicated(key(attempts, keys)) ||
    !setequal(key(attempts, keys), key(expected, keys))) stop("Faltan/sobran artefactos: no recodificar como fallo.")
reg <- registry[match(attempts$instance_sha256, registry$instance_sha256), ]
for (field in c("geometry_group_sha256", "n_customers")) if (any(attempts[[field]] != reg[[field]])) stop("Identidad final alterada.")
for (field in c("search_objective_minutes", "scenario_strict_feasibility", "mean_travel_minutes", "cvar_travel_minutes",
                "mean_waiting_minutes", "mean_recourse_reorders", "search_total_seconds", "scenario_success_per_scheduled_attempt")) attempts[[field]] <- num(attempts[[field]])
if (any(evaluated & is.na(attempts$scenario_strict_feasibility)) ||
    any(!evaluated & (!is.na(attempts$scenario_strict_feasibility) | !is.na(attempts$search_objective_minutes))) ||
    anyNA(attempts$scenario_success_per_scheduled_attempt)) stop("Estado de evaluación incompatible con mediciones registradas.")
expected_success <- ifelse(evaluated, attempts$scenario_strict_feasibility, 0)
if (any(abs(attempts$scenario_success_per_scheduled_attempt - expected_success) > 1e-12)) stop("Éxito end-to-end incompatible con evaluación/no_solution.")
attempts$evaluated <- evaluated
attempts$valid <- evaluated & structural & conservative
if (any(attempts$valid & (is.na(attempts$search_objective_minutes) | attempts$search_objective_minutes < 0))) stop("Plan válido sin objetivo conservador.")
attempts$success <- ifelse(attempts$evaluated, attempts$scenario_strict_feasibility, NA_real_)
no_solution <- !is.na(attempts$evaluation_status) & attempts$evaluation_status == "no_solution" &
  !is.na(attempts$structural_valid) & !truth(attempts$structural_valid) &
  !is.na(attempts$solver_status) & attempts$solver_status != "license_pending"
attempts$success[no_solution] <- 0
if (any(attempts$success < 0 | attempts$success > 1, na.rm = TRUE)) stop("FR fuera de [0,1].")
# Filas secundarias/controles incompletas siguen visibles; coprimarios requieren datos íntegros.
write(attempts, "attempt_panel_final.csv")
primary <- attempts[attempts$policy == meta[["primary_policy"]], ]
geometry_scores <- list(); paired_seeds <- list()
for (i in seq_len(nrow(contrasts))) {
  method <- contrasts$method[[i]]; reference <- contrasts$reference_method[[i]]
  a <- primary[primary$method == method, ]; b <- primary[primary$method == reference, ]
  pairing <- c("instance_sha256", "solver_seed")
  b <- b[match(key(a, pairing), key(b, pairing)), ]
  if (nrow(a) != nrow(registry) * length(seeds) || anyNA(b$method) || anyNA(a$success) || anyNA(b$success)) stop("Coprimario incompleto: detener inferencia.")
  score <- as.numeric(a$valid) - as.numeric(b$valid)
  both <- a$valid & b$valid
  tolerance <- pmax(tie_abs, tie_rel * pmax(abs(a$search_objective_minutes), abs(b$search_objective_minutes)))
  difference <- b$search_objective_minutes - a$search_objective_minutes
  score[both] <- ifelse(abs(difference[both]) <= tolerance[both], 0, sign(difference[both]))
  # Una comparación emparejada es un estimando operacional con semillas fijas,
  # no supone que los RNG de métodos distintos generen trayectorias equivalentes.
  paired <- data.frame(a[c("instance_sha256", "geometry_group_sha256", "n_customers", "solver_seed")],
    method = method, reference_method = reference, win_score = score,
    method_valid = a$valid, reference_valid = b$valid,
    method_success = a$success, reference_success = b$success, fr_difference = a$success - b$success)
  paired_seeds[[i]] <- paired
  geometry_scores[[i]] <- do.call(rbind, lapply(split(seq_len(nrow(paired)), paired$geometry_group_sha256), function(ii) {
    x <- paired[ii, ]
    if (nrow(x) != length(seeds) || !setequal(x$solver_seed, seeds)) stop("Semillas prefijadas incompletas.")
    data.frame(x[1, c("geometry_group_sha256", "n_customers", "method", "reference_method")],
      win_score = mean(x$win_score), fr_difference = mean(x$fr_difference),
      method_valid_fraction = mean(x$method_valid), reference_valid_fraction = mean(x$reference_valid),
      expected_seeds = length(seeds), independent_geometry_count = 1L)
  }))
}
paired <- do.call(rbind, paired_seeds); geometry <- do.call(rbind, geometry_scores)
write(paired, "seed_pair_scores.csv"); write(geometry, "geometry_primary_scores.csv")
# Hoeffding para X_g en [-1,1] y pesos fijados por estrato: sum(w_g)=1.
# p <= 2 exp(-estimate^2/(2 sum(w_g^2))); intervalos conservadores sin simetría.
results <- do.call(rbind, lapply(seq_len(nrow(contrasts)), function(i) {
  x <- geometry[geometry$method == contrasts$method[[i]] & geometry$reference_method == contrasts$reference_method[[i]], ]
  counts <- table(factor(x$n_customers, levels = sizes))
  if (any(counts != quota)) stop("Contraste no cubre todo el test.")
  weights <- 1 / (length(sizes) * as.numeric(counts[match(x$n_customers, names(counts))]))
  estimate <- sum(weights * x$win_score); square_sum <- sum(weights^2)
  radius <- sqrt(2 * square_sum * log(2 / alpha))
  simultaneous <- sqrt(2 * square_sum * log(2 * nrow(contrasts) / alpha))
  fr_delta <- sum(weights * x$fr_difference)
  data.frame(method = contrasts$method[[i]], reference_method = contrasts$reference_method[[i]],
    geometry_count = nrow(x), estimate = estimate, ci95_lower = max(-1, estimate - radius),
    ci95_upper = min(1, estimate + radius), simultaneous_ci_lower = max(-1, estimate - simultaneous),
    simultaneous_ci_upper = min(1, estimate + simultaneous),
    p_two_sided = min(1, 2 * exp(-estimate^2 / (2 * square_sum))),
    fr_difference = fr_delta, fr_ci95_lower = max(-1, fr_delta - radius), fr_ci95_upper = min(1, fr_delta + radius),
    fr_guardrail = if (fr_delta < 0) "observed_degradation_report_explicitly" else "no_observed_degradation_not_noninferiority_proof",
    scope = "geometry_population_conditional_on_frozen_models_and_finite_seed_set",
    inference = "stratified_geometry_weighted_hoeffding_mean_bound", sum_squared_geometry_weights = square_sum,
    minimum_abs_score_for_first_holm_rejection = simultaneous,
    primary_evidence_is_not_global_robustness = TRUE)
}))
results$p_holm <- p.adjust(results$p_two_sided, method = "holm", n = 2L)
results$holm_reject <- results$p_holm <= alpha
results$positive_primary_evidence <- results$holm_reject & results$estimate > 0
results$negative_primary_evidence <- results$holm_reject & results$estimate < 0
write(results, "coprimary_inference.csv")
# Secundarios descriptivos; no se ensaya una batería post hoc de p-valores.
secondary <- do.call(rbind, lapply(split(seq_len(nrow(attempts)), key(attempts, c("instance_sha256", "policy", "method"))), function(ii) {
  x <- attempts[ii, ]; valid <- x$valid; known <- sum(!is.na(x$success)); expected <- nrow(x)
  data.frame(x[1, c("instance_sha256", "geometry_group_sha256", "n_customers", "policy", "method")],
    expected = expected, evaluated = sum(x$evaluated), conservative_valid = sum(valid), known_success = known,
    success = if (known == expected) mean(x$success) else NA_real_,
    success_lower = sum(x$success, na.rm = TRUE) / expected,
    success_upper = (sum(x$success, na.rm = TRUE) + expected - known) / expected,
    travel_conditional = mean_available(x$mean_travel_minutes[valid]),
    cvar_conditional = mean_available(x$cvar_travel_minutes[valid]),
    duration_conditional = mean_available((x$mean_travel_minutes + x$mean_waiting_minutes)[valid]),
    recourse_reorders = mean_available(x$mean_recourse_reorders[x$evaluated]),
    search_seconds = mean_available(x$search_total_seconds), independent_geometry_count = 1L)
}))
write(secondary, "secondary_descriptive_by_geometry.csv")
secondary_lines <- c("# Resultados secundarios descriptivos por tamaño", "",
  "Métodos en orden fijo. N cuenta geometrías. Costos condicionados a planes conservadores válidos; comparar con cobertura.",
  "Las cotas FR reflejan desconocimiento cuando falta medición; no son IC poblacionales.",
  "Para diferencias de costo pareadas con todas las semillas válidas en ambos métodos usar secondary_common_valid_costs.csv.", "",
  "| n | Política | Método | N | Válidos/intentos | Geometrías con viaje/N | Viaje | Duración | FR [cotas] |",
  "|---:|---|---|---:|---:|---:|---:|---:|---|")
fmt <- function(value) if (is.na(value)) "—" else sprintf("%.2f", value)
for (size in sizes) for (policy in policies) for (method in methods) {
  x <- secondary[secondary$n_customers == size & secondary$policy == policy & secondary$method == method, ]
  secondary_lines <- c(secondary_lines, sprintf("| %s | %s | %s | %d | %d/%d | %d/%d | %s | %s | [%.3f, %.3f] |",
    size, policy, method, nrow(x), sum(x$conservative_valid), sum(x$expected),
    sum(!is.na(x$travel_conditional)), nrow(x), fmt(mean_available(x$travel_conditional)),
    fmt(mean_available(x$duration_conditional)), mean(x$success_lower), mean(x$success_upper)))
}
secondary_lines <- c(secondary_lines, "", "No hay pruebas secundarias post hoc ni ranking global. Los CSV conservan CVaR, reordenamientos y tiempos.",
  "La memoria se informa sólo cuando hay medición: CUDA del proceso durante el intento y RSS máximo histórico del proceso; no son incrementos aislados por método. Entrenamiento/validación aparecen por checkpoint en offline_model_costs.csv.")
writeLines(secondary_lines, file.path(output, "secondary_results.md"), useBytes = TRUE)

# Costos offline se muestran una vez por checkpoint, incluido el rechazado; HBG
# sin feromona reutiliza esos pesos y no duplica el cargo de entrenamiento.
offline <- read("offline_model_costs.csv")
need(offline, c("method", "solver_seed", "model_checkpoint_sha256", "training_seconds_including_validation", "validation_seconds_already_included"))
if (anyDuplicated(key(offline, c("method", "solver_seed")))) stop("Costo offline duplicado por semilla/modelo.")
write(offline, "offline_model_costs.csv")
for (field in c("evaluation_seconds", "trace_validation_seconds", "peak_cuda_allocated_bytes", "process_max_rss_kib")) {
  if (!field %in% names(attempts)) attempts[[field]] <- NA_real_ else attempts[[field]] <- num(attempts[[field]])
}
search_keys <- c("instance_sha256", "method", "solver_seed")
for (ii in split(seq_len(nrow(attempts)), key(attempts, search_keys))) {
  values <- attempts$search_total_seconds[ii]
  if (length(unique(values)) != 1L) stop("Tiempo de búsqueda cambia entre políticas de un mismo intento.")
}
search_once <- attempts[!duplicated(key(attempts, search_keys)), ]
compute <- do.call(rbind, lapply(methods, function(method) {
  x <- search_once[search_once$method == method, ]; evaluations <- attempts[attempts$method == method, ]
  total_if_complete <- function(values) if (anyNA(values)) NA_real_ else sum(values)
  data.frame(method = method, search_attempts = nrow(x),
    search_seconds = total_if_complete(x$search_total_seconds),
    evaluated_policy_rows = sum(evaluations$evaluated),
    evaluation_time_rows_available = sum(!is.na(evaluations$evaluation_seconds)),
    evaluation_seconds = total_if_complete(evaluations$evaluation_seconds),
    trace_validation_time_rows_available = sum(!is.na(evaluations$trace_validation_seconds)),
    trace_validation_seconds = total_if_complete(evaluations$trace_validation_seconds),
    training_is_in_separate_shared_checkpoint_table = TRUE,
    missing_time_is_unknown_not_zero = TRUE,
    cuda_peak_rows_available = sum(!is.na(x$peak_cuda_allocated_bytes)),
    cuda_process_peak_during_attempt_bytes = if (all(is.na(x$peak_cuda_allocated_bytes))) NA_real_ else max(x$peak_cuda_allocated_bytes, na.rm = TRUE),
    cpu_lifetime_high_water_rows_available = sum(!is.na(x$process_max_rss_kib)),
    cpu_process_lifetime_high_water_kib = if (all(is.na(x$process_max_rss_kib))) NA_real_ else max(x$process_max_rss_kib, na.rm = TRUE),
    memory_scope = "process_memory_not_isolated_method_increment;_training_memory_not_measured")
}))
write(compute, "compute_costs_without_policy_search_duplication.csv")

# Costos secundarios sólo sobre la misma geometría con todas las semillas válidas
# en ambos métodos; cada exclusión conserva su fila, denominador y motivo.
cost_pairs <- unique(rbind(contrasts, data.frame(method = setdiff(methods, "gurobi"), reference_method = "gurobi")))
paired_costs <- list(); counter <- 0L
for (i in seq_len(nrow(cost_pairs))) for (policy in policies) for (geometry_id in registry$geometry_group_sha256) {
  a <- attempts[attempts$geometry_group_sha256 == geometry_id & attempts$policy == policy & attempts$method == cost_pairs$method[[i]], ]
  b <- attempts[attempts$geometry_group_sha256 == geometry_id & attempts$policy == policy & attempts$method == cost_pairs$reference_method[[i]], ]
  b <- b[match(a$solver_seed, b$solver_seed), ]
  for (metric in c("search_objective_minutes", "mean_travel_minutes", "cvar_travel_minutes", "duration_minutes")) {
    av <- if (metric == "duration_minutes") a$mean_travel_minutes + a$mean_waiting_minutes else a[[metric]]
    bv <- if (metric == "duration_minutes") b$mean_travel_minutes + b$mean_waiting_minutes else b[[metric]]
    eligible <- nrow(a) == length(seeds) && nrow(b) == length(seeds) && all(a$valid) && all(b$valid) && !anyNA(av) && !anyNA(bv)
    counter <- counter + 1L
    paired_costs[[counter]] <- data.frame(geometry_group_sha256 = geometry_id,
      n_customers = registry$n_customers[match(geometry_id, registry$geometry_group_sha256)], policy = policy,
      method = cost_pairs$method[[i]], reference_method = cost_pairs$reference_method[[i]], metric = metric,
      expected_seeds = length(seeds), method_valid = sum(a$valid), reference_valid = sum(b$valid),
      status = if (eligible) "common_complete_valid_geometry" else "excluded_incomplete_or_invalid",
      method_mean = if (eligible) mean(av) else NA_real_, reference_mean = if (eligible) mean(bv) else NA_real_,
      delta = if (eligible) mean(av) - mean(bv) else NA_real_, independent_geometry_count = 1L,
      scope = "secondary_descriptive_conditional_common_complete_valid_geometry")
  }
}
write(do.call(rbind, paired_costs), "secondary_common_valid_costs.csv")
# Cota conservadora frente a costo conservador, sin restarla al viaje estocástico.
if ("best_bound" %in% names(attempts)) {
  attempts$best_bound <- num(attempts$best_bound)
  gaps <- lapply(seq_len(nrow(attempts)), function(i) {
    x <- attempts[i, ]; exact <- attempts[attempts$instance_sha256 == x$instance_sha256 & attempts$method == "gurobi", ]
    bound <- if (all(is.na(exact$best_bound))) NA_real_ else max(exact$best_bound, na.rm = TRUE)
    objective <- if (x$valid) x$search_objective_minutes else NA_real_
    compatible <- !is.na(bound) && !is.na(objective) && objective > 0 && bound <= objective + max(tie_abs, tie_rel * abs(objective))
    data.frame(x[c("instance_sha256", "geometry_group_sha256", "n_customers", "policy", "method", "solver_seed")],
      conservative_objective_minutes = objective, gurobi_conservative_bound_minutes = bound,
      gap_upperbound_pct = if (compatible) 100 * max(0, objective - bound) / objective else NA_real_,
      status = if (compatible) "available_conservative_bound_gap" else "unavailable_zero_or_inconsistent",
      bound_envelope_is_not_single_attempt_runtime = TRUE, not_stochastic_optimality = TRUE,
      gap_denominator = "candidate_conservative_objective_not_unknown_optimum")
  })
  write(do.call(rbind, gaps), "secondary_conservative_gap_audit.csv")
}

lines <- c("# Comparación confirmatoria de geometrías independientes", "",
  paste("Release:", meta[["final_release_id"]]), "",
  "Win-score: factibilidad conservadora primero; después costo conservador; ambos fallidos empatan.",
  "Se promedian las semillas prefijadas dentro de geometría, con peso igual de los cuatro estratos.",
  "Los IC y p-valores Hoeffding son conservadores y suponen independencia de geometrías. Holm controla los dos coprimarios.",
  "HBG sin feromona aísla sólo la memoria de feromona EN INFERENCIA con los mismos pesos entrenados; no identifica efectos de TB/DB/replay ni ausencia de feromona durante entrenamiento.",
  "La guardia FR registra degradación observada; no se ha fijado un margen de no inferioridad y no prueba ausencia de daño.", "",
  "| Método - control | N geometrías | Efecto | IC95% | IC simultáneo | p Holm | Diferencia FR | Evidencia positiva |",
  "|---|---:|---:|---|---|---:|---:|---|")
for (i in seq_len(nrow(results))) {
  x <- results[i, ]
  lines <- c(lines, sprintf("| %s - %s | %d | %.4f | [%.4f, %.4f] | [%.4f, %.4f] | %.5f | %.4f | %s |",
    x$method, x$reference_method, x$geometry_count, x$estimate, x$ci95_lower, x$ci95_upper,
    x$simultaneous_ci_lower, x$simultaneous_ci_upper, x$p_holm, x$fr_difference, x$positive_primary_evidence))
}
lines <- c(lines, "", "No significancia no demuestra equivalencia. Estos datos no identifican optimalidad estocástica, c+Q ni superioridad general.",
  "El alcance es la población generada con condicionamiento de ventanas y reglas selladas, condicionado a modelos congelados y tres semillas de solver.",
  "Fuentes: Hoeffding (1963), https://doi.org/10.1080/01621459.1963.10500830; R stats::p.adjust, https://search.r-project.org/R/refmans/stats/html/p.adjust.html")
writeLines(lines, file.path(output, "confirmatory_results.md"), useBytes = TRUE)
draw <- function() {
  par(mar = c(5, 12, 4, 1), las = 1)
  y <- seq_len(nrow(results))
  plot(results$estimate, y, xlim = c(-1, 1), ylim = c(.5, nrow(results) + .5), yaxt = "n", pch = 19,
       xlab = "Win-score medio con IC simultaneo Hoeffding", ylab = "", main = "Dos contrastes coprimarios; N por geometria")
  axis(2, y, paste0(results$method, " - ", results$reference_method, " (N=", results$geometry_count, ")"), cex.axis = .7)
  segments(results$simultaneous_ci_lower, y, results$simultaneous_ci_upper, y, lwd = 2)
  abline(v = 0, lty = 2, col = "#777777")
}
pdf(file.path(output, "coprimary_effects.pdf"), width = 11, height = 5); draw(); dev.off()
png(file.path(output, "coprimary_effects.png"), width = 3300, height = 1500, res = 300); draw(); dev.off()
verify()
writeLines(capture.output(sessionInfo()), file.path(output, "sessionInfo.txt"))
write(data.frame(check = c("source_hashes_before_after", "all_scheduled_attempts_present", "complete_primary_seed_pairs",
                          "independent_geometry_quotas", "two_coprimaries_holm", "no_secondary_pvalue_search"), passed = TRUE), "processing_checks.csv")
cat("R: inferencia coprimaria completa; el notebook debe sellar el reporte.\n")


Overwriting confirmatory_results_08.R


In [72]:
import importlib
if str(RUNTIME) not in sys.path:
    sys.path.insert(0,str(RUNTIME))
for name in list(sys.modules):
    if name == 'svrplab' or name.startswith('svrplab.'):
        del sys.modules[name]
importlib.invalidate_caches()
from svrplab import pilot as p
from svrplab import experiments as exp
from svrplab import final_experiment as final
from google.colab import drive
import torch
import pandas as pd
from IPython.display import display
drive.mount('/content/drive')
(RUNTIME / 'analysis_08.json').write_text(ANALYSIS_CONFIG_SOURCE)
final.validate_config(CONFIG, DESIGN)
print('Fuentes incluidas:', len(final._sources()), '| dispositivo solicitado:', TRAIN_DEVICE)


Mounted at /content/drive
Fuentes incluidas: 21 | dispositivo solicitado: cuda


## Validación funcional obligatoria en Colab
Estas pruebas pequeñas no son el experimento ni amplían su población. Deben pasar
con las mismas fuentes antes de sellar el protocolo. Un fallo se guarda y detiene
el proceso. El código nuevo sólo recibió revisión estructural local.


In [73]:
TEST_FILES = ['tests/test_final_experiment.py', 'tests/test_hybrid.py', 'tests/test_reinforcement.py', 'tests/test_hbg.py', 'tests/test_pomo.py', 'tests/test_attention.py', 'tests/test_baselines.py', 'tests/test_metaheuristics.py', 'tests/test_experiments.py', 'tests/test_exact.py', 'tests/test_research_design.py']
source_sha256 = final._sources()
check_id = 'contract10_' + p.digest(source_sha256)[:20]
CHECKS = WORK / 'checks' / check_id
REMOTE_CHECKS = DRIVE_ROOT / 'final_method_checks' / check_id
if final.recover_contract_validation(CHECKS, REMOTE_CHECKS, source_sha256):
    print('Acta Colab original recuperada, sin repetir ni alterar pytest:', check_id)
else:
    result = subprocess.run([sys.executable,'-m','pytest','-q',*TEST_FILES],cwd=RUNTIME,capture_output=True,text=True)
    CHECKS.mkdir(parents=True,exist_ok=True)
    (CHECKS / 'pytest.txt').write_text(result.stdout + '\n' + result.stderr)
    p.write_json(CHECKS / 'validation.json', {'status':'passed' if result.returncode == 0 else 'failed',
        'source_sha256':source_sha256,'test_files':TEST_FILES,'returncode':result.returncode,'environment':p.environment_report()})
    exp._commit(CHECKS, REMOTE_CHECKS)
    print(result.stdout[-6000:])
    if result.returncode != 0:
        raise RuntimeError('Contrato 10 falló; revisar informe en Drive antes de entrenar/abrir test.')


Acta Colab original recuperada, sin repetir ni alterar pytest: contract10_2888f4a9ce1d4bf27995


## Preparación y recursos antes del test
La puerta final de selección se fija ahora: alguna trayectoria neuronal completa
en al menos una geometría de **cada tamaño de validación**, y cobertura de todas
las geometrías train por el checkpoint elegido. Se prioriza cobertura de instancias,
luego fracción de trayectorias completas y costo. No se relaja tras observar test.
Los seis intentos se conservan; una validación insuficiente devuelve ausencia de
solución en test, sin inferir con pesos rechazados. Fallos técnicos detienen el flujo.

Para entrenar, completar la referencia de población y fijar `RUN_TRAINING=True`.
El protocolo incluye hashes de código, datos, configuración, análisis R, escenarios
y comprobaciones. Cada familia tiene un máximo de 12 h/modelo y muestreo barajado
sin reemplazo de geometrías; la validación completa puede sobrepasar el reloj y se registra.


In [74]:
# Auditoría de infraestructura: sólo lectura del protocolo/entrenamientos originales.
import math

T4_AMENDMENT_ID = 't4_restart_20260928_v1'
T4_REPLICATE_ID = 'independent_final_holdout_t4_restart_20260928_v1'
T4_PARENT_ID = 'final10_d4ae8593792f0f4854f5'
_t4_pop = POPULATION_REPORT_SHA256
if not isinstance(_t4_pop, str) or len(_t4_pop) != 64 or any(c not in '0123456789abcdef' for c in _t4_pop):
    raise ValueError('Falta el SHA-256 íntegro de la misma población aceptada.')
if RUN_TRAINING or OPEN_FINAL_TEST:
    if TRAIN_DEVICE != 'cuda' or not torch.cuda.is_available() or torch.cuda.get_device_name(0) != 'Tesla T4':
        raise RuntimeError('Esta enmienda exige TRAIN_DEVICE=cuda y GPU Tesla T4 para entrenar/abrir test.')
_t4_parent_dir = DRIVE_ROOT / 'final_method_runs' / _t4_pop / T4_PARENT_ID
_t4_parent_bundle = p.verify_bundle(_t4_parent_dir / 'protocol')
_t4_parent = final._read(_t4_parent_dir / 'protocol/protocol.json')
_t4_parent_core = {k: v for k, v in _t4_parent.items() if k != 'sealed_at'}
if 'final10_' + p.digest(_t4_parent_core)[:20] != T4_PARENT_ID:
    raise ValueError('La identidad del protocolo padre no corresponde a su contenido.')
_t4_sources = final._sources()
_t4_validation_hash = p.verify_bundle(CHECKS)
_t4_validation = final._read(CHECKS / 'validation.json')
_t4_r_hash = p.file_hash(RUNTIME / 'confirmatory_results_08.R')
_t4_analysis_hash = p.file_hash(RUNTIME / 'analysis_08.json')
if (_t4_parent['population_sha256'] != _t4_pop or _t4_parent['design'] != DESIGN
        or _t4_parent['source_sha256'] != _t4_sources
        or _t4_parent['evaluator_sha256'] != p.file_hash(p.__file__)
        or _t4_parent['analysis_source_sha256'] != _t4_r_hash
        or _t4_parent['analysis_r_sha256'] != _t4_r_hash
        or _t4_parent['analysis_config_sha256'] != _t4_analysis_hash
        or _t4_parent['validation_bundle_sha256'] != _t4_validation_hash
        or _t4_validation.get('status') != 'passed' or _t4_validation.get('returncode') != 0
        or _t4_validation.get('source_sha256') != _t4_sources):
    raise ValueError('Población, diseño, fuentes, evaluador, R o acta Colab difieren del padre.')
if p.file_hash(_t4_parent_dir / 'protocol/confirmatory_results_08.R') != _t4_r_hash or p.file_hash(_t4_parent_dir / 'protocol/analysis_08.json') != _t4_analysis_hash:
    raise ValueError('Las fuentes de análisis del padre no coinciden.')
if p.digest(_t4_parent['registry']) != _t4_parent['registry_sha256']:
    raise ValueError('Registro de geometrías del padre incompatible.')
if any((_t4_parent_dir / name).exists() for name in ('test', 'report')):
    raise ValueError('El padre contiene evidencia de exposición/reporte: conservarla y detener.')
_t4_claim_root = DRIVE_ROOT / 'final_test_exposure_registry'
_t4_claim_paths = [_t4_claim_root / _t4_pop / 'exposure.json'] + [
    _t4_claim_root / 'geometries' / f'{group}.json' for group in sorted({
        row['geometry_group_sha256'] for row in _t4_parent['registry'] if row['role'] == 'test'})]
_t4_relative = Path('infrastructure_amendments') / _t4_pop / T4_AMENDMENT_ID
T4_AMENDMENT_LOCAL, T4_AMENDMENT_REMOTE = WORK / _t4_relative, DRIVE_ROOT / _t4_relative
_t4_recovered = exp._recover(T4_AMENDMENT_LOCAL, T4_AMENDMENT_REMOTE)
_t4_base = {
    'schema_version': 'infrastructure-restart-amendment-v1', 'amendment_id': T4_AMENDMENT_ID,
    'parent_protocol_id': T4_PARENT_ID, 'population_sha256': _t4_pop,
    'parent_protocol_bundle_sha256': _t4_parent_bundle,
    'parent_protocol_file_sha256': p.file_hash(_t4_parent_dir / 'protocol/protocol.json'),
    'parent_config_sha256': p.digest(_t4_parent['config']), 'source_sha256': _t4_sources,
    'analysis_r_sha256': _t4_r_hash, 'analysis_config_sha256': _t4_analysis_hash,
    'validation_bundle_sha256': _t4_validation_hash,
    'reason': 'Autor solicita continuar únicamente con T4 disponible, sin comprar acceso A100.',
    'change': 'Reiniciar los seis entrenamientos; mismos datos, métodos, semillas, presupuestos y puertas.',
    'checkpoint_transfer': False, 'prior_artifacts_preserved': True,
    'prior_cost_scope': 'Último acumulado sellado por intento; cota inferior del gasto previo real.',
    'prior_cost_exclusions': 'Persistencia, interrupciones no selladas e inactividad; no sumar validación otra vez.',
    'new_export_excludes_prior_costs': True, 'new_replicate_id': T4_REPLICATE_ID}
if not _t4_recovered:
    if CONFIG != _t4_parent['config']:
        raise ValueError('Antes de crear la enmienda, CONFIG debe ser idéntica a la del padre.')
    if any(path.exists() for path in _t4_claim_paths):
        raise ValueError('Población/geometrías test ya reservadas; no crear otro protocolo.')
    _t4_inventory = []
    for _t4_method in ('hbg_facs', 'pomo'):
        for _t4_seed in _t4_parent['config']['solver_seeds']:
            _t4_dir = _t4_parent_dir / 'training' / _t4_method / f'seed_{_t4_seed}'
            _t4_row = {'method': _t4_method, 'seed': _t4_seed, 'status': 'no_sealed_training_evidence',
                       'epochs_completed': 0, 'recorded_training_seconds_lower_bound': 0.0}
            _t4_epochs = sorted((int(x.name[6:]), x) for x in _t4_dir.glob('epoch_*')
                               if x.name[6:].isdigit() and (x / 'CHECKSUMS.json').is_file())
            _t4_artifact = _t4_dir / 'final' if (_t4_dir / 'final/CHECKSUMS.json').is_file() else (_t4_epochs[-1][1] if _t4_epochs else None)
            if _t4_artifact is not None:
                _t4_bundle = p.verify_bundle(_t4_artifact)
                _t4_is_final = _t4_artifact.name == 'final'
                _t4_record_file = _t4_artifact / ('summary.json' if _t4_is_final else 'progress.json')
                _t4_record = final._read(_t4_record_file)
                _t4_seconds = _t4_record['training_seconds']
                if isinstance(_t4_seconds, bool) or not isinstance(_t4_seconds, (int, float)) or not math.isfinite(_t4_seconds) or _t4_seconds < 0:
                    raise ValueError('Tiempo acumulado previo inválido.')
                _t4_row.update(status=_t4_record['selection_status'] if _t4_is_final else 'partial_sealed_checkpoint',
                    epochs_completed=_t4_record['epochs_completed'] if _t4_is_final else _t4_record['epoch'],
                    recorded_training_seconds_lower_bound=_t4_seconds,
                    artifact_relative_to_parent=str(_t4_artifact.relative_to(_t4_parent_dir)),
                    artifact_bundle_sha256=_t4_bundle, record_file_sha256=p.file_hash(_t4_record_file))
            _t4_inventory.append(_t4_row)
    if len(_t4_inventory) != 6:
        raise ValueError('La enmienda requiere los seis intentos originales.')
    _t4_amendment = {**_t4_base, 'prior_training_inventory': _t4_inventory,
        'recorded_prior_training_seconds_lower_bound': sum(row['recorded_training_seconds_lower_bound'] for row in _t4_inventory)}
    exp.write_json(T4_AMENDMENT_LOCAL / 'amendment.json', _t4_amendment)
    exp._commit(T4_AMENDMENT_LOCAL, T4_AMENDMENT_REMOTE)
else:
    _t4_amendment = final._read(T4_AMENDMENT_LOCAL / 'amendment.json')
    if any(_t4_amendment.get(k) != v for k, v in _t4_base.items()):
        raise ValueError('La enmienda guardada pertenece a otro padre/código/configuración.')
_t4_amendment_hash = p.verify_bundle(T4_AMENDMENT_LOCAL)
_t4_config = json.loads(json.dumps(_t4_parent['config']))
for _t4_method in ('hbg_facs', 'pomo'):
    _t4_config['training'][_t4_method]['replicate_id'] = T4_REPLICATE_ID
_t4_config['infrastructure_amendment'] = {
    'amendment_id': T4_AMENDMENT_ID, 'bundle_sha256': _t4_amendment_hash,
    'drive_relative_path': str(_t4_relative), 'parent_protocol_id': T4_PARENT_ID,
    'parent_protocol_file_sha256': _t4_base['parent_protocol_file_sha256']}
if CONFIG not in (_t4_parent['config'], _t4_config):
    raise ValueError('CONFIG contiene cambios adicionales a los dos replicate_id y la enmienda.')
final.validate_config(_t4_config, DESIGN)
if final.resource_plan(_t4_parent['registry'], _t4_config) != _t4_parent['resource_plan']:
    raise ValueError('El presupuesto de recursos ya no coincide con el padre.')
_t4_new_core = {**_t4_parent_core, 'config': _t4_config}
EXPECTED_PROTOCOL_ID = 'final10_' + p.digest(_t4_new_core)[:20]
for _t4_claim_path in _t4_claim_paths:
    if _t4_claim_path.exists():
        _t4_claim = final._read(_t4_claim_path)
        if (not _t4_recovered or _t4_claim.get('schema_version') != 'single-population-test-exposure-v1'
                or _t4_claim.get('population_sha256') != _t4_pop
                or _t4_claim.get('release_id') != EXPECTED_PROTOCOL_ID):
            raise ValueError('Existe exposición ajena al nuevo protocolo T4; detener sin alterarla.')
CONFIG = _t4_config
print('Enmienda inmutable:', T4_AMENDMENT_REMOTE, '| SHA-256 del bundle:', _t4_amendment_hash)
print('Protocolo esperado:', EXPECTED_PROTOCOL_ID, '| reinicio de seis modelos, sin transferencia de checkpoints.')
display(pd.DataFrame(_t4_amendment['prior_training_inventory']))
print('Gasto previo registrado (cota inferior):', _t4_amendment['recorded_prior_training_seconds_lower_bound'] / 3600, 'horas.')
print('El reporte R del nuevo protocolo excluye ese gasto previo: conservar esta auditoría junto al reporte.')
print('Presupuesto nuevo:', _t4_parent['resource_plan']['planned_training_budget_hours'], 'h de cómputo de entrenamiento; persistencia e interrupciones adicionales.')
print('Test sigue sujeto al freeze y reclamo exacto de protocolo/modelos del motor original.')


Enmienda inmutable: /content/drive/MyDrive/TESIS-EHBG-FACS/infrastructure_amendments/ce05f08892e92f3cff035f887bdf4604e54f9ac124e4af9ae048eeaf2541acfe/t4_restart_20260928_v1 | SHA-256 del bundle: c370ff60afc2769cc797e1c81589c154020f228135b7d22b4b00ba8af1d0894e
Protocolo esperado: final10_802d2255f89bb6ccacb6 | reinicio de seis modelos, sin transferencia de checkpoints.


,method,seed,status,epochs_completed,recorded_training_seconds_lower_bound,artifact_relative_to_parent,artifact_bundle_sha256,record_file_sha256
0,hbg_facs,11,validation_gate_failed,5,43562.152146,training/hbg_facs/seed_11/final,0ead0a76fc39f39f68f8fd2b27220533e20140bab4c7ac...,029934eaa5a1f26e3d57abb3d29e47492ada799669e6a8...
1,hbg_facs,29,partial_sealed_checkpoint,3,26707.956510,training/hbg_facs/seed_29/epoch_0003,7a1b69836bed872aaece56fd24aa4b0b2549d0359117db...,eba582391c52cb7d3e182f8b1a4d8d1b37e67f8af60a88...
2,hbg_facs,47,no_sealed_training_evidence,0,0.000000,NaN,NaN,NaN
3,pomo,11,no_sealed_training_evidence,0,0.000000,NaN,NaN,NaN
4,pomo,29,no_sealed_training_evidence,0,0.000000,NaN,NaN,NaN
5,pomo,47,no_sealed_training_evidence,0,0.000000,NaN,NaN,NaN


Gasto previo registrado (cota inferior): 19.51947462655472 horas.
El reporte R del nuevo protocolo excluye ese gasto previo: conservar esta auditoría junto al reporte.
Presupuesto nuevo: 72.0 h de cómputo de entrenamiento; persistencia e interrupciones adicionales.
Test sigue sujeto al freeze y reclamo exacto de protocolo/modelos del motor original.


In [75]:
ROOT_RUN = REMOTE_RUN = INSTANCES = None
if PREPARE_PROTOCOL or RUN_TRAINING or OPEN_FINAL_TEST:
    if not POPULATION_RELATIVE_PATH or len(POPULATION_REPORT_SHA256) != 64:
        raise ValueError('Fijar ruta y hash del acta de población aceptada de 07.')
    population = final._safe(DRIVE_ROOT, POPULATION_RELATIVE_PATH)
    ROOT_RUN, REMOTE_RUN, INSTANCES = final.prepare_protocol(population, POPULATION_REPORT_SHA256,
        DESIGN, CONFIG, RUNTIME / 'confirmatory_results_08.R', RUNTIME / 'analysis_08.json', CHECKS,
        WORK / 'final_method_runs', DRIVE_ROOT / 'final_method_runs')
    if ROOT_RUN.name != EXPECTED_PROTOCOL_ID:
        raise ValueError('La identidad cambió: detener sin entrenar ni abrir otra prueba.')
    display(pd.DataFrame([final._protocol(ROOT_RUN)['resource_plan']]))
    print('Protocolo:', ROOT_RUN.name, '| remoto:', REMOTE_RUN)
if RUN_TRAINING:
    final.train_and_freeze(ROOT_RUN, REMOTE_RUN, INSTANCES, device=TRAIN_DEVICE)
    frozen = final._read(ROOT_RUN / 'freeze/freeze.json')
    display(pd.DataFrame([{'method':row['method'],'seed':row['seed'],
        'selection_status':row['summary']['selection_status'],'best_epoch':row['summary']['best_epoch'],
        'training_seconds':row['summary']['training_seconds']} for row in frozen['models']]))
else:
    print('PREPARACIÓN COMPLETADA. No se entrenó. Para iniciar la réplica nueva: elegir T4, fijar RUN_TRAINING=True y ejecutar desde el inicio. Mantener OPEN_FINAL_TEST=False.')


,test_instances,solver_attempts,planned_search_budget_hours,planned_training_budget_hours,scenario_metric_rows_upper_bound,client_visit_event_rows_avoided,passes_declared_resource_limits,limits_are_compute_budgets_not_wall_time_guarantees,evaluation_io_validation_and_restart_overheads_are_additional
0,120,3600,120.0,72.0,7200000,1170000000,True,True,True


Protocolo: final10_802d2255f89bb6ccacb6 | remoto: /content/drive/MyDrive/TESIS-EHBG-FACS/final_method_runs/ce05f08892e92f3cff035f887bdf4604e54f9ac124e4af9ae048eeaf2541acfe/final10_802d2255f89bb6ccacb6
PREPARACIÓN COMPLETADA. No se entrenó. Para iniciar la réplica nueva: elegir T4, fijar RUN_TRAINING=True y ejecutar desde el inicio. Mantener OPEN_FINAL_TEST=False.


## Exposición final deliberada y reanudación
Sólo después de revisar recursos, freeze y sus resultados de validación se fija
`OPEN_FINAL_TEST=True`. Se exige WLS Academic en Secrets de Colab: `WLSACCESSID`,
`WLSSECRET`, `LICENSEID`. No colocar credenciales en celdas ni Drive.

Un marcador por población y por geometría permite sólo esta identidad de test.
Una reanudación recupera rutas y evaluaciones selladas; no escoge otra corrida o
semilla por sus métricas. Al reanudar otro runtime, dejar `RUN_TRAINING=True` recupera
el freeze existente y sus modelos sin entrenarlos otra vez. Un cambio de fuentes,
modelos, hiperparámetros o análisis requiere una población test independiente.


In [76]:
REPORT = None
if OPEN_FINAL_TEST:
    from google.colab import userdata
    from svrplab import exact
    credentials = {key: userdata.get(key) for key in ['WLSACCESSID','WLSSECRET','LICENSEID']}
    env = exact.create_wls_env(credentials)
    del credentials
    try:
        REPORT = final.run_test(ROOT_RUN, REMOTE_RUN, INSTANCES,
            DRIVE_ROOT / 'final_test_exposure_registry', device=TRAIN_DEVICE, gurobi_env=env, open_test=True)
    finally:
        env.dispose()
    print('Reporte para 08:', REMOTE_RUN / 'report')
    print('SHA256 de CHECKSUMS.json:', p.file_hash(REPORT / 'CHECKSUMS.json'))
else:
    print('Test final cerrado: no se entrenó ni ejecutó ningún método sobre test en esta celda.')


Test final cerrado: no se entrenó ni ejecutó ningún método sobre test en esta celda.


## Evidencia lista para R y tesis
08 procesa `report/` con su SHA-256 externo, verifica cadena población→protocolo→
modelos→exposición y cada resultado contra su bundle original. Los reportes finales
no se mezclan con los cuatro casos de desarrollo de 06. Cada geometría test aporta
una unidad independiente; las tres semillas y 1000 escenarios quedan anidados.
Un resultado negativo o sin diferencia es un resultado válido. El mecanismo de
aprendizaje/feromona sólo se atribuye si sus controles y diagnósticos lo sostienen.
